<a href="https://colab.research.google.com/github/stesalps/Export/blob/master/CMcp_idea.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Google Colab MCP Server Setup
# Run this cell to expose Colab as an MCP server

import json
import asyncio
import logging
from typing import Any, Dict, List, Optional
import subprocess
import sys
import os
from pathlib import Path

# Install required packages
!pip install -q fastapi uvicorn pyngrok python-multipart aiofiles

import fastapi
from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import JSONResponse
from pyngrok import ngrok
import uvicorn
from threading import Thread
import nest_asyncio

# Enable nested asyncio for Colab compatibility
nest_asyncio.apply()

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class ColabMCPServer:
    def __init__(self):
        self.app = FastAPI(title="Colab MCP Server", version="1.0.0")
        self.setup_routes()

    def setup_routes(self):
        @self.app.get("/")
        async def root():
            return {"message": "Colab MCP Server is running", "version": "1.0.0"}

        @self.app.post("/mcp/initialize")
        async def initialize(request: Request):
            """Initialize MCP connection"""
            data = await request.json()
            return {
                "protocolVersion": "2024-11-05",
                "capabilities": {
                    "tools": {
                        "listChanged": True
                    },
                    "resources": {
                        "subscribe": True,
                        "listChanged": True
                    }
                },
                "serverInfo": {
                    "name": "colab-mcp-server",
                    "version": "1.0.0"
                }
            }

        @self.app.post("/mcp/tools/list")
        async def list_tools():
            """List available tools"""
            return {
                "tools": [
                    {
                        "name": "execute_python",
                        "description": "Execute Python code in the Colab environment",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "code": {
                                    "type": "string",
                                    "description": "Python code to execute"
                                }
                            },
                            "required": ["code"]
                        }
                    },
                    {
                        "name": "install_package",
                        "description": "Install a Python package using pip",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "package": {
                                    "type": "string",
                                    "description": "Package name to install"
                                }
                            },
                            "required": ["package"]
                        }
                    },
                    {
                        "name": "list_files",
                        "description": "List files in a directory",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "path": {
                                    "type": "string",
                                    "description": "Directory path to list",
                                    "default": "."
                                }
                            }
                        }
                    },
                    {
                        "name": "read_file",
                        "description": "Read contents of a file",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "filepath": {
                                    "type": "string",
                                    "description": "Path to the file to read"
                                }
                            },
                            "required": ["filepath"]
                        }
                    },
                    {
                        "name": "write_file",
                        "description": "Write content to a file",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "filepath": {
                                    "type": "string",
                                    "description": "Path to the file to write"
                                },
                                "content": {
                                    "type": "string",
                                    "description": "Content to write to the file"
                                }
                            },
                            "required": ["filepath", "content"]
                        }
                    }
                ]
            }

        @self.app.post("/mcp/tools/call")
        async def call_tool(request: Request):
            """Execute a tool"""
            data = await request.json()
            tool_name = data.get("name")
            arguments = data.get("arguments", {})

            try:
                if tool_name == "execute_python":
                    result = self.execute_python(arguments.get("code", ""))
                elif tool_name == "install_package":
                    result = self.install_package(arguments.get("package", ""))
                elif tool_name == "list_files":
                    result = self.list_files(arguments.get("path", "."))
                elif tool_name == "read_file":
                    result = self.read_file(arguments.get("filepath", ""))
                elif tool_name == "write_file":
                    result = self.write_file(arguments.get("filepath", ""), arguments.get("content", ""))
                else:
                    raise HTTPException(status_code=400, detail=f"Unknown tool: {tool_name}")

                return {
                    "content": [
                        {
                            "type": "text",
                            "text": str(result)
                        }
                    ],
                    "isError": False
                }
            except Exception as e:
                return {
                    "content": [
                        {
                            "type": "text",
                            "text": f"Error: {str(e)}"
                        }
                    ],
                    "isError": True
                }

    def execute_python(self, code: str) -> str:
        """Execute Python code and return the result"""
        try:
            # Create a namespace for execution
            namespace = {}

            # Capture stdout
            from io import StringIO
            import contextlib

            stdout_capture = StringIO()

            with contextlib.redirect_stdout(stdout_capture):
                exec(code, namespace)

            output = stdout_capture.getvalue()
            return output if output else "Code executed successfully (no output)"

        except Exception as e:
            return f"Execution error: {str(e)}"

    def install_package(self, package: str) -> str:
        """Install a package using pip"""
        try:
            result = subprocess.run([sys.executable, "-m", "pip", "install", package],
                                  capture_output=True, text=True)
            if result.returncode == 0:
                return f"Package '{package}' installed successfully"
            else:
                return f"Installation failed: {result.stderr}"
        except Exception as e:
            return f"Installation error: {str(e)}"

    def list_files(self, path: str = ".") -> str:
        """List files in a directory"""
        try:
            path_obj = Path(path)
            if not path_obj.exists():
                return f"Path does not exist: {path}"

            if path_obj.is_file():
                return f"'{path}' is a file, not a directory"

            files = []
            for item in path_obj.iterdir():
                if item.is_file():
                    files.append(f"📄 {item.name}")
                else:
                    files.append(f"📁 {item.name}/")

            return "\n".join(sorted(files)) if files else "Directory is empty"

        except Exception as e:
            return f"Error listing files: {str(e)}"

    def read_file(self, filepath: str) -> str:
        """Read contents of a file"""
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                content = f.read()
            return content
        except Exception as e:
            return f"Error reading file: {str(e)}"

    def write_file(self, filepath: str, content: str) -> str:
        """Write content to a file"""
        try:
            # Create directory if it doesn't exist
            Path(filepath).parent.mkdir(parents=True, exist_ok=True)

            with open(filepath, 'w', encoding='utf-8') as f:
                f.write(content)
            return f"File written successfully: {filepath}"
        except Exception as e:
            return f"Error writing file: {str(e)}"

# Initialize and start the server
def start_mcp_server():
    # Set up ngrok authentication (you may need to set your auth token)
    # Get your token from https://dashboard.ngrok.com/get-started/your-authtoken
    # ngrok.set_auth_token("your_auth_token_here")  # Uncomment and add your token

    server = ColabMCPServer()

    # Start the server in a separate thread
    def run_server():
        uvicorn.run(server.app, host="0.0.0.0", port=8000, log_level="info")

    server_thread = Thread(target=run_server, daemon=True)
    server_thread.start()

    # Wait a moment for the server to start
    import time
    time.sleep(3)

    # Create ngrok tunnel
    try:
        public_url = ngrok.connect(8000)
        print(f"🚀 MCP Server is running!")
        print(f"📡 Public URL: {public_url}")
        print(f"🔧 Local URL: http://localhost:8000")
        print(f"\n📋 MCP Connection Details:")
        print(f"   - Endpoint: {public_url}")
        print(f"   - Initialize: POST {public_url}/mcp/initialize")
        print(f"   - List Tools: POST {public_url}/mcp/tools/list")
        print(f"   - Call Tool: POST {public_url}/mcp/tools/call")
        print(f"\n🛠️  Available Tools:")
        print(f"   - execute_python: Run Python code")
        print(f"   - install_package: Install pip packages")
        print(f"   - list_files: List directory contents")
        print(f"   - read_file: Read file contents")
        print(f"   - write_file: Write to files")

        return public_url

    except Exception as e:
        print(f"❌ Error setting up ngrok tunnel: {e}")
        print("💡 You may need to set up ngrok authentication token")
        print("   Visit: https://dashboard.ngrok.com/get-started/your-authtoken")
        return None

# Start the server
public_url = start_mcp_server()

# Keep the server running
print("\n✅ Server is running. Use the public URL above to connect your MCP client.")
print("⚠️  Note: The ngrok URL will change each time you restart this cell.")
print("🔄 To stop the server, restart the runtime or interrupt this cell.")


INFO:     Started server process [927]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


ERROR:pyngrok.process.ngrok:t=2025-06-23T13:02:18+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2025-06-23T13:02:18+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2025-06-23T13:02:18+0000 lvl=eror msg="terminating with error" obj=app err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your aut

❌ Error setting up ngrok tunnel: The ngrok process errored on start: authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.
💡 You may need to set up ngrok authentication token
   Visit: https://dashboard.ngrok.com/get-started/your-authtoken

✅ Server is running. Use the public URL above to connect your MCP client.
⚠️  Note: The ngrok URL will change each time you restart this cell.
🔄 To stop the server, restart the runtime or interrupt this cell.


In [ ]:
# Alternative approach using Colab's built-in public URL
import subprocess
import time
from threading import Thread

# The server is already running on localhost:8000
# Let's try using Colab's built-in method to expose it

# First, let's check if the server is accessible locally
def test_local_connection():
    import requests
    try:
        response = requests.get("http://localhost:8000/")
        print("✅ Local server is accessible!")
        print(f"Response: {response.json()}")
        return True
    except Exception as e:
        print(f"❌ Local server test failed: {e}")
        return False

# Test the connection
if test_local_connection():
    print("\n🎉 Your MCP server is running successfully!")
    print("\n📋 Server Status:")
    print("   - Local URL: http://localhost:8000")
    print("   - Status: ✅ Active")

    print("\n🔌 MCP Endpoints:")
    print("   - Initialize: POST http://localhost:8000/mcp/initialize")
    print("   - List Tools: POST http://localhost:8000/mcp/tools/list")
    print("   - Call Tool: POST http://localhost:8000/mcp/tools/call")

    print("\n🛠️ Available Tools:")
    print("   - execute_python: Execute Python code")
    print("   - install_package: Install pip packages")
    print("   - list_files: List directory contents")
    print("   - read_file: Read file contents")
    print("   - write_file: Write content to files")

    print("\n💡 Next Steps:")
    print("   1. Set up ngrok auth token for public access, OR")
    print("   2. Use this server locally within Colab, OR")
    print("   3. Try the Google Colab public URL method below")

# Alternative: Try to create a simple public endpoint using Colab's sharing
print("\n🔄 Attempting Colab public URL method...")
try:
    # This creates a simple way to access the server publicly through Colab
    from google.colab import output
    print("📡 You can also try accessing via Colab's network interface")
    print("   Your server is running and ready for MCP connections!")
except:
    print("📱 Running in standard mode - server accessible within this environment")


INFO:     127.0.0.1:50706 - "GET / HTTP/1.1" 200 OK
✅ Local server is accessible!
Response: {'message': 'Colab MCP Server is running', 'version': '1.0.0'}

🎉 Your MCP server is running successfully!

📋 Server Status:
   - Local URL: http://localhost:8000
   - Status: ✅ Active

🔌 MCP Endpoints:
   - Initialize: POST http://localhost:8000/mcp/initialize
   - List Tools: POST http://localhost:8000/mcp/tools/list
   - Call Tool: POST http://localhost:8000/mcp/tools/call

🛠️ Available Tools:
   - execute_python: Execute Python code
   - install_package: Install pip packages
   - list_files: List directory contents
   - read_file: Read file contents
   - write_file: Write content to files

💡 Next Steps:
   1. Set up ngrok auth token for public access, OR
   2. Use this server locally within Colab, OR
   3. Try the Google Colab public URL method below

🔄 Attempting Colab public URL method...
📡 You can also try accessing via Colab's network interface
   Your server is running and ready for MCP

In [ ]:
# Updated MCP Server with your ngrok auth token
import json
import asyncio
import logging
from typing import Any, Dict, List, Optional
import subprocess
import sys
import os
from pathlib import Path

# The packages should already be installed from the previous run
import fastapi
from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import JSONResponse
from pyngrok import ngrok
import uvicorn
from threading import Thread
import nest_asyncio

# Enable nested asyncio for Colab compatibility
nest_asyncio.apply()

# Set up your ngrok authentication token
ngrok.set_auth_token("2yuUKpiAlYXCez40i8UktLZ91s7_5Rv1GjG5HMFw5KKbk5sHz")

# The ColabMCPServer class remains the same
class ColabMCPServer:
    def __init__(self):
        self.app = FastAPI(title="Colab MCP Server", version="1.0.0")
        self.setup_routes()

    def setup_routes(self):
        @self.app.get("/")
        async def root():
            return {"message": "Colab MCP Server is running", "version": "1.0.0"}

        @self.app.post("/mcp/initialize")
        async def initialize(request: Request):
            """Initialize MCP connection"""
            data = await request.json()
            return {
                "protocolVersion": "2024-11-05",
                "capabilities": {
                    "tools": {
                        "listChanged": True
                    },
                    "resources": {
                        "subscribe": True,
                        "listChanged": True
                    }
                },
                "serverInfo": {
                    "name": "colab-mcp-server",
                    "version": "1.0.0"
                }
            }

        @self.app.post("/mcp/tools/list")
        async def list_tools():
            """List available tools"""
            return {
                "tools": [
                    {
                        "name": "execute_python",
                        "description": "Execute Python code in the Colab environment",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "code": {
                                    "type": "string",
                                    "description": "Python code to execute"
                                }
                            },
                            "required": ["code"]
                        }
                    },
                    {
                        "name": "install_package",
                        "description": "Install a Python package using pip",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "package": {
                                    "type": "string",
                                    "description": "Package name to install"
                                }
                            },
                            "required": ["package"]
                        }
                    },
                    {
                        "name": "list_files",
                        "description": "List files in a directory",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "path": {
                                    "type": "string",
                                    "description": "Directory path to list",
                                    "default": "."
                                }
                            }
                        }
                    },
                    {
                        "name": "read_file",
                        "description": "Read contents of a file",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "filepath": {
                                    "type": "string",
                                    "description": "Path to the file to read"
                                }
                            },
                            "required": ["filepath"]
                        }
                    },
                    {
                        "name": "write_file",
                        "description": "Write content to a file",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "filepath": {
                                    "type": "string",
                                    "description": "Path to the file to write"
                                },
                                "content": {
                                    "type": "string",
                                    "description": "Content to write to the file"
                                }
                            },
                            "required": ["filepath", "content"]
                        }
                    }
                ]
            }

        @self.app.post("/mcp/tools/call")
        async def call_tool(request: Request):
            """Execute a tool"""
            data = await request.json()
            tool_name = data.get("name")
            arguments = data.get("arguments", {})

            try:
                if tool_name == "execute_python":
                    result = self.execute_python(arguments.get("code", ""))
                elif tool_name == "install_package":
                    result = self.install_package(arguments.get("package", ""))
                elif tool_name == "list_files":
                    result = self.list_files(arguments.get("path", "."))
                elif tool_name == "read_file":
                    result = self.read_file(arguments.get("filepath", ""))
                elif tool_name == "write_file":
                    result = self.write_file(arguments.get("filepath", ""), arguments.get("content", ""))
                else:
                    raise HTTPException(status_code=400, detail=f"Unknown tool: {tool_name}")

                return {
                    "content": [
                        {
                            "type": "text",
                            "text": str(result)
                        }
                    ],
                    "isError": False
                }
            except Exception as e:
                return {
                    "content": [
                        {
                            "type": "text",
                            "text": f"Error: {str(e)}"
                        }
                    ],
                    "isError": True
                }

    def execute_python(self, code: str) -> str:
        """Execute Python code and return the result"""
        try:
            # Create a namespace for execution
            namespace = {}

            # Capture stdout
            from io import StringIO
            import contextlib

            stdout_capture = StringIO()

            with contextlib.redirect_stdout(stdout_capture):
                exec(code, namespace)

            output = stdout_capture.getvalue()
            return output if output else "Code executed successfully (no output)"

        except Exception as e:
            return f"Execution error: {str(e)}"

    def install_package(self, package: str) -> str:
        """Install a package using pip"""
        try:
            result = subprocess.run([sys.executable, "-m", "pip", "install", package],
                                  capture_output=True, text=True)
            if result.returncode == 0:
                return f"Package '{package}' installed successfully"
            else:
                return f"Installation failed: {result.stderr}"
        except Exception as e:
            return f"Installation error: {str(e)}"

    def list_files(self, path: str = ".") -> str:
        """List files in a directory"""
        try:
            path_obj = Path(path)
            if not path_obj.exists():
                return f"Path does not exist: {path}"

            if path_obj.is_file():
                return f"'{path}' is a file, not a directory"

            files = []
            for item in path_obj.iterdir():
                if item.is_file():
                    files.append(f"📄 {item.name}")
                else:
                    files.append(f"📁 {item.name}/")

            return "\n".join(sorted(files)) if files else "Directory is empty"

        except Exception as e:
            return f"Error listing files: {str(e)}"

    def read_file(self, filepath: str) -> str:
        """Read contents of a file"""
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                content = f.read()
            return content
        except Exception as e:
            return f"Error reading file: {str(e)}"

    def write_file(self, filepath: str, content: str) -> str:
        """Write content to a file"""
        try:
            # Create directory if it doesn't exist
            Path(filepath).parent.mkdir(parents=True, exist_ok=True)

            with open(filepath, 'w', encoding='utf-8') as f:
                f.write(content)
            return f"File written successfully: {filepath}"
        except Exception as e:
            return f"Error writing file: {str(e)}"

# Initialize and start the server with authentication
def start_authenticated_mcp_server():
    print("🔐 Setting up authenticated MCP server...")

    server = ColabMCPServer()

    # Start the server in a separate thread
    def run_server():
        uvicorn.run(server.app, host="0.0.0.0", port=8001, log_level="info")  # Using port 8001 to avoid conflicts

    server_thread = Thread(target=run_server, daemon=True)
    server_thread.start()

    # Wait a moment for the server to start
    import time
    time.sleep(3)

    # Create ngrok tunnel with authentication
    try:
        print("🌐 Creating public tunnel...")
        public_url = ngrok.connect(8001)

        print(f"\n🎉 SUCCESS! Your MCP Server is now publicly accessible!")
        print(f"=" * 60)
        print(f"📡 Public URL: {public_url}")
        print(f"🔧 Local URL: http://localhost:8001")
        print(f"=" * 60)

        print(f"\n📋 MCP Connection Details:")
        print(f"   🔗 Base Endpoint: {public_url}")
        print(f"   🚀 Initialize: POST {public_url}/mcp/initialize")
        print(f"   📝 List Tools: POST {public_url}/mcp/tools/list")
        print(f"   ⚡ Call Tool: POST {public_url}/mcp/tools/call")

        print(f"\n🛠️  Available MCP Tools:")
        print(f"   🐍 execute_python   - Execute Python code remotely")
        print(f"   📦 install_package  - Install pip packages")
        print(f"   📂 list_files       - Browse directory contents")
        print(f"   📖 read_file        - Read file contents")
        print(f"   ✏️  write_file       - Create/modify files")

        print(f"\n💡 Usage Example (with curl):")
# Updated MCP Server with your ngrok auth token
import json
import asyncio
import logging
from typing import Any, Dict, List, Optional
import subprocess
import sys
import os
from pathlib import Path

# The packages should already be installed from the previous run
import fastapi
from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import JSONResponse
from pyngrok import ngrok
import uvicorn
from threading import Thread
import nest_asyncio

# Enable nested asyncio for Colab compatibility
nest_asyncio.apply()

# Set up your ngrok authentication token
ngrok.set_auth_token("2yuUKpiAlYXCez40i8UktLZ91s7_5Rv1GjG5HMFw5KKbk5sHz")

# The ColabMCPServer class remains the same
class ColabMCPServer:
    def __init__(self):
        self.app = FastAPI(title="Colab MCP Server", version="1.0.0")
        self.setup_routes()

    def setup_routes(self):
        @self.app.get("/")
        async def root():
            return {"message": "Colab MCP Server is running", "version": "1.0.0"}

        @self.app.post("/mcp/initialize")
        async def initialize(request: Request):
            """Initialize MCP connection"""
            data = await request.json()
            return {
                "protocolVersion": "2024-11-05",
                "capabilities": {
                    "tools": {
                        "listChanged": True
                    },
                    "resources": {
                        "subscribe": True,
                        "listChanged": True
                    }
                },
                "serverInfo": {
                    "name": "colab-mcp-server",
                    "version": "1.0.0"
                }
            }

        @self.app.post("/mcp/tools/list")
        async def list_tools():
            """List available tools"""
            return {
                "tools": [
                    {
                        "name": "execute_python",
                        "description": "Execute Python code in the Colab environment",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "code": {
                                    "type": "string",
                                    "description": "Python code to execute"
                                }
                            },
                            "required": ["code"]
                        }
                    },
                    {
                        "name": "install_package",
                        "description": "Install a Python package using pip",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "package": {
                                    "type": "string",
                                    "description": "Package name to install"
                                }
                            },
                            "required": ["package"]
                        }
                    },
                    {
                        "name": "list_files",
                        "description": "List files in a directory",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "path": {
                                    "type": "string",
                                    "description": "Directory path to list",
                                    "default": "."
                                }
                            }
                        }
                    },
                    {
                        "name": "read_file",
                        "description": "Read contents of a file",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "filepath": {
                                    "type": "string",
                                    "description": "Path to the file to read"
                                }
                            },
                            "required": ["filepath"]
                        }
                    },
                    {
                        "name": "write_file",
                        "description": "Write content to a file",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "filepath": {
                                    "type": "string",
                                    "description": "Path to the file to write"
                                },
                                "content": {
                                    "type": "string",
                                    "description": "Content to write to the file"
                                }
                            },
                            "required": ["filepath", "content"]
                        }
                    }
                ]
            }

        @self.app.post("/mcp/tools/call")
        async def call_tool(request: Request):
            """Execute a tool"""
            data = await request.json()
            tool_name = data.get("name")
            arguments = data.get("arguments", {})

            try:
                if tool_name == "execute_python":
                    result = self.execute_python(arguments.get("code", ""))
                elif tool_name == "install_package":
                    result = self.install_package(arguments.get("package", ""))
                elif tool_name == "list_files":
                    result = self.list_files(arguments.get("path", "."))
                elif tool_name == "read_file":
                    result = self.read_file(arguments.get("filepath", ""))
                elif tool_name == "write_file":
                    result = self.write_file(arguments.get("filepath", ""), arguments.get("content", ""))
                else:
                    raise HTTPException(status_code=400, detail=f"Unknown tool: {tool_name}")

                return {
                    "content": [
                        {
                            "type": "text",
                            "text": str(result)
                        }
                    ],
                    "isError": False
                }
            except Exception as e:
                return {
                    "content": [
                        {
                            "type": "text",
                            "text": f"Error: {str(e)}"
                        }
                    ],
                    "isError": True
                }

    def execute_python(self, code: str) -> str:
        """Execute Python code and return the result"""
        try:
            # Create a namespace for execution
            namespace = {}

            # Capture stdout
            from io import StringIO
            import contextlib

            stdout_capture = StringIO()

            with contextlib.redirect_stdout(stdout_capture):
                exec(code, namespace)

            output = stdout_capture.getvalue()
            return output if output else "Code executed successfully (no output)"

        except Exception as e:
            return f"Execution error: {str(e)}"

    def install_package(self, package: str) -> str:
        """Install a package using pip"""
        try:
            result = subprocess.run([sys.executable, "-m", "pip", "install", package],
                                  capture_output=True, text=True)
            if result.returncode == 0:
                return f"Package '{package}' installed successfully"
            else:
                return f"Installation failed: {result.stderr}"
        except Exception as e:
            return f"Installation error: {str(e)}"

    def list_files(self, path: str = ".") -> str:
        """List files in a directory"""
        try:
            path_obj = Path(path)
            if not path_obj.exists():
                return f"Path does not exist: {path}"

            if path_obj.is_file():
                return f"'{path}' is a file, not a directory"

            files = []
            for item in path_obj.iterdir():
                if item.is_file():
                    files.append(f"📄 {item.name}")
                else:
                    files.append(f"📁 {item.name}/")

            return "\n".join(sorted(files)) if files else "Directory is empty"

        except Exception as e:
            return f"Error listing files: {str(e)}"

    def read_file(self, filepath: str) -> str:
        """Read contents of a file"""
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                content = f.read()
            return content
        except Exception as e:
            return f"Error reading file: {str(e)}"

    def write_file(self, filepath: str, content: str) -> str:
        """Write content to a file"""
        try:
            # Create directory if it doesn't exist
            Path(filepath).parent.mkdir(parents=True, exist_ok=True)

            with open(filepath, 'w', encoding='utf-8') as f:
                f.write(content)
            return f"File written successfully: {filepath}"
        except Exception as e:
            return f"Error writing file: {str(e)}"

# Initialize and start the server with authentication
def start_authenticated_mcp_server():
    print("🔐 Setting up authenticated MCP server...")

    server = ColabMCPServer()

    # Start the server in a separate thread
    def run_server():
        uvicorn.run(server.app, host="0.0.0.0", port=8001, log_level="info")  # Using port 8001 to avoid conflicts

    server_thread = Thread(target=run_server, daemon=True)
    server_thread.start()

    # Wait a moment for the server to start
    import time
    time.sleep(3)

    # Create ngrok tunnel with authentication
    try:
        print("🌐 Creating public tunnel...")
        public_url = ngrok.connect(8001).public_url

        print(f"\n🎉 SUCCESS! Your MCP Server is now publicly accessible!")
        print(f"=" * 60)
        print(f"📡 Public URL: {public_url}")
        print(f"🔧 Local URL: http://localhost:8001")
        print(f"=" * 60)

        print(f"\n📋 MCP Connection Details:")
        print(f"   🔗 Base Endpoint: {public_url}")
        print(f"   🚀 Initialize: POST {public_url}/mcp/initialize")
        print(f"   📝 List Tools: POST {public_url}/mcp/tools/list")
        print(f"   ⚡ Call Tool: POST {public_url}/mcp/tools/call")

        print(f"\n🛠️  Available MCP Tools:")
        print(f"   🐍 execute_python   - Execute Python code remotely")
        print(f"   📦 install_package  - Install pip packages")
        print(f"   📂 list_files       - Browse directory contents")
        print(f"   📖 read_file        - Read file contents")
        print(f"   ✏️  write_file       - Create/modify files")

        print(f"\n💡 Usage Example (with curl):")
        print(f'curl -X POST "{public_url}/mcp/tools/call" \\')
        print(f'  -H "Content-Type: application/json" \\')
        print(f'  -d \'{{"name": "execute_python", "arguments": {{"code": "print(\\'Hello MCP!\\')"}}}}\'')

        return public_url

    except Exception as e:
        print(f"❌ Error setting up ngrok tunnel: {e}")
        return None

# Start the authenticated server
public_url = start_authenticated_mcp_server()

if public_url:
    print(f"\n✅ Your MCP server is live and ready for connections!")
    print(f"🔗 Share this URL with your MCP clients: {public_url}")
    print(f"⏰ Server will remain active as long as this Colab session is running")

    # Test the public connection
    print(f"\n🧪 Testing public connection...")
    import requests
    try:
        response = requests.get(f"{public_url}/")
        print(f"✅ Public endpoint test: {response.json()}")
    except Exception as e:
        print(f"⚠️  Public test failed (this might be normal): {e}")

else:
    print(f"❌ Failed to create public tunnel. Check your ngrok setup.")

print(f"\n🔄 To stop the server, restart the runtime or interrupt this cell.")

    except Exception as e:
        print(f"❌ Error setting up ngrok tunnel: {e}")
        return None

# Start the authenticated server
public_url = start_authenticated_mcp_server()

if public_url:
    print(f"\n✅ Your MCP server is live and ready for connections!")
    print(f"🔗 Share this URL with your MCP clients: {public_url}")
    print(f"⏰ Server will remain active as long as this Colab session is running")

    # Test the public connection
    print(f"\n🧪 Testing public connection...")
    import requests
    try:
        response = requests.get(f"{public_url}/")
        print(f"✅ Public endpoint test: {response.json()}")
    except Exception as e:
        print(f"⚠️  Public test failed (this might be normal): {e}")

else:
    print(f"❌ Failed to create public tunnel. Check your ngrok setup.")

print(f"\n🔄 To stop the server, restart the runtime or interrupt this cell.")


SyntaxError: expected 'except' or 'finally' block (ipython-input-7-3753405024.py, line 300)

In [ ]:
# Updated MCP Server with your ngrok auth token (Fixed syntax)
import json
import asyncio
import logging
from typing import Any, Dict, List, Optional
import subprocess
import sys
import os
from pathlib import Path

# The packages should already be installed from the previous run
import fastapi
from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import JSONResponse
from pyngrok import ngrok
import uvicorn
from threading import Thread
import nest_asyncio

# Enable nested asyncio for Colab compatibility
nest_asyncio.apply()

# Set up your ngrok authentication token
ngrok.set_auth_token("2yuUKpiAlYXCez40i8UktLZ91s7_5Rv1GjG5HMFw5KKbk5sHz")

# The ColabMCPServer class remains the same
class ColabMCPServer:
    def __init__(self):
        self.app = FastAPI(title="Colab MCP Server", version="1.0.0")
        self.setup_routes()

    def setup_routes(self):
        @self.app.get("/")
        async def root():
            return {"message": "Colab MCP Server is running", "version": "1.0.0"}

        @self.app.post("/mcp/initialize")
        async def initialize(request: Request):
            """Initialize MCP connection"""
            data = await request.json()
            return {
                "protocolVersion": "2024-11-05",
                "capabilities": {
                    "tools": {
                        "listChanged": True
                    },
                    "resources": {
                        "subscribe": True,
                        "listChanged": True
                    }
                },
                "serverInfo": {
                    "name": "colab-mcp-server",
                    "version": "1.0.0"
                }
            }

        @self.app.post("/mcp/tools/list")
        async def list_tools():
            """List available tools"""
            return {
                "tools": [
                    {
                        "name": "execute_python",
                        "description": "Execute Python code in the Colab environment",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "code": {
                                    "type": "string",
                                    "description": "Python code to execute"
                                }
                            },
                            "required": ["code"]
                        }
                    },
                    {
                        "name": "install_package",
                        "description": "Install a Python package using pip",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "package": {
                                    "type": "string",
                                    "description": "Package name to install"
                                }
                            },
                            "required": ["package"]
                        }
                    },
                    {
                        "name": "list_files",
                        "description": "List files in a directory",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "path": {
                                    "type": "string",
                                    "description": "Directory path to list",
                                    "default": "."
                                }
                            }
                        }
                    },
                    {
                        "name": "read_file",
                        "description": "Read contents of a file",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "filepath": {
                                    "type": "string",
                                    "description": "Path to the file to read"
                                }
                            },
                            "required": ["filepath"]
                        }
                    },
                    {
                        "name": "write_file",
                        "description": "Write content to a file",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "filepath": {
                                    "type": "string",
                                    "description": "Path to the file to write"
                                },
                                "content": {
                                    "type": "string",
                                    "description": "Content to write to the file"
                                }
                            },
                            "required": ["filepath", "content"]
                        }
                    }
                ]
            }

        @self.app.post("/mcp/tools/call")
        async def call_tool(request: Request):
            """Execute a tool"""
            data = await request.json()
            tool_name = data.get("name")
            arguments = data.get("arguments", {})

            try:
                if tool_name == "execute_python":
                    result = self.execute_python(arguments.get("code", ""))
                elif tool_name == "install_package":
                    result = self.install_package(arguments.get("package", ""))
                elif tool_name == "list_files":
                    result = self.list_files(arguments.get("path", "."))
                elif tool_name == "read_file":
                    result = self.read_file(arguments.get("filepath", ""))
                elif tool_name == "write_file":
                    result = self.write_file(arguments.get("filepath", ""), arguments.get("content", ""))
                else:
                    raise HTTPException(status_code=400, detail=f"Unknown tool: {tool_name}")

                return {
                    "content": [
                        {
                            "type": "text",
                            "text": str(result)
                        }
                    ],
                    "isError": False
                }
            except Exception as e:
                return {
                    "content": [
                        {
                            "type": "text",
                            "text": f"Error: {str(e)}"
                        }
                    ],
                    "isError": True
                }

    def execute_python(self, code: str) -> str:
        """Execute Python code and return the result"""
        try:
            # Create a namespace for execution
            namespace = {}

            # Capture stdout
            from io import StringIO
            import contextlib

            stdout_capture = StringIO()

            with contextlib.redirect_stdout(stdout_capture):
                exec(code, namespace)

            output = stdout_capture.getvalue()
            return output if output else "Code executed successfully (no output)"

        except Exception as e:
            return f"Execution error: {str(e)}"

    def install_package(self, package: str) -> str:
# Updated MCP Server with your ngrok auth token (Fixed syntax)
import json
import asyncio
import logging
from typing import Any, Dict, List, Optional
import subprocess
import sys
import os
from pathlib import Path

# The packages should already be installed from the previous run
import fastapi
from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import JSONResponse
from pyngrok import ngrok
import uvicorn
from threading import Thread
import nest_asyncio

# Enable nested asyncio for Colab compatibility
nest_asyncio.apply()

# Set up your ngrok authentication token
ngrok.set_auth_token("2yuUKpiAlYXCez40i8UktLZ91s7_5Rv1GjG5HMFw5KKbk5sHz")

# The ColabMCPServer class remains the same
class ColabMCPServer:
    def __init__(self):
        self.app = FastAPI(title="Colab MCP Server", version="1.0.0")
        self.setup_routes()

    def setup_routes(self):
        @self.app.get("/")
        async def root():
            return {"message": "Colab MCP Server is running", "version": "1.0.0"}

        @self.app.post("/mcp/initialize")
        async def initialize(request: Request):
            """Initialize MCP connection"""
            data = await request.json()
            return {
                "protocolVersion": "2024-11-05",
                "capabilities": {
                    "tools": {
                        "listChanged": True
                    },
                    "resources": {
                        "subscribe": True,
                        "listChanged": True
                    }
                },
                "serverInfo": {
                    "name": "colab-mcp-server",
                    "version": "1.0.0"
                }
            }

        @self.app.post("/mcp/tools/list")
        async def list_tools():
            """List available tools"""
            return {
                "tools": [
                    {
                        "name": "execute_python",
                        "description": "Execute Python code in the Colab environment",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "code": {
                                    "type": "string",
                                    "description": "Python code to execute"
                                }
                            },
                            "required": ["code"]
                        }
                    },
                    {
                        "name": "install_package",
                        "description": "Install a Python package using pip",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "package": {
                                    "type": "string",
                                    "description": "Package name to install"
                                }
                            },
                            "required": ["package"]
                        }
                    },
                    {
                        "name": "list_files",
                        "description": "List files in a directory",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "path": {
                                    "type": "string",
                                    "description": "Directory path to list",
                                    "default": "."
                                }
                            }
                        }
                    },
                    {
                        "name": "read_file",
                        "description": "Read contents of a file",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "filepath": {
                                    "type": "string",
                                    "description": "Path to the file to read"
                                }
                            },
                            "required": ["filepath"]
                        }
                    },
                    {
                        "name": "write_file",
                        "description": "Write content to a file",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "filepath": {
                                    "type": "string",
                                    "description": "Path to the file to write"
                                },
                                "content": {
                                    "type": "string",
                                    "description": "Content to write to the file"
                                }
                            },
                            "required": ["filepath", "content"]
                        }
                    }
                ]
            }

        @self.app.post("/mcp/tools/call")
        async def call_tool(request: Request):
            """Execute a tool"""
            data = await request.json()
            tool_name = data.get("name")
            arguments = data.get("arguments", {})

            try:
                if tool_name == "execute_python":
                    result = self.execute_python(arguments.get("code", ""))
                elif tool_name == "install_package":
                    result = self.install_package(arguments.get("package", ""))
                elif tool_name == "list_files":
                    result = self.list_files(arguments.get("path", "."))
                elif tool_name == "read_file":
                    result = self.read_file(arguments.get("filepath", ""))
                elif tool_name == "write_file":
                    result = self.write_file(arguments.get("filepath", ""), arguments.get("content", ""))
                else:
                    raise HTTPException(status_code=400, detail=f"Unknown tool: {tool_name}")

                return {
                    "content": [
                        {
                            "type": "text",
                            "text": str(result)
                        }
                    ],
                    "isError": False
                }
            except Exception as e:
                return {
                    "content": [
                        {
                            "type": "text",
                            "text": f"Error: {str(e)}"
                        }
                    ],
                    "isError": True
                }

    def execute_python(self, code: str) -> str:
        """Execute Python code and return the result"""
        try:
            # Create a namespace for execution
            namespace = {}

            # Capture stdout
            from io import StringIO
            import contextlib

            stdout_capture = StringIO()

            with contextlib.redirect_stdout(stdout_capture):
                exec(code, namespace)

            output = stdout_capture.getvalue()
            return output if output else "Code executed successfully (no output)"

        except Exception as e:
            return f"Execution error: {str(e)}"

    def install_package(self, package: str) -> str:
        """Install a package using pip"""
        try:
            result = subprocess.run([sys.executable, "-m", "pip", "install", package],
                                  capture_output=True, text=True)
            if result.returncode == 0:
                return f"Successfully installed package: {package}\n{result.stdout}"
            else:
                return f"Error installing package '{package}': {result.stderr or (result.stdout)}"
        except Exception as e:
            return f"Error executing pip command: {str(e)}"

    def list_files(self, path: str = ".") -> str:
        """List files in a directory"""
        try:
            items = os.listdir(path)
            return "\n".join(items) if items else f"Directory '{path}' is empty or does not exist."
        except FileNotFoundError:
            return f"Error: Directory '{path}' not found."
        except Exception as e:
            return f"Error listing files: {str(e)}"

    def read_file(self, filepath: str) -> str:
        """Read contents of a file"""
        try:
            with open(filepath, 'r') as f:
                content = f.read()
            return content
        except FileNotFoundError:
            return f"Error: File '{filepath}' not found."
        except Exception as e:
            return f"Error reading file: {str(e)}"

    def write_file(self, filepath: str, content: str) -> str:
        """Write content to a file"""
        try:
            with open(filepath, 'w') as f:
                f.write(content)
            return f"Successfully wrote to file '{filepath}'"
        except Exception as e:
            return f"Error writing to file: {str(e)}"

    def run(self, port: int = 8000):
        """Run the FastAPI server using uvicorn"""
        # Function to run uvicorn in a separate thread
        def run_server():
            uvicorn.run(self.app, host="0.0.0.0", port=port)

        # Start uvicorn in a thread
        thread = Thread(target=run_server)
        thread.daemon = True  # Allow the main program to exit even if the thread is running
        thread.start()

        # Set up ngrok tunnel
        try:
            print(f"Attempting to create ngrok tunnel for port {port}...")
            public_url = ngrok.connect(port).public_url
            print(f"ngrok tunnel created at: {public_url}")
            print(f"To connect MCP, use this URL: {public_url}")

            # Keep the main thread alive to keep the server and ngrok running
            try:
                # Use asyncio.get_event_loop() to get the current event loop
                loop = asyncio.get_event_loop()
                # Run forever - Ctrl+C to stop
                loop.run_forever()
            except RuntimeError:
                # Handle case where loop is already running (e.g., in some Colab environments)
                # In this case, just let the threads run
                print("Asyncio event loop already running.")
                # Keep the main thread alive by sleeping or waiting for thread
                while thread.is_alive():
                    import time
                    time.sleep(1)

        except Exception as e:
            print(f"Failed to create ngrok tunnel or run server: {e}")
            print("Please check your ngrok auth token and try again.")

# Example usage:
if __name__ == "__main__":
    server = ColabMCPServer()
    # You can specify a different port if needed, e.g., server.run(port=8001)
    server.run()
import asyncio
import logging
from typing import Any, Dict, List, Optional
import subprocess
import sys
import os
from pathlib import Path

# The packages should already be installed from the previous run
import fastapi
from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import JSONResponse
from pyngrok import ngrok
import uvicorn
from threading import Thread
import nest_asyncio

# Enable nested asyncio for Colab compatibility
nest_asyncio.apply()

# Set up your ngrok authentication token
ngrok.set_auth_token("2yuUKpiAlYXCez40i8UktLZ91s7_5Rv1GjG5HMFw5KKbk5sHz")

# The ColabMCPServer class remains the same
class ColabMCPServer:
    def __init__(self):
        self.app = FastAPI(title="Colab MCP Server", version="1.0.0")
        self.setup_routes()

    def setup_routes(self):
        @self.app.get("/")
        async def root():
            return {"message": "Colab MCP Server is running", "version": "1.0.0"}

        @self.app.post("/mcp/initialize")
        async def initialize(request: Request):
            """Initialize MCP connection"""
            data = await request.json()
            return {
                "protocolVersion": "2024-11-05",
                "capabilities": {
                    "tools": {
                        "listChanged": True
                    },
                    "resources": {
                        "subscribe": True,
                        "listChanged": True
                    }
                },
                "serverInfo": {
                    "name": "colab-mcp-server",
                    "version": "1.0.0"
                }
            }

        @self.app.post("/mcp/tools/list")
        async def list_tools():
            """List available tools"""
            return {
                "tools": [
                    {
                        "name": "execute_python",
                        "description": "Execute Python code in the Colab environment",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "code": {
                                    "type": "string",
                                    "description": "Python code to execute"
                                }
                            },
                            "required": ["code"]
                        }
                    },
                    {
                        "name": "install_package",
                        "description": "Install a Python package using pip",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "package": {
                                    "type": "string",
                                    "description": "Package name to install"
                                }
                            },
                            "required": ["package"]
                        }
                    },
                    {
                        "name": "list_files",
                        "description": "List files in a directory",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "path": {
                                    "type": "string",
                                    "description": "Directory path to list",
                                    "default": "."
                                }
                            }
                        }
                    },
                    {
                        "name": "read_file",
                        "description": "Read contents of a file",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "filepath": {
                                    "type": "string",
                                    "description": "Path to the file to read"
                                }
                            },
                            "required": ["filepath"]
                        }
                    },
                    {
                        "name": "write_file",
                        "description": "Write content to a file",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "filepath": {
                                    "type": "string",
                                    "description": "Path to the file to write"
                                },
                                "content": {
                                    "type": "string",
                                    "description": "Content to write to the file"
                                }
                            },
                            "required": ["filepath", "content"]
                        }
                    }
                ]
            }

        @self.app.post("/mcp/tools/call")
        async def call_tool(request: Request):
            """Execute a tool"""
            data = await request.json()
            tool_name = data.get("name")
            arguments = data.get("arguments", {})

            try:
                if tool_name == "execute_python":
                    result = self.execute_python(arguments.get("code", ""))
                elif tool_name == "install_package":
                    result = self.install_package(arguments.get("package", ""))
                elif tool_name == "list_files":
                    result = self.list_files(arguments.get("path", "."))
                elif tool_name == "read_file":
                    result = self.read_file(arguments.get("filepath", ""))
                elif tool_name == "write_file":
                    result = self.write_file(arguments.get("filepath", ""), arguments.get("content", ""))
                else:
                    raise HTTPException(status_code=400, detail=f"Unknown tool: {tool_name}")

                return {
                    "content": [
                        {
                            "type": "text",
                            "text": str(result)
                        }
                    ],
                    "isError": False
                }
            except Exception as e:
                return {
                    "content": [
                        {
                            "type": "text",
                            "text": f"Error: {str(e)}"
                        }
                    ],
                    "isError": True
                }

    def execute_python(self, code: str) -> str:
        """Execute Python code and return the result"""
        try:
            # Create a namespace for execution
            namespace = {}

            # Capture stdout
            from io import StringIO
            import contextlib

            stdout_capture = StringIO()

            with contextlib.redirect_stdout(stdout_capture):
                exec(code, namespace)

            output = stdout_capture.getvalue()
            return output if output else "Code executed successfully (no output)"

        except Exception as e:
            return f"Execution error: {str(e)}"

    def install_package(self, package: str) -> str:
        """Install a package using pip"""
        try:
            result = subprocess.run([sys.executable, "-m", "pip", "install", package],
                                  capture_output=True, text=True)
            if result.returncode == 0:
                return f"Successfully installed package: {package}\n{result.stdout}"
            else:
                return f"Error installing package '{package}': {result.stderr or (result.stdout)}"
        except Exception as e:
            return f"Error executing pip command: {str(e)}"

    def list_files(self, path: str = ".") -> str:
        """List files in a directory"""
        try:
            items = os.listdir(path)
            return "\n".join(items) if items else f"Directory '{path}' is empty or does not exist."
        except FileNotFoundError:
            return f"Error: Directory '{path}' not found."
        except Exception as e:
            return f"Error listing files: {str(e)}"

    def read_file(self, filepath: str) -> str:
        """Read contents of a file"""
        try:
            with open(filepath, 'r') as f:
                content = f.read()
            return content
        except FileNotFoundError:
            return f"Error: File '{filepath}' not found."
        except Exception as e:
            return f"Error reading file: {str(e)}"

    def write_file(self, filepath: str, content: str) -> str:
        """Write content to a file"""
        try:
            with open(filepath, 'w') as f:
                f.write(content)
            return f"Successfully wrote to file '{filepath}'"
        except Exception as e:
            return f"Error writing to file: {str(e)}"

    def run(self, port: int = 8000):
        """Run the FastAPI server using uvicorn"""
        # Function to run uvicorn in a separate thread
        def run_server():
            uvicorn.run(self.app, host="0.0.0.0", port=port)

        # Start uvicorn in a thread
        thread = Thread(target=run_server)
        thread.daemon = True  # Allow the main program to exit even if the thread is running
        thread.start()

        # Set up ngrok tunnel
        try:
            print(f"Attempting to create ngrok tunnel for port {port}...")
            public_url = ngrok.connect(port).public_url
            print(f"ngrok tunnel created at: {public_url}")
            print(f"To connect MCP, use this URL: {public_url}")

            # Keep the main thread alive to keep the server and ngrok running
            try:
                # Use asyncio.get_event_loop() to get the current event loop
                loop = asyncio.get_event_loop()
                # Run forever - Ctrl+C to stop
                loop.run_forever()
            except RuntimeError:
                # Handle case where loop is already running (e.g., in some Colab environments)
                # In this case, just let the threads run
                print("Asyncio event loop already running.")
                # Keep the main thread alive by sleeping or waiting for thread
                while thread.is_alive():
                    import time
                    time.sleep(1)

        except Exception as e:
            print(f"Failed to create ngrok tunnel or run server: {e}")
            print("Please check your ngrok auth token and try again.")

# Example usage:
if __name__ == "__main__":
    server = ColabMCPServer()
    # You can specify a different port if needed, e.g., server.run(port=8001)
    server.run()
            items = os.listdir(path)
            return "\n".join(items) if items else f"Directory '{path}' is empty or does not exist."
        except FileNotFoundError:
            return f"Error: Directory '{path}' not found."
        except Exception as e:
            return f"Error listing files: {str(e)}"

    def read_file(self, filepath: str) -> str:
        """Read contents of a file"""
        try:
            with open(filepath, 'r') as f:
                content = f.read()
            return content
        except FileNotFoundError:
            return f"Error: File '{filepath}' not found."
        except Exception as e:
            return f"Error reading file: {str(e)}"

    def write_file(self, filepath: str, content: str) -> str:
        """Write content to a file"""
        try:
            with open(filepath, 'w') as f:
                f.write(content)
            return f"Successfully wrote to file '{filepath}'"
        except Exception as e:
            return f"Error writing to file: {str(e)}"

    def run(self, port: int = 8000):
        """Run the FastAPI server using uvicorn"""
        # Function to run uvicorn in a separate thread
        def run_server():
            uvicorn.run(self.app, host="0.0.0.0", port=port)

        # Start uvicorn in a thread
        thread = Thread(target=run_server)
        thread.daemon = True  # Allow the main program to exit even if the thread is running
        thread.start()

        # Set up ngrok tunnel
        try:
            print(f"Attempting to create ngrok tunnel for port {port}...")
            public_url = ngrok.connect(port).public_url
            print(f"ngrok tunnel created at: {public_url}")
            print(f"To connect MCP, use this URL: {public_url}")

            # Keep the main thread alive to keep the server and ngrok running
            try:
                # Use asyncio.get_event_loop() to get the current event loop
                loop = asyncio.get_event_loop()
                # Run forever - Ctrl+C to stop
                loop.run_forever()
            except RuntimeError:
                # Handle case where loop is already running (e.g., in some Colab environments)
                # In this case, just let the threads run
                print("Asyncio event loop already running.")
                # Keep the main thread alive by sleeping or waiting for thread
                while thread.is_alive():
                    import time
                    time.sleep(1)

        except Exception as e:
            print(f"Failed to create ngrok tunnel or run server: {e}")
            print("Please check your ngrok auth token and try again.")

# Example usage:
if __name__ == "__main__":
    server = ColabMCPServer()
    # You can specify a different port if needed, e.g., server.run(port=8001)
    server.run()

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 773)

In [ ]:
# Fixed MCP Server for Google Colab with ngrok authentication

import json
import asyncio
import logging
from typing import Any, Dict, List, Optional
import subprocess
import sys
import os
from pathlib import Path

# Install required packages if not already installed

try:
import fastapi
import uvicorn
from pyngrok import ngrok
import nest_asyncio
except ImportError:
print(“Installing required packages…”)
subprocess.run([sys.executable, “-m”, “pip”, “install”, “fastapi”, “uvicorn”, “pyngrok”, “nest-asyncio”], check=True)
import fastapi
import uvicorn
from pyngrok import ngrok
import nest_asyncio

from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import JSONResponse
from threading import Thread

# Enable nested asyncio for Colab compatibility

nest_asyncio.apply()

# Set up your ngrok authentication token

ngrok.set_auth_token(“2yuUKpiAlYXCez40i8UktLZ91s7_5Rv1GjG5HMFw5KKbk5sHz”)

class ColabMCPServer:
def **init**(self):
self.app = FastAPI(title=“Colab MCP Server”, version=“1.0.0”)
self.setup_routes()

```
def setup_routes(self):
    @self.app.get("/")
    async def root():
        return {"message": "Colab MCP Server is running", "version": "1.0.0"}

    @self.app.post("/mcp/initialize")
    async def initialize(request: Request):
        """Initialize MCP connection"""
        data = await request.json()
        return {
            "protocolVersion": "2024-11-05",
            "capabilities": {
                "tools": {
                    "listChanged": True
                },
                "resources": {
                    "subscribe": True,
                    "listChanged": True
                }
            },
            "serverInfo": {
                "name": "colab-mcp-server",
                "version": "1.0.0"
            }
        }

    @self.app.post("/mcp/tools/list")
    async def list_tools():
        """List available tools"""
        return {
            "tools": [
                {
                    "name": "execute_python",
                    "description": "Execute Python code in the Colab environment",
                    "inputSchema": {
                        "type": "object",
                        "properties": {
                            "code": {
                                "type": "string",
                                "description": "Python code to execute"
                            }
                        },
                        "required": ["code"]
                    }
                },
                {
                    "name": "install_package",
                    "description": "Install a Python package using pip",
                    "inputSchema": {
                        "type": "object",
                        "properties": {
                            "package": {
                                "type": "string",
                                "description": "Package name to install"
                            }
                        },
                        "required": ["package"]
                    }
                },
                {
                    "name": "list_files",
                    "description": "List files in a directory",
                    "inputSchema": {
                        "type": "object",
                        "properties": {
                            "path": {
                                "type": "string",
                                "description": "Directory path to list",
                                "default": "."
                            }
                        }
                    }
                },
                {
                    "name": "read_file",
                    "description": "Read contents of a file",
                    "inputSchema": {
                        "type": "object",
                        "properties": {
                            "filepath": {
                                "type": "string",
                                "description": "Path to the file to read"
                            }
                        },
                        "required": ["filepath"]
                    }
                },
                {
                    "name": "write_file",
                    "description": "Write content to a file",
                    "inputSchema": {
                        "type": "object",
                        "properties": {
                            "filepath": {
                                "type": "string",
                                "description": "Path to the file to write"
                            },
                            "content": {
                                "type": "string",
                                "description": "Content to write to the file"
                            }
                        },
                        "required": ["filepath", "content"]
                    }
                }
            ]
        }

    @self.app.post("/mcp/tools/call")
    async def call_tool(request: Request):
        """Execute a tool"""
        data = await request.json()
        tool_name = data.get("name")
        arguments = data.get("arguments", {})

        try:
            if tool_name == "execute_python":
                result = self.execute_python(arguments.get("code", ""))
            elif tool_name == "install_package":
                result = self.install_package(arguments.get("package", ""))
            elif tool_name == "list_files":
                result = self.list_files(arguments.get("path", "."))
            elif tool_name == "read_file":
                result = self.read_file(arguments.get("filepath", ""))
            elif tool_name == "write_file":
                result = self.write_file(arguments.get("filepath", ""), arguments.get("content", ""))
            else:
                raise HTTPException(status_code=400, detail=f"Unknown tool: {tool_name}")

            return {
                "content": [
                    {
                        "type": "text",
                        "text": str(result)
                    }
                ],
                "isError": False
            }
        except Exception as e:
            return {
                "content": [
                    {
                        "type": "text",
                        "text": f"Error: {str(e)}"
                    }
                ],
                "isError": True
            }

def execute_python(self, code: str) -> str:
    """Execute Python code and return the result"""
    try:
        # Create a namespace for execution
        namespace = globals().copy()

        # Capture stdout
        from io import StringIO
        import contextlib

        stdout_capture = StringIO()

        with contextlib.redirect_stdout(stdout_capture):
            exec(code, namespace)

        output = stdout_capture.getvalue()
        return output if output else "Code executed successfully (no output)"

    except Exception as e:
        return f"Execution error: {str(e)}"

def install_package(self, package: str) -> str:
    """Install a package using pip"""
    try:
        result = subprocess.run([sys.executable, "-m", "pip", "install", package],
                              capture_output=True, text=True)
        if result.returncode == 0:
            return f"Package '{package}' installed successfully"
        else:
            return f"Installation failed: {result.stderr}"
    except Exception as e:
        return f"Installation error: {str(e)}"

def list_files(self, path: str = ".") -> str:
    """List files in a directory"""
    try:
        path_obj = Path(path)
        if not path_obj.exists():
            return f"Path does not exist: {path}"

        if path_obj.is_file():
            return f"'{path}' is a file, not a directory"

        files = []
        for item in path_obj.iterdir():
            if item.is_file():
                files.append(f"📄 {item.name}")
            else:
                files.append(f"📁 {item.name}/")

        return "\n".join(sorted(files)) if files else "Directory is empty"

    except Exception as e:
        return f"Error listing files: {str(e)}"

def read_file(self, filepath: str) -> str:
    """Read contents of a file"""
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            content = f.read()
        return content
    except Exception as e:
        return f"Error reading file: {str(e)}"

def write_file(self, filepath: str, content: str) -> str:
    """Write content to a file"""
    try:
        # Create directory if it doesn't exist
        Path(filepath).parent.mkdir(parents=True, exist_ok=True)

        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(content)
        return f"File written successfully: {filepath}"
    except Exception as e:
        return f"Error writing file: {str(e)}"
```

# Initialize and start the server

def start_mcp_server():
print(“🔐 Setting up MCP server…”)

```
server = ColabMCPServer()

# Start the server in a separate thread
def run_server():
    uvicorn.run(server.app, host="0.0.0.0", port=8001, log_level="info")

server_thread = Thread(target=run_server, daemon=True)
server_thread.start()

# Wait for the server to start
import time
time.sleep(3)

# Create ngrok tunnel
try:
    print("🌐 Creating public tunnel...")
    public_url = ngrok.connect(8001)

    print(f"\n🎉 SUCCESS! Your MCP Server is now publicly accessible!")
    print(f"=" * 60)
    print(f"📡 Public URL: {public_url}")
    print(f"🔧 Local URL: http://localhost:8001")
    print(f"=" * 60)

    print(f"\n📋 MCP Connection Details:")
    print(f"   🔗 Base Endpoint: {public_url}")
    print(f"   🚀 Initialize: POST {public_url}/mcp/initialize")
    print(f"   📝 List Tools: POST {public_url}/mcp/tools/list")
    print(f"   ⚡ Call Tool: POST {public_url}/mcp/tools/call")

    print(f"\n🛠️  Available MCP Tools:")
    print(f"   🐍 execute_python   - Execute Python code remotely")
    print(f"   📦 install_package  - Install pip packages")
    print(f"   📂 list_files       - Browse directory contents")
    print(f"   📖 read_file        - Read file contents")
    print(f"   ✏️  write_file       - Create/modify files")

    print(f"\n💡 Usage Example (with curl):")
    curl_command = f'''curl -X POST "{public_url}/mcp/tools/call" \\
```

-H “Content-Type: application/json” \
-d ‘{{“name”: “execute_python”, “arguments”: {{“code”: “print(\“Hello MCP!\”)”}}}}’
‘’’
print(curl_command)

```
    return public_url

except Exception as e:
    print(f"❌ Error setting up ngrok tunnel: {e}")
    return None
```

# Start the server

if **name** == “**main**”:
public_url = start_mcp_server()

```
if public_url:
    print(f"\n✅ Your MCP server is live and ready for connections!")
    print(f"🔗 Share this URL with your MCP clients: {public_url}")
    print(f"⏰ Server will remain active as long as this Colab session is running")

    # Test the public connection
    print(f"\n🧪 Testing public connection...")
    import requests
    try:
        response = requests.get(f"{public_url}/")
        print(f"✅ Public endpoint test: {response.json()}")
    except Exception as e:
        print(f"⚠️  Public test failed (this might be normal): {e}")

else:
    print(f"❌ Failed to create public tunnel. Check your ngrok setup.")

print(f"\n🔄 To stop the server, restart the runtime or interrupt this cell.")

# Keep the server running
try:
    while True:
        import time
        time.sleep(60)
except KeyboardInterrupt:
    print("\n👋 Server stopped.")
```

SyntaxError: invalid character '“' (U+201C) (ipython-input-20-362580492.py, line 20)

In [ ]:
# Fixed MCP Server for Google Colab with ngrok authentication

import json
import asyncio
import logging
from typing import Any, Dict, List, Optional
import subprocess
import sys
import os
from pathlib import Path

# Install required packages if not already installed

try:
import fastapi
import uvicorn
from pyngrok import ngrok
import nest_asyncio
except ImportError:
print(“Installing required packages…”)
subprocess.run([sys.executable, “-m”, “pip”, “install”, “fastapi”, “uvicorn”, “pyngrok”, “nest-asyncio”], check=True)
import fastapi
import uvicorn
from pyngrok import ngrok
import nest_asyncio

from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import JSONResponse
from threading import Thread

# Enable nested asyncio for Colab compatibility

nest_asyncio.apply()

# Set up your ngrok authentication token

ngrok.set_auth_token(“2yuUKpiAlYXCez40i8UktLZ91s7_5Rv1GjG5HMFw5KKbk5sHz”)

class ColabMCPServer:
def **init**(self):
self.app = FastAPI(title=“Colab MCP Server”, version=“1.0.0”)
self.setup_routes()

```
def setup_routes(self):
    @self.app.get("/")
    async def root():
        return {"message": "Colab MCP Server is running", "version": "1.0.0"}

    @self.app.post("/mcp/initialize")
    async def initialize(request: Request):
        """Initialize MCP connection"""
        data = await request.json()
        return {
            "protocolVersion": "2024-11-05",
            "capabilities": {
                "tools": {
                    "listChanged": True
                },
                "resources": {
                    "subscribe": True,
                    "listChanged": True
                }
            },
            "serverInfo": {
                "name": "colab-mcp-server",
                "version": "1.0.0"
            }
        }

    @self.app.post("/mcp/tools/list")
    async def list_tools():
        """List available tools"""
        return {
            "tools": [
                {
                    "name": "execute_python",
                    "description": "Execute Python code in the Colab environment",
                    "inputSchema": {
                        "type": "object",
                        "properties": {
                            "code": {
                                "type": "string",
                                "description": "Python code to execute"
                            }
                        },
                        "required": ["code"]
                    }
                },
                {
                    "name": "install_package",
                    "description": "Install a Python package using pip",
                    "inputSchema": {
                        "type": "object",
                        "properties": {
                            "package": {
                                "type": "string",
                                "description": "Package name to install"
                            }
                        },
                        "required": ["package"]
                    }
                },
                {
                    "name": "list_files",
                    "description": "List files in a directory",
                    "inputSchema": {
                        "type": "object",
                        "properties": {
                            "path": {
                                "type": "string",
                                "description": "Directory path to list",
                                "default": "."
                            }
                        }
                    }
                },
                {
                    "name": "read_file",
                    "description": "Read contents of a file",
                    "inputSchema": {
                        "type": "object",
                        "properties": {
                            "filepath": {
                                "type": "string",
                                "description": "Path to the file to read"
                            }
                        },
                        "required": ["filepath"]
                    }
                },
                {
                    "name": "write_file",
                    "description": "Write content to a file",
                    "inputSchema": {
                        "type": "object",
                        "properties": {
                            "filepath": {
                                "type": "string",
                                "description": "Path to the file to write"
                            },
                            "content": {
                                "type": "string",
                                "description": "Content to write to the file"
                            }
                        },
                        "required": ["filepath", "content"]
                    }
                }
            ]
        }

    @self.app.post("/mcp/tools/call")
    async def call_tool(request: Request):
        """Execute a tool"""
        data = await request.json()
        tool_name = data.get("name")
        arguments = data.get("arguments", {})

        try:
            if tool_name == "execute_python":
                result = self.execute_python(arguments.get("code", ""))
            elif tool_name == "install_package":
                result = self.install_package(arguments.get("package", ""))
            elif tool_name == "list_files":
                result = self.list_files(arguments.get("path", "."))
            elif tool_name == "read_file":
                result = self.read_file(arguments.get("filepath", ""))
            elif tool_name == "write_file":
                result = self.write_file(arguments.get("filepath", ""), arguments.get("content", ""))
            else:
                raise HTTPException(status_code=400, detail=f"Unknown tool: {tool_name}")

            return {
                "content": [
                    {
                        "type": "text",
                        "text": str(result)
                    }
                ],
                "isError": False
            }
        except Exception as e:
            return {
                "content": [
                    {
                        "type": "text",
                        "text": f"Error: {str(e)}"
                    }
                ],
                "isError": True
            }

def execute_python(self, code: str) -> str:
    """Execute Python code and return the result"""
    try:
        # Create a namespace for execution
        namespace = globals().copy()

        # Capture stdout
        from io import StringIO
        import contextlib

        stdout_capture = StringIO()

        with contextlib.redirect_stdout(stdout_capture):
            exec(code, namespace)

        output = stdout_capture.getvalue()
        return output if output else "Code executed successfully (no output)"

    except Exception as e:
        return f"Execution error: {str(e)}"

def install_package(self, package: str) -> str:
    """Install a package using pip"""
    try:
        result = subprocess.run([sys.executable, "-m", "pip", "install", package],
                              capture_output=True, text=True)
        if result.returncode == 0:
            return f"Package '{package}' installed successfully"
        else:
            return f"Installation failed: {result.stderr}"
    except Exception as e:
        return f"Installation error: {str(e)}"

def list_files(self, path: str = ".") -> str:
    """List files in a directory"""
    try:
        path_obj = Path(path)
        if not path_obj.exists():
            return f"Path does not exist: {path}"

        if path_obj.is_file():
            return f"'{path}' is a file, not a directory"

        files = []
        for item in path_obj.iterdir():
            if item.is_file():
                files.append(f"📄 {item.name}")
            else:
                files.append(f"📁 {item.name}/")

        return "\n".join(sorted(files)) if files else "Directory is empty"

    except Exception as e:
        return f"Error listing files: {str(e)}"

def read_file(self, filepath: str) -> str:
    """Read contents of a file"""
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            content = f.read()
        return content
    except Exception as e:
        return f"Error reading file: {str(e)}"

def write_file(self, filepath: str, content: str) -> str:
    """Write content to a file"""
    try:
        # Create directory if it doesn't exist
        Path(filepath).parent.mkdir(parents=True, exist_ok=True)

        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(content)
        return f"File written successfully: {filepath}"
    except Exception as e:
        return f"Error writing file: {str(e)}"
```

# Initialize and start the server

def start_mcp_server():
print(“🔐 Setting up MCP server…”)

```
server = ColabMCPServer()

# Start the server in a separate thread
def run_server():
    uvicorn.run(server.app, host="0.0.0.0", port=8001, log_level="info")

server_thread = Thread(target=run_server, daemon=True)
server_thread.start()

# Wait for the server to start
import time
time.sleep(3)

# Create ngrok tunnel
try:
    print("🌐 Creating public tunnel...")
    public_url = ngrok.connect(8001)

    print(f"\n🎉 SUCCESS! Your MCP Server is now publicly accessible!")
    print("=" * 60)
    print(f"📡 Public URL: {public_url}")
    print("🔧 Local URL: http://localhost:8001")
    print("=" * 60)

    print("\n📋 MCP Connection Details:")
    print(f"   🔗 Base Endpoint: {public_url}")
    print(f"   🚀 Initialize: POST {public_url}/mcp/initialize")
    print(f"   📝 List Tools: POST {public_url}/mcp/tools/list")
    print(f"   ⚡ Call Tool: POST {public_url}/mcp/tools/call")

    print("\n🛠️  Available MCP Tools:")
    print("   🐍 execute_python   - Execute Python code remotely")
    print("   📦 install_package  - Install pip packages")
    print("   📂 list_files       - Browse directory contents")
    print("   📖 read_file        - Read file contents")
    print("   ✏️  write_file       - Create/modify files")

    print("\n💡 Usage Example (with curl):")
    curl_command = f'''curl -X POST "{public_url}/mcp/tools/call" \\
```

-H “Content-Type: application/json” \
-d ‘{{“name”: “execute_python”, “arguments”: {{“code”: “print(\“Hello MCP!\”)”}}}}’
‘’’
print(curl_command)

```
    return public_url

except Exception as e:
    print(f"❌ Error setting up ngrok tunnel: {e}")
    return None
```

# Start the server

if **name** == “**main**”:
public_url = start_mcp_server()

```
if public_url:
    print("\n✅ Your MCP server is live and ready for connections!")
    print(f"🔗 Share this URL with your MCP clients: {public_url}")
    print("⏰ Server will remain active as long as this Colab session is running")

    # Test the public connection
    print("\n🧪 Testing public connection...")
    import requests
    try:
        response = requests.get(f"{public_url}/")
        print(f"✅ Public endpoint test: {response.json()}")
    except Exception as e:
        print(f"⚠️  Public test failed (this might be normal): {e}")

else:
    print("❌ Failed to create public tunnel. Check your ngrok setup.")

print("\n🔄 To stop the server, restart the runtime or interrupt this cell.")

# Keep the server running
try:
    while True:
        import time
        time.sleep(60)
except KeyboardInterrupt:
    print("\n👋 Server stopped.")


SyntaxError: invalid character '“' (U+201C) (ipython-input-22-3986050408.py, line 20)

In [ ]:
# Fixed MCP Server for Google Colab with ngrok authentication

import json
import asyncio
import logging
from typing import Any, Dict, List, Optional
import subprocess
import sys
import os
from pathlib import Path

# Install required packages if not already installed

try:
import fastapi
import uvicorn
from pyngrok import ngrok
import nest_asyncio
except ImportError:
print(“Installing required packages…”)
subprocess.run([sys.executable, “-m”, “pip”, “install”, “fastapi”, “uvicorn”, “pyngrok”, “nest-asyncio”], check=True)
import fastapi
import uvicorn
from pyngrok import ngrok
import nest_asyncio

from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import JSONResponse
from threading import Thread

# Enable nested asyncio for Colab compatibility

nest_asyncio.apply()

# Set up your ngrok authentication token

ngrok.set_auth_token(“2yuUKpiAlYXCez40i8UktLZ91s7_5Rv1GjG5HMFw5KKbk5sHz”)

class ColabMCPServer:
def **init**(self):
self.app = FastAPI(title=“Colab MCP Server”, version=“1.0.0”)
self.setup_routes()

```
def setup_routes(self):
    @self.app.get("/")
    async def root():
        return {"message": "Colab MCP Server is running", "version": "1.0.0"}

    @self.app.post("/mcp/initialize")
    async def initialize(request: Request):
        """Initialize MCP connection"""
        data = await request.json()
        return {
            "protocolVersion": "2024-11-05",
            "capabilities": {
                "tools": {
                    "listChanged": True
                },
                "resources": {
                    "subscribe": True,
                    "listChanged": True
                }
            },
            "serverInfo": {
                "name": "colab-mcp-server",
                "version": "1.0.0"
            }
        }

    @self.app.post("/mcp/tools/list")
    async def list_tools():
        """List available tools"""
        return {
            "tools": [
                {
                    "name": "execute_python",
                    "description": "Execute Python code in the Colab environment",
                    "inputSchema": {
                        "type": "object",
                        "properties": {
                            "code": {
                                "type": "string",
                                "description": "Python code to execute"
                            }
                        },
                        "required": ["code"]
                    }
                },
                {
                    "name": "install_package",
                    "description": "Install a Python package using pip",
                    "inputSchema": {
                        "type": "object",
                        "properties": {
                            "package": {
                                "type": "string",
                                "description": "Package name to install"
                            }
                        },
                        "required": ["package"]
                    }
                },
                {
                    "name": "list_files",
                    "description": "List files in a directory",
                    "inputSchema": {
                        "type": "object",
                        "properties": {
                            "path": {
                                "type": "string",
                                "description": "Directory path to list",
                                "default": "."
                            }
                        }
                    }
                },
                {
                    "name": "read_file",
                    "description": "Read contents of a file",
                    "inputSchema": {
                        "type": "object",
                        "properties": {
                            "filepath": {
                                "type": "string",
                                "description": "Path to the file to read"
                            }
                        },
                        "required": ["filepath"]
                    }
                },
                {
                    "name": "write_file",
                    "description": "Write content to a file",
                    "inputSchema": {
                        "type": "object",
                        "properties": {
                            "filepath": {
                                "type": "string",
                                "description": "Path to the file to write"
                            },
                            "content": {
                                "type": "string",
                                "description": "Content to write to the file"
                            }
                        },
                        "required": ["filepath", "content"]
                    }
                }
            ]
        }

    @self.app.post("/mcp/tools/call")
    async def call_tool(request: Request):
        """Execute a tool"""
        data = await request.json()
        tool_name = data.get("name")
        arguments = data.get("arguments", {})

        try:
            if tool_name == "execute_python":
                result = self.execute_python(arguments.get("code", ""))
            elif tool_name == "install_package":
                result = self.install_package(arguments.get("package", ""))
            elif tool_name == "list_files":
                result = self.list_files(arguments.get("path", "."))
            elif tool_name == "read_file":
                result = self.read_file(arguments.get("filepath", ""))
            elif tool_name == "write_file":
                result = self.write_file(arguments.get("filepath", ""), arguments.get("content", ""))
            else:
                raise HTTPException(status_code=400, detail=f"Unknown tool: {tool_name}")

            return {
                "content": [
                    {
                        "type": "text",
                        "text": str(result)
                    }
                ],
                "isError": False
            }
        except Exception as e:
            return {
                "content": [
                    {
                        "type": "text",
                        "text": f"Error: {str(e)}"
                    }
                ],
                "isError": True
            }

def execute_python(self, code: str) -> str:
    """Execute Python code and return the result"""
    try:
        # Create a namespace for execution
        namespace = globals().copy()

        # Capture stdout
        from io import StringIO
        import contextlib

        stdout_capture = StringIO()

        with contextlib.redirect_stdout(stdout_capture):
            exec(code, namespace)

        output = stdout_capture.getvalue()
        return output if output else "Code executed successfully (no output)"

    except Exception as e:
        return f"Execution error: {str(e)}"

def install_package(self, package: str) -> str:
    """Install a package using pip"""
    try:
        result = subprocess.run([sys.executable, "-m", "pip", "install", package],
                              capture_output=True, text=True)
        if result.returncode == 0:
            return f"Package '{package}' installed successfully"
        else:
            return f"Installation failed: {result.stderr}"
    except Exception as e:
        return f"Installation error: {str(e)}"

def list_files(self, path: str = ".") -> str:
    """List files in a directory"""
    try:
        path_obj = Path(path)
        if not path_obj.exists():
            return f"Path does not exist: {path}"

        if path_obj.is_file():
            return f"'{path}' is a file, not a directory"

        files = []
        for item in path_obj.iterdir():
            if item.is_file():
                files.append(f"📄 {item.name}")
            else:
                files.append(f"📁 {item.name}/")

        return "\n".join(sorted(files)) if files else "Directory is empty"

    except Exception as e:
        return f"Error listing files: {str(e)}"

def read_file(self, filepath: str) -> str:
    """Read contents of a file"""
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            content = f.read()
        return content
    except Exception as e:
        return f"Error reading file: {str(e)}"

def write_file(self, filepath: str, content: str) -> str:
    """Write content to a file"""
    try:
        # Create directory if it doesn't exist
        Path(filepath).parent.mkdir(parents=True, exist_ok=True)

        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(content)
        return f"File written successfully: {filepath}"
    except Exception as e:
        return f"Error writing file: {str(e)}"
```

# Initialize and start the server

def start_mcp_server():
print(“🔐 Setting up MCP server…”)

```
server = ColabMCPServer()

# Start the server in a separate thread
def run_server():
    uvicorn.run(server.app, host="0.0.0.0", port=8001, log_level="info")

server_thread = Thread(target=run_server, daemon=True)
server_thread.start()

# Wait for the server to start
import time
time.sleep(3)

# Create ngrok tunnel
try:
    print("🌐 Creating public tunnel...")
    public_url = ngrok.connect(8001)

    print(f"\n🎉 SUCCESS! Your MCP Server is now publicly accessible!")
    print("=" * 60)
    print(f"📡 Public URL: {public_url}")
    print("🔧 Local URL: http://localhost:8001")
    print("=" * 60)

    print("\n📋 MCP Connection Details:")
    print(f"   🔗 Base Endpoint: {public_url}")
    print(f"   🚀 Initialize: POST {public_url}/mcp/initialize")
    print(f"   📝 List Tools: POST {public_url}/mcp/tools/list")
    print(f"   ⚡ Call Tool: POST {public_url}/mcp/tools/call")

    print("\n🛠️  Available MCP Tools:")
    print("   🐍 execute_python   - Execute Python code remotely")
    print("   📦 install_package  - Install pip packages")
    print("   📂 list_files       - Browse directory contents")
    print("   📖 read_file        - Read file contents")
    print("   ✏️  write_file       - Create/modify files")

    print("\n💡 Usage Example (with curl):")
    curl_command = f'''curl -X POST "{public_url}/mcp/tools/call" \\
```

-H “Content-Type: application/json” \
-d ‘{{“name”: “execute_python”, “arguments”: {{“code”: “print(\“Hello MCP!\”)”}}}}’
‘’’
print(curl_command)

```
    return public_url

except Exception as e:
    print(f"❌ Error setting up ngrok tunnel: {e}")
    return None
```

# Start the server

if **name** == “**main**”:
public_url = start_mcp_server()

```
if public_url:
    print("\n✅ Your MCP server is live and ready for connections!")
    print(f"🔗 Share this URL with your MCP clients: {public_url}")
    print("⏰ Server will remain active as long as this Colab session is running")

    # Test the public connection
    print("\n🧪 Testing public connection...")
    import requests
    try:
        response = requests.get(f"{public_url}/")
        print(f"✅ Public endpoint test: {response.json()}")
    except Exception as e:
        print(f"⚠️  Public test failed (this might be normal): {e}")

else:
    print("❌ Failed to create public tunnel. Check your ngrok setup.")

print("\n🔄 To stop the server, restart the runtime or interrupt this cell.")

# Keep the server running
try:
    while True:
        import time
        time.sleep(60)
except KeyboardInterrupt:
    print("\n👋 Server stopped.")
```

SyntaxError: invalid character '“' (U+201C) (ipython-input-23-480069480.py, line 20)

In [ ]:
# Fixed MCP Server for Google Colab with ngrok authentication

import json
import asyncio
import logging
import subprocess
import sys
import os
import time
from pathlib import Path
from threading import Thread

# Install required packages if not already installed
try:
    import fastapi
    import uvicorn
    from pyngrok import ngrok
    import nest_asyncio
    import requests
except ImportError:
    print("Installing required packages…")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "fastapi", "uvicorn", "pyngrok", "nest-asyncio", "requests"],
        check=True
    )
    import fastapi
    import uvicorn
    from pyngrok import ngrok
    import nest_asyncio
    import requests

from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import JSONResponse

# Enable nested asyncio for Colab compatibility
nest_asyncio.apply()

# Set up your ngrok authentication token
ngrok.set_auth_token("2yuUKpiAlYXCez40i8UktLZ91s7_5Rv1GjG5HMFw5KKbk5sHz")

class ColabMCPServer:
    def __init__(self):
        self.app = FastAPI(title="Colab MCP Server", version="1.0.0")
        self.setup_routes()

    def setup_routes(self):
        @self.app.get("/")
        async def root():
            return {"message": "Colab MCP Server is running", "version": "1.0.0"}

        @self.app.post("/mcp/initialize")
        async def initialize(request: Request):
            """Initialize MCP connection"""
            data = await request.json()
            return {
                "protocolVersion": "2024-11-05",
                "capabilities": {
                    "tools": {
                        "listChanged": True
                    },
                    "resources": {
                        "subscribe": True,
                        "listChanged": True
                    }
                },
                "serverInfo": {
                    "name": "colab-mcp-server",
                    "version": "1.0.0"
                }
            }

        @self.app.post("/mcp/tools/list")
        async def list_tools():
            """List available tools"""
            return {
                "tools": [
                    {
                        "name": "execute_python",
                        "description": "Execute Python code in the Colab environment",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "code": {
                                    "type": "string",
                                    "description": "Python code to execute"
                                }
                            },
                            "required": ["code"]
                        }
                    },
                    {
                        "name": "install_package",
                        "description": "Install a Python package using pip",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "package": {
                                    "type": "string",
                                    "description": "Package name to install"
                                }
                            },
                            "required": ["package"]
                        }
                    },
                    {
                        "name": "list_files",
                        "description": "List files in a directory",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "path": {
                                    "type": "string",
                                    "description": "Directory path to list",
                                    "default": "."
                                }
                            }
                        }
                    },
                    {
                        "name": "read_file",
                        "description": "Read contents of a file",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "filepath": {
                                    "type": "string",
                                    "description": "Path to the file to read"
                                }
                            },
                            "required": ["filepath"]
                        }
                    },
                    {
                        "name": "write_file",
                        "description": "Write content to a file",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "filepath": {
                                    "type": "string",
                                    "description": "Path to the file to write"
                                },
                                "content": {
                                    "type": "string",
                                    "description": "Content to write to the file"
                                }
                            },
                            "required": ["filepath", "content"]
                        }
                    }
                ]
            }

        @self.app.post("/mcp/tools/call")
        async def call_tool(request: Request):
            """Execute a tool"""
            data = await request.json()
            tool_name = data.get("name")
            arguments = data.get("arguments", {})

            try:
                if tool_name == "execute_python":
                    result = self.execute_python(arguments.get("code", ""))
                elif tool_name == "install_package":
                    result = self.install_package(arguments.get("package", ""))
                elif tool_name == "list_files":
                    result = self.list_files(arguments.get("path", "."))
                elif tool_name == "read_file":
                    result = self.read_file(arguments.get("filepath", ""))
                elif tool_name == "write_file":
                    result = self.write_file(arguments.get("filepath", ""), arguments.get("content", ""))
                else:
                    raise HTTPException(status_code=400, detail=f"Unknown tool: {tool_name}")

                return {
                    "content": [
                        {
                            "type": "text",
                            "text": str(result)
                        }
                    ],
                    "isError": False
                }
            except Exception as e:
                return {
                    "content": [
                        {
                            "type": "text",
                            "text": f"Error: {str(e)}"
                        }
                    ],
                    "isError": True
                }

    def execute_python(self, code: str) -> str:
        """Execute Python code and return the result"""
        try:
            namespace = globals().copy()
            from io import StringIO
            import contextlib

            stdout_capture = StringIO()
            with contextlib.redirect_stdout(stdout_capture):
                exec(code, namespace)
            output = stdout_capture.getvalue()
            return output if output else "Code executed successfully (no output)"
        except Exception as e:
            return f"Execution error: {str(e)}"

    def install_package(self, package: str) -> str:
        """Install a package using pip"""
        try:
            result = subprocess.run(
                [sys.executable, "-m", "pip", "install", package],
                capture_output=True,
                text=True
            )
            if result.returncode == 0:
                return f"Package '{package}' installed successfully"
            else:
                return f"Installation failed: {result.stderr}"
        except Exception as e:
            return f"Installation error: {str(e)}"

    def list_files(self, path: str = ".") -> str:
        """List files in a directory"""
        try:
            path_obj = Path(path)
            if not path_obj.exists():
                return f"Path does not exist: {path}"
            if path_obj.is_file():
                return f"'{path}' is a file, not a directory"

            entries = []
            for item in sorted(path_obj.iterdir()):
                prefix = "📄" if item.is_file() else "📁"
                suffix = "" if item.is_file() else "/"
                entries.append(f"{prefix} {item.name}{suffix}")
            return "\n".join(entries) if entries else "Directory is empty"
        except Exception as e:
            return f"Error listing files: {str(e)}"

    def read_file(self, filepath: str) -> str:
        """Read contents of a file"""
        try:
            with open(filepath, "r", encoding="utf-8") as f:
                return f.read()
        except Exception as e:
            return f"Error reading file: {str(e)}"

    def write_file(self, filepath: str, content: str) -> str:
        """Write content to a file"""
        try:
            Path(filepath).parent.mkdir(parents=True, exist_ok=True)
            with open(filepath, "w", encoding="utf-8") as f:
                f.write(content)
            return f"File written successfully: {filepath}"
        except Exception as e:
            return f"Error writing file: {str(e)}"

def start_mcp_server() -> Optional[str]:
    print("🔐 Setting up MCP server…")
    server = ColabMCPServer()

    # Start the server in a separate thread
    def run_server():
        uvicorn.run(server.app, host="0.0.0.0", port=8001, log_level="info")

    server_thread = Thread(target=run_server, daemon=True)
    server_thread.start()

    # Wait for the server to start
    time.sleep(3)

    # Create ngrok tunnel
    try:
        print("🌐 Creating public tunnel...")
        public_url = ngrok.connect(8001).public_url
        print(f"\n🎉 SUCCESS! Your MCP Server is now publicly accessible!")
        print("=" * 60)
        print(f"📡 Public URL: {public_url}")
        print("🔧 Local URL: http://localhost:8001")
        print("=" * 60)
        print("\n📋 MCP Connection Details:")
        print(f"   🔗 Base Endpoint: {public_url}")
        print(f"   🚀 Initialize: POST {public_url}/mcp/initialize")
        print(f"   📝 List Tools: POST {public_url}/mcp/tools/list")
        print(f"   ⚡ Call Tool: POST {public_url}/mcp/tools/call")
        print("\n🛠️  Available MCP Tools:")
        print("   🐍 execute_python   - Execute Python code remotely")
        print("   📦 install_package  - Install pip packages")
        print("   📂 list_files       - Browse directory contents")
        print("   📖 read_file        - Read file contents")
        print("   ✏️  write_file       - Create/modify files")
        print("\n💡 Usage Example (with curl):")
        curl_command = (
            f'curl -X POST "{public_url}/mcp/tools/call" '
            '-H "Content-Type: application/json" '
            '-d \'{"name": "execute_python", "arguments": {"code": "print(\\\"Hello MCP!\\\")"}}\''
        )
        print(curl_command)
        return public_url
    except Exception as e:
        print(f"❌ Error setting up ngrok tunnel: {e}")
        return None

if __name__ == "__main__":
    public_url = start_mcp_server()

    if public_url:
        print("\n✅ Your MCP server is live and ready for connections!")
        print(f"🔗 Share this URL with your MCP clients: {public_url}")
        print("⏰ Server will remain active as long as this Colab session is running")
        print("\n🧪 Testing public connection...")
        try:
            response = requests.get(f"{public_url}/")
            print(f"✅ Public endpoint test: {response.json()}")
        except Exception as e:
            print(f"⚠️  Public test failed (this might be normal): {e}")
    else:
        print("❌ Failed to create public tunnel. Check your ngrok setup.")

    # Keep the server running until interrupted
    try:
        while True:
            time.sleep(60)
    except KeyboardInterrupt:
        print("\n👋 Server stopped.")

🔐 Setting up MCP server…


INFO:     Started server process [927]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8001 (Press CTRL+C to quit)


🌐 Creating public tunnel...

🎉 SUCCESS! Your MCP Server is now publicly accessible!
📡 Public URL: https://02e0-34-124-231-165.ngrok-free.app
🔧 Local URL: http://localhost:8001

📋 MCP Connection Details:
   🔗 Base Endpoint: https://02e0-34-124-231-165.ngrok-free.app
   🚀 Initialize: POST https://02e0-34-124-231-165.ngrok-free.app/mcp/initialize
   📝 List Tools: POST https://02e0-34-124-231-165.ngrok-free.app/mcp/tools/list
   ⚡ Call Tool: POST https://02e0-34-124-231-165.ngrok-free.app/mcp/tools/call

🛠️  Available MCP Tools:
   🐍 execute_python   - Execute Python code remotely
   📦 install_package  - Install pip packages
   📂 list_files       - Browse directory contents
   📖 read_file        - Read file contents
   ✏️  write_file       - Create/modify files

💡 Usage Example (with curl):
curl -X POST "https://02e0-34-124-231-165.ngrok-free.app/mcp/tools/call" -H "Content-Type: application/json" -d '{"name": "execute_python", "arguments": {"code": "print(\"Hello MCP!\")"}}'

✅ Your MCP s

In [ ]:
# streaming_mcp_server.py

import subprocess
import sys
import time
import nest_asyncio
from pathlib import Path
from threading import Thread

from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import StreamingResponse, JSONResponse
from pyngrok import ngrok
import uvicorn

# Enable nested asyncio for Colab compatibility
nest_asyncio.apply()

# Set your ngrok auth token
ngrok.set_auth_token("2yuUKpiAlYXCez40i8UktLZ91s7_5Rv1GjG5HMFw5KKbk5sHz")

app = FastAPI(title="Colab MCP Server (with Streaming)", version="1.1.0")


@app.get("/")
async def root():
    return {"message": "Colab MCP Server is running (streaming enabled)", "version": "1.1.0"}


@app.post("/mcp/initialize")
async def initialize(request: Request):
    await request.json()
    return {
        "protocolVersion": "2024-11-05",
        "capabilities": {
            "tools": {"listChanged": True},
            "resources": {"subscribe": True, "listChanged": True}
        },
        "serverInfo": {"name": "colab-mcp-server", "version": "1.1.0"}
    }


@app.post("/mcp/tools/list")
async def list_tools():
    return {
        "tools": [
            {"name": "execute_python", "description": "Execute Python code and return full output"},
            {"name": "execute_python_stream", "description": "Execute Python code and stream stdout line-by-line"}
        ]
    }


@app.post("/mcp/tools/call")
async def call_tool(request: Request):
    data = await request.json()
    name = data.get("name")
    args = data.get("arguments", {})
    try:
        if name == "execute_python":
            output = execute_python_full(args.get("code", ""))
            return {"content": [{"type": "text", "text": output}], "isError": False}
        else:
            raise HTTPException(status_code=400, detail=f"Unknown tool: {name}")
    except Exception as e:
        return {"content": [{"type": "text", "text": f"Error: {str(e)}"}], "isError": True}


@app.post("/mcp/tools/execute_python_stream")
async def execute_python_stream(request: Request):
    """
    Streams stdout of the given code back to the client one line at a time.
    """
    data = await request.json()
    code = data.get("code", "")

    def generator():
        # Write code to a temporary file
        tmp_file = Path("__mcp_tmp.py")
        tmp_file.write_text(code, encoding="utf-8")

        # Spawn a subprocess with unbuffered output
        process = subprocess.Popen(
            [sys.executable, "-u", str(tmp_file)],
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )

        # Stream each line as it arrives
        for line in process.stdout:
            yield line
        process.wait()
        tmp_file.unlink(missing_ok=True)

    return StreamingResponse(generator(), media_type="text/plain")


def execute_python_full(code: str) -> str:
    """
    Execute Python code in-process and capture all stdout.
    """
    try:
        from io import StringIO
        import contextlib

        buf = StringIO()
        namespace = globals().copy()
        with contextlib.redirect_stdout(buf):
            exec(code, namespace)
        return buf.getvalue() or "Code executed successfully (no output)"
    except Exception as e:
        return f"Execution error: {str(e)}"


def start_server():
    # Start FastAPI + ngrok
    def run():
        uvicorn.run(app, host="0.0.0.0", port=8001, log_level="info")

    Thread(target=run, daemon=True).start()
    time.sleep(2)
    url = ngrok.connect(8001).public_url
    print(f"🔗 Public URL: {url}")
    return url


if __name__ == "__main__":
    public_url = start_server()
    print(f"🚀 Server live at {public_url}")
    try:
        while True:
            time.sleep(60)
    except KeyboardInterrupt:
        print("👋 Shutting down.")

ModuleNotFoundError: No module named 'pyngrok'

In [ ]:
"""
Colab MCP Server with Live Activity Dashboard and File Download

This script sets up:
 1. A FastAPI–based MCP server for remote tool execution (execute_python, install_package, list_files, read_file, write_file).
 2. A Server-Sent Events (SSE) “activity” dashboard at /activity that shows every tool call and its result in real time.
 3. A file-download endpoint at /download/{file_path} so you can fetch any file from the Colab filesystem (or mounted Drive).

To use:
 - Paste this entire cell into a Colab notebook.
 - Run the cell. It will install dependencies automatically and print your public URLs.
 - Open the “Dashboard” URL in your browser to watch activity live.
 - Use the /mcp/... endpoints as before to call tools.
"""

# ─── Install dependencies ───────────────────────────────────────────────────────
import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet",
     "fastapi", "uvicorn", "pyngrok", "nest-asyncio", "sse-starlette", "requests"],
    check=True
)

# ─── Standard imports and setup ─────────────────────────────────────────────────
import time
import logging
import asyncio
import nest_asyncio
from threading import Thread
from pathlib import Path

from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import HTMLResponse, FileResponse
from sse_starlette.sse import EventSourceResponse
from pyngrok import ngrok
import uvicorn

# ─── Enable nested asyncio for Colab compatibility ──────────────────────────────
nest_asyncio.apply()

# ─── Configure ngrok authentication ─────────────────────────────────────────────
# Replace the token value below with your own ngrok auth token if desired.
ngrok.set_auth_token("2yuUKpiAlYXCez40i8UktLZ91s7_5Rv1GjG5HMFw5KKbk5sHz")

# ─── Create the FastAPI app and logging queue ───────────────────────────────────
app = FastAPI(title="Colab MCP + Dashboard", version="1.0.0")
log_queue: asyncio.Queue = asyncio.Queue()

# ─── Logger that pushes messages into the asyncio queue for SSE ────────────────
class QueueHandler(logging.Handler):
    def emit(self, record):
        msg = self.format(record)
        try:
            asyncio.get_event_loop().call_soon_threadsafe(log_queue.put_nowait, msg)
        except RuntimeError:
            pass  # event loop not set up yet

logger = logging.getLogger("mcp")
logger.setLevel(logging.INFO)
handler = QueueHandler()
handler.setFormatter(logging.Formatter("%(asctime)s %(message)s"))
logger.addHandler(handler)

# ─── MCP Endpoints ──────────────────────────────────────────────────────────────

@app.get("/")
async def root():
    return {"message": "Colab MCP Server is running", "version": "1.0.0"}

@app.post("/mcp/initialize")
async def initialize(request: Request):
    await request.json()  # ignore incoming payload
    return {
        "protocolVersion": "2024-11-05",
        "capabilities": {
            "tools": {"listChanged": True},
            "resources": {"subscribe": True, "listChanged": True}
        },
        "serverInfo": {"name": "colab-mcp", "version": "1.0.0"}
    }

@app.post("/mcp/tools/list")
async def list_tools():
    return {
        "tools": [
            {"name": "execute_python",  "description": "Run Python code and return stdout"},
            {"name": "install_package", "description": "Install a pip package"},
            {"name": "list_files",      "description": "List files in a directory"},
            {"name": "read_file",       "description": "Read the contents of a file"},
            {"name": "write_file",      "description": "Write content to a file"}
        ]
    }

@app.post("/mcp/tools/call")
async def call_tool(request: Request):
    data = await request.json()
    tool = data.get("name")
    args = data.get("arguments", {})
    logger.info(f"CALL: {tool} args={args}")

    try:
        if tool == "execute_python":
            result = execute_python(args.get("code", ""))
        elif tool == "install_package":
            result = install_package(args.get("package", ""))
        elif tool == "list_files":
            result = list_files(args.get("path", "."))
        elif tool == "read_file":
            result = read_file(args.get("filepath", ""))
        elif tool == "write_file":
            result = write_file(args.get("filepath", ""), args.get("content", ""))
        else:
            raise HTTPException(status_code=400, detail=f"Unknown tool: {tool}")

        logger.info(f"RESULT: {tool} → {result!r}")
        return {"content":[{"type":"text","text":result}], "isError": False}

    except Exception as e:
        logger.error(f"ERROR in {tool}: {e}")
        return {"content":[{"type":"text","text":f"Error: {e}"}], "isError": True}

# ─── Tool implementations ────────────────────────────────────────────────────────

def execute_python(code: str) -> str:
    """Execute Python in-process and capture stdout."""
    from io import StringIO
    import contextlib
    buf = StringIO()
    try:
        with contextlib.redirect_stdout(buf):
            exec(code, {})
        return buf.getvalue() or "✓ executed (no output)"
    except Exception as e:
        return f"Execution error: {e}"

def install_package(package: str) -> str:
    """Install a pip package and return output."""
    proc = subprocess.run(
        [sys.executable, "-m", "pip", "install", package],
        capture_output=True, text=True
    )
    return proc.stdout or proc.stderr

def list_files(path: str = ".") -> str:
    """List files and subdirectories."""
    p = Path(path)
    if not p.exists():
        return f"Path not found: {path}"
    if p.is_file():
        return p.read_text(encoding="utf-8")
    items = []
    for it in sorted(p.iterdir()):
        prefix = "📄" if it.is_file() else "📁"
        suffix = "" if it.is_file() else "/"
        items.append(f"{prefix} {it.name}{suffix}")
    return "\n".join(items) or "— empty"

def read_file(filepath: str) -> str:
    """Read text from a file."""
    try:
        return Path(filepath).read_text(encoding="utf-8")
    except Exception as e:
        return f"Read error: {e}"

def write_file(filepath: str, content: str) -> str:
    """Write text to a file, creating parent dirs."""
    try:
        Path(filepath).parent.mkdir(parents=True, exist_ok=True)
        Path(filepath).write_text(content, encoding="utf-8")
        return f"Written: {filepath}"
    except Exception as e:
        return f"Write error: {e}"

# ─── Activity Dashboard & SSE ─────────────────────────────────────────────────

@app.get("/activity", response_class=HTMLResponse)
async def activity_dashboard():
    # Simple HTML page that connects to /activity/stream via EventSource
    return """
<!doctype html>
<html>
<head>
  <title>MCP Activity Dashboard</title>
  <style>
    body { font-family: sans-serif; margin: 1em; }
    #log { white-space: pre-wrap; background:#f5f5f5; padding:1em;
           height:70vh; overflow:auto; border:1px solid #ccc; }
  </style>
</head>
<body>
  <h1>MCP Server Activity</h1>
  <div id="log">Connecting…</div>
  <script>
    const logDiv = document.getElementById("log");
    const evtSrc = new EventSource("/activity/stream");
    evtSrc.onmessage = e => {
      logDiv.textContent += "\\n" + e.data;
      logDiv.scrollTop = logDiv.scrollHeight;
    };
    evtSrc.onerror = () => logDiv.textContent += "\\n[Stream closed]";
  </script>
</body>
</html>
"""

@app.get("/activity/stream")
async def activity_stream():
    async def event_generator():
        while True:
            msg = await log_queue.get()
            yield {"data": msg}
    return EventSourceResponse(event_generator())

# ─── File download endpoint ────────────────────────────────────────────────────

@app.get("/download/{file_path:path}")
async def download_file(file_path: str):
    p = Path(file_path)
    if not p.exists():
        raise HTTPException(404, f"File not found: {file_path}")
    if p.is_dir():
        raise HTTPException(400, "Cannot download a directory")
    return FileResponse(path=str(p), filename=p.name, media_type="application/octet-stream")

# ─── Server startup ─────────────────────────────────────────────────────────────

def start_server():
    """Starts uvicorn + ngrok tunnel and prints public URLs."""
    def _run():
        uvicorn.run(app, host="0.0.0.0", port=8001, log_level="info")
    Thread(target=_run, daemon=True).start()
    time.sleep(2)
    public_url = ngrok.connect(8001).public_url
    print(f"🔗 MCP API + Dashboard live at: {public_url}")
    print(f" • API Root:      {public_url}/")
    print(f" • Dashboard:     {public_url}/activity")
    print(f" • Download files:{public_url}/download/<your_path>")
    return public_url

if __name__ == "__main__":
    start_server()
    try:
        while True:
            time.sleep(60)
    except KeyboardInterrupt:
        print("👋 Shutting down.")

INFO:     Started server process [1514]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8001 (Press CTRL+C to quit)


🔗 MCP API + Dashboard live at: https://e48a-104-155-235-158.ngrok-free.app
 • API Root:      https://e48a-104-155-235-158.ngrok-free.app/
 • Dashboard:     https://e48a-104-155-235-158.ngrok-free.app/activity
 • Download files:https://e48a-104-155-235-158.ngrok-free.app/download/<your_path>
INFO:     86.100.109.247:0 - "GET /activity HTTP/1.1" 200 OK
INFO:     86.100.109.247:0 - "GET /activity/stream HTTP/1.1" 200 OK
INFO:     34.168.192.174:0 - "GET / HTTP/1.1" 200 OK
INFO:     86.100.109.247:0 - "GET /activity HTTP/1.1" 200 OK
INFO:     86.100.109.247:0 - "GET /activity/stream HTTP/1.1" 200 OK
INFO:     86.100.109.247:0 - "GET /activity HTTP/1.1" 200 OK
INFO:     86.100.109.247:0 - "GET /activity/stream HTTP/1.1" 200 OK
INFO:     34.168.192.174:0 - "POST /mcp/initialize HTTP/1.1" 200 OK
INFO:     86.100.109.247:0 - "GET /activity HTTP/1.1" 200 OK
INFO:     86.100.109.247:0 - "GET /activity/stream HTTP/1.1" 200 OK
INFO:     34.168.192.174:0 - "POST /mcp/tools/list HTTP/1.1" 200 OK
INF

INFO:mcp:CALL: None args={}
ERROR:mcp:ERROR in None: 400: Unknown tool: None


INFO:     34.168.192.174:0 - "POST /mcp/tools/call HTTP/1.1" 200 OK


INFO:mcp:CALL: execute_python args={'code': '\n# Test Python execution via MCP FastAPI server\nimport sys\nimport platform\nimport datetime\nimport json\nimport math\n\nprint("🚀 Python execution test via MCP FastAPI server")\nprint(f"Python version: {sys.version}")\nprint(f"Platform: {platform.platform()}")\nprint(f"Current time: {datetime.datetime.now().isoformat()}")\n\n# Perform calculations\ncalculation_result = math.sqrt(144) + math.pi\nprint(f"Calculation: sqrt(144) + π = {calculation_result:.4f}")\n\n# Create test data\ntest_data = {\n    "calculation": calculation_result,\n    "timestamp": datetime.datetime.now().isoformat(),\n    "python_version": f"{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}",\n    "platform": platform.platform(),\n    "test_status": "success"\n}\n\nprint(f"Test data: {json.dumps(test_data, indent=2)}")\nprint("✅ MCP FastAPI execution test completed successfully!")\n\n# Return the result for the MCP server\ntest_data\n'}
INFO:mc

...

INFO:     34.168.192.174:0 - "POST /mcp/tools/call HTTP/1.1" 200 OK


INFO:mcp:CALL: list_files args={'path': '/tmp'}
INFO:mcp:RESULT: list_files → '📁 colab_runtime.sock/\n📄 dap_multiplexer.INFO\n📄 dap_multiplexer.a4ababeb2314.root.log.INFO.20250623-134518.125\n📁 debugger_y7049jxv0/\n📁 initgoogle_syslog_dir.0/\n📄 language_service.INFO\n📄 language_service.a4ababeb2314.root.log.INFO.20250623-135056.1583\n📄 language_service.a4ababeb2314.root.log.INFO.20250623-135246.2063\n📄 mcp_parameter_test.txt\n📄 ngrok-v3-stable-linux-amd64.zip\n📁 pyright-1601-VqWKsWQRaF15/\n📁 pyright-2074-omhvbe0rzcP0/\n📁 python-languageserver-cancellation/\n📄 python_test_output.json'


INFO:     34.168.192.174:0 - "POST /mcp/tools/call HTTP/1.1" 200 OK


INFO:mcp:CALL: execute_python args={'code': '\n# Parameter correction verification test\nimport datetime\nimport json\n\nprint("🔧 MCP Parameter Correction Verification")\nprint(f"Test timestamp: {datetime.datetime.now().isoformat()}")\n\n# Verify the parameter corrections worked\ncorrections = {\n    "install_package": "package (not package_name)",\n    "write_file": "filepath (not path)",\n    "read_file": "filepath (not path)",\n    "list_files": "path (verification needed)"\n}\n\nprint("\\nParameter corrections applied:")\nfor tool, correction in corrections.items():\n    print(f"  {tool}: {correction}")\n\n# Test file operations from Python\nimport os\ntest_file = "/tmp/mcp_parameter_test.txt"\nif os.path.exists(test_file):\n    print(f"\\n✅ Test file exists: {test_file}")\n    with open(test_file, \'r\') as f:\n        content_length = len(f.read())\n    print(f"✅ File content length: {content_length} characters")\nelse:\n    print(f"\\n❌ Test file not found: {test_file}")\n\nprin

INFO:     34.168.192.174:0 - "POST /mcp/tools/call HTTP/1.1" 200 OK
INFO:     34.168.192.174:0 - "GET / HTTP/1.1" 200 OK
INFO:     34.168.192.174:0 - "POST /mcp/initialize HTTP/1.1" 200 OK
INFO:     34.168.192.174:0 - "GET /dashboard HTTP/1.1" 404 Not Found
👋 Shutting down.


In [ ]:
"""
Colab MCP Server with Live Activity Dashboard, Log Viewer, Chat Interface, and File Download

This script sets up:
 1. A FastAPI–based MCP server for remote tool execution:
    - execute_python: run arbitrary Python code
    - install_package: install pip packages
    - list_files: list directory contents
    - read_file: read file contents
    - write_file: write content to a file
 2. A Server-Sent Events (SSE) activity dashboard at /activity
    that streams every tool call and result in real time.
 3. A static site under /site/:
    - /site/log.html: refreshable server log viewer
    - /site/chat.html: lightweight chat interface
 4. File download endpoint at /download/{file_path}
    to fetch any file from the Colab environment.

Usage:
 - Paste this entire cell into a Colab notebook.
 - Run the cell; it will install dependencies automatically.
 - It will print your public ngrok URLs for API, dashboard, and site.
"""

# ─── Install dependencies ───────────────────────────────────────────────────────
import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet",
     "fastapi", "uvicorn", "pyngrok", "nest-asyncio", "sse-starlette", "requests"],
    check=True
)

# ─── Standard imports and setup ─────────────────────────────────────────────────
import time
import logging
import asyncio
import nest_asyncio
from threading import Thread
from pathlib import Path
import subprocess
import sys

from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import HTMLResponse, FileResponse
from fastapi.staticfiles import StaticFiles
from sse_starlette.sse import EventSourceResponse
from pyngrok import ngrok
import uvicorn

# ─── Enable nested asyncio for Colab compatibility ──────────────────────────────
nest_asyncio.apply()

# ─── Configure ngrok authentication ─────────────────────────────────────────────
# Replace this token with your own if desired
ngrok.set_auth_token("2yuUKpiAlYXCez40i8UktLZ91s7_5Rv1GjG5HMFw5KKbk5sHz")

# ─── Create FastAPI app and logging setup ────────────────────────────────────────
app = FastAPI(title="Colab MCP + Dashboard + Site", version="1.0.0")

# In-memory broadcast subscribers for SSE
subscribers: set[asyncio.Queue] = set()

# File paths for persistent logs
SERVER_LOG_PATH = Path("server_log.txt")
CHAT_LOG_PATH = Path("chat_log.txt")
SERVER_LOG_PATH.touch(exist_ok=True)
CHAT_LOG_PATH.touch(exist_ok=True)

# Logger that sends to both file and SSE subscribers
class BroadcastHandler(logging.Handler):
    def emit(self, record):
        msg = self.format(record)
        # Write to server log file
        with SERVER_LOG_PATH.open("a", encoding="utf-8") as f:
            f.write(msg + "\n")
        # Broadcast to all SSE subscribers
        for q in list(subscribers):
            try:
                asyncio.get_event_loop().call_soon_threadsafe(q.put_nowait, msg)
            except Exception:
                pass

logger = logging.getLogger("mcp")
logger.setLevel(logging.INFO)
handler = BroadcastHandler()
handler.setFormatter(logging.Formatter("%(asctime)s %(message)s"))
logger.addHandler(handler)

# ─── MCP API Endpoints ──────────────────────────────────────────────────────────

@app.get("/")
async def root():
    return {"message": "Colab MCP Server is running", "version": "1.0.0"}

@app.post("/mcp/initialize")
async def initialize(request: Request):
    await request.json()
    return {
        "protocolVersion": "2024-11-05",
        "capabilities": {
            "tools": {"listChanged": True},
            "resources": {"subscribe": True, "listChanged": True}
        },
        "serverInfo": {"name": "colab-mcp", "version": "1.0.0"}
    }

@app.post("/mcp/tools/list")
async def list_tools():
    return {
        "tools": [
            {"name": "execute_python",  "description": "Run Python code and return output"},
            {"name": "install_package", "description": "Install a pip package"},
            {"name": "list_files",      "description": "List files in a directory"},
            {"name": "read_file",       "description": "Read the contents of a file"},
            {"name": "write_file",      "description": "Write content to a file"}
        ]
    }

@app.post("/mcp/tools/call")
async def call_tool(request: Request):
    data = await request.json()
    tool = data.get("name")
    args = data.get("arguments", {})
    logger.info(f"CALL: {tool} args={args!r}")

    try:
        if tool == "execute_python":
            result = execute_python(args.get("code", ""))
        elif tool == "install_package":
            result = install_package(args.get("package", ""))
        elif tool == "list_files":
            result = list_files(args.get("path", "."))
        elif tool == "read_file":
            result = read_file(args.get("filepath", ""))
        elif tool == "write_file":
            result = write_file(args.get("filepath", ""), args.get("content", ""))
        else:
            raise HTTPException(status_code=400, detail=f"Unknown tool: {tool}")

        logger.info(f"RESULT: {tool} → {result!r}")
        return {"content":[{"type":"text","text":result}], "isError": False}

    except Exception as e:
        logger.error(f"ERROR in {tool}: {e}")
        return {"content":[{"type":"text","text":f"Error: {e}"}], "isError": True}

# ─── Tool Implementations ────────────────────────────────────────────────────────

def execute_python(code: str) -> str:
    """Execute Python in-process and capture stdout."""
    from io import StringIO
    import contextlib
    buf = StringIO()
    try:
        with contextlib.redirect_stdout(buf):
            exec(code, {})
        return buf.getvalue() or "✓ executed (no output)"
    except Exception as e:
        return f"Execution error: {e}"

def install_package(package: str) -> str:
    """Install a pip package and return output."""
    proc = subprocess.run(
        [sys.executable, "-m", "pip", "install", package],
        capture_output=True, text=True
    )
    return proc.stdout or proc.stderr

def list_files(path: str = ".") -> str:
    """List files and subdirectories."""
    p = Path(path)
    if not p.exists():
        return f"Path not found: {path}"
    if p.is_file():
        return p.read_text(encoding="utf-8")
    items = []
    for it in sorted(p.iterdir()):
        prefix = "📄" if it.is_file() else "📁"
        suffix = "" if it.is_file() else "/"
        items.append(f"{prefix} {it.name}{suffix}")
    return "\n".join(items) or "— empty"

def read_file(filepath: str) -> str:
    """Read text from a file."""
    try:
        return Path(filepath).read_text(encoding="utf-8")
    except Exception as e:
        return f"Read error: {e}"

def write_file(filepath: str, content: str) -> str:
    """Write text to a file, creating parent dirs."""
    try:
        Path(filepath).parent.mkdir(parents=True, exist_ok=True)
        Path(filepath).write_text(content, encoding="utf-8")
        return f"Written: {filepath}"
    except Exception as e:
        return f"Write error: {e}"

# ─── Activity Dashboard (SSE) ──────────────────────────────────────────────────

@app.get("/activity", response_class=HTMLResponse)
async def activity_dashboard():
    return """
<!doctype html>
<html>
<head>
  <title>MCP Activity Dashboard</title>
  <style>
    body { font-family: sans-serif; margin: 1em; }
    #log { white-space: pre-wrap; background:#f5f5f5; padding:1em;
           height:70vh; overflow:auto; border:1px solid #ccc; }
  </style>
</head>
<body>
  <h1>MCP Server Activity</h1>
  <div id="log">Connecting…</div>
  <script>
    const logDiv = document.getElementById("log");
    const evtSrc = new EventSource("/activity/stream");
    evtSrc.onmessage = e => {
      logDiv.textContent += "\\n" + e.data;
      logDiv.scrollTop = logDiv.scrollHeight;
    };
    evtSrc.onerror = () => logDiv.textContent += "\\n[Stream closed]";
  </script>
</body>
</html>
"""

@app.get("/activity/stream")
async def activity_stream():
    q: asyncio.Queue = asyncio.Queue()
    subscribers.add(q)
    try:
        while True:
            msg = await q.get()
            yield {"data": msg}
    finally:
        subscribers.discard(q)

# ─── Static Site: Log Viewer & Chat Interface ──────────────────────────────────

# Ensure site directory exists
site_dir = Path("site")
site_dir.mkdir(exist_ok=True)

# chat.html
(site_dir / "chat.html").write_text("""
<!DOCTYPE html>
<html>
<head><title>Lightweight Chat</title></head>
<body>
  <h2>Chat</h2>
  <div id="chatlog" style="border:1px solid #ccc; padding:10px; height:200px; overflow:auto;"></div>
  <input id="msg" type="text" placeholder="Say something..." style="width:80%;" />
  <button onclick="send()">Send</button>
  <script>
    async function send() {
      const msg = document.getElementById('msg').value;
      if (!msg) return;
      await fetch('/site/chat/send', {
        method: 'POST',
        headers: {'Content-Type': 'application/json'},
        body: JSON.stringify({ message: msg })
      });
      document.getElementById('msg').value = '';
      load();
    }
    async function load() {
      const res = await fetch('/site/chat/log');
      const lines = await res.text();
      document.getElementById('chatlog').textContent = lines;
    }
    load();
    setInterval(load, 3000);
  </script>
</body>
</html>
""", encoding="utf-8")

# log.html
(site_dir / "log.html").write_text("""
<!DOCTYPE html>
<html>
<head><title>Server Log</title></head>
<body>
  <h2>Server Log</h2>
  <pre id="log" style="background:#f0f0f0; border:1px solid #ccc; padding:10px; height:300px; overflow:auto;"></pre>
  <script>
    async function refresh() {
      const res = await fetch('/site/log/serverlog');
      const txt = await res.text();
      document.getElementById('log').textContent = txt;
    }
    refresh();
    setInterval(refresh, 3000);
  </script>
</body>
</html>
""", encoding="utf-8")

# Mount static files at /site
app.mount("/site", StaticFiles(directory="site"), name="site")

# Chat POST endpoint
@app.post("/site/chat/send")
async def post_chat(request: Request):
    data = await request.json()
    msg = data.get("message", "").strip()
    if msg:
        line = f"[chat {time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}"
        # append to chat log file
        with CHAT_LOG_PATH.open("a", encoding="utf-8") as f:
            f.write(line + "\n")
        # also log via logger
        logger.info(f"CHAT: {msg}")
    return {"ok": True}

# Serve chat log
@app.get("/site/chat/log")
async def get_chat_log():
    return CHAT_LOG_PATH.read_text(encoding="utf-8")

# Serve server log (for log.html)
@app.get("/site/log/serverlog")
async def get_server_log():
    return SERVER_LOG_PATH.read_text(encoding="utf-8")

# ─── File Download Endpoint ────────────────────────────────────────────────────

@app.get("/download/{file_path:path}")
async def download_file(file_path: str):
    p = Path(file_path)
    if not p.exists():
        raise HTTPException(404, f"File not found: {file_path}")
    if p.is_dir():
        raise HTTPException(400, "Cannot download a directory")
    return FileResponse(path=str(p), filename=p.name, media_type="application/octet-stream")

# ─── Server Startup ─────────────────────────────────────────────────────────────

def start_server():
    def _run():
        uvicorn.run(app, host="0.0.0.0", port=8001, log_level="info")
    Thread(target=_run, daemon=True).start()
    time.sleep(2)
    public_url = ngrok.connect(8001).public_url
    print(f"🔗 MCP API + Dashboard + Site live at: {public_url}")
    print(f" • API Root:      {public_url}/")
    print(f" • Activity Dash: {public_url}/activity")
    print(f" • Log Viewer:    {public_url}/site/log.html")
    print(f" • Chat UI:       {public_url}/site/chat.html")
    print(f" • Download files:{public_url}/download/<your_path>")
    return public_url

if __name__ == "__main__":
    start_server()
    try:
        while True:
            time.sleep(60)
    except KeyboardInterrupt:
        print("👋 Shutting down.")

INFO:     Started server process [1514]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8001): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


🔗 MCP API + Dashboard + Site live at: https://349d-104-155-235-158.ngrok-free.app
 • API Root:      https://349d-104-155-235-158.ngrok-free.app/
 • Activity Dash: https://349d-104-155-235-158.ngrok-free.app/activity
 • Log Viewer:    https://349d-104-155-235-158.ngrok-free.app/site/log.html
 • Chat UI:       https://349d-104-155-235-158.ngrok-free.app/site/chat.html
 • Download files:https://349d-104-155-235-158.ngrok-free.app/download/<your_path>


In [ ]:
"""
Colab MCP Server + Dashboard + Site + Tool-Call Chat

Adds a structured “Tool Chat” UI so human or AI agents can choose a tool,
see its JSON schema, supply arguments, and run it at the click of a button.
"""

# ─── Install dependencies ───────────────────────────────────────────────────────
import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet",
     "fastapi", "uvicorn", "pyngrok", "nest-asyncio", "sse-starlette", "requests"],
    check=True
)

# ─── Imports & Setup ────────────────────────────────────────────────────────────
import time, logging, asyncio, nest_asyncio
from threading import Thread
from pathlib import Path
from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import HTMLResponse, FileResponse
from fastapi.staticfiles import StaticFiles
from sse_starlette.sse import EventSourceResponse
from pyngrok import ngrok
import uvicorn, subprocess

nest_asyncio.apply()
ngrok.set_auth_token("2yuUKpiAlYXCez40i8UktLZ91s7_5Rv1GjG5HMFw5KKbk5sHz")

app = FastAPI(title="Colab MCP + Dashboard + Tool Chat", version="1.1.0")

# ─── Logging & Broadcast ────────────────────────────────────────────────────────
subscribers: set[asyncio.Queue] = set()
SERVER_LOG = Path("server_log.txt"); SERVER_LOG.touch(exist_ok=True)
CHAT_LOG   = Path("chat_log.txt");   CHAT_LOG.touch(exist_ok=True)

class BroadcastHandler(logging.Handler):
    def emit(self, record):
        msg = self.format(record)
        with SERVER_LOG.open("a") as f: f.write(msg+"\n")
        for q in list(subscribers):
            try: asyncio.get_event_loop().call_soon_threadsafe(q.put_nowait, msg)
            except: pass

logger = logging.getLogger("mcp")
logger.setLevel(logging.INFO)
h = BroadcastHandler(); h.setFormatter(logging.Formatter("%(asctime)s %(message)s"))
logger.addHandler(h)

# ─── MCP API ────────────────────────────────────────────────────────────────────
@app.get("/")
async def root(): return {"message":"running","version":"1.1.0"}

@app.post("/mcp/initialize")
async def initialize(r: Request):
    await r.json()
    return {
        "protocolVersion":"2024-11-05",
        "capabilities":{
            "tools":{"listChanged":True},
            "resources":{"subscribe":True,"listChanged":True}
        },
        "serverInfo":{"name":"colab-mcp","version":"1.1.0"}
    }

@app.post("/mcp/tools/list")
async def list_tools():
    return {
        "tools":[
            {"name":"execute_python",
             "description":"Run Python code and return stdout",
             "inputSchema":{
               "type":"object",
               "properties":{
                 "code":{"type":"string","description":"Python code to execute"}
               },
               "required":["code"]
             }},
            {"name":"install_package",
             "description":"Install a pip package",
             "inputSchema":{
               "type":"object",
               "properties":{
                 "package":{"type":"string","description":"Name of package"}
               },
               "required":["package"]
             }},
            {"name":"list_files",
             "description":"List files in a directory",
             "inputSchema":{
               "type":"object",
               "properties":{
                 "path":{"type":"string","default":".","description":"Directory path"}
               }
             }},
            {"name":"read_file",
             "description":"Read contents of a file",
             "inputSchema":{
               "type":"object",
               "properties":{
                 "filepath":{"type":"string","description":"File path to read"}
               },
               "required":["filepath"]
             }},
            {"name":"write_file",
             "description":"Write content to a file",
             "inputSchema":{
               "type":"object",
               "properties":{
                 "filepath":{"type":"string","description":"File path to write"},
                 "content":{"type":"string","description":"Content to write"}
               },
               "required":["filepath","content"]
             }}
        ]
    }

@app.post("/mcp/tools/call")
async def call_tool(r: Request):
    data = await r.json()
    tool = data.get("name"); args = data.get("arguments",{})
    logger.info(f"CALL {tool} {args!r}")
    try:
        if tool=="execute_python":  out=exec_py(args.get("code",""))
        elif tool=="install_package": out=inst_pkg(args.get("package",""))
        elif tool=="list_files":    out=ls_files(args.get("path","."))
        elif tool=="read_file":     out=rd_file(args.get("filepath",""))
        elif tool=="write_file":    out=wr_file(args.get("filepath",""),args.get("content",""))
        else: raise HTTPException(400,f"Unknown tool {tool}")
        logger.info(f"RESULT {tool} -> {out!r}")
        return {"content":[{"type":"text","text":out}],"isError":False}
    except Exception as e:
        logger.error(f"ERR {tool} {e}")
        return {"content":[{"type":"text","text":f"Error: {e}"}],"isError":True}

def exec_py(code):
    from io import StringIO; import contextlib
    buf=StringIO()
    try:
        with contextlib.redirect_stdout(buf): exec(code,{})
        return buf.getvalue() or "✓ no output"
    except Exception as e: return f"Execution error: {e}"

def inst_pkg(pkg):
    p=subprocess.run([sys.executable,"-m","pip","install",pkg],capture_output=True,text=True)
    return p.stdout or p.stderr

def ls_files(path):
    p=Path(path)
    if not p.exists(): return f"Not found: {path}"
    if p.is_file(): return p.read_text()
    return "\n".join(
        f"{'📄' if it.is_file() else '📁'} {it.name}{'' if it.is_file() else '/'}"
        for it in sorted(p.iterdir())
    ) or "— empty"

def rd_file(fp):
    try: return Path(fp).read_text()
    except Exception as e: return f"Read error: {e}"

def wr_file(fp,content):
    try:
        Path(fp).parent.mkdir(parents=True,exist_ok=True)
        Path(fp).write_text(content)
        return f"Written {fp}"
    except Exception as e: return f"Write error: {e}"

# ─── Activity Dashboard (SSE) ─────────────────────────────────────────────────
@app.get("/activity",response_class=HTMLResponse)
async def dash():
    return """<!doctype html><html><head><meta charset="utf-8"><title>Activity</title>
<style>body{font:14px sans;}#log{white-space:pre-wrap;background:#eee;padding:8px;height:70vh;overflow:auto;border:1px solid #ccc;}</style>
</head><body><h1>Activity</h1><div id="log">Connecting…</div>
<script>
const log=document.getElementById('log'),src=new EventSource('/activity/stream');
src.onmessage=e=>{log.textContent+='\\n'+e.data;log.scrollTop=log.scrollHeight;}
src.onerror=_=>log.textContent+='\\n[stream closed]';
</script></body></html>"""

@app.get("/activity/stream")
async def stream():
    q=asyncio.Queue(); subscribers.add(q)
    try:
        while True:
            m=await q.get(); yield {"data":m}
    finally:
        subscribers.discard(q)

# ─── Static Site & Tool-Call Chat ──────────────────────────────────────────────
site=Path("site"); site.mkdir(exist_ok=True)

# toolchat.html
(site/"toolchat.html").write_text(f"""
<!DOCTYPE html>
<html><head>
  <meta charset="utf-8"><title>Tool Chat</title>
  <style>body{{font:14px sans;max-width:800px;margin:1em auto;}}textarea{{width:100%;height:4em;}}</style>
</head><body>
  <h1>Structured Tool Chat</h1>
  <p>Pick a tool, fill its arguments as JSON, and click <strong>Run</strong>.</p>
  <h2>Available Tools &amp; Schemas</h2>
  <pre style="background:#f4f4f4;padding:8px;border:1px solid #ccc;">{{
{app.dependency_overrides.get('','')}
}}</pre>
  <h2>Example</h2>
  <code>{{"name":"execute_python","arguments":{{"code":"print(2+2)"}}}}</code>
  <h2>Your Request</h2>
  <textarea id="req">{{"name":"","arguments":{{}}}}</textarea><br>
  <button onclick="run()">Run</button>
  <h2>Response</h2>
  <pre id="out" style="background:#eef;padding:8px;border:1px solid #999;height:200px;overflow:auto;"></pre>
  <script>
    async function run(){{
      const r=JSON.parse(document.getElementById('req').value);
      const res=await fetch('/mcp/tools/call',{{method:'POST',headers:{{'Content-Type':'application/json'}},body:JSON.stringify(r)}});
      const j=await res.json();
      document.getElementById('out').textContent=JSON.stringify(j,null,2);
    }}
  </script>
</body></html>
""",encoding="utf-8")

# log.html & chat.html (unchanged from before)
(site/"log.html").write_text("""<!DOCTYPE html><html><head><title>Log</title></head><body>
<h2>Server Log</h2><pre id="log" style="background:#f0f0f0;padding:10px;height:300px;overflow:auto;"></pre>
<script>
async function refresh(){let t=await fetch('/site/log/serverlog').then(r=>r.text());document.getElementById('log').textContent=t;}
refresh();setInterval(refresh,3000);
</script></body></html>""",encoding="utf-8")
(site/"chat.html").write_text("""<!DOCTYPE html><html><head><title>Chat</title></head><body>
<h2>Chat</h2><div id="chatlog" style="border:1px solid #ccc;padding:10px;height:200px;overflow:auto;"></div>
<input id="msg" style="width:80%;" placeholder="Type message..."><button onclick="send()">Send</button>
<script>
async function send(){
  let m=document.getElementById('msg').value;
  if(!m) return;
  await fetch('/site/chat/send',{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify({message:m})});
  document.getElementById('msg').value='';
  load();
}
async function load(){
  let t=await fetch('/site/chat/log').then(r=>r.text());
  document.getElementById('chatlog').textContent=t;
}
load();setInterval(load,3000);
</script></body></html>""",encoding="utf-8")

app.mount("/site",StaticFiles(directory="site"),name="site")

@app.post("/site/chat/send")
async def post_chat(r:Request):
    d=await r.json(); m=d.get("message","").strip()
    if m:
        line=f"[chat {time.strftime('%Y-%m-%d %H:%M:%S')}] {m}"
        with CHAT_LOG.open("a") as f: f.write(line+"\n")
        logger.info(f"CHAT {m}")
    return{"ok":True}

@app.get("/site/chat/log")
async def get_chat(): return CHAT_LOG.read_text()

@app.get("/site/log/serverlog")
async def get_slog(): return SERVER_LOG.read_text()

# ─── Download ───────────────────────────────────────────────────────────────────
@app.get("/download/{fp:path}")
async def dl(fp:str):
    p=Path(fp)
    if not p.exists(): raise HTTPException(404)
    if p.is_dir():      raise HTTPException(400)
    return FileResponse(str(p),filename=p.name)

# ─── Startup ───────────────────────────────────────────────────────────────────
def start():
    Thread(target=lambda:uvicorn.run(app,host="0.0.0.0",port=8001,log_level="info"),daemon=True).start()
    time.sleep(2)
    url=ngrok.connect(8001).public_url
    print("🔗 Public at",url); return url

if __name__=="__main__":
    start()
    try:
        while True: time.sleep(60)
    except KeyboardInterrupt:
        print("Shutdown")

In [ ]:
# ╔════════════════════════════════════════════════════════════════════════════╗
# ║     🤖 Colab MCP Server + Tool Call UI + Live Dashboard + GDrive Mount    ║
# ║                                                                            ║
# ║  Launches a public FastAPI-based server for structured tool calls.         ║
# ║  - Supports LLM agents and humans interacting via tool chat UI             ║
# ║  - Provides install_package, execute_python, read/write/list files         ║
# ║  - All files stored in a safe Google Drive folder                          ║
# ║  - /site/toolchat.html offers a JSON tool caller interface                 ║
# ║  - /activity shows real-time logs (SSE)                                    ║
# ║  - /site/chat.html supports structured human-agent messaging               ║
# ╚════════════════════════════════════════════════════════════════════════════╝

# ─── Install Dependencies ──────────────────────────────────────────────────────
import subprocess, sys, os, json, time, logging, asyncio, nest_asyncio
from threading import Thread
from pathlib import Path
from fastapi import FastAPI
import uvicorn

# Install required Python packages
subprocess.run([
    sys.executable, "-m", "pip", "install", "--quiet",
    "fastapi", "uvicorn", "pyngrok", "nest-asyncio", "sse-starlette", "requests"
], check=True)

# Enable async in Colab notebooks
nest_asyncio.apply()

# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

# ─── File Access Setup ────────────────────────────────────────────────────────
SAFE_DIR = Path("/content/drive/MyDrive/ColabMCPFiles")
SAFE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(SAFE_DIR)  # Lock working directory to this folder

# Ngrok for public access — replace with your token
from pyngrok import ngrok
ngrok.set_auth_token("REPLACE_WITH_YOUR_NGROK_TOKEN")

# ─── FastAPI App Initialization ───────────────────────────────────────────────
app = FastAPI(title="Colab MCP Tool Server", version="1.2")

# ─── Logging & Broadcast Handler ───────────────────────────────────────────────
subscribers: set[asyncio.Queue] = set()
SERVER_LOG = SAFE_DIR / "server_log.txt"; SERVER_LOG.touch()
CHAT_LOG   = SAFE_DIR / "chat_log.txt";   CHAT_LOG.touch()

class BroadcastHandler(logging.Handler):
    def emit(self, record):
        msg = self.format(record)
        with SERVER_LOG.open("a") as f: f.write(msg + "\n")
        for q in list(subscribers):
            try:
                asyncio.get_event_loop().call_soon_threadsafe(q.put_nowait, msg)
            except:
                pass

logger = logging.getLogger("mcp")
logger.setLevel(logging.INFO)
handler = BroadcastHandler()
handler.setFormatter(logging.Formatter("%(asctime)s %(message)s"))
logger.addHandler(handler)

# ─── Tool Definitions ─────────────────────────────────────────────────────────
TOOLS = [
    {
        "name": "execute_python",
        "description": "Run Python code and return stdout",
        "inputSchema": {
            "type": "object",
            "properties": {
                "code": {"type": "string", "description": "Python code to execute"}
            },
            "required": ["code"]
        }
    },
    {
        "name": "install_package",
        "description": "Install a pip package",
        "inputSchema": {
            "type": "object",
            "properties": {
                "package": {"type": "string", "description": "Name of package"}
            },
            "required": ["package"]
        }
    },
    {
        "name": "list_files",
        "description": "List files in a directory",
        "inputSchema": {
            "type": "object",
            "properties": {
                "path": {"type": "string", "default": ".", "description": "Directory path"}
            }
        }
    },
    {
        "name": "read_file",
        "description": "Read contents of a file",
        "inputSchema": {
            "type": "object",
            "properties": {
                "filepath": {"type": "string", "description": "File path to read"}
            },
            "required": ["filepath"]
        }
    },
    {
        "name": "write_file",
        "description": "Write content to a file",
        "inputSchema": {
            "type": "object",
            "properties": {
                "filepath": {"type": "string", "description": "File path to write"},
                "content": {"type": "string", "description": "Content to write"}
            },
            "required": ["filepath", "content"]
        }
    }
]

# ─── Tool Execution Functions ─────────────────────────────────────────────────
def exec_py(code: str) -> str:
    from io import StringIO; import contextlib
    buf = StringIO()
    try:
        with contextlib.redirect_stdout(buf):
            exec(code, {})
        return buf.getvalue() or "✓ no output"
    except Exception as e:
        return f"Execution error: {e}"

def inst_pkg(pkg: str) -> str:
    p = subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg],
        capture_output=True, text=True
    )
    return p.stdout or p.stderr

def safe_path(p: str) -> Path:
    target = SAFE_DIR / p
    if not target.resolve().is_relative_to(SAFE_DIR.resolve()):
        raise ValueError("Unsafe path.")
    return target

def ls_files(path: str = ".") -> str:
    p = safe_path(path)
    if not p.exists():
        return f"Not found: {path}"
    if p.is_file():
        return p.read_text()
    return "\n".join(
        f"{'📄' if it.is_file() else '📁'} {it.name}{'/' if it.is_dir() else ''}"
        for it in sorted(p.iterdir())
    ) or "— empty"

def rd_file(fp: str) -> str:
    return safe_path(fp).read_text()

def wr_file(fp: str, content: str) -> str:
    p = safe_path(fp)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content)
    return f"Written {p.name}"

# ─── Part 2: API, Memory, Schema, UI ───────────────────────────────────────────

from fastapi import FastAPI, Request, HTTPException
from fastapi.responses import HTMLResponse, FileResponse
from fastapi.staticfiles import StaticFiles
from sse_starlette.sse import EventSourceResponse
import asyncio, time, json, logging
from threading import Thread
import uvicorn
from pathlib import Path

# app = FastAPI(title="Colab MCP Tool Server", version="1.2") # App already initialized above

#─── Reuse these from Part 1 ────────────────────────────────────────────────────
# SAFE_DIR = Path("/content/drive/MyDrive/ColabMCPFiles")
# logger: configured BroadcastHandler logging
# TOOLS, exec_py, inst_pkg, ls_files, rd_file, wr_file, subscribers, SERVER_LOG, CHAT_LOG

# ─── 1. STANDARD MCP ENDPOINTS ────────────────────────────────────────────────
@app.get("/")
async def root():
    return {"message": "MCP Server Running", "version": "1.2"}

@app.post("/mcp/tools/list")
async def list_tools():
    """Return list of available tools with JSON schemas."""
    return {"tools": TOOLS}

@app.post("/mcp/tools/call")
async def call_tool(req: Request):
    """Invoke a tool by name with arguments, log call & result."""
    data = await req.json()
    name = data.get("name"); args = data.get("arguments", {})
    logger.info(f"CALL {name} {args!r}")
    try:
        if name == "execute_python":
            out = exec_py(args["code"])
        elif name == "install_package":
            out = inst_pkg(args["package"])
        elif name == "list_files":
            out = ls_files(args.get("path", "."))
        elif name == "read_file":
            out = rd_file(args["filepath"])
        elif name == "write_file":
            out = wr_file(args["filepath"], args["content"])
        else:
            raise Exception("Unknown tool")
        logger.info(f"RESULT {name} -> {out[:120]!r}")
        return {"content": [{"type": "text", "text": out}], "isError": False}
    except Exception as e:
        logger.error(f"ERR {name} {e}")
        return {"content": [{"type": "text", "text": f"Error: {e}"}], "isError": True}


# ─── 2. AGENT MEMORY (JSON) ────────────────────────────────────────────────────
MEMORY_FILE = SAFE_DIR / "memory.json"
if not MEMORY_FILE.exists():
    MEMORY_FILE.write_text("{}")

def load_memory():
    try:
        return json.loads(MEMORY_FILE.read_text())
    except:
        return {}

def save_memory(mem: dict):
    MEMORY_FILE.write_text(json.dumps(mem, indent=2))

@app.get("/memory/view")
async def memory_view():
    """Return the full memory JSON object."""
    return load_memory()

@app.get("/memory/summary")
async def memory_summary():
    """Return only the 'summary' field from memory (if any)."""
    mem = load_memory()
    return {"summary": mem.get("summary")}

@app.post("/memory/update")
async def memory_update(req: Request):
    """
    Merge provided JSON into memory.
    e.g. POST {"task":"summarize","summary":"pending"} updates memory.
    """
    data = await req.json()
    mem = load_memory()
    mem.update(data)
    save_memory(mem)
    logger.info(f"Memory updated: {data}")
    return mem

@app.post("/memory/clear")
async def memory_clear():
    """Clear all memory (reset to empty JSON {})."""
    save_memory({})
    logger.info("Memory cleared")
    return {}


# ─── 3. OPENAPI / TOOL SCHEMA ──────────────────────────────────────────────────
@app.get("/mcp/schema")
async def mcp_schema():
    """Return the full OpenAPI schema for tool discovery by agents."""
    return app.openapi()

@app.get("/.well-known/tool-schema.json")
async def well_known_schema():
    """Alias for the OpenAPI schema (for standardized discovery)."""
    return app.openapi()


# ─── 4. LIVE ACTIVITY DASHBOARD ───────────────────────────────────────────────
@app.get("/activity", response_class=HTMLResponse)
async def dash():
    """
    Live SSE dashboard of all server activity:
    tool calls, results, chat messages.
    """
    return """<!DOCTYPE html><html><head><title>Live Activity</title>
<meta charset="utf-8"><meta name="description" content="Real-time MCP tool activity log.">
<style>body{font:14px sans;}#log{white-space:pre-wrap;background:#eee;padding:8px;height:70vh;overflow:auto;border:1px solid #ccc;}</style>
</head><body><h1>Live Tool Activity</h1><p>Real-time activity (calls, results, chat).</p>
<div id="log">Connecting…</div>
<script>
const log=document.getElementById('log'),
      src=new EventSource('/activity/stream');
src.onmessage=e=>{log.textContent+='\\n'+e.data;log.scrollTop=log.scrollHeight;}
src.onerror=_=>log.textContent+='\\n[stream closed]';
</script></body></html>"""

@app.get("/activity/stream")
async def stream():
    """SSE endpoint streaming every log line to connected dashboards."""
    q = asyncio.Queue(); subscribers.add(q)
    async def event_gen():
        while True:
            yield {"data": await q.get()}
    return EventSourceResponse(event_gen())


# ─── 5. STATIC PAGES: TOOL CHAT & HUMAN CHAT ──────────────────────────────────
SITE = SAFE_DIR / "site"; SITE.mkdir(exist_ok=True)

# 5a. Structured Tool Chat UI at /site/toolchat.html
(SITE / "toolchat.html").write_text(f"""
<!DOCTYPE html><html><head><meta charset="utf-8"><title>Tool Chat</title>
<meta name="description" content="JSON UI to call MCP tools (execute_python, install_package, file ops).">
<style>body{{font:14px sans;max-width:700px;margin:1em auto;}}textarea{{width:100%;height:6em;}}</style>
</head><body>
  <h1>Structured Tool Chat</h1>
  <p>Fill in a tool call as JSON and click <strong>Run</strong>:</p>
  <pre style="background:#f8f8f8;padding:8px;border:1px solid #ccc;">{json.dumps(TOOLS, indent=2)}</pre>
  <textarea id="req">{{"name":"execute_python","arguments":{{"code":"print(2+2)"}}}}</textarea><br>
  <button onclick="run()">Run</button>
  <h3>Response</h3>
  <pre id="out" style="background:#eef;border:1px solid #ccc;padding:8px;height:200px;overflow:auto;"></pre>
  <script>
    async function run() {{
      const r = JSON.parse(document.getElementById('req').value);
      const res = await fetch('/mcp/tools/call', {{
        method:'POST', headers:{{'Content-Type':'application/json'}}, body: JSON.stringify(r)
      }});
      const j = await res.json();
      document.getElementById('out').textContent = JSON.stringify(j, null, 2);
    }}
  </script>
</body></html>
""", encoding="utf-8")

# 5b. Simple Chat UI at /site/chat.html
(SITE / "chat.html").write_text("""
<!DOCTYPE html><html><head><title>Chat</title>
<meta name="description" content="Simple human↔agent messaging log.">
<style>body{font:14px sans;max-width:600px;margin:1em auto;}#chatlog{height:200px;overflow:auto;border:1px solid #ccc;padding:8px;}</style>
</head><body>
  <h2>Chat Log</h2>
  <div id="chatlog"></div>
  <input id="msg" placeholder="Type message..." style="width:80%;"><button onclick="send()">Send</button>
  <script>
    async function send() {{
      let m = document.getElementById('msg').value;
      if (!m) return;
      await fetch('/site/chat/send', {{
        method:'POST', headers:{{'Content-Type':'application/json'}}, body: JSON.stringify({{message:m}})
      }});
      document.getElementById('msg').value = '';
      load();
    }}
    async function load() {{
      const t = await fetch('/site/chat/log').then(r => r.text());
      document.getElementById('chatlog').textContent = t;
    }}
    load(); setInterval(load, 3000);
  </script>
</body></html>
""", encoding="utf-8")

# Mount static files under /site
app.mount("/site", StaticFiles(directory=SITE), name="site")

@app.post("/site/chat/send")
async def post_chat(req: Request):
    msg = (await req.json()).get("message","").strip()
    if msg:
        line = f"[chat {time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}"
        with CHAT_LOG.open("a") as f: f.write(line+"\n")
        logger.info(f"CHAT {msg}")
    return {"ok": True}

@app.get("/site/chat/log")
async def get_chat():
    return CHAT_LOG.read_text()

@app.get("/site/log/serverlog")
async def get_slog():
    return SERVER_LOG.read_text()


# ─── 6. STARTUP VIA NGROK ─────────────────────────────────────────────────────
def start():
    Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8001, log_level="info"),
           daemon=True).start()
    time.sleep(2)
    try:
        url = ngrok.connect(8001).public_url
        print("🔗 Public URL:", url)
    except Exception as e:
        print("⚠️ ngrok failed:", e)
        print("🔗 Local URL: http://localhost:8001")

start()
# ║  Launches a public FastAPI-based server for structured tool calls.         ║
# ║  - Supports LLM agents and humans interacting via tool chat UI             ║
# ║  - Provides install_package, execute_python, read/write/list files         ║
# ║  - All files stored in a safe Google Drive folder                          ║
# ║  - /site/toolchat.html offers a JSON tool caller interface                 ║
# ║  - /activity shows real-time logs (SSE)                                    ║
# ║  - /site/chat.html supports structured human-agent messaging               ║
# ╚════════════════════════════════════════════════════════════════════════════╝

# ─── Install Dependencies ──────────────────────────────────────────────────────
import subprocess, sys, os, json, time, logging, asyncio, nest_asyncio
from threading import Thread
from pathlib import Path
from fastapi import FastAPI
import uvicorn

# Install required Python packages
subprocess.run([
    sys.executable, "-m", "pip", "install", "--quiet",
    "fastapi", "uvicorn", "pyngrok", "nest-asyncio", "sse-starlette", "requests"
], check=True)

# Enable async in Colab notebooks
nest_asyncio.apply()

# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

# ─── File Access Setup ────────────────────────────────────────────────────────
SAFE_DIR = Path("/content/drive/MyDrive/ColabMCPFiles")
SAFE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(SAFE_DIR)  # Lock working directory to this folder

# Ngrok for public access — replace with your token
from pyngrok import ngrok
ngrok.set_auth_token("REPLACE_WITH_YOUR_NGROK_TOKEN")

# ─── FastAPI App Initialization ───────────────────────────────────────────────
app = FastAPI(title="Colab MCP Tool Server", version="1.2")

# ─── Logging & Broadcast Handler ───────────────────────────────────────────────
subscribers: set[asyncio.Queue] = set()
SERVER_LOG = SAFE_DIR / "server_log.txt"; SERVER_LOG.touch()
CHAT_LOG   = SAFE_DIR / "chat_log.txt";   CHAT_LOG.touch()

class BroadcastHandler(logging.Handler):
    def emit(self, record):
        msg = self.format(record)
        with SERVER_LOG.open("a") as f: f.write(msg + "\n")
        for q in list(subscribers):
            try:
                asyncio.get_event_loop().call_soon_threadsafe(q.put_nowait, msg)
            except:
                pass

logger = logging.getLogger("mcp")
logger.setLevel(logging.INFO)
handler = BroadcastHandler()
handler.setFormatter(logging.Formatter("%(asctime)s %(message)s"))
logger.addHandler(handler)

# ─── Tool Definitions ─────────────────────────────────────────────────────────
TOOLS = [
    {
        "name": "execute_python",
        "description": "Run Python code and return stdout",
        "inputSchema": {
            "type": "object",
            "properties": {
                "code": {"type": "string", "description": "Python code to execute"}
            },
            "required": ["code"]
        }
    },
    {
        "name": "install_package",
        "description": "Install a pip package",
        "inputSchema": {
            "type": "object",
            "properties": {
                "package": {"type": "string", "description": "Name of package"}
            },
            "required": ["package"]
        }
    },
    {
        "name": "list_files",
        "description": "List files in a directory",
        "inputSchema": {
            "type": "object",
            "properties": {
                "path": {"type": "string", "default": ".", "description": "Directory path"}
            }
        }
    },
    {
        "name": "read_file",
        "description": "Read contents of a file",
        "inputSchema": {
            "type": "object",
            "properties": {
                "filepath": {"type": "string", "description": "File path to read"}
            },
            "required": ["filepath"]
        }
    },
    {
        "name": "write_file",
        "description": "Write content to a file",
        "inputSchema": {
            "type": "object",
            "properties": {
                "filepath": {"type": "string", "description": "File path to write"},
                "content": {"type": "string", "description": "Content to write"}
            },
            "required": ["filepath", "content"]
        }
    }
]

# ─── Tool Execution Functions ─────────────────────────────────────────────────
def exec_py(code: str) -> str:
    from io import StringIO; import contextlib
    buf = StringIO()
    try:
        with contextlib.redirect_stdout(buf):
            exec(code, {})
        return buf.getvalue() or "✓ no output"
    except Exception as e:
        return f"Execution error: {e}"

def inst_pkg(pkg: str) -> str:
    p = subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg],
        capture_output=True, text=True
    )
    return p.stdout or p.stderr

def safe_path(p: str) -> Path:
    target = SAFE_DIR / p
    if not target.resolve().is_relative_to(SAFE_DIR.resolve()):
        raise ValueError("Unsafe path.")
    return target

def ls_files(path: str = ".") -> str:
    p = safe_path(path)
    if not p.exists():
        return f"Not found: {path}"
    if p.is_file():
        return p.read_text()
    return "\n".join(
        f"{'📄' if it.is_file() else '📁'} {it.name}{'/' if it.is_dir() else ''}"
        for it in sorted(p.iterdir())
    ) or "— empty"

def rd_file(fp: str) -> str:
    return safe_path(fp).read_text()

def wr_file(fp: str, content: str) -> str:
    p = safe_path(fp)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content)
    return f"Written {p.name}"

    # ─── Part 2: API, Memory, Schema, UI ───────────────────────────────────────────

from fastapi import FastAPI, Request, HTTPException
from fastapi.responses import HTMLResponse, FileResponse
from fastapi.staticfiles import StaticFiles
from sse_starlette.sse import EventSourceResponse
import asyncio, time, json, logging
from threading import Thread
import uvicorn
from pathlib import Path

app = FastAPI(title="Colab MCP Tool Server", version="1.2")

#─── Reuse these from Part 1 ────────────────────────────────────────────────────
# SAFE_DIR = Path("/content/drive/MyDrive/ColabMCPFiles")
# logger: configured BroadcastHandler logging
# TOOLS, exec_py, inst_pkg, ls_files, rd_file, wr_file, subscribers, SERVER_LOG, CHAT_LOG

# ─── 1. STANDARD MCP ENDPOINTS ────────────────────────────────────────────────
@app.get("/")
async def root():
    return {"message": "MCP Server Running", "version": "1.2"}

@app.post("/mcp/tools/list")
async def list_tools():
    """Return list of available tools with JSON schemas."""
    return {"tools": TOOLS}

@app.post("/mcp/tools/call")
async def call_tool(req: Request):
    """Invoke a tool by name with arguments, log call & result."""
    data = await req.json()
    name = data.get("name"); args = data.get("arguments", {})
    logger.info(f"CALL {name} {args!r}")
    try:
        if name == "execute_python":
            out = exec_py(args["code"])
        elif name == "install_package":
            out = inst_pkg(args["package"])
        elif name == "list_files":
            out = ls_files(args.get("path", "."))
        elif name == "read_file":
            out = rd_file(args["filepath"])
        elif name == "write_file":
            out = wr_file(args["filepath"], args["content"])
        else:
            raise Exception("Unknown tool")
        logger.info(f"RESULT {name} -> {out[:120]!r}")
        return {"content": [{"type": "text", "text": out}], "isError": False}
    except Exception as e:
        logger.error(f"ERR {name} {e}")
        return {"content": [{"type": "text", "text": f"Error: {e}"}], "isError": True}


# ─── 2. AGENT MEMORY (JSON) ────────────────────────────────────────────────────
MEMORY_FILE = SAFE_DIR / "memory.json"
if not MEMORY_FILE.exists():
    MEMORY_FILE.write_text("{}")

def load_memory():
    try:
        return json.loads(MEMORY_FILE.read_text())
    except:
        return {}

def save_memory(mem: dict):
    MEMORY_FILE.write_text(json.dumps(mem, indent=2))

@app.get("/memory/view")
async def memory_view():
    """Return the full memory JSON object."""
    return load_memory()

@app.get("/memory/summary")
async def memory_summary():
    """Return only the 'summary' field from memory (if any)."""
    mem = load_memory()
    return {"summary": mem.get("summary")}

@app.post("/memory/update")
async def memory_update(req: Request):
    """
    Merge provided JSON into memory.
    e.g. POST {"task":"summarize","summary":"pending"} updates memory.
    """
    data = await req.json()
    mem = load_memory()
    mem.update(data)
    save_memory(mem)
    logger.info(f"Memory updated: {data}")
    return mem

@app.post("/memory/clear")
async def memory_clear():
    """Clear all memory (reset to empty JSON {})."""
    save_memory({})
    logger.info("Memory cleared")
    return {}


# ─── 3. OPENAPI / TOOL SCHEMA ──────────────────────────────────────────────────
@app.get("/mcp/schema")
async def mcp_schema():
    """Return the full OpenAPI schema for tool discovery by agents."""
    return app.openapi()

@app.get("/.well-known/tool-schema.json")
async def well_known_schema():
    """Alias for the OpenAPI schema (for standardized discovery)."""
    return app.openapi()


# ─── 4. LIVE ACTIVITY DASHBOARD ───────────────────────────────────────────────
@app.get("/activity", response_class=HTMLResponse)
async def dash():
    """
    Live SSE dashboard of all server activity:
    tool calls, results, chat messages.
    """
    return """<!DOCTYPE html><html><head><title>Live Activity</title>
<meta charset="utf-8"><meta name="description" content="Real-time MCP tool activity log.">
<style>body{font:14px sans;}#log{white-space:pre-wrap;background:#eee;padding:8px;height:70vh;overflow:auto;border:1px solid #ccc;}</style>
</head><body><h1>Live Tool Activity</h1><p>Real-time activity (calls, results, chat).</p>
<div id="log">Connecting…</div>
<script>
const log=document.getElementById('log'),
      src=new EventSource('/activity/stream');
src.onmessage=e=>{log.textContent+='\\n'+e.data;log.scrollTop=log.scrollHeight;}
src.onerror=_=>log.textContent+='\\n[stream closed]';
</script></body></html>"""

@app.get("/activity/stream")
async def stream():
    """SSE endpoint streaming every log line to connected dashboards."""
    q = asyncio.Queue(); subscribers.add(q)
    async def event_gen():
        while True:
            yield {"data": await q.get()}
    return EventSourceResponse(event_gen())


# ─── 5. STATIC PAGES: TOOL CHAT & HUMAN CHAT ──────────────────────────────────
SITE = SAFE_DIR / "site"; SITE.mkdir(exist_ok=True)

# 5a. Structured Tool Chat UI at /site/toolchat.html
(SITE / "toolchat.html").write_text(f"""
<!DOCTYPE html><html><head><meta charset="utf-8"><title>Tool Chat</title>
<meta name="description" content="JSON UI to call MCP tools (execute_python, install_package, file ops).">
<style>body{{font:14px sans;max-width:700px;margin:1em auto;}}textarea{{width:100%;height:6em;}}</style>
</head><body>
  <h1>Structured Tool Chat</h1>
  <p>Fill in a tool call as JSON and click <strong>Run</strong>:</p>
  <pre style="background:#f8f8f8;padding:8px;border:1px solid #ccc;">{json.dumps(TOOLS, indent=2)}</pre>
  <textarea id="req">{{"name":"execute_python","arguments":{{"code":"print(2+2)"}}}}</textarea><br>
  <button onclick="run()">Run</button>
  <h3>Response</h3>
  <pre id="out" style="background:#eef;border:1px solid #ccc;padding:8px;height:200px;overflow:auto;"></pre>
  <script>
    async function run() {{
      const r = JSON.parse(document.getElementById('req').value);
      const res = await fetch('/mcp/tools/call', {{
        method:'POST', headers:{{'Content-Type':'application/json'}}, body: JSON.stringify(r)
      }});
      const j = await res.json();
      document.getElementById('out').textContent = JSON.stringify(j, null, 2);
    }}
  </script>
</body></html>
""", encoding="utf-8")

# 5b. Simple Chat UI at /site/chat.html
(SITE / "chat.html").write_text("""
<!DOCTYPE html><html><head><title>Chat</title>
<meta name="description" content="Simple human↔agent messaging log.">
<style>body{font:14px sans;max-width:600px;margin:1em auto;}#chatlog{height:200px;overflow:auto;border:1px solid #ccc;padding:8px;}</style>
</head><body>
  <h2>Chat Log</h2>
  <div id="chatlog"></div>
  <input id="msg" placeholder="Type message..." style="width:80%;"><button onclick="send()">Send</button>
  <script>
    async function send() {{
      let m = document.getElementById('msg').value;
      if (!m) return;
      await fetch('/site/chat/send', {{
        method:'POST', headers:{{'Content-Type':'application/json'}}, body: JSON.stringify({{message:m}})
      }});
      document.getElementById('msg').value = '';
      load();
    }}
    async function load() {{
      const t = await fetch('/site/chat/log').then(r => r.text());
      document.getElementById('chatlog').textContent = t;
    }}
    load(); setInterval(load, 3000);
  </script>
</body></html>
""", encoding="utf-8")

# Mount static files under /site
app.mount("/site", StaticFiles(directory=SITE), name="site")

@app.post("/site/chat/send")
async def post_chat(req: Request):
    msg = (await req.json()).get("message","").strip()
    if msg:
        line = f"[chat {time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}"
        with CHAT_LOG.open("a") as f: f.write(line+"\n")
        logger.info(f"CHAT {msg}")
    return {"ok": True}

@app.get("/site/chat/log")
async def get_chat():
    return CHAT_LOG.read_text()

@app.get("/site/log/serverlog")
async def get_slog():
    return SERVER_LOG.read_text()


# ─── 6. STARTUP VIA NGROK ─────────────────────────────────────────────────────
def start():
    Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8001, log_level="info"),
           daemon=True).start()
    time.sleep(2)
    try:
        url = ngrok.connect(8001).public_url
        print("🔗 Public URL:", url)
    except Exception as e:
        print("⚠️ ngrok failed:", e)
        print("🔗 Local URL: http://localhost:8001")

start()


Mounted at /content/drive


INFO:     Started server process [2694]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8001 (Press CTRL+C to quit)
ERROR:pyngrok.process.ngrok:t=2025-06-23T14:51:37+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: The authtoken you specified does not look like a proper ngrok tunnel authtoken.\nYour authtoken: REPLACE_WITH_YOUR_NGROK_TOKEN\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_105\r\n"
ERROR:pyngrok.process.ngrok:t=2025-06-23T14:51:37+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: The authtoken you specified does not look like a proper ngrok tunnel authtoken.\nYour authtoken: REPLACE_WITH_YOUR_NGROK_TOKEN\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-autht

⚠️ ngrok failed: The ngrok process errored on start: authentication failed: The authtoken you specified does not look like a proper ngrok tunnel authtoken.\nYour authtoken: REPLACE_WITH_YOUR_NGROK_TOKEN\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_105\r\n.
🔗 Local URL: http://localhost:8001
Mounted at /content/drive


INFO:     Started server process [2694]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8001): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
ERROR:pyngrok.process.ngrok:t=2025-06-23T14:51:47+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: The authtoken you specified does not look like a proper ngrok tunnel authtoken.\nYour authtoken: REPLACE_WITH_YOUR_NGROK_TOKEN\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_105\r\n"
ERROR:pyngrok.process.ngrok:t=2025-06-23T14:51:47+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: The authtoken you specified does not look like a proper ngrok tunnel authtoken.\nYour authtoken: REPLACE_WITH_YOUR_NGROK_TOKEN\nI

⚠️ ngrok failed: The ngrok process errored on start: authentication failed: The authtoken you specified does not look like a proper ngrok tunnel authtoken.\nYour authtoken: REPLACE_WITH_YOUR_NGROK_TOKEN\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_105\r\n.
🔗 Local URL: http://localhost:8001


In [ ]:
# ╔════════════════════════════════════════════════════════════════════════════╗
# ║     🤖 Colab MCP Server + Tool Call UI + Live Dashboard + GDrive Mount    ║
# ║                                                                            ║
# ║  Launches a public FastAPI-based server for structured tool calls.         ║
# ║  - Supports LLM agents and humans interacting via tool chat UI             ║
# ║  - Provides install_package, execute_python, read/write/list files         ║
# ║  - All files stored in a safe Google Drive folder                          ║
# ║  - /site/toolchat.html offers a JSON tool caller interface                 ║
# ║  - /activity shows real-time logs (SSE)                                    ║
# ║  - /site/chat.html supports structured human-agent messaging               ║
# ╚════════════════════════════════════════════════════════════════════════════╝

# ─── Install Dependencies ──────────────────────────────────────────────────────
import subprocess, sys, os, json, time, logging, asyncio, nest_asyncio
from threading import Thread
from pathlib import Path
from fastapi import FastAPI, Request, HTTPException
from fastapi.responses import HTMLResponse, FileResponse
from fastapi.staticfiles import StaticFiles
from sse_starlette.sse import EventSourceResponse
from pyngrok import ngrok
import uvicorn

# Install required Python packages
subprocess.run([
    sys.executable, "-m", "pip", "install", "--quiet",
    "fastapi", "uvicorn", "pyngrok", "nest-asyncio", "sse-starlette", "requests"
], check=True)

# Enable async in Colab notebooks
nest_asyncio.apply()

# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

# ─── File Access Setup ────────────────────────────────────────────────────────
SAFE_DIR = Path("/content/drive/MyDrive/ColabMCPFiles")
SAFE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(SAFE_DIR)  # Lock working directory to this folder

# Ngrok for public access — replace with your actual ngrok token after testing
ngrok.set_auth_token("2yuUKpiAlYXCez40i8UktLZ91s7_5Rv1GjG5HMFw5KKbk5sHz")

# ─── FastAPI App Initialization ───────────────────────────────────────────────
app = FastAPI(title="Colab MCP Tool Server", version="1.2")

# ─── Logging & Broadcast Handler ───────────────────────────────────────────────
subscribers: set[asyncio.Queue] = set()
SERVER_LOG = SAFE_DIR / "server_log.txt"; SERVER_LOG.touch()
CHAT_LOG   = SAFE_DIR / "chat_log.txt";   CHAT_LOG.touch()

class BroadcastHandler(logging.Handler):
    def emit(self, record):
        msg = self.format(record)
        with SERVER_LOG.open("a") as f:
            f.write(msg + "\n")
        for q in list(subscribers):
            try:
                asyncio.get_event_loop().call_soon_threadsafe(q.put_nowait, msg)
            except:
                pass

logger = logging.getLogger("mcp")
logger.setLevel(logging.INFO)
handler = BroadcastHandler()
handler.setFormatter(logging.Formatter("%(asctime)s %(message)s"))
logger.addHandler(handler)

# ─── Tool Definitions ─────────────────────────────────────────────────────────
TOOLS = [
    {
        "name": "execute_python",
        "description": "Run Python code and return stdout",
        "inputSchema": {
            "type": "object",
            "properties": {
                "code": {"type": "string", "description": "Python code to execute"}
            },
            "required": ["code"]
        }
    },
    {
        "name": "install_package",
        "description": "Install a pip package",
        "inputSchema": {
            "type": "object",
            "properties": {
                "package": {"type": "string", "description": "Name of package"}
            },
            "required": ["package"]
        }
    },
    {
        "name": "list_files",
        "description": "List files in a directory",
        "inputSchema": {
            "type": "object",
            "properties": {
                "path": {"type": "string", "default": ".", "description": "Directory path"}
            }
        }
    },
    {
        "name": "read_file",
        "description": "Read contents of a file",
        "inputSchema": {
            "type": "object",
            "properties": {
                "filepath": {"type": "string", "description": "File path to read"}
            },
            "required": ["filepath"]
        }
    },
    {
        "name": "write_file",
        "description": "Write content to a file",
        "inputSchema": {
            "type": "object",
            "properties": {
                "filepath": {"type": "string", "description": "File path to write"},
                "content": {"type": "string", "description": "Content to write"}
            },
            "required": ["filepath", "content"]
        }
    }
]

# ─── Tool Execution Functions ─────────────────────────────────────────────────
def exec_py(code: str) -> str:
    from io import StringIO; import contextlib
    buf = StringIO()
    try:
        with contextlib.redirect_stdout(buf):
            exec(code, {})
        return buf.getvalue() or "✓ no output"
    except Exception as e:
        return f"Execution error: {e}"

def inst_pkg(pkg: str) -> str:
    p = subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg],
        capture_output=True, text=True
    )
    return p.stdout or p.stderr

def safe_path(p: str) -> Path:
    target = SAFE_DIR / p
    if not target.resolve().is_relative_to(SAFE_DIR.resolve()):
        raise ValueError("Unsafe path.")
    return target

def ls_files(path: str = ".") -> str:
    p = safe_path(path)
    if not p.exists():
        return f"Not found: {path}"
    if p.is_file():
        return p.read_text()
    return "\n".join(
        f"{'📄' if it.is_file() else '📁'} {it.name}{'/' if it.is_dir() else ''}"
        for it in sorted(p.iterdir())
    ) or "— empty"

def rd_file(fp: str) -> str:
    return safe_path(fp).read_text()

def wr_file(fp: str, content: str) -> str:
    p = safe_path(fp)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content)
    return f"Written {p.name}"

# ─── Part 2: API Endpoints, Memory, Schema, UI & Startup ───────────────────────

# 1. MCP Endpoints
@app.get("/")
async def root():
    return {"message": "MCP Server Running", "version": "1.2"}

@app.post("/mcp/tools/list")
async def list_tools():
    """Return list of available tools with JSON schemas."""
    return {"tools": TOOLS}

@app.post("/mcp/tools/call")
async def call_tool(req: Request):
    """Invoke a tool by name with arguments, log call & result."""
    data = await req.json()
    name = data.get("name"); args = data.get("arguments", {})
    logger.info(f"CALL {name} {args!r}")
    try:
        if name == "execute_python":
            out = exec_py(args["code"])
        elif name == "install_package":
            out = inst_pkg(args["package"])
        elif name == "list_files":
            out = ls_files(args.get("path", "."))
        elif name == "read_file":
            out = rd_file(args["filepath"])
        elif name == "write_file":
            out = wr_file(args["filepath"], args["content"])
        else:
            raise Exception("Unknown tool")
        logger.info(f"RESULT {name} -> {out[:120]!r}")
        return {"content": [{"type": "text", "text": out}], "isError": False}
    except Exception as e:
        logger.error(f"ERR {name} {e}")
        return {"content": [{"type": "text", "text": f"Error: {e}"}], "isError": True}

# 2. Agent Memory (JSON)
MEMORY_FILE = SAFE_DIR / "memory.json"
if not MEMORY_FILE.exists():
    MEMORY_FILE.write_text("{}")

def load_memory():
    try:
        return json.loads(MEMORY_FILE.read_text())
    except:
        return {}

def save_memory(mem: dict):
    MEMORY_FILE.write_text(json.dumps(mem, indent=2))

@app.get("/memory/view")
async def memory_view():
    """Return the full memory JSON object."""
    return load_memory()

@app.get("/memory/summary")
async def memory_summary():
    """Return only the 'summary' field from memory (if any)."""
    mem = load_memory()
    return {"summary": mem.get("summary")}

@app.post("/memory/update")
async def memory_update(req: Request):
    """Merge provided JSON into memory."""
    data = await req.json()
    mem = load_memory(); mem.update(data)
    save_memory(mem)
    logger.info(f"Memory updated: {data}")
    return mem

@app.post("/memory/clear")
async def memory_clear():
    """Clear all memory (reset to empty)."""
    save_memory({})
    logger.info("Memory cleared")
    return {}

# 3. OpenAPI & Tool Schema
@app.get("/mcp/schema")
async def mcp_schema():
    """Return the full OpenAPI schema for tool discovery."""
    return app.openapi()

@app.get("/.well-known/tool-schema.json")
async def well_known_schema():
    """Alias for the OpenAPI schema."""
    return app.openapi()

# 4. Live Activity Dashboard
@app.get("/activity", response_class=HTMLResponse)
async def dash():
    """Live SSE dashboard of all server activity."""
    return """<!DOCTYPE html><html><head><title>Live Activity</title>
<meta charset="utf-8"><meta name="description" content="Real-time MCP tool activity log.">
<style>body{font:14px sans;}#log{white-space:pre-wrap;background:#eee;padding:8px;height:70vh;overflow:auto;border:1px solid #ccc;}</style>
</head><body><h1>Live Tool Activity</h1><p>Real-time activity (calls, results, chat).</p>
<div id="log">Connecting…</div>
<script>
const log=document.getElementById('log'),
      src=new EventSource('/activity/stream');
src.onmessage=e=>{log.textContent+='\\n'+e.data;log.scrollTop=log.scrollHeight;}
src.onerror=_=>log.textContent+='\\n[stream closed]';
</script></body></html>"""

@app.get("/activity/stream")
async def stream():
    """SSE endpoint streaming each log line."""
    q = asyncio.Queue(); subscribers.add(q)
    async def event_gen():
        while True:
            yield {"data": await q.get()}
    return EventSourceResponse(event_gen())

# 5. Static Tool Chat & Human Chat UI
SITE = SAFE_DIR / "site"; SITE.mkdir(exist_ok=True)

# 5a. Structured Tool Chat UI
(SITE / "toolchat.html").write_text(f"""<!DOCTYPE html><html><head><meta charset="utf-8"><title>Tool Chat</title>
<meta name="description" content="JSON UI to call MCP tools.">
<style>body{{font:14px sans;max-width:700px;margin:1em auto;}}textarea{{width:100%;height:6em;}}</style>
</head><body>
  <h1>Structured Tool Chat</h1>
  <p>Paste a tool call JSON and click <strong>Run</strong>:</p>
  <pre style="background:#f8f8f8;padding:8px;border:1px solid #ccc;">{json.dumps(TOOLS, indent=2)}</pre>
  <textarea id="req">{{"name":"execute_python","arguments":{{"code": "print(2+2)"}}}}</textarea><br>
  <button onclick="run()">Run</button>
  <pre id="out" style="background:#eef;border:1px solid #ccc;padding:8px;height:200px;overflow:auto;"></pre>
  <script>
    async function run() {{
      const req = JSON.parse(document.getElementById('req').value);
      const res = await fetch('/mcp/tools/call', {{
        method:'POST', headers:{{'Content-Type':'application/json'}}, body: JSON.stringify(req)
      }});
      const j = await res.json();
      document.getElementById('out').textContent = JSON.stringify(j, null, 2);
    }}
  </script>
</body></html>""", encoding="utf-8")

# 5b. Simple Chat UI
(SITE / "chat.html").write_text("""<!DOCTYPE html><html><head><title>Chat</title>
<meta name="description" content="Simple human↔agent messaging log.">
<style>body{font:14px sans;max-width:600px;margin:1em auto;}#chatlog{height:200px;overflow:auto;border:1px solid #ccc;padding:8px;}</style>
</head><body>
  <h2>Chat Log</h2><div id="chatlog"></div>
  <input id="msg" placeholder="Type message..." style="width:80%;"><button onclick="send()">Send</button>
  <script>
    async function send() {{
      let m = document.getElementById('msg').value; if(!m) return;
      await fetch('/site/chat/send', {{
        method:'POST', headers:{{'Content-Type':'application/json'}}, body: JSON.stringify({{message: m}})
      }});
      document.getElementById('msg').value = ''; load();
    }}
    async function load() {{
      const t = await fetch('/site/chat/log').then(r => r.text());
      document.getElementById('chatlog').textContent = t;
    }}
    load(); setInterval(load, 3000);
  </script>
</body></html>""", encoding="utf-8")

app.mount("/site", StaticFiles(directory=SITE), name="site")

@app.post("/site/chat/send")
async def post_chat(req: Request):
    msg = (await req.json()).get("message","").strip()
    if msg:
        line = f"[chat {time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}"
        with CHAT_LOG.open("a") as f:
            f.write(line + "\n")
        logger.info(f"CHAT {msg}")
    return {"ok": True}

@app.get("/site/chat/log")
async def get_chat():
    return CHAT_LOG.read_text()

@app.get("/site/log/serverlog")
async def get_slog():
    return SERVER_LOG.read_text()

# ─── Startup via Ngrok ────────────────────────────────────────────────────────
def start():
    Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8001, log_level="info"), daemon=True).start()
    time.sleep(2)
    try:
        url = ngrok.connect(8001).public_url
        print("🔗 Public URL:", url)
    except Exception as e:
        print("⚠️ ngrok failed:", e)
        print("🔗 Local URL: http://localhost:8001")

start()

ERROR:asyncio:Task exception was never retrieved
future: <Task finished name='Task-4' coro=<Server.serve() done, defined at /usr/local/lib/python3.11/dist-packages/uvicorn/server.py:68> exception=SystemExit(1)>
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/uvicorn/server.py", line 163, in startup
    server = await loop.create_server(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.11/asyncio/base_events.py", line 1536, in create_server
    raise OSError(err.errno, msg) from None
OSError: [Errno 98] error while attempting to bind on address ('0.0.0.0', 8001): address already in use

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/lib/python3.11/threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.11/threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipython-input-4-2469852543.py", line 787, in 

Mounted at /content/drive


INFO:     Started server process [2694]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8001): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


🔗 Public URL: https://6736-34-80-226-174.ngrok-free.app


In [ ]:
# ╔════════════════════════════════════════════════════════════════════════════╗
# ║     🤖 Colab MCP Server + Tool Call UI + Live Dashboard + GDrive Mount    ║
# ║                                                                            ║
# ║  Launches a public FastAPI-based server for structured tool calls.         ║
# ║  - Supports LLM agents and humans interacting via tool chat UI             ║
# ║  - Provides install_package, execute_python, read/write/list files         ║
# ║  - All files stored in a safe Google Drive folder                          ║
# ║  - /site/toolchat.html offers a JSON tool caller interface                 ║
# ║  - /activity shows real-time logs (SSE)                                    ║
# ║  - /site/chat.html supports structured human-agent messaging               ║
# ╚════════════════════════════════════════════════════════════════════════════╝

# ─── Install Dependencies ──────────────────────────────────────────────────────
import subprocess, sys, os, json, time, logging, asyncio, nest_asyncio
from threading import Thread
from pathlib import Path
from fastapi import FastAPI, Request, HTTPException
from fastapi.responses import HTMLResponse, FileResponse
from fastapi.staticfiles import StaticFiles
from sse_starlette.sse import EventSourceResponse
from pyngrok import ngrok
import uvicorn

# Install required Python packages
subprocess.run([
    sys.executable, "-m", "pip", "install", "--quiet",
    "fastapi", "uvicorn", "pyngrok", "nest-asyncio", "sse-starlette", "requests"
], check=True)

# Enable async in Colab notebooks
nest_asyncio.apply()

# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

# ─── File Access Setup ────────────────────────────────────────────────────────
SAFE_DIR = Path("/content/drive/MyDrive/ColabMCPFiles")
SAFE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(SAFE_DIR)  # Lock working directory to this folder

# Ngrok for public access — replace with your actual ngrok token after testing
ngrok.set_auth_token("2yuUKpiAlYXCez40i8UktLZ91s7_5Rv1GjG5HMFw5KKbk5sHz")

# ─── FastAPI App Initialization ───────────────────────────────────────────────
app = FastAPI(title="Colab MCP Tool Server", version="1.2")

# ─── Logging & Broadcast Handler ───────────────────────────────────────────────
subscribers: set[asyncio.Queue] = set()
SERVER_LOG = SAFE_DIR / "server_log.txt"; SERVER_LOG.touch()
CHAT_LOG   = SAFE_DIR / "chat_log.txt";   CHAT_LOG.touch()

class BroadcastHandler(logging.Handler):
    def emit(self, record):
        msg = self.format(record)
        with SERVER_LOG.open("a") as f:
            f.write(msg + "\n")
        for q in list(subscribers):
            try:
                asyncio.get_event_loop().call_soon_threadsafe(q.put_nowait, msg)
            except:
                pass

logger = logging.getLogger("mcp")
logger.setLevel(logging.INFO)
handler = BroadcastHandler()
handler.setFormatter(logging.Formatter("%(asctime)s %(message)s"))
logger.addHandler(handler)

# ─── Tool Definitions ─────────────────────────────────────────────────────────
TOOLS = [
    {
        "name": "execute_python",
        "description": "Run Python code and return stdout",
        "inputSchema": {
            "type": "object",
            "properties": {
                "code": {"type": "string", "description": "Python code to execute"}
            },
            "required": ["code"]
        }
    },
    {
        "name": "install_package",
        "description": "Install a pip package",
        "inputSchema": {
            "type": "object",
            "properties": {
                "package": {"type": "string", "description": "Name of package"}
            },
            "required": ["package"]
        }
    },
    {
        "name": "list_files",
        "description": "List files in a directory",
        "inputSchema": {
            "type": "object",
            "properties": {
                "path": {"type": "string", "default": ".", "description": "Directory path"}
            }
        }
    },
    {
        "name": "read_file",
        "description": "Read contents of a file",
        "inputSchema": {
            "type": "object",
            "properties": {
                "filepath": {"type": "string", "description": "File path to read"}
            },
            "required": ["filepath"]
        }
    },
    {
        "name": "write_file",
        "description": "Write content to a file",
        "inputSchema": {
            "type": "object",
            "properties": {
                "filepath": {"type": "string", "description": "File path to write"},
                "content": {"type": "string", "description": "Content to write"}
            },
            "required": ["filepath", "content"]
        }
    }
]

# ─── Tool Execution Functions ─────────────────────────────────────────────────
def exec_py(code: str) -> str:
    from io import StringIO; import contextlib
    buf = StringIO()
    try:
        with contextlib.redirect_stdout(buf):
            exec(code, {})
        return buf.getvalue() or "✓ no output"
    except Exception as e:
        return f"Execution error: {e}"

def inst_pkg(pkg: str) -> str:
    p = subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg],
        capture_output=True, text=True
    )
    return p.stdout or p.stderr

def safe_path(p: str) -> Path:
    target = SAFE_DIR / p
    if not target.resolve().is_relative_to(SAFE_DIR.resolve()):
        raise ValueError("Unsafe path.")
    return target

def ls_files(path: str = ".") -> str:
    p = safe_path(path)
    if not p.exists():
        return f"Not found: {path}"
    if p.is_file():
        return p.read_text()
    return "\n".join(
        f"{'📄' if it.is_file() else '📁'} {it.name}{'/' if it.is_dir() else ''}"
        for it in sorted(p.iterdir())
    ) or "— empty"

def rd_file(fp: str) -> str:
    return safe_path(fp).read_text()

def wr_file(fp: str, content: str) -> str:
    p = safe_path(fp)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content)
    return f"Written {p.name}"

# ─── Part 2: API Endpoints, Memory, Schema, UI & Startup ───────────────────────

# 1. MCP Endpoints
@app.get("/")
async def root():
    return {"message": "MCP Server Running", "version": "1.2"}

@app.post("/mcp/tools/list")
async def list_tools():
    """Return list of available tools with JSON schemas."""
    return {"tools": TOOLS}

@app.post("/mcp/tools/call")
async def call_tool(req: Request):
    """Invoke a tool by name with arguments, log call & result."""
    data = await req.json()
    name = data.get("name"); args = data.get("arguments", {})
    logger.info(f"CALL {name} {args!r}")
    try:
        if name == "execute_python":
            out = exec_py(args["code"])
        elif name == "install_package":
            out = inst_pkg(args["package"])
        elif name == "list_files":
            out = ls_files(args.get("path", "."))
        elif name == "read_file":
            out = rd_file(args["filepath"])
        elif name == "write_file":
            out = wr_file(args["filepath"], args["content"])
        else:
            raise Exception("Unknown tool")
        logger.info(f"RESULT {name} -> {out[:120]!r}")
        return {"content": [{"type": "text", "text": out}], "isError": False}
    except Exception as e:
        logger.error(f"ERR {name} {e}")
        return {"content": [{"type": "text", "text": f"Error: {e}"}], "isError": True}

# 2. Agent Memory (JSON)
MEMORY_FILE = SAFE_DIR / "memory.json"
if not MEMORY_FILE.exists():
    MEMORY_FILE.write_text("{}")

def load_memory():
    try:
        return json.loads(MEMORY_FILE.read_text())
    except:
        return {}

def save_memory(mem: dict):
    MEMORY_FILE.write_text(json.dumps(mem, indent=2))

@app.get("/memory/view")
async def memory_view():
    """Return the full memory JSON object."""
    return load_memory()

@app.get("/memory/summary")
async def memory_summary():
    """Return only the 'summary' field from memory (if any)."""
    mem = load_memory()
    return {"summary": mem.get("summary")}

@app.post("/memory/update")
async def memory_update(req: Request):
    """Merge provided JSON into memory."""
    data = await req.json()
    mem = load_memory(); mem.update(data)
    save_memory(mem)
    logger.info(f"Memory updated: {data}")
    return mem

@app.post("/memory/clear")
async def memory_clear():
    """Clear all memory (reset to empty)."""
    save_memory({})
    logger.info("Memory cleared")
    return {}

# 3. OpenAPI & Tool Schema
@app.get("/mcp/schema")
async def mcp_schema():
    """Return the full OpenAPI schema for tool discovery."""
    return app.openapi()

@app.get("/.well-known/tool-schema.json")
async def well_known_schema():
    """Alias for the OpenAPI schema."""
    return app.openapi()

# 4. Live Activity Dashboard
@app.get("/activity", response_class=HTMLResponse)
async def dash():
    """Live SSE dashboard of all server activity."""
    return """<!DOCTYPE html><html><head><title>Live Activity</title>
<meta charset="utf-8"><meta name="description" content="Real-time MCP tool activity log.">
<style>body{font:14px sans;}#log{white-space:pre-wrap;background:#eee;padding:8px;height:70vh;overflow:auto;border:1px solid #ccc;}</style>
</head><body><h1>Live Tool Activity</h1><p>Real-time activity (calls, results, chat).</p>
<div id="log">Connecting…</div>
<script>
const log=document.getElementById('log'),
      src=new EventSource('/activity/stream');
src.onmessage=e=>{log.textContent+='\\n'+e.data;log.scrollTop=log.scrollHeight;}
src.onerror=_=>log.textContent+='\\n[stream closed]';
</script></body></html>"""

@app.get("/activity/stream")
async def stream():
    """SSE endpoint streaming each log line."""
    q = asyncio.Queue(); subscribers.add(q)
    async def event_gen():
        while True:
            yield {"data": await q.get()}
    return EventSourceResponse(event_gen())

# 5. Static Tool Chat & Human Chat UI
SITE = SAFE_DIR / "site"; SITE.mkdir(exist_ok=True)

# 5a. Structured Tool Chat UI at /site/toolchat.html
(SITE / "toolchat.html").write_text(f"""<!DOCTYPE html><html><head><meta charset="utf-8"><title>Tool Chat</title>
<meta name="description" content="JSON UI to call MCP tools.">
<style>body{{font:14px sans;max-width:700px;margin:1em auto;}}textarea{{width:100%;height:6em;}}</style>
</head><body>
  <h1>Structured Tool Chat</h1>
  <p>Paste a tool call JSON and click <strong>Run</strong>:</p>
  <pre style="background:#f8f8f8;padding:8px;border:1px solid #ccc;">{json.dumps(TOOLS, indent=2)}</pre>
  <textarea id="req">{{"name":"execute_python","arguments":{{"code": "print(2+2)"}}}}</textarea><br>
  <button onclick="run()">Run</button>
  <pre id="out" style="background:#eef;border:1px solid #ccc;padding:8px;height:200px;overflow:auto;"></pre>
  <script>
    async function run() {{
      const req = JSON.parse(document.getElementById('req').value);
      const res = await fetch('/mcp/tools/call', {{
        method:'POST', headers:{{'Content-Type':'application/json'}}, body: JSON.stringify(req)
      }});
      const j = await res.json();
      document.getElementById('out').textContent = JSON.stringify(j, null, 2);
    }}
  </script>
</body></html>""", encoding="utf-8")

# 5b. Simple Chat UI at /site/chat.html
(SITE / "chat.html").write_text("""<!DOCTYPE html><html><head><title>Chat</title>
<meta name="description" content="Simple human↔agent messaging log.">
<style>body{font:14px sans;max-width:600px;margin:1em auto;}#chatlog{height:200px;overflow:auto;border:1px solid #ccc;padding:8px;}</style>
</head><body>
  <h2>Chat Log</h2><div id="chatlog"></div>
  <input id="msg" placeholder="Type message..." style="width:80%;"><button onclick="send()">Send</button>
  <script>
    async function send() {{
      let m = document.getElementById('msg').value; if(!m) return;
      await fetch('/site/chat/send', {{
        method:'POST', headers:{{'Content-Type':'application/json'}}, body: JSON.stringify({{message: m}})
      }});
      document.getElementById('msg').value = ''; load();
    }}
    async function load() {{
      const t = await fetch('/site/chat/log').then(r => r.text());
      document.getElementById('chatlog').textContent = t;
    }}
    load(); setInterval(load, 3000);
  </script>
</body></html>""", encoding="utf-8")

app.mount("/site", StaticFiles(directory=SITE), name="site")

@app.post("/site/chat/send")
async def post_chat(req: Request):
    msg = (await req.json()).get("message","").strip()
    if msg:
        line = f"[chat {time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}"
        with CHAT_LOG.open("a") as f:
            f.write(line + "\n")
        logger.info(f"CHAT {msg}")
    return {"ok": True}

@app.get("/site/chat/log")
async def get_chat():
    return CHAT_LOG.read_text()

@app.get("/site/log/serverlog")
async def get_slog():
    return SERVER_LOG.read_text()

# ─── Startup via Ngrok using port 8000 ─────────────────────────────────────────
def start():
    Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info"), daemon=True).start()
    time.sleep(2)
    try:
        url = ngrok.connect(8000).public_url
        print("🔗 Public URL:", url)
    except Exception as e:
        print("⚠️ ngrok failed:", e)
        print("🔗 Local URL: http://localhost:8000")

start()

ModuleNotFoundError: No module named 'sse_starlette'

In [ ]:
# ╔════════════════════════════════════════════════════════════════════════════╗
# ║     🤖 Colab MCP Server + Tool Call UI + Live Dashboard + GDrive Mount    ║
# ╚════════════════════════════════════════════════════════════════════════════╝

# ─── Colab Setup & Dependencies ───────────────────────────────────────────────
import subprocess, sys, os, json, time, logging, asyncio, nest_asyncio
from threading import Thread
from pathlib import Path

# Install core dependencies
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
  "fastapi", "uvicorn", "pyngrok", "nest-asyncio", "sse-starlette", "requests"
], check=True)

# Enable nest_asyncio in Colab
nest_asyncio.apply()

# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

# ─── Safe Working Directory ───────────────────────────────────────────────────
SAFE_DIR = Path("/content/drive/MyDrive/ColabMCPFiles")
SAFE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(SAFE_DIR)

# ─── Ngrok Setup ───────────────────────────────────────────────────────────────
from pyngrok import ngrok
# Placeholder token for testing; replace with your own for production
ngrok.set_auth_token("2yuUKpiAlYXCez40i8UktLZ91s7_5Rv1GjG5HMFw5KKbk5sHz")

# ─── Imports for FastAPI & SSE ────────────────────────────────────────────────
from fastapi import FastAPI, Request, HTTPException
from fastapi.responses import HTMLResponse, FileResponse
from fastapi.staticfiles import StaticFiles
import uvicorn

# Ensure sse-starlette is importable, else install
try:
    from sse_starlette.sse import EventSourceResponse
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "sse-starlette"], check=True)
    from sse_starlette.sse import EventSourceResponse

# ─── App & Logging ─────────────────────────────────────────────────────────────
app = FastAPI(title="Colab MCP Tool Server", version="1.2")
subscribers: set[asyncio.Queue] = set()
SERVER_LOG = SAFE_DIR/"server_log.txt"; SERVER_LOG.touch()
CHAT_LOG   = SAFE_DIR/"chat_log.txt";   CHAT_LOG.touch()

class BroadcastHandler(logging.Handler):
    def emit(self, record):
        msg = self.format(record)
        with SERVER_LOG.open("a") as f: f.write(msg+"\n")
        for q in list(subscribers):
            try: asyncio.get_event_loop().call_soon_threadsafe(q.put_nowait, msg)
            except: pass

logger = logging.getLogger("mcp")
logger.setLevel(logging.INFO)
h = BroadcastHandler(); h.setFormatter(logging.Formatter("%(asctime)s %(message)s"))
logger.addHandler(h)

# ─── Tool Definitions ─────────────────────────────────────────────────────────
TOOLS = [
  {"name":"execute_python","description":"Run Python code",
   "inputSchema":{"type":"object","properties":{"code":{"type":"string","description":"Python code to execute"}},"required":["code"]}},
  {"name":"install_package","description":"Install a pip package",
   "inputSchema":{"type":"object","properties":{"package":{"type":"string","description":"Package name"}},"required":["package"]}},
  {"name":"list_files","description":"List directory contents",
   "inputSchema":{"type":"object","properties":{"path":{"type":"string","default":".","description":"Directory path"}}}},
  {"name":"read_file","description":"Read file contents",
   "inputSchema":{"type":"object","properties":{"filepath":{"type":"string","description":"File to read"}},"required":["filepath"]}},
  {"name":"write_file","description":"Write to a file",
   "inputSchema":{"type":"object","properties":{"filepath":{"type":"string","description":"File to write"},"content":{"type":"string","description":"Text content"}},"required":["filepath","content"]}}
]

# ─── Tool Execution Functions ─────────────────────────────────────────────────
def exec_py(code: str) -> str:
    from io import StringIO; import contextlib
    buf = StringIO()
    try:
        with contextlib.redirect_stdout(buf): exec(code,{})
        return buf.getvalue() or "✓ no output"
    except Exception as e: return f"Execution error: {e}"

def inst_pkg(pkg: str) -> str:
    p = subprocess.run([sys.executable, "-m", "pip", "install", pkg], capture_output=True, text=True)
    return p.stdout or p.stderr

def safe_path(p: str) -> Path:
    t = SAFE_DIR / p
    if not t.resolve().is_relative_to(SAFE_DIR.resolve()):
        raise ValueError("Unsafe path")
    return t

def ls_files(path: str=".") -> str:
    p = safe_path(path)
    if not p.exists(): return f"Not found: {path}"
    if p.is_file(): return p.read_text()
    return "\n".join(f"{'📄' if f.is_file() else '📁'} {f.name}{'/' if f.is_dir() else ''}" for f in sorted(p.iterdir())) or "— empty"

def rd_file(fp: str) -> str: return safe_path(fp).read_text()
def wr_file(fp: str, content: str) -> str:
    p = safe_path(fp); p.parent.mkdir(parents=True, exist_ok=True); p.write_text(content)
    return f"Written {p.name}"
%env OPENAI_API_KEY=123456789
# ╔════════════════════════════════════════════════════════════════════════════╗
# ║     🤖 Colab MCP Server + Tool Call UI + Live Dashboard + GDrive Mount    ║
# ╚════════════════════════════════════════════════════════════════════════════╝

# ─── Colab Setup & Dependencies ───────────────────────────────────────────────
import subprocess, sys, os, json, time, logging, asyncio, nest_asyncio
from threading import Thread
from pathlib import Path

# Install core dependencies
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
  "fastapi", "uvicorn", "pyngrok", "nest-asyncio", "sse-starlette", "requests"
], check=True)

# Enable nest_asyncio in Colab
nest_asyncio.apply()

# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

# ─── Safe Working Directory ───────────────────────────────────────────────────
SAFE_DIR = Path("/content/drive/MyDrive/ColabMCPFiles")
SAFE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(SAFE_DIR)

# ─── Ngrok Setup ───────────────────────────────────────────────────────────────
from pyngrok import ngrok
# Placeholder token for testing; replace with your own for production
ngrok.set_auth_token("2yuUKpiAlYXCez40i8UktLZ91s7_5Rv1GjG5HMFw5KKbk5sHz")

# ─── Imports for FastAPI & SSE ────────────────────────────────────────────────
from fastapi import FastAPI, Request, HTTPException
from fastapi.responses import HTMLResponse, FileResponse
from fastapi.staticfiles import StaticFiles
import uvicorn

# Ensure sse-starlette is importable, else install
try:
    from sse_starlette.sse import EventSourceResponse
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "sse-starlette"], check=True)
    from sse_starlette.sse import EventSourceResponse

# ─── App & Logging ─────────────────────────────────────────────────────────────
app = FastAPI(title="Colab MCP Tool Server", version="1.2")
subscribers: set[asyncio.Queue] = set()
SERVER_LOG = SAFE_DIR/"server_log.txt"; SERVER_LOG.touch()
CHAT_LOG   = SAFE_DIR/"chat_log.txt";   CHAT_LOG.touch()

class BroadcastHandler(logging.Handler):
    def emit(self, record):
        msg = self.format(record)
        with SERVER_LOG.open("a") as f: f.write(msg+"\n")
        for q in list(subscribers):
            try: asyncio.get_event_loop().call_soon_threadsafe(q.put_nowait, msg)
            except: pass

logger = logging.getLogger("mcp")
logger.setLevel(logging.INFO)
h = BroadcastHandler(); h.setFormatter(logging.Formatter("%(asctime)s %(message)s"))
logger.addHandler(h)

# ─── Tool Definitions ─────────────────────────────────────────────────────────
TOOLS = [
  {"name":"execute_python","description":"Run Python code",
   "inputSchema":{"type":"object","properties":{"code":{"type":"string","description":"Python code to execute"}},"required":["code"]}},
  {"name":"install_package","description":"Install a pip package",
   "inputSchema":{"type":"object","properties":{"package":{"type":"string","description":"Package name"}},"required":["package"]}},
  {"name":"list_files","description":"List directory contents",
   "inputSchema":{"type":"object","properties":{"path":{"type":"string","default":".","description":"Directory path"}}}},
  {"name":"read_file","description":"Read file contents",
   "inputSchema":{"type":"object","properties":{"filepath":{"type":"string","description":"File to read"}},"required":["filepath"]}},
  {"name":"write_file","description":"Write to a file",
   "inputSchema":{"type":"object","properties":{"filepath":{"type":"string","description":"File to write"},"content":{"type":"string","description":"Text content"}},"required":["filepath","content"]}}
]

# ─── Tool Execution Functions ─────────────────────────────────────────────────
def exec_py(code: str) -> str:
    from io import StringIO; import contextlib
    buf = StringIO()
    try:
        with contextlib.redirect_stdout(buf): exec(code,{})
        return buf.getvalue() or "✓ no output"
    except Exception as e: return f"Execution error: {e}"

def inst_pkg(pkg: str) -> str:
    p = subprocess.run([sys.executable, "-m", "pip", "install", pkg], capture_output=True, text=True)
    return p.stdout or p.stderr

def safe_path(p: str) -> Path:
    t = SAFE_DIR / p
    if not t.resolve().is_relative_to(SAFE_DIR.resolve()):
        raise ValueError("Unsafe path")
    return t

def ls_files(path: str=".") -> str:
    p = safe_path(path)
    if not p.exists(): return f"Not found: {path}"
    if p.is_file(): return p.read_text()
    return "\n".join(f"{'📄' if f.is_file() else '📁'} {f.name}{'/' if f.is_dir() else ''}" for f in sorted(p.iterdir())) or "— empty"

def rd_file(fp: str) -> str: return safe_path(fp).read_text()
def wr_file(fp: str, content: str) -> str:
    p = safe_path(fp); p.parent.mkdir(parents=True, exist_ok=True); p.write_text(content)
    return f"Written {p.name}"

# ─── Part 2: API Endpoints, Memory, Schema, UI ────────────────────────────────
@app.get("/") async def root(): return {"message":"MCP Server Running","version":"1.2"}

@app.post("/mcp/tools/list")
async def list_tools(): return {"tools": TOOLS}

@app.post("/mcp/tools/call")
async def call_tool(req:Request):
    data = await req.json(); name=data.get("name"); args=data.get("arguments",{})
    logger.info(f"CALL {name} {args!r}")
    try:
        if   name=="execute_python": out=exec_py(args["code"])
        elif name=="install_package": out=inst_pkg(args["package"])
        elif name=="list_files":    out=ls_files(args.get("path","."))
        elif name=="read_file":     out=rd_file(args["filepath"])
        elif name=="write_file":    out=wr_file(args["filepath"],args["content"])
        else: raise Exception("Unknown tool")
        logger.info(f"RESULT {name} -> {out[:120]!r}")
        return {"content":[{"type":"text","text":out}],"isError":False}
    except Exception as e:
        logger.error(f"ERR {name} {e}")
        return {"content":[{"type":"text","text":f"Error: {e}"}],"isError":True}

# Agent Memory (JSON)
MEMORY_FILE=SAFE_DIR/"memory.json"
if not MEMORY_FILE.exists(): MEMORY_FILE.write_text("{}")
def load_memory():
    try: return json.loads(MEMORY_FILE.read_text())
    except: return {}
def save_memory(mem): MEMORY_FILE.write_text(json.dumps(mem, indent=2))

@app.get("/memory/view")    async def memory_view():    return load_memory()
@app.get("/memory/summary") async def memory_summary(): return {"summary": load_memory().get("summary")}
@app.post("/memory/update")async def memory_update(req:Request):
    data=await req.json(); mem=load_memory(); mem.update(data); save_memory(mem)
    logger.info(f"Memory updated: {data}"); return mem
@app.post("/memory/clear") async def memory_clear():
    save_memory({}); logger.info("Memory cleared"); return {}

# OpenAPI & Tool Schema
@app.get("/mcp/schema")                 async def mcp_schema():        return app.openapi()
@app.get("/.well-known/tool-schema.json") async def well_known_schema(): return app.openapi()

# Live Activity Dashboard
@app.get("/activity", response_class=HTMLResponse)
async def dash():
    return """<!DOCTYPE html><html><head><title>Live Activity</title>
<meta charset="utf-8"><meta name="description" content="Real-time log of MCP usage.">
<style>body{font:14px sans;}#log{background:#eee;padding:8px;height:70vh;overflow:auto;white-space:pre-wrap;border:1px solid #ccc;}</style>
</head><body><h1>Live Activity</h1><div id="log">Connecting…</div>
<script>
src=new EventSource('/activity/stream');
src.onmessage=e=>{let l=document.getElementById('log');l.textContent+=`\\n${e.data}`;l.scrollTop=l.scrollHeight;}
</script></body></html>"""

@app.get("/activity/stream")
async def stream():
    q=asyncio.Queue(); subscribers.add(q)
    async def gen():
        while True: yield {"data":await q.get()}
    return EventSourceResponse(gen())

# Static Tool Chat & Human Chat UI
SITE=SAFE_DIR/"site"; SITE.mkdir(exist_ok=True)
# Tool Chat
(SITE/"toolchat.html").write_text(f"""<!DOCTYPE html><html><head><meta charset="utf-8"><title>Tool Chat</title>
<meta name="description" content="JSON UI for MCP tools.">
<style>body{{font:14px sans;max-width:700px;margin:1em auto}}textarea{{width:100%;height:6em}}</style>
</head><body>
<h1>Structured Tool Chat</h1>
<pre style="background:#f8f8f8;padding:8px;border:1px solid #ccc">{json.dumps(TOOLS,indent=2)}</pre>
<textarea id="req">{{"name":"execute_python","arguments":{{"code":"print(2+2)"}}}}</textarea><br>
<button onclick="run()">Run</button>
<pre id="out" style="background:#eef;border:1px solid #ccc;padding:8px;height:200px;overflow:auto"></pre>
<script>
async function run(){{
  const r=JSON.parse(document.getElementById('req').value);
  const res=await fetch('/mcp/tools/call',{{method:'POST',headers:{{'Content-Type':'application/json'}},body:JSON.stringify(r)}});
  document.getElementById('out').textContent=JSON.stringify(await res.json(),null,2);
}}
</script></body></html>""", encoding="utf-8")

# Human Chat
(SITE/"chat.html").write_text("""<!DOCTYPE html><html><head><title>Chat</title>
<meta name="description" content="Human↔Agent chat log.">
<style>body{font:14px sans;max-width:600px;margin:1em auto}#chatlog{height:200px;overflow:auto;border:1px solid #ccc;padding:8px}</style>
</head><body>
<h2>Chat Log</h2><div id="chatlog"></div>
<input id="msg" placeholder="Type message..." style="width:80%"><button onclick="send()">Send</button>
<script>
async function send(){let m=document.getElementById('msg').value; if(!m)return;
await fetch('/site/chat/send',{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify({message:m})});
document.getElementById('msg').value='';load();}
async function load(){document.getElementById('chatlog').textContent=await fetch('/site/chat/log').then(r=>r.text());}
load(); setInterval(load,3000);
</script></body></html>""", encoding="utf-8")

app.mount("/site", StaticFiles(directory=SITE), name="site")

@app.post("/site/chat/send")
async def post_chat(req:Request):
    msg=(await req.json()).get("message","").strip()
    if msg:
        line=f"[chat {time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}"
        with CHAT_LOG.open("a") as f: f.write(line+"\n")
        logger.info(f"CHAT {msg}")
    return {"ok":True}

@app.get("/site/chat/log")
async def get_chat(): return CHAT_LOG.read_text()

@app.get("/site/log/serverlog")
async def get_slog(): return SERVER_LOG.read_text()

# ─── Startup (port 8000) ───────────────────────────────────────────────────────
def start():
    Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info"),daemon=True).start()
    time.sleep(2)
    try:
        url=ngrok.connect(8000).public_url
        print("🔗 Public URL:",url)
    except Exception as e:
        print("⚠️ ngrok failed:",e)
        print("🔗 Local URL: http://localhost:8000")

start()
async def list_tools(): return {"tools": TOOLS}

@app.post("/mcp/tools/call")
async def call_tool(req:Request):
    data = await req.json(); name=data.get("name"); args=data.get("arguments",{})
    logger.info(f"CALL {name} {args!r}")
    try:
        if   name=="execute_python": out=exec_py(args["code"])
        elif name=="install_package": out=inst_pkg(args["package"])
        elif name=="list_files":    out=ls_files(args.get("path","."))
        elif name=="read_file":     out=rd_file(args["filepath"])
        elif name=="write_file":    out=wr_file(args["filepath"],args["content"])
        else: raise Exception("Unknown tool")
        logger.info(f"RESULT {name} -> {out[:120]!r}")
        return {"content":[{"type":"text","text":out}],"isError":False}
    except Exception as e:
        logger.error(f"ERR {name} {e}")
        return {"content":[{"type":"text","text":f"Error: {e}"}],"isError":True}

# Agent Memory (JSON)
MEMORY_FILE=SAFE_DIR/"memory.json"
if not MEMORY_FILE.exists(): MEMORY_FILE.write_text("{}")
def load_memory():
    try: return json.loads(MEMORY_FILE.read_text())
    except: return {}
def save_memory(mem): MEMORY_FILE.write_text(json.dumps(mem, indent=2))

@app.get("/memory/view")    async def memory_view():    return load_memory()
@app.get("/memory/summary") async def memory_summary(): return {"summary": load_memory().get("summary")}
@app.post("/memory/update")async def memory_update(req:Request):
    data=await req.json(); mem=load_memory(); mem.update(data); save_memory(mem)
    logger.info(f"Memory updated: {data}"); return mem
@app.post("/memory/clear") async def memory_clear():
    save_memory({}); logger.info("Memory cleared"); return {}

# OpenAPI & Tool Schema
@app.get("/mcp/schema")                 async def mcp_schema():        return app.openapi()
@app.get("/.well-known/tool-schema.json") async def well_known_schema(): return app.openapi()

# Live Activity Dashboard
@app.get("/activity", response_class=HTMLResponse)
async def dash():
    return """<!DOCTYPE html><html><head><title>Live Activity</title>
<meta charset="utf-8"><meta name="description" content="Real-time log of MCP usage.">
<style>body{font:14px sans;}#log{background:#eee;padding:8px;height:70vh;overflow:auto;white-space:pre-wrap;border:1px solid #ccc;}</style>
</head><body><h1>Live Activity</h1><div id="log">Connecting…</div>
<script>
src=new EventSource('/activity/stream');
src.onmessage=e=>{let l=document.getElementById('log');l.textContent+=`\\n${e.data}`;l.scrollTop=l.scrollHeight;}
</script></body></html>"""

@app.get("/activity/stream")
async def stream():
    q=asyncio.Queue(); subscribers.add(q)
    async def gen():
        while True: yield {"data":await q.get()}
    return EventSourceResponse(gen())

# Static Tool Chat & Human Chat UI
SITE=SAFE_DIR/"site"; SITE.mkdir(exist_ok=True)
# Tool Chat
(SITE/"toolchat.html").write_text(f"""<!DOCTYPE html><html><head><meta charset="utf-8"><title>Tool Chat</title>
<meta name="description" content="JSON UI for MCP tools.">
<style>body{{font:14px sans;max-width:700px;margin:1em auto}}textarea{{width:100%;height:6em}}</style>
</head><body>
<h1>Structured Tool Chat</h1>
<pre style="background:#f8f8f8;padding:8px;border:1px solid #ccc">{json.dumps(TOOLS,indent=2)}</pre>
<textarea id="req">{{"name":"execute_python","arguments":{{"code":"print(2+2)"}}}}</textarea><br>
<button onclick="run()">Run</button>
<pre id="out" style="background:#eef;border:1px solid #ccc;padding:8px;height:200px;overflow:auto"></pre>
<script>
async function run(){{
  const r=JSON.parse(document.getElementById('req').value);
  const res=await fetch('/mcp/tools/call',{{method:'POST',headers:{{'Content-Type':'application/json'}},body:JSON.stringify(r)}});
  document.getElementById('out').textContent=JSON.stringify(await res.json(),null,2);
}}
</script></body></html>""", encoding="utf-8")

# Human Chat
(SITE/"chat.html").write_text("""<!DOCTYPE html><html><head><title>Chat</title>
<meta name="description" content="Human↔Agent chat log.">
<style>body{font:14px sans;max-width:600px;margin:1em auto}#chatlog{height:200px;overflow:auto;border:1px solid #ccc;padding:8px}</style>
</head><body>
<h2>Chat Log</h2><div id="chatlog"></div>
<input id="msg" placeholder="Type message..." style="width:80%"><button onclick="send()">Send</button>
<script>
async function send(){let m=document.getElementById('msg').value; if(!m)return;
await fetch('/site/chat/send',{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify({message:m})});
document.getElementById('msg').value='';load();}
async function load(){document.getElementById('chatlog').textContent=await fetch('/site/chat/log').then(r=>r.text());}
load(); setInterval(load,3000);
</script></body></html>""", encoding="utf-8")

app.mount("/site", StaticFiles(directory=SITE), name="site")

@app.post("/site/chat/send")
async def post_chat(req:Request):
    msg=(await req.json()).get("message","").strip()
    if msg:
        line=f"[chat {time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}"
        with CHAT_LOG.open("a") as f: f.write(line+"\n")
        logger.info(f"CHAT {msg}")
    return {"ok":True}

@app.get("/site/chat/log")
async def get_chat(): return CHAT_LOG.read_text()

@app.get("/site/log/serverlog")
async def get_slog(): return SERVER_LOG.read_text()

# ─── Startup (port 8000) ───────────────────────────────────────────────────────
def start():
    Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info"),daemon=True).start()
    time.sleep(2)
    try:
        url=ngrok.connect(8000).public_url
        print("🔗 Public URL:",url)
    except Exception as e:
        print("⚠️ ngrok failed:",e)
        print("🔗 Local URL: http://localhost:8000")

start()

SyntaxError: invalid syntax (ipython-input-6-3225216786.py, line 216)

In [ ]:
# ╔════════════════════════════════════════════════════════════════════════════╗
# ║     🤖 Colab MCP Server + Tool Call UI + Live Dashboard + GDrive Mount    ║
# ║                                                                            ║
# ║  Structured tool-calling server for humans & LLM agents in one cell.      ║
# ╚════════════════════════════════════════════════════════════════════════════╝

import subprocess, sys, os, json, time, logging, asyncio, nest_asyncio
from threading import Thread
from pathlib import Path

# ─── Install Core Dependencies ────────────────────────────────────────────────
packages = [
    "fastapi", "uvicorn", "pyngrok", "nest-asyncio", "requests"
]
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet"] + packages, check=True)

# Ensure sse-starlette is present
try:
    from sse_starlette.sse import EventSourceResponse
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "sse-starlette"], check=True)
    from sse_starlette.sse import EventSourceResponse

# Enable asyncio patch for Colab
nest_asyncio.apply()

# ─── Mount Google Drive ───────────────────────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

# ─── Safe Working Directory ───────────────────────────────────────────────────
SAFE_DIR = Path("/content/drive/MyDrive/ColabMCPFiles")
SAFE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(SAFE_DIR)

# ─── Ngrok Setup ──────────────────────────────────────────────────────────────
from pyngrok import ngrok
ngrok.set_auth_token("2yuUKpiAlYXCez40i8UktLZ91s7_5Rv1GjG5HMFw5KKbk5sHz")  # TEST TOKEN; replace with yours

# ─── FastAPI & Logging Setup ──────────────────────────────────────────────────
from fastapi import FastAPI, Request, HTTPException
from fastapi.responses import HTMLResponse
from fastapi.staticfiles import StaticFiles
import uvicorn

app = FastAPI(title="Colab MCP Tool Server", version="1.2")

subscribers: set[asyncio.Queue] = set()
SERVER_LOG = SAFE_DIR / "server_log.txt"; SERVER_LOG.touch()
CHAT_LOG   = SAFE_DIR / "chat_log.txt";   CHAT_LOG.touch()

class BroadcastHandler(logging.Handler):
    def emit(self, record):
        msg = self.format(record)
        with SERVER_LOG.open("a") as f:
            f.write(msg + "\n")
        for q in list(subscribers):
            try:
                asyncio.get_event_loop().call_soon_threadsafe(q.put_nowait, msg)
            except:
                pass

logger = logging.getLogger("mcp")
logger.setLevel(logging.INFO)
handler = BroadcastHandler()
handler.setFormatter(logging.Formatter("%(asctime)s %(message)s"))
logger.addHandler(handler)

# ─── Tool Definitions ─────────────────────────────────────────────────────────
TOOLS = [
    { "name": "execute_python",
      "description": "Run Python code and return stdout",
      "inputSchema": {
        "type": "object",
        "properties": {
          "code": {"type":"string","description":"Python code to execute"}
        },
        "required": ["code"]
      }
    },
    { "name":"install_package",
      "description":"Install a pip package",
      "inputSchema": {
        "type":"object",
        "properties": {
          "package":{"type":"string","description":"Name of package"}
        },
        "required":["package"]
      }
    },
    { "name":"list_files",
      "description":"List files in a directory",
      "inputSchema":{
        "type":"object",
        "properties":{
          "path":{"type":"string","default":".","description":"Directory path"}
        }
      }
    },
    { "name":"read_file",
      "description":"Read contents of a file",
      "inputSchema":{
        "type":"object",
        "properties":{
          "filepath":{"type":"string","description":"File path to read"}
        },
        "required":["filepath"]
      }
    },
    { "name":"write_file",
      "description":"Write content to a file",
      "inputSchema":{
        "type":"object",
        "properties":{
          "filepath":{"type":"string","description":"File to write"},
          "content":{"type":"string","description":"Content to write"}
        },
        "required":["filepath","content"]
      }
    }
]

# ─── Tool Execution Functions ─────────────────────────────────────────────────
def exec_py(code: str) -> str:
    from io import StringIO; import contextlib
    buf = StringIO()
    try:
        with contextlib.redirect_stdout(buf):
            exec(code, {})
        return buf.getvalue() or "✓ no output"
    except Exception as e:
        return f"Execution error: {e}"

def inst_pkg(pkg: str) -> str:
    p = subprocess.run([sys.executable, "-m", "pip", "install", pkg],
                       capture_output=True, text=True)
    return p.stdout or p.stderr

def safe_path(p: str) -> Path:
    target = SAFE_DIR / p
    if not target.resolve().is_relative_to(SAFE_DIR.resolve()):
        raise ValueError("Unsafe path.")
    return target

def ls_files(path: str = ".") -> str:
    p = safe_path(path)
    if not p.exists(): return f"Not found: {path}"
    if p.is_file(): return p.read_text()
    return "\n".join(
        f"{'📄' if it.is_file() else '📁'} {it.name}{'/' if it.is_dir() else ''}"
        for it in sorted(p.iterdir())
    ) or "— empty"

def rd_file(fp: str) -> str:
    return safe_path(fp).read_text()

def wr_file(fp: str, content: str) -> str:
    p = safe_path(fp)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content)
    return f"Written {p.name}"

# ─── Part 2: API Routes ────────────────────────────────────────────────────────
@app.get("/")
async def root():
    return {"message": "MCP Server Running", "version": "1.2"}

@app.post("/mcp/tools/list")
async def list_tools():
    return {"tools": TOOLS}

@app.post("/mcp/tools/call")
async def call_tool(req: Request):
    data = await req.json()
    name = data.get("name"); args = data.get("arguments", {})
    logger.info(f"CALL {name} {args!r}")
    try:
        if name=="execute_python":
            out = exec_py(args["code"])
        elif name=="install_package":
            out = inst_pkg(args["package"])
        elif name=="list_files":
            out = ls_files(args.get("path", "."))
        elif name=="read_file":
            out = rd_file(args["filepath"])
        elif name=="write_file":
            out = wr_file(args["filepath"], args["content"])
        else:
            raise Exception("Unknown tool")
        logger.info(f"RESULT {name} -> {out[:120]!r}")
        return {"content":[{"type":"text","text":out}], "isError":False}
    except Exception as e:
        logger.error(f"ERR {name} {e}")
        return {"content":[{"type":"text","text":f"Error: {e}"}], "isError":True}

# ─── Agent Memory (JSON) ──────────────────────────────────────────────────────
MEMORY_FILE = SAFE_DIR / "memory.json"
if not MEMORY_FILE.exists(): MEMORY_FILE.write_text("{}")

def load_memory():
    try: return json.loads(MEMORY_FILE.read_text())
    except: return {}

def save_memory(mem: dict):
    MEMORY_FILE.write_text(json.dumps(mem, indent=2))

@app.get("/memory/view")
async def memory_view():
    return load_memory()

@app.get("/memory/summary")
async def memory_summary():
    mem = load_memory()
    return {"summary": mem.get("summary")}

@app.post("/memory/update")
async def memory_update(req: Request):
    data = await req.json()
    mem = load_memory(); mem.update(data)
    save_memory(mem)
    logger.info(f"Memory updated: {data}")
    return mem

@app.post("/memory/clear")
async def memory_clear():
    save_memory({})
    logger.info("Memory cleared")
    return {}

# ─── OpenAPI & Tool Schema ────────────────────────────────────────────────────
@app.get("/mcp/schema")
async def mcp_schema():
    return app.openapi()

@app.get("/.well-known/tool-schema.json")
async def well_known_schema():
    return app.openapi()

# ─── Live Activity Dashboard ─────────────────────────────────────────────────
@app.get("/activity", response_class=HTMLResponse)
async def dash():
    return """<!DOCTYPE html><html><head><title>Live Activity</title>
<meta charset="utf-8"><meta name="description" content="Real-time MCP activity.">
<style>body{font:14px sans;}#log{background:#eee;padding:8px;height:70vh;overflow:auto;white-space:pre-wrap;border:1px solid #ccc}</style>
</head><body><h1>Live Activity</h1><div id="log">Connecting…</div>
<script>
const src = new EventSource('/activity/stream');
src.onmessage = e => {
  const log = document.getElementById('log');
  log.textContent += '\\n' + e.data;
  log.scrollTop = log.scrollHeight;
};
</script></body></html>"""

@app.get("/activity/stream")
async def stream():
    q = asyncio.Queue(); subscribers.add(q)
    async def gen():
        while True: yield {"data": await q.get()}
    return EventSourceResponse(gen())

# ─── Static UI: Tool Chat & Human Chat ────────────────────────────────────────
SITE = SAFE_DIR / "site"; SITE.mkdir(exist_ok=True)
# Tool Chat UI
(SITE/"toolchat.html").write_text(f"""<!DOCTYPE html><html><head><meta charset="utf-8"><title>Tool Chat</title>
<meta name="description" content="JSON UI for MCP tools.">
<style>body{{font:14px sans;max-width:700px;margin:1em auto}}textarea{{width:100%;height:6em}}</style>
</head><body>
<h1>Structured Tool Chat</h1>
<pre style="background:#f8f8f8;padding:8px;border:1px solid #ccc">{json.dumps(TOOLS,indent=2)}</pre>
<textarea id="req">{{"name":"execute_python","arguments":{{"code":"print(2+2)"}}}}</textarea><br>
<button onclick="run()">Run</button>
<pre id="out" style="background:#eef;border:1px solid #ccc;padding:8px;height:200px;overflow:auto"></pre>
<script>
async function run(){{
  const r = JSON.parse(document.getElementById('req').value);
  const res = await fetch('/mcp/tools/call', {{
    method:'POST', headers:{{'Content-Type':'application/json'}}, body: JSON.stringify(r)
  }});
  document.getElementById('out').textContent = JSON.stringify(await res.json(),null,2);
}}
</script></body></html>""", encoding="utf-8")
# Human Chat UI
(SITE/"chat.html").write_text("""<!DOCTYPE html><html><head><title>Chat</title>
<meta name="description" content="Human↔Agent chat log.">
<style>body{font:14px sans;max-width:600px;margin:1em auto}#chatlog{height:200px;overflow:auto;border:1px solid #ccc;padding:8px}</style>
</head><body>
<h2>Chat Log</h2><div id="chatlog"></div>
<input id="msg" placeholder="Type message..." style="width:80%"><button onclick="send()">Send</button>
<script>
async function send(){{
  let m=document.getElementById('msg').value; if(!m)return;
  await fetch('/site/chat/send',{method:'POST',headers:{{'Content-Type':'application/json'}},body:JSON.stringify({{message:m}})});
  document.getElementById('msg').value=''; load();
}}
async function load(){{
  document.getElementById('chatlog').textContent = await fetch('/site/chat/log').then(r=>r.text());
}}
load(); setInterval(load,3000);
</script></body></html>""", encoding="utf-8")

app.mount("/site", StaticFiles(directory=SITE), name="site")

@app.post("/site/chat/send")
async def post_chat(req: Request):
    msg = (await req.json()).get("message","").strip()
    if msg:
        line = f"[chat {time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}"
        with CHAT_LOG.open("a") as f: f.write(line+"\n")
        logger.info(f"CHAT {msg}")
    return {"ok": True}

@app.get("/site/chat/log")
async def get_chat():
    return CHAT_LOG.read_text()

@app.get("/site/log/serverlog")
async def get_slog():
    return SERVER_LOG.read_text()

# ─── Startup on port 8000 ─────────────────────────────────────────────────────
def start():
    Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info"), daemon=True).start()
    time.sleep(2)
    try:
        url = ngrok.connect(8000).public_url
        print("🔗 Public URL:", url)
    except Exception as e:
        print("⚠️ ngrok failed:", e)
        print("🔗 Local URL: http://localhost:8000")

# ─── Endpoint Information Printer ──────────────────────────────────────────────
def print_endpoints(public_url: str):
    """
    Print all the key URLs for both local and public (ngrok) access.
    Call this with the ngrok URL returned in start().
    """
    local = "http://localhost:8000"
    print("\n╔════════════════════════════════════════╗")
    print("║        MCP Server Endpoints           ║")
    print("╚════════════════════════════════════════╝\n")
    print(f"• Root Health Check:       {local}/")
    print(f"• Tools List (JSON):       {local}/mcp/tools/list")
    print(f"• Call a Tool (POST):      {local}/mcp/tools/call")
    print(f"• OpenAPI Schema:          {local}/mcp/schema")
    print(f"• Well-Known Schema:       {local}/.well-known/tool-schema.json")
    print(f"• Memory View:             {local}/memory/view")
    print(f"• Memory Summary:          {local}/memory/summary")
    print(f"• Memory Update (POST):    {local}/memory/update")
    print(f"• Memory Clear (POST):     {local}/memory/clear")
    print(f"• Activity Dashboard:      {local}/activity")
    print(f"• Raw Activity Stream:     {local}/activity/stream")
    print(f"• Tool Chat UI:            {local}/site/toolchat.html")
    print(f"• Human Chat UI:           {local}/site/chat.html")
    print(f"• Chat Log (text):         {local}/site/chat/log")
    print(f"• Server Log (text):       {local}/site/log/serverlog")
    print(f"• Download Files:          {local}/download/<relative-path>\n")
    print("Replace 'localhost:8000' with your ngrok URL for public access:")
    print(f"  e.g. Tool Chat UI:      {public_url}/site/toolchat.html\n")

# ─── STARTUP (with endpoint printing) ──────────────────────────────────────────
def start():
    Thread(
        target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info"),
        daemon=True
    ).start()
    time.sleep(2)
    try:
        url = ngrok.connect(8000).public_url
        print(f"🔗 Public URL: {url}")
        print_endpoints(url)
    except Exception as e:
        print(f"⚠️ ngrok failed: {e}")
        print_endpoints("http://localhost:8000")

# ─── Endpoint Information Printer ──────────────────────────────────────────────
def print_endpoints(public_url: str, port: int = 8000):
    """
    Print all the key URLs for both local (localhost) and public (ngrok) access.
    """
    local_base = f"http://localhost:{port}"
    print("\n╔════════════════════════════════════════╗")
    print("║        MCP Server Endpoints           ║")
    print("╚════════════════════════════════════════╝\n")
    entries = [
        ("Root Health Check",         "/"),
        ("Tools List (JSON)",         "/mcp/tools/list"),
        ("Call a Tool (POST)",        "/mcp/tools/call"),
        ("OpenAPI Schema",            "/mcp/schema"),
        ("Well-Known Schema",         "/.well-known/tool-schema.json"),
        ("Memory View",               "/memory/view"),
        ("Memory Summary",            "/memory/summary"),
        ("Memory Update (POST)",      "/memory/update"),
        ("Memory Clear (POST)",       "/memory/clear"),
        ("Activity Dashboard (UI)",   "/activity"),
        ("Activity Stream (SSE)",     "/activity/stream"),
        ("Tool Chat UI",              "/site/toolchat.html"),
        ("Human Chat UI",             "/site/chat.html"),
        ("Chat Log (text)",           "/site/chat/log"),
        ("Server Log (text)",         "/site/log/serverlog"),
        ("Download Files",            "/download/<relative-path>"),
    ]
    for name, path in entries:
        print(f"• {name:<25} Local: {local_base}{path}")
        print(f"  {'':25} Public: {public_url}{path}\n")

# ─── STARTUP (with endpoint printing) ──────────────────────────────────────────
def start():
    Thread(
        target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info"),
        daemon=True
    ).start()
    time.sleep(2)
    try:
        public_url = ngrok.connect(8000).public_url
        print(f"🔗 Public URL: {public_url}\n")
        print_endpoints(public_url, port=8000)
    except Exception as e:
        print(f"⚠️ ngrok failed: {e}")
        print_endpoints("http://localhost:8000", port=8000)

start()

"""
Your MCP server is running on port 8000. Here are the available endpoints and UIs:

• Root Health Check
  http://localhost:8000/
  → { "message": "MCP Server Running", "version": "1.2" }

• List Available Tools (with JSON schemas)
  http://localhost:8000/mcp/tools/list

• Call a Tool
  POST to http://localhost:8000/mcp/tools/call
  Body example:
    {
      "name": "execute_python",
      "arguments": { "code": "print(2+2)" }
    }

• OpenAPI / Tool Schema
  http://localhost:8000/mcp/schema
  http://localhost:8000/.well-known/tool-schema.json

• Agent Memory (JSON)
  • View full memory:    http://localhost:8000/memory/view
  • View summary only:   http://localhost:8000/memory/summary
  • Update memory:       POST http://localhost:8000/memory/update
  • Clear memory:        POST http://localhost:8000/memory/clear

• Live Activity Dashboard (SSE)
  http://localhost:8000/activity
  → Streams live log of tool calls, results, and chats

• Activity Stream (raw SSE)
  http://localhost:8000/activity/stream

• Structured Tool Chat UI
  http://localhost:8000/site/toolchat.html
  → JSON textarea to pick a tool, supply arguments, and run it

• Human Chat UI
  http://localhost:8000/site/chat.html
  → Simple chat log for human ↔ agent messaging

• Chat Log (text)
  http://localhost:8000/site/chat/log

• Server Log (text)
  http://localhost:8000/site/log/serverlog

• Download Any File in SAFE_DIR
  http://localhost:8000/download/<relative-path>
  e.g. http://localhost:8000/download/server_log.txt

— If you’re using the ngrok public URL (printed in the logs),
  simply replace “localhost:8000” with your ngrok address.
"""


Mounted at /content/drive


INFO:     Started server process [1584]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


🔗 Public URL: https://8122-35-186-146-131.ngrok-free.app


╔════════════════════════════════════════╗
║        MCP Server Endpoints           ║
╚════════════════════════════════════════╝

• Root Health Check         Local: http://localhost:8000/
                            Public: https://8122-35-186-146-131.ngrok-free.app/

• Tools List (JSON)         Local: http://localhost:8000/mcp/tools/list
                            Public: https://8122-35-186-146-131.ngrok-free.app/mcp/tools/list

• Call a Tool (POST)        Local: http://localhost:8000/mcp/tools/call
                            Public: https://8122-35-186-146-131.ngrok-free.app/mcp/tools/call

• OpenAPI Schema            Local: http://localhost:8000/mcp/schema
                            Public: https://8122-35-186-146-131.ngrok-free.app/mcp/schema

• Well-Known Schema         Local: http://localhost:8000/.well-known/tool-schema.json
                            Public: https://8122-35-186-146-131.ngrok-free.app/.well-known/too

'\nYour MCP server is running on port 8000. Here are the available endpoints and UIs:\n\n• Root Health Check\n  http://localhost:8000/\n  → { "message": "MCP Server Running", "version": "1.2" }\n\n• List Available Tools (with JSON schemas)\n  http://localhost:8000/mcp/tools/list\n\n• Call a Tool\n  POST to http://localhost:8000/mcp/tools/call\n  Body example:\n    {\n      "name": "execute_python",\n      "arguments": { "code": "print(2+2)" }\n    }\n\n• OpenAPI / Tool Schema\n  http://localhost:8000/mcp/schema\n  http://localhost:8000/.well-known/tool-schema.json\n\n• Agent Memory (JSON)\n  • View full memory:    http://localhost:8000/memory/view\n  • View summary only:   http://localhost:8000/memory/summary\n  • Update memory:       POST http://localhost:8000/memory/update\n  • Clear memory:        POST http://localhost:8000/memory/clear\n\n• Live Activity Dashboard (SSE)\n  http://localhost:8000/activity\n  → Streams live log of tool calls, results, and chats\n\n• Activity Stream (r

In [ ]:
# ╔════════════════════════════════════════════════════════════════════════════╗
# ║     🤖 Colab MCP Server + Tool Call UI + Live Dashboard + GDrive Mount    ║
# ║                                                                            ║
# ║  Self-contained Google Colab cell: installs deps, mounts Drive, defines   ║
# ║  the full FastAPI MCP server (with health, GET/POST tools list, memory    ║
# ║  CRUD, live dashboard, file download, chat UIs), generates Claude & OpenAI║
# ║  plugin manifests, and launches Uvicorn behind ngrok — all in one place.   ║
# ╚════════════════════════════════════════════════════════════════════════════╝

# ───────────────────────────────────────────────────────────────────────────────
# 1) Install & import core dependencies
# ───────────────────────────────────────────────────────────────────────────────
import subprocess, sys
subprocess.run([
    sys.executable, "-m", "pip", "install", "--quiet",
    "fastapi", "uvicorn", "pyngrok", "nest-asyncio", "requests"
], check=True)

# Ensure SSE support package is present
try:
    from sse_starlette.sse import EventSourceResponse
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "sse-starlette"], check=True)
    from sse_starlette.sse import EventSourceResponse

# ───────────────────────────────────────────────────────────────────────────────
# 2) Enable asyncio patching (needed in Colab) and mount Google Drive
# ───────────────────────────────────────────────────────────────────────────────
import nest_asyncio; nest_asyncio.apply()
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

# ───────────────────────────────────────────────────────────────────────────────
# 3) Set up a safe working directory & switch into it
# ───────────────────────────────────────────────────────────────────────────────
import os
from pathlib import Path
SAFE_DIR = Path("/content/drive/MyDrive/ColabMCPFiles")
SAFE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(SAFE_DIR)

# ───────────────────────────────────────────────────────────────────────────────
# 4) Configure ngrok for public tunneling
# ───────────────────────────────────────────────────────────────────────────────
from pyngrok import ngrok
# Replace with your actual ngrok auth token from https://dashboard.ngrok.com
ngrok.set_auth_token("2yuUKpiAlYXCez40i8UktLZ91s7_5Rv1GjG5HMFw5KKbk5sHz")

# ───────────────────────────────────────────────────────────────────────────────
# 5) Import FastAPI, Uvicorn, StaticFiles, logging, etc.
# ───────────────────────────────────────────────────────────────────────────────
import time, json, logging, asyncio
from threading import Thread
from fastapi import FastAPI, Request, HTTPException
from fastapi.responses import HTMLResponse, FileResponse
from fastapi.staticfiles import StaticFiles
import uvicorn

# ───────────────────────────────────────────────────────────────────────────────
# 6) Initialize FastAPI app & broadcast-capable logging
# ───────────────────────────────────────────────────────────────────────────────
app = FastAPI(title="Colab MCP Tool Server", version="1.2")
# Log files for server events and chat messages
SERVER_LOG = SAFE_DIR / "server_log.txt"; SERVER_LOG.touch()
CHAT_LOG   = SAFE_DIR / "chat_log.txt";   CHAT_LOG.touch()
# Set of asyncio queues to broadcast SSE messages
subscribers: set[asyncio.Queue] = set()

class BroadcastHandler(logging.Handler):
    def emit(self, record):
        msg = self.format(record)
        # Append to persistent server log file
        with SERVER_LOG.open("a") as f:
            f.write(msg + "\n")
        # Broadcast in real-time to any connected SSE clients
        for q in list(subscribers):
            try:
                asyncio.get_event_loop().call_soon_threadsafe(q.put_nowait, msg)
            except:
                pass

logger = logging.getLogger("mcp")
logger.setLevel(logging.INFO)
handler = BroadcastHandler()
handler.setFormatter(logging.Formatter("%(asctime)s %(message)s"))
logger.addHandler(handler)

# ───────────────────────────────────────────────────────────────────────────────
# 7) Define all available tools and their JSON schemas
# ───────────────────────────────────────────────────────────────────────────────
TOOLS = [
    {
        "name": "execute_python",
        "description": "Run Python code and return stdout",
        "inputSchema": {
            "type": "object",
            "properties": {
                "code": {"type":"string", "description":"Python code to execute"}
            },
            "required": ["code"]
        }
    },
    {
        "name": "install_package",
        "description": "Install a pip package",
        "inputSchema": {
            "type":"object",
            "properties": {
                "package": {"type":"string", "description":"Name of package"}
            },
            "required": ["package"]
        }
    },
    {
        "name": "list_files",
        "description": "List files in a directory",
        "inputSchema": {
            "type":"object",
            "properties": {
                "path": {"type":"string", "default":".", "description":"Directory path"}
            }
        }
    },
    {
        "name": "read_file",
        "description": "Read contents of a file",
        "inputSchema": {
            "type":"object",
            "properties": {
                "filepath": {"type":"string", "description":"File path to read"}
            },
            "required": ["filepath"]
        }
    },
    {
        "name": "write_file",
        "description": "Write content to a file",
        "inputSchema": {
            "type":"object",
            "properties": {
                "filepath": {"type":"string", "description":"File to write"},
                "content":  {"type":"string", "description":"Content to write"}
            },
            "required": ["filepath", "content"]
        }
    }
]

# ───────────────────────────────────────────────────────────────────────────────
# 8) Implement helper functions for each tool
# ───────────────────────────────────────────────────────────────────────────────
def exec_py(code: str) -> str:
    """Execute Python code and capture stdout."""
    from io import StringIO
    import contextlib
    buf = StringIO()
    try:
        with contextlib.redirect_stdout(buf):
            exec(code, {})
        return buf.getvalue() or "✓ no output"
    except Exception as e:
        return f"Execution error: {e}"

def inst_pkg(pkg: str) -> str:
    """Install a pip package and return pip output."""
    p = subprocess.run([sys.executable, "-m", "pip", "install", pkg],
                       capture_output=True, text=True)
    return p.stdout or p.stderr

def safe_path(p: str) -> Path:
    """Ensure file operations stay within SAFE_DIR."""
    target = SAFE_DIR / p
    if not target.resolve().is_relative_to(SAFE_DIR.resolve()):
        raise ValueError("Unsafe path.")
    return target

def ls_files(path: str = ".") -> str:
    """List files under a directory or read a file if path is file."""
    p = safe_path(path)
    if not p.exists(): return f"Not found: {path}"
    if p.is_file(): return p.read_text()
    return "\n".join(
        f"{'📄' if it.is_file() else '📁'} {it.name}{'/' if it.is_dir() else ''}"
        for it in sorted(p.iterdir())
    ) or "— empty"

def rd_file(fp: str) -> str:
    """Read and return the contents of a file."""
    return safe_path(fp).read_text()

def wr_file(fp: str, content: str) -> str:
    """Write content to a file and return confirmation."""
    p = safe_path(fp)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content)
    return f"Written {p.name}"

# ───────────────────────────────────────────────────────────────────────────────
# 9) Prepare agent memory (JSON) file & loader/saver
# ───────────────────────────────────────────────────────────────────────────────
MEMORY_FILE = SAFE_DIR / "memory.json"
if not MEMORY_FILE.exists():
    MEMORY_FILE.write_text("{}")

def load_memory():
    """Load memory dict from disk."""
    try:
        return json.loads(MEMORY_FILE.read_text())
    except:
        return {}

def save_memory(mem: dict):
    """Persist memory dict to disk."""
    MEMORY_FILE.write_text(json.dumps(mem, indent=2))

# ───────────────────────────────────────────────────────────────────────────────
# 10) Define all API routes
# ───────────────────────────────────────────────────────────────────────────────

# Root health check (used in tests)
@app.get("/")
async def root():
    return {"message": "MCP Server Running", "version": "1.2"}

# Extended health endpoint
@app.get("/health")
async def health_check():
    mem = load_memory()
    return {
        "status": "healthy",
        "version": "1.2",
        "tools_count": len(TOOLS),
        "memory_size": len(mem),
        "uptime": time.time()
    }

# Tools list (supports GET & POST)
@app.get("/mcp/tools/list")
@app.post("/mcp/tools/list")
async def list_tools():
    return {"tools": TOOLS}

# Invoke a tool
@app.post("/mcp/tools/call")
async def call_tool(req: Request):
    data = await req.json()
    name, args = data.get("name"), data.get("arguments", {})
    logger.info(f"CALL {name} {args!r}")
    try:
        if name == "execute_python":
            out = exec_py(args["code"])
        elif name == "install_package":
            out = inst_pkg(args["package"])
        elif name == "list_files":
            out = ls_files(args.get("path", "."))
        elif name == "read_file":
            out = rd_file(args["filepath"])
        elif name == "write_file":
            out = wr_file(args["filepath"], args["content"])
        else:
            raise Exception("Unknown tool")
        logger.info(f"RESULT {name} -> {repr(out)[:120]}")
        return {"content":[{"type":"text","text":out}], "isError":False}
    except Exception as e:
        logger.error(f"ERR {name} {e}")
        return {"content":[{"type":"text","text":f"Error: {e}"}], "isError":True}

# Agent memory endpoints
@app.get("/memory/view")
async def memory_view():
    return load_memory()

@app.get("/memory/summary")
async def memory_summary():
    return {"summary": load_memory().get("summary")}

@app.post("/memory/update")
async def memory_update(req: Request):
    data = await req.json()
    mem = load_memory(); mem.update(data)
    save_memory(mem)
    logger.info(f"Memory updated: {data}")
    return mem

@app.post("/memory/clear")
async def memory_clear():
    save_memory({})
    logger.info("Memory cleared")
    return {}

# OpenAPI & tool-schema endpoints
@app.get("/mcp/schema")
async def mcp_schema():
    return app.openapi()

@app.get("/.well-known/tool-schema.json")
async def well_known_schema():
    return app.openapi()

# Live Activity Dashboard (HTML + SSE)
@app.get("/activity", response_class=HTMLResponse)
async def activity_dashboard():
    return """
<!DOCTYPE html><html><head><title>Live Activity</title>
<meta charset="utf-8"><style>body{font:14px sans;}#log{background:#eee;
padding:8px;height:70vh;overflow:auto;white-space:pre-wrap;border:1px solid #ccc}
</style></head><body><h1>Live Activity</h1><div id="log">Connecting…</div>
<script>
const src = new EventSource('/activity/stream');
src.onmessage = e => {
  const log = document.getElementById('log');
  log.textContent += '\\n' + e.data;
  log.scrollTop = log.scrollHeight;
};
</script></body></html>
"""

@app.get("/activity/stream")
async def activity_stream():
    q = asyncio.Queue(); subscribers.add(q)
    async def gen():
        while True:
            yield {"data": await q.get()}
    return EventSourceResponse(gen())

# Static UI: Tool Chat & Human Chat
SITE = SAFE_DIR / "site"
SITE.mkdir(exist_ok=True)

# Structured Tool Chat UI (JSON-driven)
(SITE/"toolchat.html").write_text(f"""<!DOCTYPE html>
<html><head><meta charset="utf-8"><title>Tool Chat</title>
<style>body{{font:14px sans;max-width:700px;margin:1em auto}}textarea{{width:100%;height:6em}}</style>
</head><body>
<h1>Structured Tool Chat</h1>
<pre style="background:#f8f8f8;padding:8px;border:1px solid #ccc">
{json.dumps(TOOLS, indent=2)}
</pre>
<textarea id="req">{{"name":"execute_python","arguments":{{"code":"print(2+2)"}}}}</textarea><br>
<button onclick="run()">Run</button>
<pre id="out" style="background:#eef;border:1px solid #ccc;
padding:8px;height:200px;overflow:auto"></pre>
<script>
async function run() {{
  const r = JSON.parse(document.getElementById('req').value);
  const res = await fetch('/mcp/tools/call', {{
    method:'POST', headers:{{'Content-Type':'application/json'}}, body: JSON.stringify(r)
  }});
  document.getElementById('out').textContent = JSON.stringify(await res.json(), null, 2);
}}
</script>
</body></html>""", encoding="utf-8")

# Simple Human Chat UI
(SITE/"chat.html").write_text("""<!DOCTYPE html>
<html><head><title>Chat</title>
<style>body{font:14px sans;max-width:600px;margin:1em auto}
#chatlog{height:200px;overflow:auto;border:1px solid #ccc;padding:8px}</style>
</head><body>
<h2>Chat Log</h2><div id="chatlog"></div>
<input id="msg" placeholder="Type message..." style="width:80%"><button onclick="send()">Send</button>
<script>
async function send() {{
  let m = document.getElementById('msg').value; if(!m) return;
  await fetch('/site/chat/send', {{
    method:'POST', headers:{{'Content-Type':'application/json'}}, body:JSON.stringify({{message:m}})
  }});
  document.getElementById('msg').value=''; load();
}}
async function load() {{
  document.getElementById('chatlog').textContent =
    await fetch('/site/chat/log').then(r=>r.text());
}}
load(); setInterval(load,3000);
</script>
</body></html>""", encoding="utf-8")

# Mount the static UI directory
app.mount("/site", StaticFiles(directory=SITE), name="site")

# Chat endpoints for human UI
@app.post("/site/chat/send")
async def post_chat(req: Request):
    msg = (await req.json()).get("message","").strip()
    if msg:
        line = f"[chat {time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}"
        with CHAT_LOG.open("a") as f: f.write(line + "\n")
        logger.info(f"CHAT {msg}")
    return {"ok": True}

@app.get("/site/chat/log")
async def get_chat_log():
    return CHAT_LOG.read_text()

@app.get("/site/log/serverlog")
async def get_server_log():
    return SERVER_LOG.read_text()

# File download endpoint
@app.get("/download/{file_path:path}")
async def download_file(file_path: str):
    fp = safe_path(file_path)
    if not fp.exists() or not fp.is_file():
        raise HTTPException(status_code=404, detail="File not found")
    return FileResponse(fp)

# ───────────────────────────────────────────────────────────────────────────────
# 11) Function to start Uvicorn + ngrok, print endpoints, and return public URL
# ───────────────────────────────────────────────────────────────────────────────
def start_server() -> str:
    # Launch Uvicorn in a background thread
    Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info"), daemon=True).start()
    time.sleep(2)  # allow server to spin up
    tunnel = ngrok.connect(8000)
    public_url = tunnel.public_url
    # Print key URLs
    print(f"🔗 Public URL: {public_url}")
    print(f"• Root:            {public_url}/")
    print(f"• Health:          {public_url}/health")
    print(f"• Tools list:      {public_url}/mcp/tools/list")
    print(f"• Tool call:       {public_url}/mcp/tools/call")
    print(f"• Memory view:     {public_url}/memory/view")
    print(f"• Memory update:   {public_url}/memory/update")
    print(f"• Memory clear:    {public_url}/memory/clear")
    print(f"• Tool chat UI:    {public_url}/site/toolchat.html")
    print(f"• Human chat UI:   {public_url}/site/chat.html")
    print(f"• Download files:  {public_url}/download/<relative-path>")
    return public_url

# ───────────────────────────────────────────────────────────────────────────────
# 12) Launch the server and capture its public URL
# ───────────────────────────────────────────────────────────────────────────────
public_url = start_server()

# ───────────────────────────────────────────────────────────────────────────────
# 13) Generate Claude & OpenAI plugin manifests (using actual public_url)
# ───────────────────────────────────────────────────────────────────────────────

# Claude plugin manifest
CLAUDE_YAML = SAFE_DIR / "claude.yaml"
CLAUDE_YAML.write_text(f"""schema_version: "1.0"
name_for_human: "MCP Tool Server"
name_for_model: "mcp_tool_server"
description_for_human: |
  Interact with the Colab MCP Tool Server to:
    - List tools
    - Execute Python
    - Install packages
    - Manage files
    - Inspect/update memory
    - Health checks & uptime
description_for_model: |
  Plugin endpoints:
    • GET {public_url}/mcp/tools/list
    • POST {public_url}/mcp/tools/call
    • GET {public_url}/health
    • GET/POST memory endpoints
    • Schema: {public_url}/.well-known/tool-schema.json
    • Download: {public_url}/download/{{file_path}}
auth:
  type: none
api:
  type: openapi
  url: "{public_url}/.well-known/tool-schema.json"
  is_user_authenticated: false
logo_url: "{public_url}/logo.png"
contact_email: "support@your-domain.com"
legal_info_url: "https://your-domain.com/legal.html"
""", encoding="utf-8")
print(f"Wrote Claude manifest to: {CLAUDE_YAML}")

# OpenAI ChatGPT plugin manifest (ai-plugin.json)
AI_PLUGIN = SAFE_DIR / "ai-plugin.json"
AI_PLUGIN.write_text(json.dumps({
    "schema_version": "v1",
    "name_for_human": "MCP Tool Server",
    "name_for_model": "mcp_tool_server",
    "description_for_human": "API to execute Python, manage files, memory, health",
    "description_for_model": "Plugin to call MCP Tool Server endpoints",
    "auth": {"type": "none"},
    "api": {
        "type": "openapi",
        "url": f"{public_url}/openai.yaml",
        "is_user_authenticated": False
    },
    "logo_url": f"{public_url}/logo.png",
    "contact_email": "support@your-domain.com",
    "legal_info_url": "https://your-domain.com/legal.html"
}, indent=2), encoding="utf-8")
print(f"Wrote OpenAI ai-plugin.json to: {AI_PLUGIN}")

# OpenAI OpenAPI spec (openai.yaml)
OPENAPI_YAML = SAFE_DIR / "openai.yaml"
OPENAPI_YAML.write_text(f"""openapi: "3.0.1"
info:
  title: "MCP Tool Server API"
  version: "1.2"
  description: |
    Execute Python, install packages, manage files & memory, health checks.
servers:
  - url: "{public_url}"
paths:
  /mcp/tools/list:
    get:
      summary: "List all tools"
  /mcp/tools/call:
    post:
      summary: "Invoke a tool"
  /health:
    get:
      summary: "Health check"
  /memory/view:
    get:
      summary: "View memory"
  /memory/summary:
    get:
      summary: "Memory summary"
  /memory/update:
    post:
      summary: "Update memory"
  /memory/clear:
    post:
      summary: "Clear memory"
  /.well-known/tool-schema.json:
    get:
      summary: "Tool schema JSON"
  /download/{{file_path}}:
    get:
      summary: "Download a file"
""", encoding="utf-8")
print(f"Wrote OpenAPI spec to: {OPENAPI_YAML}")

# ───────────────────────────────────────────────────────────────────────────────
# ✅ Everything is now in place and running!                                   ┑
# • Check the printed URLs above to start interacting with your MCP server.    ┑
# • Plugin manifests written to claude.yaml, ai-plugin.json, and openai.yaml.  ┑
# ───────────────────────────────────────────────────────────────────────────────

Mounted at /content/drive


INFO:     Started server process [4209]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


🔗 Public URL: https://7842-35-198-225-173.ngrok-free.app
• Root:            https://7842-35-198-225-173.ngrok-free.app/
• Health:          https://7842-35-198-225-173.ngrok-free.app/health
• Tools list:      https://7842-35-198-225-173.ngrok-free.app/mcp/tools/list
• Tool call:       https://7842-35-198-225-173.ngrok-free.app/mcp/tools/call
• Memory view:     https://7842-35-198-225-173.ngrok-free.app/memory/view
• Memory update:   https://7842-35-198-225-173.ngrok-free.app/memory/update
• Memory clear:    https://7842-35-198-225-173.ngrok-free.app/memory/clear
• Tool chat UI:    https://7842-35-198-225-173.ngrok-free.app/site/toolchat.html
• Human chat UI:   https://7842-35-198-225-173.ngrok-free.app/site/chat.html
• Download files:  https://7842-35-198-225-173.ngrok-free.app/download/<relative-path>
Wrote Claude manifest to: /content/drive/MyDrive/ColabMCPFiles/claude.yaml
Wrote OpenAI ai-plugin.json to: /content/drive/MyDrive/ColabMCPFiles/ai-plugin.json
Wrote OpenAPI spec to: /cont

In [ ]:
import subprocess
import sys
import os
import json
import time
import logging
import asyncio
from threading import Thread
from pathlib import Path

from fastapi import FastAPI, Request, HTTPException
from fastapi.responses import HTMLResponse, FileResponse
from fastapi.staticfiles import StaticFiles
import uvicorn
from pyngrok import ngrok
from sse_starlette.sse import EventSourceResponse

# ─── Mute pyngrok logs ─────────────────────────────────────────────────────────
logging.getLogger("pyngrok").setLevel(logging.ERROR)

# ─── Mount Google Drive (if in Colab) ──────────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

# ─── Safe Working Directory ───────────────────────────────────────────────────
SAFE_DIR = Path("/content/drive/MyDrive/ColabMCPFiles")
SAFE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(SAFE_DIR)

# ─── FastAPI App & Logging ─────────────────────────────────────────────────────
app = FastAPI(title="Colab MCP Tool Server", version="1.2")
subscribers: set[asyncio.Queue] = set()
SERVER_LOG = SAFE_DIR / "server_log.txt"
SERVER_LOG.touch()
CHAT_LOG = SAFE_DIR / "chat_log.txt"
CHAT_LOG.touch()

class BroadcastHandler(logging.Handler):
    def emit(self, record):
        msg = self.format(record)
        with SERVER_LOG.open("a") as f:
            f.write(msg + "\n")
        for q in list(subscribers):
            try:
                asyncio.get_event_loop().call_soon_threadsafe(q.put_nowait, msg)
            except:
                pass

logger = logging.getLogger("mcp")
logger.setLevel(logging.INFO)
h = BroadcastHandler()
h.setFormatter(logging.Formatter("%(asctime)s %(message)s"))
logger.addHandler(h)

# ─── Tool Definitions ─────────────────────────────────────────────────────────
TOOLS = [
    {
        "name": "execute_python",
        "description": "Run Python code",
        "inputSchema": {
            "type": "object",
            "properties": {"code": {"type": "string", "description": "Python code to execute"}},
            "required": ["code"],
        },
    },
    {
        "name": "install_package",
        "description": "Install a pip package",
        "inputSchema": {
            "type": "object",
            "properties": {"package": {"type": "string", "description": "Package name"}},
            "required": ["package"],
        },
    },
    {
        "name": "list_files",
        "description": "List directory contents",
        "inputSchema": {
            "type": "object",
            "properties": {"path": {"type": "string", "description": "Directory path", "default": "."}},
        },
    },
    {
        "name": "read_file",
        "description": "Read file contents",
        "inputSchema": {
            "type": "object",
            "properties": {"filepath": {"type": "string", "description": "File to read"}},
            "required": ["filepath"],
        },
    },
    {
        "name": "write_file",
        "description": "Write to a file",
        "inputSchema": {
            "type": "object",
            "properties": {
                "filepath": {"type": "string", "description": "File to write"},
                "content": {"type": "string", "description": "Text content"},
            },
            "required": ["filepath", "content"],
        },
    },
]

# ─── Helper Functions ──────────────────────────────────────────────────────────
def exec_py(code: str) -> str:
    from io import StringIO
    import contextlib

    buf = StringIO()
    try:
        with contextlib.redirect_stdout(buf):
            exec(code, {})
        return buf.getvalue() or "✓ no output"
    except Exception as e:
        return f"Execution error: {e}"

def inst_pkg(pkg: str) -> str:
    p = subprocess.run([sys.executable, "-m", "pip", "install", pkg], capture_output=True, text=True)
    return p.stdout or p.stderr

def safe_path(p: str) -> Path:
    t = SAFE_DIR / p
    if not t.resolve().is_relative_to(SAFE_DIR.resolve()):
        raise ValueError("Unsafe path")
    return t

def ls_files(path: str = ".") -> str:
    p = safe_path(path)
    if not p.exists():
        return f"Not found: {path}"
    if p.is_file():
        return p.read_text()
    return "\n".join(
        f"{'📄' if f.is_file() else '📁'} {f.name}{'/' if f.is_dir() else ''}"
        for f in sorted(p.iterdir())
    ) or "— empty"

def rd_file(fp: str) -> str:
    return safe_path(fp).read_text()

def wr_file(fp: str, content: str) -> str:
    p = safe_path(fp)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content)
    return f"Written {p.name}"

# ─── API Endpoints ─────────────────────────────────────────────────────────────
@app.get("/")
async def root():
    return {"message": "MCP Server Running", "version": "1.2"}

@app.get("/mcp/tools/list")
@app.post("/mcp/tools/list")
async def list_tools():
    return {"tools": TOOLS}

@app.post("/mcp/tools/call")
async def call_tool(req: Request):
    data = await req.json()
    name = data.get("name")
    args = data.get("arguments", {})
    logger.info(f"CALL {name} {args!r}")
    try:
        if name == "execute_python":
            out = exec_py(args["code"])
        elif name == "install_package":
            out = inst_pkg(args["package"])
        elif name == "list_files":
            out = ls_files(args.get("path", "."))
        elif name == "read_file":
            out = rd_file(args["filepath"])
        elif name == "write_file":
            out = wr_file(args["filepath"], args["content"])
        else:
            raise Exception("Unknown tool")
        logger.info(f"RESULT {name} -> {out[:120]!r}")
        return {"content": [{"type": "text", "text": out}], "isError": False}
    except Exception as e:
        logger.error(f"ERR {name} {e}")
        return {"content": [{"type": "text", "text": f"Error: {e}"}], "isError": True}

@app.get("/mcp/schema")
async def mcp_schema():
    return app.openapi()

@app.get("/.well-known/tool-schema.json")
async def well_known_schema():
    return app.openapi()

@app.get("/memory/view")
async def memory_view():
    mem = json.loads((SAFE_DIR / "memory.json").read_text()) if (SAFE_DIR / "memory.json").exists() else {}
    return mem

@app.get("/memory/summary")
async def memory_summary():
    mem = json.loads((SAFE_DIR / "memory.json").read_text()) if (SAFE_DIR / "memory.json").exists() else {}
    return {"summary": mem.get("summary")}

@app.post("/memory/update")
async def memory_update(req: Request):
    data = await req.json()
    mem_path = SAFE_DIR / "memory.json"
    mem = json.loads(mem_path.read_text()) if mem_path.exists() else {}
    mem.update(data)
    mem_path.write_text(json.dumps(mem, indent=2))
    logger.info(f"Memory updated: {data}")
    return mem

@app.post("/memory/clear")
async def memory_clear():
    mem_path = SAFE_DIR / "memory.json"
    mem_path.write_text("{}")
    logger.info("Memory cleared")
    return {}

@app.get("/activity", response_class=HTMLResponse)
async def activity_dashboard():
    return """<!DOCTYPE html>
<html><head><title>Live Activity</title><meta charset="utf-8"></head>
<body><h1>Live Activity</h1><div id="log"></div>
<script>
let src = new EventSource('/activity/stream');
src.onmessage = e => {
  let l = document.getElementById('log');
  l.textContent += `\n${e.data}`;
  l.scrollTop = l.scrollHeight;
};
</script></body></html>"""

@app.get("/activity/stream")
async def activity_stream():
    q = asyncio.Queue()
    subscribers.add(q)
    async def gen():
        while True:
            yield {"data": await q.get()}
    return EventSourceResponse(gen())

# ─── Static UIs ────────────────────────────────────────────────────────────────
SITE = SAFE_DIR / "site"
SITE.mkdir(exist_ok=True)

# Tool Chat UI (with skip-browser-warning header in fetch)
toolchat_html = f"""<!DOCTYPE html>
<html><head><meta charset="utf-8"><title>Tool Chat</title></head>
<body>
  <h1>Structured Tool Chat</h1>
  <pre id="tools"></pre>
  <textarea id="req" style="width:100%;height:6em"></textarea><br>
  <button onclick="run()">Run</button>
  <pre id="out" style="background:#eef;border:1px solid #ccc;padding:8px;height:200px;overflow:auto"></pre>

  <script>
    const TOOLS = {json.dumps(TOOLS, indent=2)};
    document.getElementById('tools').textContent = JSON.stringify(TOOLS, null, 2);
    document.getElementById('req').value = JSON.stringify({{name:TOOLS[0].name,arguments:{{}}}}, null, 2);

    async function run() {{
      const r = JSON.parse(document.getElementById('req').value);
      const res = await fetch('/mcp/tools/call', {{
        method: 'POST',
        headers: {{
          'Content-Type': 'application/json',
          'ngrok-skip-browser-warning': 'true'
        }},
        body: JSON.stringify(r)
      }});
      const data = await res.json();
      document.getElementById('out').textContent = JSON.stringify(data, null, 2);
    }}
  </script>
</body></html>
"""
(SITE / "toolchat.html").write_text(toolchat_html)

# Human Chat UI (with skip-browser-warning header in fetch)
chat_html = """<!DOCTYPE html>
<html><head><meta charset="utf-8"><title>Chat</title></head>
<body>
  <h2>Chat Log</h2>
  <div id="chatlog" style="height:200px;overflow:auto;border:1px solid #ccc;padding:8px"></div>
  <input id="msg" placeholder="Type message..." style="width:80%">
  <button onclick="send()">Send</button>

  <script>
    async function send() {{
      const msg = document.getElementById('msg').value;
      if (!msg) return;
      await fetch('/site/chat/send', {{
        method: 'POST',
        headers: {{
          'Content-Type': 'application/json',
          'ngrok-skip-browser-warning': 'true'
        }},
        body: JSON.stringify({{message: msg}})
      }});
      document.getElementById('msg').value = '';
      load();
    }}

    async function load() {{
      const res = await fetch('/site/chat/log', {{
        headers: {{'ngrok-skip-browser-warning': 'true'}}
      }});
      document.getElementById('chatlog').textContent = await res.text();
    }}

    load(); setInterval(load, 3000);
  </script>
</body></html>
"""
(SITE / "chat.html").write_text(chat_html)

app.mount("/site", StaticFiles(directory=SITE), name="site")

@app.post("/site/chat/send")
async def post_chat(req: Request):
    msg = (await req.json()).get("message", "").strip()
    if msg:
        line = f"[chat {time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}"
        with CHAT_LOG.open("a") as f:
            f.write(line + "\n")
        logger.info(f"CHAT {msg}")
    return {"ok": True}

@app.get("/site/chat/log")
async def get_chat_log():
    return CHAT_LOG.read_text()

@app.get("/site/log/serverlog")
async def get_server_log():
    return SERVER_LOG.read_text()

# ─── Startup ──────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    # Open ngrok tunnel silently
    ngrok.connect(8000)
    # Start Uvicorn in the main thread
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

ModuleNotFoundError: No module named 'pyngrok'

In [ ]:
# ╔════════════════════════════════════════════════════════════════════════════╗
# ║     🤖 Colab MCP Server + Tool Call UI + Live Dashboard + GDrive Mount    ║
# ║                                                                            ║
# ║  Self-contained Google Colab cell: installs deps, mounts Drive, defines   ║
# ║  the full FastAPI MCP server (with health, GET/POST tools list, memory    ║
# ║  CRUD, live dashboard, file download, chat UIs), generates Claude & OpenAI║
# ║  plugin manifests, and launches Uvicorn behind ngrok — all in one place.   ║
# ╚════════════════════════════════════════════════════════════════════════════╝

# ───────────────────────────────────────────────────────────────────────────────
# 1) Install everything (using Colab magic so imports see them immediately)
# ───────────────────────────────────────────────────────────────────────────────
!pip install --quiet fastapi uvicorn pyngrok nest-asyncio requests sse-starlette

# ───────────────────────────────────────────────────────────────────────────────
# 2) Enable asyncio patching (needed in Colab) and mount Google Drive
# ───────────────────────────────────────────────────────────────────────────────
import nest_asyncio; nest_asyncio.apply()
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

# ───────────────────────────────────────────────────────────────────────────────
# 3) Set up safe working dir & switch into it
# ───────────────────────────────────────────────────────────────────────────────
import os
from pathlib import Path
SAFE_DIR = Path("/content/drive/MyDrive/ColabMCPFiles")
SAFE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(SAFE_DIR)

# ───────────────────────────────────────────────────────────────────────────────
# 4) Configure ngrok for public tunneling
# ───────────────────────────────────────────────────────────────────────────────
from pyngrok import ngrok
# ← replace with your own token from https://dashboard.ngrok.com/get-started
ngrok.set_auth_token("2yuUKpiAlYXCez40i8UktLZ91s7_5Rv1GjG5HMFw5KKbk5sHz")

# ───────────────────────────────────────────────────────────────────────────────
# 5) Import FastAPI, Uvicorn, SSE, logging, etc.
# ───────────────────────────────────────────────────────────────────────────────
import time, json, logging, asyncio
from threading import Thread
from fastapi import FastAPI, Request, HTTPException
from fastapi.responses import HTMLResponse, FileResponse
from fastapi.staticfiles import StaticFiles
import uvicorn
from sse_starlette.sse import EventSourceResponse

# ───────────────────────────────────────────────────────────────────────────────
# 6) Initialize FastAPI app & broadcast-capable logging
# ───────────────────────────────────────────────────────────────────────────────
app = FastAPI(title="Colab MCP Tool Server", version="1.2")
SERVER_LOG = SAFE_DIR / "server_log.txt"; SERVER_LOG.touch()
CHAT_LOG   = SAFE_DIR / "chat_log.txt";   CHAT_LOG.touch()
subscribers: set[asyncio.Queue] = set()

class BroadcastHandler(logging.Handler):
    def emit(self, record):
        msg = self.format(record)
        with SERVER_LOG.open("a") as f:
            f.write(msg + "\n")
        for q in list(subscribers):
            try:
                asyncio.get_event_loop().call_soon_threadsafe(q.put_nowait, msg)
            except:
                pass

logger = logging.getLogger("mcp")
logger.setLevel(logging.INFO)
handler = BroadcastHandler()
handler.setFormatter(logging.Formatter("%(asctime)s %(message)s"))
logger.addHandler(handler)

# ───────────────────────────────────────────────────────────────────────────────
# 7) Define tools & JSON schemas
# ───────────────────────────────────────────────────────────────────────────────
TOOLS = [
    {
        "name":"execute_python",
        "description":"Run Python code and return stdout",
        "inputSchema":{
            "type":"object",
            "properties":{
                "code":{"type":"string","description":"Python code to execute"}
            },
            "required":["code"]
        }
    },
    {
        "name":"install_package",
        "description":"Install a pip package",
        "inputSchema":{
            "type":"object",
            "properties":{
                "package":{"type":"string","description":"Name of package"}
            },
            "required":["package"]
        }
    },
    {
        "name":"list_files",
        "description":"List files in a directory",
        "inputSchema":{
            "type":"object",
            "properties":{
                "path":{"type":"string","default":".","description":"Directory path"}
            }
        }
    },
    {
        "name":"read_file",
        "description":"Read contents of a file",
        "inputSchema":{
            "type":"object",
            "properties":{
                "filepath":{"type":"string","description":"File path to read"}
            },
            "required":["filepath"]
        }
    },
    {
        "name":"write_file",
        "description":"Write content to a file",
        "inputSchema":{
            "type":"object",
            "properties":{
                "filepath":{"type":"string","description":"File to write"},
                "content":{"type":"string","description":"Content to write"}
            },
            "required":["filepath","content"]
        }
    }
]

# ───────────────────────────────────────────────────────────────────────────────
# 8) Tool implementations
# ───────────────────────────────────────────────────────────────────────────────
def exec_py(code: str) -> str:
    from io import StringIO
    import contextlib
    buf = StringIO()
    try:
        with contextlib.redirect_stdout(buf):
            exec(code, {})
        return buf.getvalue() or "✓ no output"
    except Exception as e:
        return f"Execution error: {e}"

def inst_pkg(pkg: str) -> str:
    p = subprocess.run([sys.executable, "-m", "pip", "install", pkg],
                       capture_output=True, text=True)
    return p.stdout or p.stderr

def safe_path(p: str) -> Path:
    target = SAFE_DIR / p
    if not target.resolve().is_relative_to(SAFE_DIR.resolve()):
        raise ValueError("Unsafe path.")
    return target

def ls_files(path: str=".") -> str:
    p = safe_path(path)
    if not p.exists(): return f"Not found: {path}"
    if p.is_file(): return p.read_text()
    return "\n".join(
        f"{'📄' if it.is_file() else '📁'} {it.name}{'/' if it.is_dir() else ''}"
        for it in sorted(p.iterdir())
    ) or "— empty"

def rd_file(fp: str) -> str:
    return safe_path(fp).read_text()

def wr_file(fp: str, content: str) -> str:
    p = safe_path(fp)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content)
    return f"Written {p.name}"

# ───────────────────────────────────────────────────────────────────────────────
# 9) Agent memory (JSON) helpers
# ───────────────────────────────────────────────────────────────────────────────
MEMORY_FILE = SAFE_DIR / "memory.json"
if not MEMORY_FILE.exists(): MEMORY_FILE.write_text("{}")

def load_memory():
    try: return json.loads(MEMORY_FILE.read_text())
    except: return {}

def save_memory(mem: dict):
    MEMORY_FILE.write_text(json.dumps(mem, indent=2))

# ───────────────────────────────────────────────────────────────────────────────
# 10) API routes
# ───────────────────────────────────────────────────────────────────────────────
@app.get("/")
async def root():
    return {"message":"MCP Server Running","version":"1.2"}

@app.get("/health")
async def health_check():
    mem = load_memory()
    return {
        "status":"healthy",
        "version":"1.2",
        "tools_count":len(TOOLS),
        "memory_size":len(mem),
        "uptime":time.time()
    }

@app.get("/mcp/tools/list")
@app.post("/mcp/tools/list")
async def list_tools():
    return {"tools":TOOLS}

@app.post("/mcp/tools/call")
async def call_tool(req: Request):
    data = await req.json()
    name, args = data.get("name"), data.get("arguments",{})
    logger.info(f"CALL {name} {args!r}")
    try:
        if name=="execute_python": out = exec_py(args["code"])
        elif name=="install_package": out = inst_pkg(args["package"])
        elif name=="list_files": out = ls_files(args.get("path","."))
        elif name=="read_file": out = rd_file(args["filepath"])
        elif name=="write_file": out = wr_file(args["filepath"], args["content"])
        else: raise Exception("Unknown tool")
        logger.info(f"RESULT {name} -> {repr(out)[:120]}")
        return {"content":[{"type":"text","text":out}],"isError":False}
    except Exception as e:
        logger.error(f"ERR {name} {e}")
        return {"content":[{"type":"text","text":f"Error: {e}"}],"isError":True}

@app.get("/memory/view")
async def memory_view():    return load_memory()

@app.get("/memory/summary")
async def memory_summary():return {"summary":load_memory().get("summary")}

@app.post("/memory/update")
async def memory_update(req: Request):
    data = await req.json()
    mem = load_memory(); mem.update(data); save_memory(mem)
    logger.info(f"Memory updated: {data}")
    return mem

@app.post("/memory/clear")
async def memory_clear():
    save_memory({}); logger.info("Memory cleared"); return {}

@app.get("/mcp/schema")
async def mcp_schema():     return app.openapi()

@app.get("/.well-known/tool-schema.json")
async def well_known_schema(): return app.openapi()

@app.get("/activity", response_class=HTMLResponse)
async def activity_dashboard():
    return """<!DOCTYPE html><html><head><title>Live Activity</title>
<meta charset="utf-8"><style>body{font:14px sans;}#log{background:#eee;
padding:8px;height:70vh;overflow:auto;white-space:pre-wrap;border:1px solid #ccc}
</style></head><body><h1>Live Activity</h1><div id="log">Connecting…</div>
<script>
const src=new EventSource('/activity/stream');
src.onmessage=e=>{const log=document.getElementById('log');
log.textContent+='\\n'+e.data;log.scrollTop=log.scrollHeight;};
</script></body></html>"""

@app.get("/activity/stream")
async def activity_stream():
    q=asyncio.Queue(); subscribers.add(q)
    async def gen():
        while True: yield {"data":await q.get()}
    return EventSourceResponse(gen())

# Static UI folder
SITE = SAFE_DIR/"site"; SITE.mkdir(exist_ok=True)
# Tool chat
(SITE/"toolchat.html").write_text(f"""<!DOCTYPE html><html><head><meta charset="utf-8"><title>Tool Chat</title>
<style>body{{font:14px sans;max-width:700px;margin:1em auto}}textarea{{width:100%;height:6em}}</style>
</head><body><h1>Structured Tool Chat</h1><pre style="background:#f8f8f8;padding:8px;border:1px solid #ccc">
{json.dumps(TOOLS,indent=2)}
</pre><textarea id="req">{{"name":"execute_python","arguments":{{"code":"print(2+2)"}}}}</textarea><br>
<button onclick="run()">Run</button><pre id="out" style="background:#eef;border:1px solid #ccc;
padding:8px;height:200px;overflow:auto"></pre><script>
async function run(){{const r=JSON.parse(document.getElementById('req').value);
const res=await fetch('/mcp/tools/call',{{method:'POST',headers:{{'Content-Type':'application/json'}},body:JSON.stringify(r)}});
document.getElementById('out').textContent=JSON.stringify(await res.json(),null,2);}}
</script></body></html>""",encoding="utf-8")
# Human chat
(SITE/"chat.html").write_text("""<!DOCTYPE html><html><head><title>Chat</title>
<style>body{font:14px sans;max-width:600px;margin:1em auto}#chatlog{height:200px;overflow:auto;border:1px solid #ccc;padding:8px}</style>
</head><body><h2>Chat Log</h2><div id="chatlog"></div>
<input id="msg" placeholder="Type message..." style="width:80%"><button onclick="send()">Send</button>
<script>
async function send(){{let m=document.getElementById('msg').value; if(!m) return;
await fetch('/site/chat/send',{method:'POST',headers:{{'Content-Type':'application/json'}},body:JSON.stringify({message:m})});
document.getElementById('msg').value='';load();}}
async function load(){{document.getElementById('chatlog').textContent=
await fetch('/site/chat/log').then(r=>r.text());}}
load();setInterval(load,3000);
</script></body></html>""",encoding="utf-8")
app.mount("/site",StaticFiles(directory=SITE),name="site")

@app.post("/site/chat/send")
async def post_chat(req: Request):
    msg=(await req.json()).get("message","").strip()
    if msg:
        line=f"[chat {time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}"
        with CHAT_LOG.open("a") as f: f.write(line+"\n")
        logger.info(f"CHAT {msg}")
    return {"ok":True}

@app.get("/site/chat/log")
async def get_chat_log(): return CHAT_LOG.read_text()
@app.get("/site/log/serverlog")
async def get_server_log(): return SERVER_LOG.read_text()

@app.get("/download/{file_path:path}")
async def download_file(file_path: str):
    fp=safe_path(file_path)
    if not fp.exists() or not fp.is_file():
        raise HTTPException(404,"File not found")
    return FileResponse(fp)

# ───────────────────────────────────────────────────────────────────────────────
# 11) Start Uvicorn + ngrok, print endpoints, return public_url
# ───────────────────────────────────────────────────────────────────────────────
def start_server() -> str:
    Thread(target=lambda: uvicorn.run(app,host="0.0.0.0",port=8000,log_level="info"),daemon=True).start()
    time.sleep(2)
    public_url = ngrok.connect(8000).public_url
    print(f"🔗 Public URL: {public_url}")
    for name,path in [
        ("Root", "/"),("Health","/health"),
        ("Tools list","/mcp/tools/list"),("Tool call","/mcp/tools/call"),
        ("Memory view","/memory/view"),("Memory update","/memory/update"),
        ("Memory clear","/memory/clear"),("Tool chat UI","/site/toolchat.html"),
        ("Human chat UI","/site/chat.html"),("Download","/download/<relative-path>")
    ]:
        print(f"• {name:<13} {public_url}{path}")
    return public_url

public_url = start_server()

# ───────────────────────────────────────────────────────────────────────────────
# 12) Generate Claude & OpenAI plugin manifests using actual public_url
# ───────────────────────────────────────────────────────────────────────────────
CLAUDE_YAML = SAFE_DIR/"claude.yaml"
CLAUDE_YAML.write_text(f"""schema_version: "1.0"
name_for_human: "MCP Tool Server"
name_for_model: "mcp_tool_server"
description_for_human: |
  Interact with the Colab MCP Tool Server to:
    - List tools
    - Execute Python
    - Install packages
    - Manage files
    - Inspect/update memory
    - Health checks & uptime
description_for_model: |
  Plugin endpoints:
    • GET {public_url}/mcp/tools/list
    • POST {public_url}/mcp/tools/call
    • GET {public_url}/health
    • Memory: /memory/view, /memory/summary, /memory/update, /memory/clear
    • Schema: {public_url}/.well-known/tool-schema.json
    • Download: {public_url}/download/{{file_path}}
auth:
  type: none
api:
  type: openapi
  url: "{public_url}/.well-known/tool-schema.json"
  is_user_authenticated: false
logo_url: "{public_url}/logo.png"
contact_email: "support@your-domain.com"
legal_info_url: "https://your-domain.com/legal.html"
""",encoding="utf-8")
print(f"Wrote Claude manifest to: {CLAUDE_YAML}")

AI_PLUGIN = SAFE_DIR/"ai-plugin.json"
AI_PLUGIN.write_text(json.dumps({
    "schema_version":"v1",
    "name_for_human":"MCP Tool Server",
    "name_for_model":"mcp_tool_server",
    "description_for_human":"API to execute Python, manage files, memory and health",
    "description_for_model":"Plugin to call MCP Tool Server endpoints",
    "auth":{"type":"none"},
    "api":{"type":"openapi","url":f"{public_url}/openai.yaml","is_user_authenticated":False},
    "logo_url":f"{public_url}/logo.png",
    "contact_email":"support@your-domain.com",
    "legal_info_url":"https://your-domain.com/legal.html"
},indent=2),encoding="utf-8")
print(f"Wrote OpenAI ai-plugin.json to: {AI_PLUGIN}")

OPENAPI_YAML = SAFE_DIR/"openai.yaml"
OPENAPI_YAML.write_text(f"""openapi: "3.0.1"
info:
  title: "MCP Tool Server API"
  version: "1.2"
  description: |
    Execute Python, install packages, manage files & memory, health checks.
servers:
  - url: "{public_url}"
paths:
  /mcp/tools/list:
    get:
      summary: "List all tools"
  /mcp/tools/call:
    post:
      summary: "Invoke a tool"
  /health:
    get:
      summary: "Health check"
  /memory/view:
    get:
      summary: "View memory"
  /memory/summary:
    get:
      summary: "Memory summary"
  /memory/update:
    post:
      summary: "Update memory"
  /memory/clear:
    post:
      summary: "Clear memory"
  /.well-known/tool-schema.json:
    get:
      summary: "Tool schema JSON"
  /download/{{file_path}}:
    get:
      summary: "Download a file"
""",encoding="utf-8")
print(f"Wrote OpenAPI spec to: {OPENAPI_YAML}")

# ───────────────────────────────────────────────────────────────────────────────
# ✅ All set! Use the printed URLs above to interact with your MCP server.
# ───────────────────────────────────────────────────────────────────────────────

Mounted at /content/drive


INFO:     Started server process [1632]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


🔗 Public URL: https://abaa-34-34-75-175.ngrok-free.app
• Root          https://abaa-34-34-75-175.ngrok-free.app/
• Health        https://abaa-34-34-75-175.ngrok-free.app/health
• Tools list    https://abaa-34-34-75-175.ngrok-free.app/mcp/tools/list
• Tool call     https://abaa-34-34-75-175.ngrok-free.app/mcp/tools/call
• Memory view   https://abaa-34-34-75-175.ngrok-free.app/memory/view
• Memory update https://abaa-34-34-75-175.ngrok-free.app/memory/update
• Memory clear  https://abaa-34-34-75-175.ngrok-free.app/memory/clear
• Tool chat UI  https://abaa-34-34-75-175.ngrok-free.app/site/toolchat.html
• Human chat UI https://abaa-34-34-75-175.ngrok-free.app/site/chat.html
• Download      https://abaa-34-34-75-175.ngrok-free.app/download/<relative-path>
Wrote Claude manifest to: /content/drive/MyDrive/ColabMCPFiles/claude.yaml
Wrote OpenAI ai-plugin.json to: /content/drive/MyDrive/ColabMCPFiles/ai-plugin.json
Wrote OpenAPI spec to: /content/drive/MyDrive/ColabMCPFiles/openai.yaml


In [ ]:
# ╔════════════════════════════════════════════════════════════════════════════╗
# ║     🤖 Colab MCP Server + Tool Call UI + Live Dashboard + GDrive Mount    ║
# ║                                                                            ║
# ║  Self-contained Google Colab cell: installs deps, mounts Drive, defines   ║
# ║  the full FastAPI MCP server (with health, GET/POST tools list, memory    ║
# ║  CRUD, live dashboard, file download, chat UIs), generates Claude & OpenAI║
# ║  plugin manifests, and launches Uvicorn behind ngrok — all in one place.   ║
# ╚════════════════════════════════════════════════════════════════════════════╝

# ───────────────────────────────────────────────────────────────────────────────
# 1) Install everything (using Colab magic so imports see them immediately)
# ───────────────────────────────────────────────────────────────────────────────
!pip install --quiet fastapi uvicorn pyngrok nest-asyncio requests sse-starlette

# ───────────────────────────────────────────────────────────────────────────────
# 2) Enable asyncio patching (needed in Colab) and mount Google Drive
# ───────────────────────────────────────────────────────────────────────────────
import nest_asyncio; nest_asyncio.apply()
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

# ───────────────────────────────────────────────────────────────────────────────
# 3) Set up safe working dir & switch into it
# ───────────────────────────────────────────────────────────────────────────────
import os
from pathlib import Path
SAFE_DIR = Path("/content/drive/MyDrive/ColabMCPFiles")
SAFE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(SAFE_DIR)

# ───────────────────────────────────────────────────────────────────────────────
# 4) Configure ngrok for public tunneling
# ───────────────────────────────────────────────────────────────────────────────
from pyngrok import ngrok
# ← replace with your own token from https://dashboard.ngrok.com/get-started/your-authtoken
ngrok.set_auth_token("2yuUKpiAlYXCez40i8UktLZ91s7_5Rv1GjG5HMFw5KKbk5sHz")

# ───────────────────────────────────────────────────────────────────────────────
# 5) Import FastAPI, Uvicorn, SSE, logging, CORS, etc.
# ───────────────────────────────────────────────────────────────────────────────
import time, json, logging, asyncio, subprocess, sys
from threading import Thread
from fastapi import FastAPI, Request, HTTPException
from fastapi.responses import HTMLResponse, FileResponse
from fastapi.staticfiles import StaticFiles
from fastapi.middleware.cors import CORSMiddleware
import uvicorn
from sse_starlette.sse import EventSourceResponse

# ───────────────────────────────────────────────────────────────────────────────
# 6) Initialize FastAPI app & add CORS for Claude.ai
# ───────────────────────────────────────────────────────────────────────────────
app = FastAPI(title="Colab MCP Tool Server", version="1.2")

# → **Absolute Minimum Requirement #4: HTTPS + CORS**
app.add_middleware(
    CORSMiddleware,
    allow_origins=["https://claude.ai"],
    allow_methods=["POST"],
    allow_headers=["*"],
)

SERVER_LOG = SAFE_DIR / "server_log.txt"; SERVER_LOG.touch()
CHAT_LOG   = SAFE_DIR / "chat_log.txt";   CHAT_LOG.touch()
subscribers: set[asyncio.Queue] = set()

class BroadcastHandler(logging.Handler):
    def emit(self, record):
        msg = self.format(record)
        with SERVER_LOG.open("a") as f:
            f.write(msg + "\n")
        for q in list(subscribers):
            try:
                asyncio.get_event_loop().call_soon_threadsafe(q.put_nowait, msg)
            except:
                pass

logger = logging.getLogger("mcp")
logger.setLevel(logging.INFO)
handler = BroadcastHandler()
handler.setFormatter(logging.Formatter("%(asctime)s %(message)s"))
logger.addHandler(handler)

# ───────────────────────────────────────────────────────────────────────────────
# 7) Define tools & JSON schemas
# ───────────────────────────────────────────────────────────────────────────────
TOOLS = [
    {
        "name":"execute_python",
        "description":"Run Python code and return stdout",
        "inputSchema":{
            "type":"object",
            "properties":{
                "code":{"type":"string","description":"Python code to execute"}
            },
            "required":["code"]
        }
    },
    {
        "name":"install_package",
        "description":"Install a pip package",
        "inputSchema":{
            "type":"object",
            "properties":{
                "package":{"type":"string","description":"Name of package"}
            },
            "required":["package"]
        }
    },
    {
        "name":"list_files",
        "description":"List files in a directory",
        "inputSchema":{
            "type":"object",
            "properties":{
                "path":{"type":"string","default":".","description":"Directory path"}
            }
        }
    },
    {
        "name":"read_file",
        "description":"Read contents of a file",
        "inputSchema":{
            "type":"object",
            "properties":{
                "filepath":{"type":"string","description":"File path to read"}
            },
            "required":["filepath"]
        }
    },
    {
        "name":"write_file",
        "description":"Write content to a file",
        "inputSchema":{
            "type":"object",
            "properties":{
                "filepath":{"type":"string","description":"File to write"},
                "content":{"type":"string","description":"Content to write"}
            },
            "required":["filepath","content"]
        }
    }
]

# ───────────────────────────────────────────────────────────────────────────────
# 8) Tool implementations
# ───────────────────────────────────────────────────────────────────────────────
def exec_py(code: str) -> str:
    from io import StringIO
    import contextlib
    buf = StringIO()
    try:
        with contextlib.redirect_stdout(buf):
            exec(code, {})
        return buf.getvalue() or "✓ no output"
    except Exception as e:
        return f"Execution error: {e}"

def inst_pkg(pkg: str) -> str:
    p = subprocess.run([sys.executable, "-m", "pip", "install", pkg],
                       capture_output=True, text=True)
    return p.stdout or p.stderr

def safe_path(p: str) -> Path:
    target = SAFE_DIR / p
    if not target.resolve().is_relative_to(SAFE_DIR.resolve()):
        raise ValueError("Unsafe path.")
    return target

def ls_files(path: str=".") -> str:
    p = safe_path(path)
    if not p.exists(): return f"Not found: {path}"
    if p.is_file(): return p.read_text()
    return "\n".join(
        f"{'📄' if it.is_file() else '📁'} {it.name}{'/' if it.is_dir() else ''}"
        for it in sorted(p.iterdir())
    ) or "— empty"

def rd_file(fp: str) -> str:
    return safe_path(fp).read_text()

def wr_file(fp: str, content: str) -> str:
    p = safe_path(fp)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content)
    return f"Written {p.name}"

# ───────────────────────────────────────────────────────────────────────────────
# 9) Agent memory (JSON) helpers
# ───────────────────────────────────────────────────────────────────────────────
MEMORY_FILE = SAFE_DIR / "memory.json"
if not MEMORY_FILE.exists(): MEMORY_FILE.write_text("{}")

def load_memory():
    try: return json.loads(MEMORY_FILE.read_text())
    except: return {}

def save_memory(mem: dict):
    MEMORY_FILE.write_text(json.dumps(mem, indent=2))

# ───────────────────────────────────────────────────────────────────────────────
# 10) API routes
# ───────────────────────────────────────────────────────────────────────────────

@app.get("/")
async def root():
    return {"message":"MCP Server Running","version":"1.2"}

@app.get("/health")
async def health_check():
    mem = load_memory()
    return {
        "status":"healthy",
        "version":"1.2",
        "tools_count":len(TOOLS),
        "memory_size":len(mem),
        "uptime":time.time()
    }

# → **Absolute Minimum Requirement #1 & #2 & #3: Single JSON-RPC /mcp endpoint**
@app.post("/mcp")
async def mcp_endpoint(request: Request):
    """
    Handle JSON-RPC 2.0 requests:
    - method "tools/list"
    - method "tools/call"
    """
    data = await request.json()
    jsonrpc = data.get("jsonrpc")
    method  = data.get("method")
    req_id  = data.get("id")
    if jsonrpc != "2.0" or not method or req_id is None:
        return {"jsonrpc":"2.0","error":{"code":-32600,"message":"Invalid Request"},"id":req_id}

    if method == "tools/list":
        result = {"tools": TOOLS}
    elif method == "tools/call":
        # expect params: {"name":..., "arguments":{...}}
        params = data.get("params") or {}
        name   = params.get("name")
        args   = params.get("arguments", {})
        try:
            logger.info(f"CALL {name} {args!r}")
            if name=="execute_python": out = exec_py(args["code"])
            elif name=="install_package": out = inst_pkg(args["package"])
            elif name=="list_files": out = ls_files(args.get("path","."))
            elif name=="read_file": out = rd_file(args["filepath"])
            elif name=="write_file": out = wr_file(args["filepath"], args["content"])
            else: raise Exception("Unknown tool")
            logger.info(f"RESULT {name} -> {repr(out)[:120]}")
            result = {"content":[{"type":"text","text":out}]}
        except Exception as e:
            logger.error(f"ERR {name} {e}")
            result = {"content":[{"type":"text","text":f"Error: {e}"}], "isError":True}
    else:
        result = {"error":{"code":-32601,"message":"Method not found"}}

    return {"jsonrpc":"2.0","result": result,"id":req_id}

# Keep existing list_tools and call_tool for backwards compatibility if needed
@app.get("/mcp/tools/list")
@app.post("/mcp/tools/list")
async def list_tools():
    return {"tools":TOOLS}

@app.post("/mcp/tools/call")
async def call_tool(req: Request):
    data = await req.json()
    name, args = data.get("name"), data.get("arguments",{})
    logger.info(f"CALL {name} {args!r}")
    try:
        if name=="execute_python": out = exec_py(args["code"])
        elif name=="install_package": out = inst_pkg(args["package"])
        elif name=="list_files": out = ls_files(args.get("path","."))
        elif name=="read_file": out = rd_file(args["filepath"])
        elif name=="write_file": out = wr_file(args["filepath"], args["content"])
        else: raise Exception("Unknown tool")
        logger.info(f"RESULT {name} -> {repr(out)[:120]}")
        return {"content":[{"type":"text","text":out}],"isError":False}
    except Exception as e:
        logger.error(f"ERR {name} {e}")
        return {"content":[{"type":"text","text":f"Error: {e}"}],"isError":True}

@app.get("/memory/view")
async def memory_view():    return load_memory()

@app.get("/memory/summary")
async def memory_summary():return {"summary":load_memory().get("summary")}

@app.post("/memory/update")
async def memory_update(req: Request):
    data = await req.json()
    mem = load_memory(); mem.update(data); save_memory(mem)
    logger.info(f"Memory updated: {data}")
    return mem

@app.post("/memory/clear")
async def memory_clear():
    save_memory({}); logger.info("Memory cleared"); return {}

@app.get("/mcp/schema")
async def mcp_schema():     return app.openapi()

@app.get("/.well-known/tool-schema.json")
async def well_known_schema(): return app.openapi()

@app.get("/activity", response_class=HTMLResponse)
async def activity_dashboard():
    return """<!DOCTYPE html><html><head><title>Live Activity</title>
<meta charset="utf-8"><style>body{font:14px sans;}#log{background:#eee;
padding:8px;height:70vh;overflow:auto;white-space:pre-wrap;border:1px solid #ccc}
</style></head><body><h1>Live Activity</h1><div id="log">Connecting…</div>
<script>
const src=new EventSource('/activity/stream');
src.onmessage=e=>{const log=document.getElementById('log');
log.textContent+='\\n'+e.data;log.scrollTop=log.scrollHeight;};
</script></body></html>"""

@app.get("/activity/stream")
async def activity_stream():
    q=asyncio.Queue(); subscribers.add(q)
    async def gen():
        while True: yield {"data":await q.get()}
    return EventSourceResponse(gen())

# Static UI folder
SITE = SAFE_DIR/"site"; SITE.mkdir(exist_ok=True)
# Tool chat
(SITE/"toolchat.html").write_text(f"""<!DOCTYPE html><html><head><meta charset="utf-8"><title>Tool Chat</title>
<style>body{{font:14px sans;max-width:700px;margin:1em auto}}textarea{{width:100%;height:6em}}</style>
</head><body><h1>Structured Tool Chat</h1><pre style="background:#f8f8f8;padding:8px;border:1px solid #ccc">
{json.dumps(TOOLS,indent=2)}
</pre><textarea id="req">{{"name":"execute_python","arguments":{{"code":"print(2+2)"}}}}</textarea><br>
<button onclick="run()">Run</button><pre id="out" style="background:#eef;border:1px solid #ccc;
padding:8px;height:200px;overflow:auto"></pre><script>
async function run(){{const r=JSON.parse(document.getElementById('req').value);
const res=await fetch('/mcp/tools/call',{{method:'POST',headers:{{'Content-Type':'application/json'}},body:JSON.stringify(r)}});
document.getElementById('out').textContent=JSON.stringify(await res.json(),null,2);}}
</script></body></html>""",encoding="utf-8")
# Human chat
(SITE/"chat.html").write_text("""<!DOCTYPE html><html><head><title>Chat</title>
<style>body{font:14px sans;max-width:600px;margin:1em auto}#chatlog{height:200px;overflow:auto;border:1px solid #ccc;padding:8px}</style>
</head><body><h2>Chat Log</h2><div id="chatlog"></div>
<input id="msg" placeholder="Type message..." style="width:80%"><button onclick="send()">Send</button>
<script>
async function send(){{let m=document.getElementById('msg').value; if(!m) return;
await fetch('/site/chat/send',{method:'POST',headers:{{'Content-Type':'application/json'}},body:JSON.stringify({message:m})});
document.getElementById('msg').value='';load();}}
async function load(){{document.getElementById('chatlog').textContent=
await fetch('/site/chat/log').then(r=>r.text());}}
load();setInterval(load,3000);
</script></body></html>""",encoding="utf-8")
app.mount("/site",StaticFiles(directory=SITE),name="site")

@app.post("/site/chat/send")
async def post_chat(req: Request):
    msg=(await req.json()).get("message","").strip()
    if msg:
        line=f"[chat {time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}"
        with CHAT_LOG.open("a") as f: f.write(line+"\n")
        logger.info(f"CHAT {msg}")
    return {"ok":True}

@app.get("/site/chat/log")
async def get_chat_log(): return CHAT_LOG.read_text()
@app.get("/site/log/serverlog")
async def get_server_log(): return SERVER_LOG.read_text()

@app.get("/download/{file_path:path}")
async def download_file(file_path: str):
    fp=safe_path(file_path)
    if not fp.exists() or not fp.is_file():
        raise HTTPException(404,"File not found")
    return FileResponse(fp)

# ───────────────────────────────────────────────────────────────────────────────
# 11) Start Uvicorn + ngrok, print endpoints, return public_url
# ───────────────────────────────────────────────────────────────────────────────
def start_server() -> str:
    Thread(target=lambda: uvicorn.run(app,host="0.0.0.0",port=8000,log_level="info"),daemon=True).start()
    time.sleep(2)
    public_url = ngrok.connect(8000).public_url
    print(f"🔗 Public URL: {public_url}")
    for name,path in [
        ("Root", "/"),("Health","/health"),
        ("MCP RPC","/mcp"),    # newly added
        ("Tools list","/mcp/tools/list"),("Tool call","/mcp/tools/call"),
        ("Memory view","/memory/view"),("Memory update","/memory/update"),
        ("Memory clear","/memory/clear"),("Tool chat UI","/site/toolchat.html"),
        ("Human chat UI","/site/chat.html"),("Download","/download/<relative-path>")
    ]:
        print(f"• {name:<13} {public_url}{path}")
    return public_url

public_url = start_server()

# ───────────────────────────────────────────────────────────────────────────────
# 12) Generate Claude & OpenAI plugin manifests using actual public_url
# ───────────────────────────────────────────────────────────────────────────────
CLAUDE_YAML = SAFE_DIR/"claude.yaml"
CLAUDE_YAML.write_text(f"""schema_version: "1.0"
name_for_human: "MCP Tool Server"
name_for_model: "mcp_tool_server"
description_for_human: |
  Interact with the Colab MCP Tool Server to:
    - List tools
    - Execute Python
    - Install packages
    - Manage files
    - Inspect/update memory
    - Health checks & uptime
description_for_model: |
  Plugin endpoints:
    • POST {public_url}/mcp
    • GET {public_url}/mcp/tools/list
    • POST {public_url}/mcp/tools/call
    • GET {public_url}/health
    • Memory: /memory/view, /memory/summary, /memory/update, /memory/clear
    • Schema: {public_url}/.well-known/tool-schema.json
    • Download: {public_url}/download/{{file_path}}
auth:
  type: none
api:
  type: openapi
  url: "{public_url}/.well-known/tool-schema.json"
  is_user_authenticated: false
logo_url: "{public_url}/logo.png"
contact_email: "support@your-domain.com"
legal_info_url: "https://your-domain.com/legal.html"
""",encoding="utf-8")
print(f"Wrote Claude manifest to: {CLAUDE_YAML}")

AI_PLUGIN = SAFE_DIR/"ai-plugin.json"
AI_PLUGIN.write_text(json.dumps({
    "schema_version":"v1",
    "name_for_human":"MCP Tool Server",
    "name_for_model":"mcp_tool_server",
    "description_for_human":"API to execute Python, manage files, memory and health",
    "description_for_model":"Plugin to call MCP Tool Server endpoints",
    "auth":{"type":"none"},
    "api":{"type":"openapi","url":f"{public_url}/openai.yaml","is_user_authenticated":False},
    "logo_url":f"{public_url}/logo.png",
    "contact_email":"support@your-domain.com",
    "legal_info_url":"https://your-domain.com/legal.html"
},indent=2),encoding="utf-8")
print(f"Wrote OpenAI ai-plugin.json to: {AI_PLUGIN}")

OPENAPI_YAML = SAFE_DIR/"openai.yaml"
OPENAPI_YAML.write_text(f"""openapi: "3.0.1"
info:
  title: "MCP Tool Server API"
  version: "1.2"
  description: |
    Execute Python, install packages, manage files & memory, health checks.
servers:
  - url: "{public_url}"
paths:
  /mcp:
    post:
      summary: "JSON-RPC 2.0 endpoint"
  /mcp/tools/list:
    get:
      summary: "List all tools"
  /mcp/tools/call:
    post:
      summary: "Invoke a tool"
  /health:
    get:
      summary: "Health check"
  /memory/view:
    get:
      summary: "View memory"
  /memory/summary:
    get:
      summary: "Memory summary"
  /memory/update:
    post:
      summary: "Update memory"
  /memory/clear:
    post:
      summary: "Clear memory"
  /.well-known/tool-schema.json:
    get:
      summary: "Tool schema JSON"
  /download/{{file_path}}:
    get:
      summary: "Download a file"
""",encoding="utf-8")
print(f"Wrote OpenAPI spec to: {OPENAPI_YAML}")

# ───────────────────────────────────────────────────────────────────────────────
# ✅ All set! Use the printed URLs above to interact with your MCP server.
# ───────────────────────────────────────────────────────────────────────────────

Mounted at /content/drive


INFO:     Started server process [1669]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


🔗 Public URL: https://b925-35-247-154-216.ngrok-free.app
• Root          https://b925-35-247-154-216.ngrok-free.app/
• Health        https://b925-35-247-154-216.ngrok-free.app/health
• MCP RPC       https://b925-35-247-154-216.ngrok-free.app/mcp
• Tools list    https://b925-35-247-154-216.ngrok-free.app/mcp/tools/list
• Tool call     https://b925-35-247-154-216.ngrok-free.app/mcp/tools/call
• Memory view   https://b925-35-247-154-216.ngrok-free.app/memory/view
• Memory update https://b925-35-247-154-216.ngrok-free.app/memory/update
• Memory clear  https://b925-35-247-154-216.ngrok-free.app/memory/clear
• Tool chat UI  https://b925-35-247-154-216.ngrok-free.app/site/toolchat.html
• Human chat UI https://b925-35-247-154-216.ngrok-free.app/site/chat.html
• Download      https://b925-35-247-154-216.ngrok-free.app/download/<relative-path>
Wrote Claude manifest to: /content/drive/MyDrive/ColabMCPFiles/claude.yaml
Wrote OpenAI ai-plugin.json to: /content/drive/MyDrive/ColabMCPFiles/ai-plugin.

In [ ]:
2yuUKpiAlYXCez40i8UktLZ91s7_5Rv1GjG5HMFw5KKbk5# ╔════════════════════════════════════════════════════════════════════════════╗
# ║     🤖 Colab MCP Server + Tool Call UI + Live Dashboard + GDrive Mount    ║
# ║                                                                            ║
# ║  Self-contained Google Colab cell: installs deps, mounts Drive, defines   ║
# ║  the full FastAPI MCP server (with health, GET/POST tools list, memory    ║
# ║  CRUD, live dashboard, file download, chat UIs), generates Claude & OpenAI║
# ║  plugin manifests, and launches Uvicorn behind ngrok — all in one place.   ║
# ╚════════════════════════════════════════════════════════════════════════════╝

# ───────────────────────────────────────────────────────────────────────────────
# 1) Install everything (using Colab magic so imports see them immediately)
# ───────────────────────────────────────────────────────────────────────────────
!pip install --quiet fastapi uvicorn pyngrok nest-asyncio requests sse-starlette

# ───────────────────────────────────────────────────────────────────────────────
# 2) Enable asyncio patching (needed in Colab) and mount Google Drive
# ───────────────────────────────────────────────────────────────────────────────
import nest_asyncio; nest_asyncio.apply()
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

# ───────────────────────────────────────────────────────────────────────────────
# 3) Set up safe working dir & switch into it
# ───────────────────────────────────────────────────────────────────────────────
import os
from pathlib import Path
SAFE_DIR = Path("/content/drive/MyDrive/ColabMCPFiles")
SAFE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(SAFE_DIR)

# ───────────────────────────────────────────────────────────────────────────────
# 4) Configure ngrok for public tunneling
# ───────────────────────────────────────────────────────────────────────────────
from pyngrok import ngrok
# ← replace with your own token from https://dashboard.ngrok.com/get-started/your-authtoken
ngrok.set_auth_token("2yuUKpiAlYXCez40i8UktLZ91s7_5Rv1GjG5HMFw5KKbk5sHz")

# ───────────────────────────────────────────────────────────────────────────────
# 5) Import FastAPI, Uvicorn, SSE, logging, CORS, etc.
# ───────────────────────────────────────────────────────────────────────────────
import time, json, logging, asyncio, subprocess, sys
from threading import Thread
from fastapi import FastAPI, Request, HTTPException
from fastapi.responses import HTMLResponse, FileResponse
from fastapi.staticfiles import StaticFiles
from fastapi.middleware.cors import CORSMiddleware
import uvicorn
from sse_starlette.sse import EventSourceResponse

# ───────────────────────────────────────────────────────────────────────────────
# 6) Initialize FastAPI app & add CORS for Claude.ai
# ───────────────────────────────────────────────────────────────────────────────
app = FastAPI(title="Colab MCP Tool Server", version="1.2")

# → **Absolute Minimum Requirement #4: HTTPS + CORS**
app.add_middleware(
    CORSMiddleware,
    allow_origins=["https://claude.ai"],
    allow_methods=["POST"],
    allow_headers=["*"],
)

SERVER_LOG = SAFE_DIR / "server_log.txt"; SERVER_LOG.touch()
CHAT_LOG   = SAFE_DIR / "chat_log.txt";   CHAT_LOG.touch()
subscribers: set[asyncio.Queue] = set()

class BroadcastHandler(logging.Handler):
    def emit(self, record):
        msg = self.format(record)
        with SERVER_LOG.open("a") as f:
            f.write(msg + "\n")
        for q in list(subscribers):
            try:
                asyncio.get_event_loop().call_soon_threadsafe(q.put_nowait, msg)
            except:
                pass

logger = logging.getLogger("mcp")
logger.setLevel(logging.INFO)
handler = BroadcastHandler()
handler.setFormatter(logging.Formatter("%(asctime)s %(message)s"))
logger.addHandler(handler)

# ───────────────────────────────────────────────────────────────────────────────
# 7) Define tools & JSON schemas
# ───────────────────────────────────────────────────────────────────────────────
TOOLS = [
    {
        "name":"execute_python",
        "description":"Run Python code and return stdout",
        "inputSchema":{
            "type":"object",
            "properties":{
                "code":{"type":"string","description":"Python code to execute"}
            },
            "required":["code"]
        }
    },
    {
        "name":"install_package",
        "description":"Install a pip package",
        "inputSchema":{
            "type":"object",
            "properties":{
                "package":{"type":"string","description":"Name of package"}
            },
            "required":["package"]
        }
    },
    {
        "name":"list_files",
        "description":"List files in a directory",
        "inputSchema":{
            "type":"object",
            "properties":{
                "path":{"type":"string","default":".","description":"Directory path"}
            }
        }
    },
    {
        "name":"read_file",
        "description":"Read contents of a file",
        "inputSchema":{
            "type":"object",
            "properties":{
                "filepath":{"type":"string","description":"File path to read"}
            },
            "required":["filepath"]
        }
    },
    {
        "name":"write_file",
        "description":"Write content to a file",
        "inputSchema":{
            "type":"object",
            "properties":{
                "filepath":{"type":"string","description":"File to write"},
                "content":{"type":"string","description":"Content to write"}
            },
            "required":["filepath","content"]
        }
    }
]

# ───────────────────────────────────────────────────────────────────────────────
# 8) Tool implementations
# ───────────────────────────────────────────────────────────────────────────────
def exec_py(code: str) -> str:
    from io import StringIO
    import contextlib
    buf = StringIO()
    try:
        with contextlib.redirect_stdout(buf):
            exec(code, {})
        return buf.getvalue() or "✓ no output"
    except Exception as e:
        return f"Execution error: {e}"

def inst_pkg(pkg: str) -> str:
    p = subprocess.run([sys.executable, "-m", "pip", "install", pkg],
                       capture_output=True, text=True)
    return p.stdout or p.stderr

def safe_path(p: str) -> Path:
    target = SAFE_DIR / p
    if not target.resolve().is_relative_to(SAFE_DIR.resolve()):
        raise ValueError("Unsafe path.")
    return target

def ls_files(path: str=".") -> str:
    p = safe_path(path)
    if not p.exists(): return f"Not found: {path}"
    if p.is_file(): return p.read_text()
    return "\n".join(
        f"{'📄' if it.is_file() else '📁'} {it.name}{'/' if it.is_dir() else ''}"
        for it in sorted(p.iterdir())
    ) or "— empty"

def rd_file(fp: str) -> str:
    return safe_path(fp).read_text()

def wr_file(fp: str, content: str) -> str:
    p = safe_path(fp)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content)
    return f"Written {p.name}"

# ───────────────────────────────────────────────────────────────────────────────
# 9) Agent memory (JSON) helpers
# ───────────────────────────────────────────────────────────────────────────────
MEMORY_FILE = SAFE_DIR / "memory.json"
if not MEMORY_FILE.exists(): MEMORY_FILE.write_text("{}")

def load_memory():
    try: return json.loads(MEMORY_FILE.read_text())
    except: return {}

def save_memory(mem: dict):
    MEMORY_FILE.write_text(json.dumps(mem, indent=2))

# ───────────────────────────────────────────────────────────────────────────────
# 10) API routes
# ───────────────────────────────────────────────────────────────────────────────

@app.get("/")
async def root():
    return {"message":"MCP Server Running","version":"1.2"}

@app.get("/health")
async def health_check():
    mem = load_memory()
    return {
        "status":"healthy",
        "version":"1.2",
        "tools_count":len(TOOLS),
        "memory_size":len(mem),
        "uptime":time.time()
    }

# → **Absolute Minimum Requirement #1 & #2 & #3: Single JSON-RPC /mcp endpoint**
@app.post("/mcp")
async def mcp_endpoint(request: Request):
    """
    Handle JSON-RPC 2.0 requests:
    - method "tools/list"
    - method "tools/call"
    """
    data = await request.json()
    jsonrpc = data.get("jsonrpc")
    method  = data.get("method")
    req_id  = data.get("id")
    if jsonrpc != "2.0" or not method or req_id is None:
        return {"jsonrpc":"2.0","error":{"code":-32600,"message":"Invalid Request"},"id":req_id}

    if method == "tools/list":
        result = {"tools": TOOLS}
    elif method == "tools/call":
        # expect params: {"name":..., "arguments":{...}}
        params = data.get("params") or {}
        name   = params.get("name")
        args   = params.get("arguments", {})
        try:
            logger.info(f"CALL {name} {args!r}")
            if name=="execute_python": out = exec_py(args["code"])
            elif name=="install_package": out = inst_pkg(args["package"])
            elif name=="list_files": out = ls_files(args.get("path","."))
            elif name=="read_file": out = rd_file(args["filepath"])
            elif name=="write_file": out = wr_file(args["filepath"], args["content"])
            else: raise Exception("Unknown tool")
            logger.info(f"RESULT {name} -> {repr(out)[:120]}")
            result = {"content":[{"type":"text","text":out}]}
        except Exception as e:
            logger.error(f"ERR {name} {e}")
            result = {"content":[{"type":"text","text":f"Error: {e}"}], "isError":True}
    else:
        result = {"error":{"code":-32601,"message":"Method not found"}}

    return {"jsonrpc":"2.0","result": result,"id":req_id}

# Keep existing list_tools and call_tool for backwards compatibility if needed
@app.get("/mcp/tools/list")
@app.post("/mcp/tools/list")
async def list_tools():
    return {"tools":TOOLS}

@app.post("/mcp/tools/call")
async def call_tool(req: Request):
    data = await req.json()
    name, args = data.get("name"), data.get("arguments",{})
    logger.info(f"CALL {name} {args!r}")
    try:
        if name=="execute_python": out = exec_py(args["code"])
        elif name=="install_package": out = inst_pkg(args["package"])
        elif name=="list_files": out = ls_files(args.get("path","."))
        elif name=="read_file": out = rd_file(args["filepath"])
        elif name=="write_file": out = wr_file(args["filepath"], args["content"])
        else: raise Exception("Unknown tool")
        logger.info(f"RESULT {name} -> {repr(out)[:120]}")
        return {"content":[{"type":"text","text":out}],"isError":False}
    except Exception as e:
        logger.error(f"ERR {name} {e}")
        return {"content":[{"type":"text","text":f"Error: {e}"}],"isError":True}

@app.get("/memory/view")
async def memory_view():    return load_memory()

@app.get("/memory/summary")
async def memory_summary():return {"summary":load_memory().get("summary")}

@app.post("/memory/update")
async def memory_update(req: Request):
    data = await req.json()
    mem = load_memory(); mem.update(data); save_memory(mem)
    logger.info(f"Memory updated: {data}")
    return mem

@app.post("/memory/clear")
async def memory_clear():
    save_memory({}); logger.info("Memory cleared"); return {}

@app.get("/mcp/schema")
async def mcp_schema():     return app.openapi()

@app.get("/.well-known/tool-schema.json")
async def well_known_schema(): return app.openapi()

@app.get("/activity", response_class=HTMLResponse)
async def activity_dashboard():
    return """<!DOCTYPE html><html><head><title>Live Activity</title>
<meta charset="utf-8"><style>body{font:14px sans;}#log{background:#eee;
padding:8px;height:70vh;overflow:auto;white-space:pre-wrap;border:1px solid #ccc}
</style></head><body><h1>Live Activity</h1><div id="log">Connecting…</div>
<script>
const src=new EventSource('/activity/stream');
src.onmessage=e=>{const log=document.getElementById('log');
log.textContent+='\\n'+e.data;log.scrollTop=log.scrollHeight;};
</script></body></html>"""

@app.get("/activity/stream")
async def activity_stream():
    q=asyncio.Queue(); subscribers.add(q)
    async def gen():
        while True: yield {"data":await q.get()}
    return EventSourceResponse(gen())

# Static UI folder
SITE = SAFE_DIR/"site"; SITE.mkdir(exist_ok=True)
# Tool chat
(SITE/"toolchat.html").write_text(f"""<!DOCTYPE html><html><head><meta charset="utf-8"><title>Tool Chat</title>
<style>body{{font:14px sans;max-width:700px;margin:1em auto}}textarea{{width:100%;height:6em}}</style>
</head><body><h1>Structured Tool Chat</h1><pre style="background:#f8f8f8;padding:8px;border:1px solid #ccc">
{json.dumps(TOOLS,indent=2)}
</pre><textarea id="req">{{"name":"execute_python","arguments":{{"code":"print(2+2)"}}}}</textarea><br>
<button onclick="run()">Run</button><pre id="out" style="background:#eef;border:1px solid #ccc;
padding:8px;height:200px;overflow:auto"></pre><script>
async function run(){{const r=JSON.parse(document.getElementById('req').value);
const res=await fetch('/mcp/tools/call',{{method:'POST',headers:{{'Content-Type':'application/json'}},body:JSON.stringify(r)}});
document.getElementById('out').textContent=JSON.stringify(await res.json(),null,2);}}
</script></body></html>""",encoding="utf-8")
# Human chat
(SITE/"chat.html").write_text("""<!DOCTYPE html><html><head><title>Chat</title>
<style>body{font:14px sans;max-width:600px;margin:1em auto}#chatlog{height:200px;overflow:auto;border:1px solid #ccc;padding:8px}</style>
</head><body><h2>Chat Log</h2><div id="chatlog"></div>
<input id="msg" placeholder="Type message..." style="width:80%"><button onclick="send()">Send</button>
<script>
async function send(){{let m=document.getElementById('msg').value; if(!m) return;
await fetch('/site/chat/send',{method:'POST',headers:{{'Content-Type':'application/json'}},body:JSON.stringify({message:m})});
document.getElementById('msg').value='';load();}}
async function load(){{document.getElementById('chatlog').textContent=
await fetch('/site/chat/log').then(r=>r.text());}}
load();setInterval(load,3000);
</script></body></html>""",encoding="utf-8")
app.mount("/site",StaticFiles(directory=SITE),name="site")

@app.post("/site/chat/send")
async def post_chat(req: Request):
    msg=(await req.json()).get("message","").strip()
    if msg:
        line=f"[chat {time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}"
        with CHAT_LOG.open("a") as f: f.write(line+"\n")
        logger.info(f"CHAT {msg}")
    return {"ok":True}

@app.get("/site/chat/log")
async def get_chat_log(): return CHAT_LOG.read_text()
@app.get("/site/log/serverlog")
async def get_server_log(): return SERVER_LOG.read_text()

@app.get("/download/{file_path:path}")
async def download_file(file_path: str):
    fp=safe_path(file_path)
    if not fp.exists() or not fp.is_file():
        raise HTTPException(404,"File not found")
    return FileResponse(fp)

# ───────────────────────────────────────────────────────────────────────────────
# 11) Start Uvicorn + ngrok, print endpoints, return public_url
# ───────────────────────────────────────────────────────────────────────────────
def start_server() -> str:
    Thread(target=lambda: uvicorn.run(app,host="0.0.0.0",port=8000,log_level="info"),daemon=True).start()
    time.sleep(2)
    public_url = ngrok.connect(8000).public_url
    print(f"🔗 Public URL: {public_url}")
    for name,path in [
        ("Root", "/"),("Health","/health"),
        ("MCP RPC","/mcp"),    # newly added
        ("Tools list","/mcp/tools/list"),("Tool call","/mcp/tools/call"),
        ("Memory view","/memory/view"),("Memory update","/memory/update"),
        ("Memory clear","/memory/clear"),("Tool chat UI","/site/toolchat.html"),
        ("Human chat UI","/site/chat.html"),("Download","/download/<relative-path>")
    ]:
        print(f"• {name:<13} {public_url}{path}")
    return public_url

public_url = start_server()

# ───────────────────────────────────────────────────────────────────────────────
# 12) Generate Claude & OpenAI plugin manifests using actual public_url
# ───────────────────────────────────────────────────────────────────────────────
CLAUDE_YAML = SAFE_DIR/"claude.yaml"
CLAUDE_YAML.write_text(f"""schema_version: "1.0"
name_for_human: "MCP Tool Server"
name_for_model: "mcp_tool_server"
description_for_human: |
  Interact with the Colab MCP Tool Server to:
    - List tools
    - Execute Python
    - Install packages
    - Manage files
    - Inspect/update memory
    - Health checks & uptime
description_for_model: |
  Plugin endpoints:
    • POST {public_url}/mcp
    • GET {public_url}/mcp/tools/list
    • POST {public_url}/mcp/tools/call
    • GET {public_url}/health
    • Memory: /memory/view, /memory/summary, /memory/update, /memory/clear
    • Schema: {public_url}/.well-known/tool-schema.json
    • Download: {public_url}/download/{{file_path}}
auth:
  type: none
api:
  type: openapi
  url: "{public_url}/.well-known/tool-schema.json"
  is_user_authenticated: false
logo_url: "{public_url}/logo.png"
contact_email: "support@your-domain.com"
legal_info_url: "https://your-domain.com/legal.html"
""",encoding="utf-8")
print(f"Wrote Claude manifest to: {CLAUDE_YAML}")

AI_PLUGIN = SAFE_DIR/"ai-plugin.json"
AI_PLUGIN.write_text(json.dumps({
    "schema_version":"v1",
    "name_for_human":"MCP Tool Server",
    "name_for_model":"mcp_tool_server",
    "description_for_human":"API to execute Python, manage files, memory and health",
    "description_for_model":"Plugin to call MCP Tool Server endpoints",
    "auth":{"type":"none"},
    "api":{"type":"openapi","url":f"{public_url}/openai.yaml","is_user_authenticated":False},
    "logo_url":f"{public_url}/logo.png",
    "contact_email":"support@your-domain.com",
    "legal_info_url":"https://your-domain.com/legal.html"
},indent=2),encoding="utf-8")
print(f"Wrote OpenAI ai-plugin.json to: {AI_PLUGIN}")

OPENAPI_YAML = SAFE_DIR/"openai.yaml"
OPENAPI_YAML.write_text(f"""openapi: "3.0.1"
info:
  title: "MCP Tool Server API"
  version: "1.2"
  description: |
    Execute Python, install packages, manage files & memory, health checks.
servers:
  - url: "{public_url}"
paths:
  /mcp:
    post:
      summary: "JSON-RPC 2.0 endpoint"
  /mcp/tools/list:
    get:
      summary: "List all tools"
  /mcp/tools/call:
    post:
      summary: "Invoke a tool"
  /health:
    get:
      summary: "Health check"
  /memory/view:
    get:
      summary: "View memory"
  /memory/summary:
    get:
      summary: "Memory summary"
  /memory/update:
    post:
      summary: "Update memory"
  /memory/clear:
    post:
      summary: "Clear memory"
  /.well-known/tool-schema.json:
    get:
      summary: "Tool schema JSON"
  /download/{{file_path}}:
    get:
      summary: "Download a file"
""",encoding="utf-8")
print(f"Wrote OpenAPI spec to: {OPENAPI_YAML}")

# ───────────────────────────────────────────────────────────────────────────────
# ✅ All set! Use the printed URLs above to interact with your MCP server.
# ───────────────────────────────────────────────────────────────────────────────

Mounted at /content/drive


INFO:     Started server process [4683]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


🔗 Public URL: https://b6ab-34-126-173-164.ngrok-free.app
• Root          https://b6ab-34-126-173-164.ngrok-free.app/
• Health        https://b6ab-34-126-173-164.ngrok-free.app/health
• MCP RPC       https://b6ab-34-126-173-164.ngrok-free.app/mcp
• Tools list    https://b6ab-34-126-173-164.ngrok-free.app/mcp/tools/list
• Tool call     https://b6ab-34-126-173-164.ngrok-free.app/mcp/tools/call
• Memory view   https://b6ab-34-126-173-164.ngrok-free.app/memory/view
• Memory update https://b6ab-34-126-173-164.ngrok-free.app/memory/update
• Memory clear  https://b6ab-34-126-173-164.ngrok-free.app/memory/clear
• Tool chat UI  https://b6ab-34-126-173-164.ngrok-free.app/site/toolchat.html
• Human chat UI https://b6ab-34-126-173-164.ngrok-free.app/site/chat.html
• Download      https://b6ab-34-126-173-164.ngrok-free.app/download/<relative-path>
Wrote Claude manifest to: /content/drive/MyDrive/ColabMCPFiles/claude.yaml
Wrote OpenAI ai-plugin.json to: /content/drive/MyDrive/ColabMCPFiles/ai-plugin.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════════════════════╗
# ║ 🧠 Colab MCP Server (v1.3) — Full FastAPI Tool Agent for Claude / OpenAI with File Support    ║
# ║                                                                                              ║
# ║ ☑ JSON-RPC endpoint at /mcp                                                                 ║
# ║ ☑ /mcp/tools/call endpoint and /mcp/tools/list for GPT-friendly fallback                    ║
# ║ ☑ Tools: Python exec, pip install, file read/write/list, memory,                             ║
# ║ ☑ NEW TOOLS: `print_file_text` and `download_file`                                           ║
# ║ ☑ Auto-generates OpenAI and Claude manifests + public ngrok endpoints                       ║
# ╚══════════════════════════════════════════════════════════════════════════════════════════════╝

# ─────────────────────────────────────────────────────────────────────────────
# 1. Install required packages
# ─────────────────────────────────────────────────────────────────────────────
!pip install --quiet fastapi uvicorn pyngrok nest-asyncio requests sse-starlette

# ─────────────────────────────────────────────────────────────────────────────
# 2. Initialize environment + mount Google Drive
# ─────────────────────────────────────────────────────────────────────────────
import nest_asyncio; nest_asyncio.apply()
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

# ─────────────────────────────────────────────────────────────────────────────
# 3. Core imports and safe working directory setup
# ─────────────────────────────────────────────────────────────────────────────
import os, time, json, logging, asyncio, subprocess, sys
from pathlib import Path
from threading import Thread
from fastapi import FastAPI, Request, HTTPException
from fastapi.responses import FileResponse, HTMLResponse
from fastapi.staticfiles import StaticFiles
from fastapi.middleware.cors import CORSMiddleware
from sse_starlette.sse import EventSourceResponse
import uvicorn
from pyngrok import ngrok

SAFE_DIR = Path("/content/drive/MyDrive/ColabMCPFiles")
SAFE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(SAFE_DIR)

# ─────────────────────────────────────────────────────────────────────────────
# 4. Configure ngrok and CORS
# ─────────────────────────────────────────────────────────────────────────────
ngrok.set_auth_token("2yuUKpiAlYXCez40i8UktLZ91s7_5Rv1GjG5HMFw5KKbk5sHz")

app = FastAPI(title="Colab MCP Tool Server", version="1.3")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["https://claude.ai"],
    allow_methods=["POST"],
    allow_headers=["*"],
)

# ─────────────────────────────────────────────────────────────────────────────
# 5. Setup logger + subscribers
# ─────────────────────────────────────────────────────────────────────────────
SERVER_LOG = SAFE_DIR / "server_log.txt"; SERVER_LOG.touch()
CHAT_LOG   = SAFE_DIR / "chat_log.txt";   CHAT_LOG.touch()
subscribers: set[asyncio.Queue] = set()

class BroadcastHandler(logging.Handler):
    def emit(self, record):
        msg = self.format(record)
        with SERVER_LOG.open("a") as f: f.write(msg + "\n")
        for q in list(subscribers):
            try: asyncio.get_event_loop().call_soon_threadsafe(q.put_nowait, msg)
            except: pass

logger = logging.getLogger("mcp")
logger.setLevel(logging.INFO)
handler = BroadcastHandler()
handler.setFormatter(logging.Formatter("%(asctime)s %(message)s"))
logger.addHandler(handler)

# ─────────────────────────────────────────────────────────────────────────────
# 6. Define tools (schemas)
# ─────────────────────────────────────────────────────────────────────────────
TOOLS = [
    {
        "name": "execute_python",
        "description": "Run Python code and return stdout",
        "inputSchema": {
            "type": "object",
            "properties": {"code": {"type": "string"}},
            "required": ["code"]
        }
    },
    {
        "name": "install_package",
        "description": "Install a pip package",
        "inputSchema": {
            "type": "object",
            "properties": {"package": {"type": "string"}},
            "required": ["package"]
        }
    },
    {
        "name": "list_files",
        "description": "List files in a directory",
        "inputSchema": {
            "type": "object",
            "properties": {"path": {"type": "string", "default": "."}}
        }
    },
    {
        "name": "read_file",
        "description": "Read contents of a file",
        "inputSchema": {
            "type": "object",
            "properties": {"filepath": {"type": "string"}},
            "required": ["filepath"]
        }
    },
    {
        "name": "write_file",
        "description": "Write content to a file",
        "inputSchema": {
            "type": "object",
            "properties": {
                "filepath": {"type": "string"},
                "content": {"type": "string"}
            },
            "required": ["filepath", "content"]
        }
    },
    {
        "name": "print_file_text",
        "description": "Print the contents of a UTF-8 text file",
        "inputSchema": {
            "type": "object",
            "properties": {"filepath": {"type": "string"}},
            "required": ["filepath"]
        }
    },
    {
        "name": "download_file",
        "description": "Return a public URL for downloading the file",
        "inputSchema": {
            "type": "object",
            "properties": {"filepath": {"type": "string"}},
            "required": ["filepath"]
        }
    }
]

# ─────────────────────────────────────────────────────────────────────────────
# 7. Tool implementations
# ─────────────────────────────────────────────────────────────────────────────
def exec_py(code: str) -> str:
    from io import StringIO
    import contextlib
    buf = StringIO()
    try:
        with contextlib.redirect_stdout(buf):
            exec(code, {})
        return buf.getvalue() or "✓ no output"
    except Exception as e:
        return f"Execution error: {e}"

def inst_pkg(pkg: str) -> str:
    p = subprocess.run([sys.executable, "-m", "pip", "install", pkg], capture_output=True, text=True)
    return p.stdout or p.stderr

def safe_path(p: str) -> Path:
    target = SAFE_DIR / p
    if not target.resolve().is_relative_to(SAFE_DIR.resolve()):
        raise ValueError("Unsafe path.")
    return target

def ls_files(path: str = ".") -> str:
    p = safe_path(path)
    if not p.exists(): return f"Not found: {path}"
    if p.is_file(): return p.read_text()
    return "\n".join(
        f"{'📄' if it.is_file() else '📁'} {it.name}{'/' if it.is_dir() else ''}"
        for it in sorted(p.iterdir())
    ) or "— empty"

def rd_file(fp: str) -> str:
    return safe_path(fp).read_text()

def wr_file(fp: str, content: str) -> str:
    p = safe_path(fp)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content)
    return f"Written {p.name}"

# ─────────────────────────────────────────────────────────────────────────────
# 8. Memory functions
# ─────────────────────────────────────────────────────────────────────────────
MEMORY_FILE = SAFE_DIR / "memory.json"
if not MEMORY_FILE.exists(): MEMORY_FILE.write_text("{}")

def load_memory(): return json.loads(MEMORY_FILE.read_text())
def save_memory(mem): MEMORY_FILE.write_text(json.dumps(mem, indent=2))

# ─────────────────────────────────────────────────────────────────────────────
# 9. Main API routes
# ─────────────────────────────────────────────────────────────────────────────
@app.get("/")
async def root(): return {"message": "MCP Server Running", "version": "1.3"}

@app.get("/health")
async def health(): return {"status": "healthy", "tools": len(TOOLS), "memory": len(load_memory())}

@app.post("/mcp")
async def mcp_endpoint(request: Request):
    data = await request.json()
    jsonrpc = data.get("jsonrpc")
    method = data.get("method")
    req_id = data.get("id")
    if jsonrpc != "2.0" or not method or req_id is None:
        return {"jsonrpc": "2.0", "error": {"code": -32600, "message": "Invalid Request"}, "id": req_id}
    try:
        if method == "tools/list":
            result = {"tools": TOOLS}
        elif method == "tools/call":
            params = data.get("params", {})
            name = params.get("name")
            args = params.get("input") or params.get("arguments") or {}
            logger.info(f"CALL {name} {args!r}")
            if name == "execute_python": out = exec_py(args["code"])
            elif name == "install_package": out = inst_pkg(args["package"])
            elif name == "list_files": out = ls_files(args.get("path", "."))
            elif name == "read_file": out = rd_file(args["filepath"])
            elif name == "write_file": out = wr_file(args["filepath"], args["content"])
            elif name == "print_file_text": out = rd_file(args["filepath"])
            elif name == "download_file":
                safe_path(args["filepath"])  # validate
                out = f"{public_url}/download/{args['filepath']}"
            else: raise Exception(f"Unknown tool: {name}")
            logger.info(f"RESULT {name} -> {repr(out)[:120]}")
            result = {"content": [{"type": "text", "text": out}]}
        else:
            result = {"error": {"code": -32601, "message": "Method not found"}}
    except Exception as e:
        logger.error(f"ERR {method} {e}")
        result = {"content": [{"type": "text", "text": f"Error: {e}"}], "isError": True}
    return {"jsonrpc": "2.0", "result": result, "id": req_id}

@app.post("/mcp/tools/call")
async def call_tool(req: Request):
    data = await req.json()
    name = data.get("name")
    args = data.get("input") or data.get("arguments") or {}
    logger.info(f"CALL {name} {args!r}")
    try:
        if name == "execute_python": out = exec_py(args["code"])
        elif name == "install_package": out = inst_pkg(args["package"])
        elif name == "list_files": out = ls_files(args.get("path", "."))
        elif name == "read_file": out = rd_file(args["filepath"])
        elif name == "write_file": out = wr_file(args["filepath"], args["content"])
        elif name == "print_file_text": out = rd_file(args["filepath"])
        elif name == "download_file":
            safe_path(args["filepath"])
            out = f"{public_url}/download/{args['filepath']}"
        else: raise Exception(f"Unknown tool: {name}")
        logger.info(f"RESULT {name} -> {repr(out)[:120]}")
        return {"content": [{"type": "text", "text": out}], "isError": False}
    except Exception as e:
        logger.error(f"ERR {name} {e}")
        return {"content": [{"type": "text", "text": f"Error: {e}"}], "isError": True}

@app.get("/memory/view")
async def memory_view(): return load_memory()

@app.post("/memory/update")
async def memory_update(req: Request):
    data = await req.json(); mem = load_memory(); mem.update(data); save_memory(mem)
    logger.info(f"Memory updated: {data}"); return mem

@app.post("/memory/clear")
async def memory_clear(): save_memory({}); logger.info("Memory cleared"); return {}

@app.get("/download/{file_path:path}")
async def download_file(file_path: str):
    fp = safe_path(file_path)
    if not fp.exists() or not fp.is_file():
        raise HTTPException(404, "File not found")
    return FileResponse(fp)

@app.get("/.well-known/tool-schema.json")
@app.get("/mcp/schema")
async def get_schema(): return app.openapi()

# ─────────────────────────────────────────────────────────────────────────────
# 10. Launch Uvicorn + ngrok + print public URLs
# ─────────────────────────────────────────────────────────────────────────────
def start_server():
    Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000), daemon=True).start()
    time.sleep(2)
    public_url = ngrok.connect(8000).public_url
    print("🔗 MCP Server Public URL:", public_url)
    print("📡 Available Endpoints:")
    for name, path in [
        ("Root", "/"),
        ("Health", "/health"),
        ("MCP RPC", "/mcp"),
        ("Tools List", "/mcp/tools/list"),
        ("Call Tool", "/mcp/tools/call"),
        ("Download", "/download/<file>"),
        ("Memory View", "/memory/view"),
        ("Memory Update", "/memory/update"),
        ("Memory Clear", "/memory/clear"),
        ("OpenAPI Schema", "/.well-known/tool-schema.json"),
    ]:
        print(f"• {name:<16} {public_url}{path}")
    return public_url

public_url = start_server()

Mounted at /content/drive


INFO:     Started server process [4683]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


🔗 MCP Server Public URL: https://7d51-34-126-173-164.ngrok-free.app
📡 Available Endpoints:
• Root             https://7d51-34-126-173-164.ngrok-free.app/
• Health           https://7d51-34-126-173-164.ngrok-free.app/health
• MCP RPC          https://7d51-34-126-173-164.ngrok-free.app/mcp
• Tools List       https://7d51-34-126-173-164.ngrok-free.app/mcp/tools/list
• Call Tool        https://7d51-34-126-173-164.ngrok-free.app/mcp/tools/call
• Download         https://7d51-34-126-173-164.ngrok-free.app/download/<file>
• Memory View      https://7d51-34-126-173-164.ngrok-free.app/memory/view
• Memory Update    https://7d51-34-126-173-164.ngrok-free.app/memory/update
• Memory Clear     https://7d51-34-126-173-164.ngrok-free.app/memory/clear
• OpenAPI Schema   https://7d51-34-126-173-164.ngrok-free.app/.well-known/tool-schema.json


In [ ]:
# ╔════════════════════════════════════════════════════════════════════════════╗
# ║ 🤖 Colab MCP Server v1.5 — Tool Call UI + GDrive + Download + Full Schema ║
# ╚════════════════════════════════════════════════════════════════════════════╝

# ─────────────────────────────────────────────────────────────────────────────
# 1. Install packages and mount Drive
# ─────────────────────────────────────────────────────────────────────────────
!pip install --quiet fastapi uvicorn pyngrok nest-asyncio sse-starlette requests

import nest_asyncio; nest_asyncio.apply()
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# ─────────────────────────────────────────────────────────────────────────────
# 2. Imports and Safe Directory Setup
# ─────────────────────────────────────────────────────────────────────────────
import os, json, asyncio, time, logging, subprocess, sys
from fastapi import FastAPI, Request, HTTPException
from fastapi.responses import HTMLResponse, FileResponse
from fastapi.middleware.cors import CORSMiddleware
from fastapi.staticfiles import StaticFiles
from sse_starlette.sse import EventSourceResponse
from pathlib import Path
from threading import Thread
from pyngrok import ngrok

SAFE_DIR = Path("/content/drive/MyDrive/ColabMCPFiles")
SAFE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(SAFE_DIR)

# ─────────────────────────────────────────────────────────────────────────────
# 3. Ngrok Setup and App Init
# ─────────────────────────────────────────────────────────────────────────────
ngrok.set_auth_token("2yuUKpiAlYXCez40i8UktLZ91s7_5Rv1GjG5HMFw5KKbk5sHz")

app = FastAPI(title="Colab MCP Tool Server", version="1.5")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"]
)

# ─────────────────────────────────────────────────────────────────────────────
# 4. Logging
# ─────────────────────────────────────────────────────────────────────────────
SERVER_LOG = SAFE_DIR / "server_log.txt"; SERVER_LOG.touch()
CHAT_LOG = SAFE_DIR / "chat_log.txt"; CHAT_LOG.touch()
subscribers: set[asyncio.Queue] = set()

class BroadcastHandler(logging.Handler):
    def emit(self, record):
        msg = self.format(record)
        with SERVER_LOG.open("a") as f: f.write(msg + "\n")
        for q in list(subscribers):
            try: asyncio.get_event_loop().call_soon_threadsafe(q.put_nowait, msg)
            except: pass

logger = logging.getLogger("mcp")
logger.setLevel(logging.INFO)
handler = BroadcastHandler()
handler.setFormatter(logging.Formatter("%(asctime)s %(message)s"))
logger.addHandler(handler)

# ─────────────────────────────────────────────────────────────────────────────
# 5. Tool Definitions
# ─────────────────────────────────────────────────────────────────────────────
TOOLS = [
    {
        "name": "execute_python",
        "description": "Run Python code and return its stdout output.",
        "inputSchema": {
            "type": "object",
            "properties": { "code": { "type": "string", "description": "Python code to execute." }},
            "required": ["code"]
        }
    },
    {
        "name": "install_package",
        "description": "Install a Python package via pip and return the install log.",
        "inputSchema": {
            "type": "object",
            "properties": { "package": { "type": "string", "description": "Name of pip package to install." }},
            "required": ["package"]
        }
    },
    {
        "name": "list_files",
        "description": "List files and folders in a specified directory (relative to working dir).",
        "inputSchema": {
            "type": "object",
            "properties": {
                "path": {
                    "type": "string",
                    "default": ".",
                    "description": "Directory path relative to root (default: '.')"
                }
            }
        }
    },
    {
        "name": "read_file",
        "description": "Read a UTF-8 text file from the working directory and return its contents.",
        "inputSchema": {
            "type": "object",
            "properties": { "filepath": { "type": "string", "description": "Path to file relative to working dir" }},
            "required": ["filepath"]
        }
    },
    {
        "name": "write_file",
        "description": "Write string content to a UTF-8 file in the working directory. Creates folders if needed.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "filepath": { "type": "string", "description": "Relative path of file to write" },
                "content": { "type": "string", "description": "Text content to write to file" }
            },
            "required": ["filepath", "content"]
        }
    },
    {
        "name": "print_file_text",
        "description": "Print the contents of a file to console or UI.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "filepath": {
                    "type": "string",
                    "description": "Relative path of the text file to print."
                }
            },
            "required": ["filepath"]
        }
    },
    {
        "name": "download_file",
        "description": "Generate a public download link for any file saved inside the working directory.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "filepath": {
                    "type": "string",
                    "description": "Path to file relative to working dir. Only valid if inside."
                }
            },
            "required": ["filepath"]
        }
    }
]

# ─────────────────────────────────────────────────────────────────────────────
# 6. Tool Logic
# ─────────────────────────────────────────────────────────────────────────────
def exec_py(code): from io import StringIO; import contextlib
    buf = StringIO()
    with contextlib.redirect_stdout(buf):
        try: exec(code, {}); return buf.getvalue() or "✓ No output"
        except Exception as e: return f"Error: {e}"

def inst_pkg(p): return subprocess.run([sys.executable, "-m", "pip", "install", p], capture_output=True, text=True).stdout

def safe_path(p): resolved = (SAFE_DIR / p).resolve(); assert resolved.is_relative_to(SAFE_DIR), "Unsafe path"; return resolved

def ls_files(path="."): p = safe_path(path)
    if p.is_file(): return p.read_text()
    return "\n".join(f"{'📄' if f.is_file() else '📁'} {f.name}" for f in sorted(p.iterdir()))

def rd_file(fp): return safe_path(fp).read_text()
def wr_file(fp, content): p = safe_path(fp); p.parent.mkdir(parents=True, exist_ok=True); p.write_text(content); return "✓ Written"
def make_dl_url(fp): safe_path(fp); return f"{public_url}/download/{fp}"

# ─────────────────────────────────────────────────────────────────────────────
# 7. FastAPI Routes
# ─────────────────────────────────────────────────────────────────────────────
@app.get("/")
async def root(): return {"message": "MCP Server v1.5 ready"}

@app.get("/health")
async def health(): return {"status": "ok", "tools": len(TOOLS)}

@app.post("/mcp")
async def mcp_rpc(req: Request):
    j = await req.json(); m = j.get("method"); a = j.get("params", {}).get("input") or {}
    try:
        if m == "tools/list": return {"jsonrpc": "2.0", "result": {"tools": TOOLS}, "id": j["id"]}
        elif m == "tools/call":
            n = j["params"]["name"]
            res = call_tool_impl(n, a)
            return {"jsonrpc": "2.0", "result": {"content": [{"type": "text", "text": res}]}, "id": j["id"]}
    except Exception as e:
        return {"jsonrpc": "2.0", "error": {"code": -1, "message": str(e)}, "id": j.get("id")}

def call_tool_impl(name, args):
    return {
        "execute_python": lambda: exec_py(args["code"]),
        "install_package": lambda: inst_pkg(args["package"]),
        "list_files": lambda: ls_files(args.get("path", ".")),
        "read_file": lambda: rd_file(args["filepath"]),
        "write_file": lambda: wr_file(args["filepath"], args["content"]),
        "print_file_text": lambda: rd_file(args["filepath"]),
        "download_file": lambda: make_dl_url(args["filepath"]),
    }[name]()

@app.post("/mcp/tools/call")
async def call_tool(req: Request):
    j = await req.json()
    try: return {"content": [{"type": "text", "text": call_tool_impl(j["name"], j.get("input", {}))}]}
    except Exception as e: return {"content": [{"type": "text", "text": f"Error: {e}"}]}

@app.get("/mcp/tools/list")
async def list_tools(): return {"tools": TOOLS}

@app.get("/download/{file_path:path}")
async def download_file(file_path: str):
    fp = safe_path(file_path)
    if not fp.exists(): raise HTTPException(404, "File not found")
    return FileResponse(fp)

@app.get("/site/toolchat.html")
async def serve_chat_ui():
    return HTMLResponse("""
    <html><body>
    <h2>MCP Tool Chat</h2>
    <form method="post" action="/mcp/tools/call">
      Tool: <input name="name"><br>
      Input (JSON): <textarea name="input" rows=6 cols=40></textarea><br>
      <input type="submit">
    </form>
    </body></html>
    """)

@app.get("/.well-known/tool-schema.json")
@app.get("/mcp/schema")
async def openapi(): return app.openapi()

# ─────────────────────────────────────────────────────────────────────────────
# 8. Start Server and Print Endpoints
# ─────────────────────────────────────────────────────────────────────────────
def start_server():
    Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000), daemon=True).start()
    time.sleep(2)
    public_url = ngrok.connect(8000).public_url
    print("\n🔗 Using ngrok:", public_url)
    print("📡 Tool Endpoints:")
    for n, p in [
        ("Tool List", "/mcp/tools/list"),
        ("Tool Call", "/mcp/tools/call"),
        ("Download File", "/download/<file>"),
        ("Tool Chat UI", "/site/toolchat.html"),
        ("Schema (Claude+OpenAI)", "/.well-known/tool-schema.json")
    ]:
        print(f"• {n:<18} {public_url}{p}")
    return public_url

public_url = start_server()

IndentationError: unexpected indent (ipython-input-3-1048110956.py, line 157)

In [ ]:
# ╔════════════════════════════════════════════════════════════════════════════╗
# ║     🤖 Colab MCP Server + Tool Chat UI + GDrive + File Tools (v1.5)       ║
# ║                                                                            ║
# ║  Colab-ready FastAPI server that exposes tool calls, file download/read,  ║
# ║  memory state, and generates OpenAPI schema with Action-friendly info.    ║
# ╚════════════════════════════════════════════════════════════════════════════╝

# ─────────────────────────────────────────────────────────────────────────────
# 1. Install required packages
# ─────────────────────────────────────────────────────────────────────────────
!pip install --quiet fastapi uvicorn pyngrok nest-asyncio sse-starlette

# ─────────────────────────────────────────────────────────────────────────────
# 2. Colab setup: asyncio patch, Google Drive mount, working dir
# ─────────────────────────────────────────────────────────────────────────────
import nest_asyncio; nest_asyncio.apply()
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

from pathlib import Path
import os
SAFE_DIR = Path("/content/drive/MyDrive/ColabMCPFiles")
SAFE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(SAFE_DIR)

# ─────────────────────────────────────────────────────────────────────────────
# 3. Imports: FastAPI, tools, logging, etc.
# ─────────────────────────────────────────────────────────────────────────────
import time, json, logging, asyncio, subprocess, sys
from fastapi import FastAPI, Request, HTTPException
from fastapi.responses import FileResponse, HTMLResponse
from fastapi.staticfiles import StaticFiles
from fastapi.middleware.cors import CORSMiddleware
from sse_starlette.sse import EventSourceResponse
import uvicorn
from threading import Thread
from pyngrok import ngrok

# ─────────────────────────────────────────────────────────────────────────────
# 4. Ngrok token
# ─────────────────────────────────────────────────────────────────────────────
ngrok.set_auth_token("YOUR_NGROK_TOKEN_HERE")

# ─────────────────────────────────────────────────────────────────────────────
# 5. Initialize FastAPI app
# ─────────────────────────────────────────────────────────────────────────────
app = FastAPI(title="Colab MCP Tool Server", version="1.5", description="A Colab-ready FastAPI server with code execution, file tools, and memory.")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"]
)

# ─────────────────────────────────────────────────────────────────────────────
# 6. Logging setup
# ─────────────────────────────────────────────────────────────────────────────
SERVER_LOG = SAFE_DIR / "server_log.txt"; SERVER_LOG.touch()
CHAT_LOG   = SAFE_DIR / "chat_log.txt";   CHAT_LOG.touch()
subscribers: set[asyncio.Queue] = set()

class BroadcastHandler(logging.Handler):
    def emit(self, record):
        msg = self.format(record)
        with SERVER_LOG.open("a") as f: f.write(msg + "\n")
        for q in list(subscribers):
            try: asyncio.get_event_loop().call_soon_threadsafe(q.put_nowait, msg)
            except: pass

logger = logging.getLogger("mcp")
logger.setLevel(logging.INFO)
handler = BroadcastHandler()
handler.setFormatter(logging.Formatter("%(asctime)s %(message)s"))
logger.addHandler(handler)

# ─────────────────────────────────────────────────────────────────────────────
# 7. Tool logic
# ─────────────────────────────────────────────────────────────────────────────
def exec_py(code: str) -> str:
    from io import StringIO
    import contextlib
    buf = StringIO()
    try:
        with contextlib.redirect_stdout(buf): exec(code, {})
        return buf.getvalue() or "✓ no output"
    except Exception as e: return f"Execution error: {e}"

def inst_pkg(pkg: str) -> str:
    p = subprocess.run([sys.executable, "-m", "pip", "install", pkg], capture_output=True, text=True)
    return p.stdout or p.stderr

def safe_path(p: str) -> Path:
    target = SAFE_DIR / p
    if not target.resolve().is_relative_to(SAFE_DIR.resolve()):
        raise ValueError("Unsafe path.")
    return target

def ls_files(path: str=".") -> str:
    p = safe_path(path)
    if not p.exists(): return f"Not found: {path}"
    if p.is_file(): return p.read_text()
    return "\n".join(f"{'📄' if it.is_file() else '📁'} {it.name}{'/' if it.is_dir() else ''}" for it in sorted(p.iterdir())) or "— empty"

def rd_file(fp: str) -> str: return safe_path(fp).read_text()

def wr_file(fp: str, content: str) -> str:
    p = safe_path(fp)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content)
    return f"Written {p.name}"

# ─────────────────────────────────────────────────────────────────────────────
# 8. Tool schema
# ─────────────────────────────────────────────────────────────────────────────
TOOLS = [
    {"name":"execute_python","description":"Run Python code and return output","inputSchema":{"type":"object","properties":{"code":{"type":"string"}},"required":["code"]}},
    {"name":"install_package","description":"Install a pip package by name","inputSchema":{"type":"object","properties":{"package":{"type":"string"}},"required":["package"]}},
    {"name":"list_files","description":"List contents of a path in the working folder","inputSchema":{"type":"object","properties":{"path":{"type":"string"}}}},
    {"name":"read_file","description":"Read a file (path must be inside working folder)","inputSchema":{"type":"object","properties":{"filepath":{"type":"string"}},"required":["filepath"]}},
    {"name":"write_file","description":"Write a string to a file","inputSchema":{"type":"object","properties":{"filepath":{"type":"string"},"content":{"type":"string"}},"required":["filepath","content"]}},
]

@app.get("/mcp/tools/list")
async def list_tools(): return {"tools": TOOLS}

@app.post("/mcp/tools/call")
async def call_tool(req: Request):
    data = await req.json()
    name, args = data.get("name"), data.get("arguments",{})
    logger.info(f"CALL {name} {args!r}")
    try:
        if name=="execute_python": out = exec_py(args["code"])
        elif name=="install_package": out = inst_pkg(args["package"])
        elif name=="list_files": out = ls_files(args.get("path","."))
        elif name=="read_file": out = rd_file(args["filepath"])
        elif name=="write_file": out = wr_file(args["filepath"], args["content"])
        else: raise Exception("Unknown tool")
        return {"content":[{"type":"text","text":out}],"isError":False}
    except Exception as e:
        return {"content":[{"type":"text","text":f"Error: {e}"}],"isError":True}

@app.get("/download/{file_path:path}")
async def download_file(file_path: str):
    fp = safe_path(file_path)
    if not fp.exists() or not fp.is_file():
        raise HTTPException(404,"File not found or outside safe directory.")
    return FileResponse(fp)

# ─────────────────────────────────────────────────────────────────────────────
# 9. Serve toolchat.html
# ─────────────────────────────────────────────────────────────────────────────
SITE = SAFE_DIR/"site"; SITE.mkdir(exist_ok=True)
(SITE/"toolchat.html").write_text(f"""
<!DOCTYPE html><html><head><meta charset="utf-8"><title>Tool Chat</title>
<style>body{{font:14px sans;max-width:700px;margin:1em auto}}textarea{{width:100%;height:6em}}</style>
</head><body><h1>Tool Chat</h1><pre style="background:#f8f8f8;padding:8px;border:1px solid #ccc">
{json.dumps(TOOLS, indent=2)}
</pre><textarea id="req">{{"name":"execute_python","arguments":{{"code":"print(2+2)"}}}}</textarea><br>
<button onclick="run()">Run</button><pre id="out" style="background:#eef;border:1px solid #ccc;padding:8px;height:200px;overflow:auto"></pre>
<script>
async function run() {{
  const r = JSON.parse(document.getElementById('req').value);
  const res = await fetch('/mcp/tools/call', {{
    method:'POST', headers:{{'Content-Type':'application/json'}}, body:JSON.stringify(r)
  }});
  document.getElementById('out').textContent = JSON.stringify(await res.json(), null, 2);
}}
</script></body></html>
""", encoding="utf-8")
app.mount("/site", StaticFiles(directory=SITE), name="site")

# ─────────────────────────────────────────────────────────────────────────────
# 10. Start server & expose
# ─────────────────────────────────────────────────────────────────────────────
def start_server():
    Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info"), daemon=True).start()
    time.sleep(2)
    url = ngrok.connect(8000).public_url
    print(f"🔗 Public URL: {url}")
    print(f"• Toolchat UI: {url}/site/toolchat.html")
    print(f"• Download: {url}/download/yourfile.txt")
    print(f"• Tool Call: {url}/mcp/tools/call")
    return url

public_url = start_server()

Mounted at /content/drive


INFO:     Started server process [238]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
ERROR:pyngrok.process.ngrok:t=2025-06-28T17:58:15+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: The authtoken you specified does not look like a proper ngrok tunnel authtoken.\nYour authtoken: YOUR_NGROK_TOKEN_HERE\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_105\r\n"
ERROR:pyngrok.process.ngrok:t=2025-06-28T17:58:15+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: The authtoken you specified does not look like a proper ngrok tunnel authtoken.\nYour authtoken: YOUR_NGROK_TOKEN_HERE\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_N

PyngrokNgrokError: The ngrok process errored on start: authentication failed: The authtoken you specified does not look like a proper ngrok tunnel authtoken.\nYour authtoken: YOUR_NGROK_TOKEN_HERE\nInstructions to install your authtoken are on your ngrok dashboard:\nhttps://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_105\r\n.

In [ ]:
# ╔════════════════════════════════════════════════════════════════════════════════════╗
# ║ 🤖 Colab MCP Server v1.5 — Full FastAPI Tool Server with Claude & OpenAI Support  ║
# ║                                                                                    ║
# ║ • Includes /mcp, /mcp/tools/call, /memory routes, /download/{file_path},          ║
# ║   /site/toolchat.html, and OpenAPI schema with readable tool descriptions.        ║
# ║ • Sandboxes file access to Google Drive.                                          ║
# ║ • Designed for compatibility with Claude and OpenAI plugins (ActionsGPT).         ║
# ╚════════════════════════════════════════════════════════════════════════════════════╝

# ─────────────────────────────────────────────────────────────────────────────
# 1. Install dependencies
# ─────────────────────────────────────────────────────────────────────────────
!pip install --quiet fastapi uvicorn pyngrok nest-asyncio sse-starlette requests

# ─────────────────────────────────────────────────────────────────────────────
# 2. Mount Google Drive
# ─────────────────────────────────────────────────────────────────────────────
import nest_asyncio; nest_asyncio.apply()
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

# ─────────────────────────────────────────────────────────────────────────────
# 3. Imports and server setup
# ─────────────────────────────────────────────────────────────────────────────
import os, time, json, asyncio, logging, subprocess, sys
from pathlib import Path
from threading import Thread
from fastapi import FastAPI, Request, HTTPException
from fastapi.responses import FileResponse, HTMLResponse
from fastapi.middleware.cors import CORSMiddleware
from fastapi.staticfiles import StaticFiles
from sse_starlette.sse import EventSourceResponse
from pyngrok import ngrok

SAFE_DIR = Path("/content/drive/MyDrive/ColabMCPFiles")
SAFE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(SAFE_DIR)

ngrok.set_auth_token("2yuUKpiAlYXCez40i8UktLZ91s7_5Rv1GjG5HMFw5KKbk5sHz")

app = FastAPI(title="Colab MCP Tool Server", version="1.5")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])
app.mount("/site", StaticFiles(directory=SAFE_DIR / "site", html=True), name="site")

# ─────────────────────────────────────────────────────────────────────────────
# 4. Logging for tool responses and memory
# ─────────────────────────────────────────────────────────────────────────────
SERVER_LOG = SAFE_DIR / "server_log.txt"; SERVER_LOG.touch()
CHAT_LOG   = SAFE_DIR / "chat_log.txt";   CHAT_LOG.touch()
subscribers: set[asyncio.Queue] = set()

class BroadcastHandler(logging.Handler):
    def emit(self, record):
        msg = self.format(record)
        with SERVER_LOG.open("a") as f: f.write(msg + "\n")
        for q in list(subscribers):
            try: asyncio.get_event_loop().call_soon_threadsafe(q.put_nowait, msg)
            except: pass

logger = logging.getLogger("mcp")
logger.setLevel(logging.INFO)
handler = BroadcastHandler()
handler.setFormatter(logging.Formatter("%(asctime)s %(message)s"))
logger.addHandler(handler)

# ─────────────────────────────────────────────────────────────────────────────
# 5. Tool definitions
# ─────────────────────────────────────────────────────────────────────────────
TOOLS = [
    {
        "name": "execute_python",
        "description": "Execute Python code and return its stdout output.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "code": {"type": "string", "description": "Python code to execute."}
            },
            "required": ["code"]
        }
    },
    {
        "name": "install_package",
        "description": "Install a pip package and return the installation output.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "package": {"type": "string", "description": "Name of pip package to install."}
            },
            "required": ["package"]
        }
    },
    {
        "name": "list_files",
        "description": "List all files and folders in a given directory inside the sandboxed area.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "path": {"type": "string", "description": "Relative directory path.", "default": "."}
            }
        }
    },
    {
        "name": "read_file",
        "description": "Read and return the contents of a UTF-8 text file from within the sandbox.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "filepath": {"type": "string", "description": "Path to file within sandbox."}
            },
            "required": ["filepath"]
        }
    },
    {
        "name": "write_file",
        "description": "Write a UTF-8 string to a file within the sandboxed directory.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "filepath": {"type": "string"},
                "content": {"type": "string"}
            },
            "required": ["filepath", "content"]
        }
    },
    {
        "name": "print_file_text",
        "description": "Print and return plain-text contents of a file (same as read_file).",
        "inputSchema": {
            "type": "object",
            "properties": {
                "filepath": {"type": "string"}
            },
            "required": ["filepath"]
        }
    },
    {
        "name": "download_file",
        "description": "Generate a secure public download link to a file stored in the sandbox. "
                       "Use this to let users download previously written files. "
                       "Files are served from: /download/<filename>.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "filepath": {"type": "string"}
            },
            "required": ["filepath"]
        }
    }
]

# ─────────────────────────────────────────────────────────────────────────────
# 6. Tool implementations
# ─────────────────────────────────────────────────────────────────────────────
def exec_py(code: str) -> str:
    from io import StringIO
    import contextlib
    buf = StringIO()
    try:
        with contextlib.redirect_stdout(buf):
            exec(code, {})
        return buf.getvalue() or "✓ Code ran with no output."
    except Exception as e:
        return f"❌ Error: {e}"

def inst_pkg(pkg: str) -> str:
    result = subprocess.run([sys.executable, "-m", "pip", "install", pkg], capture_output=True, text=True)
    return result.stdout or result.stderr

def safe_path(p: str) -> Path:
    target = SAFE_DIR / p
    if not target.resolve().is_relative_to(SAFE_DIR.resolve()):
        raise ValueError("Unsafe path.")
    return target

def ls_files(path: str = ".") -> str:
    p = safe_path(path)
    if not p.exists(): return "⚠️ Not found."
    if p.is_file(): return p.read_text()
    return "\n".join(
        f"{'📄' if it.is_file() else '📁'} {it.name}/" if it.is_dir() else f"{it.name}"
        for it in sorted(p.iterdir())
    ) or "(empty folder)"

def rd_file(fp: str) -> str:
    return safe_path(fp).read_text()

def wr_file(fp: str, content: str) -> str:
    p = safe_path(fp)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content)
    return f"✅ Written to {p.name}"

# ─────────────────────────────────────────────────────────────────────────────
# 7. API routes
# ─────────────────────────────────────────────────────────────────────────────
@app.get("/")
async def root(): return {"message": "MCP server running.", "version": "1.5"}

@app.get("/health")
async def health(): return {"status": "ok", "tools": len(TOOLS)}

@app.get("/memory/view")
async def memory_view():
    mem_path = SAFE_DIR / "memory.json"
    if not mem_path.exists(): mem_path.write_text("{}")
    return json.loads(mem_path.read_text())

@app.post("/memory/update")
async def memory_update(req: Request):
    mem_path = SAFE_DIR / "memory.json"
    mem = json.loads(mem_path.read_text()) if mem_path.exists() else {}
    update = await req.json()
    mem.update(update)
    mem_path.write_text(json.dumps(mem, indent=2))
    return {"message": "Memory updated."}

@app.post("/memory/clear")
async def memory_clear():
    mem_path = SAFE_DIR / "memory.json"
    mem_path.write_text("{}")
    return {"message": "Memory cleared."}

@app.get("/download/{file_path:path}")
async def download_file(file_path: str):
    fp = safe_path(file_path)
    if not fp.exists(): raise HTTPException(404, "File not found")
    return FileResponse(fp)

@app.post("/mcp/tools/call")
async def call_tool(req: Request):
    data = await req.json()
    name = data.get("name")
    args = data.get("input") or {}
    logger.info(f"CALL {name} {args!r}")
    try:
        if name == "execute_python": out = exec_py(args["code"])
        elif name == "install_package": out = inst_pkg(args["package"])
        elif name == "list_files": out = ls_files(args.get("path", "."))
        elif name == "read_file": out = rd_file(args["filepath"])
        elif name == "write_file": out = wr_file(args["filepath"], args["content"])
        elif name == "print_file_text": out = rd_file(args["filepath"])
        elif name == "download_file":
            safe_path(args["filepath"])  # validate
            out = f"{public_url}/download/{args['filepath']}"
        else: raise Exception("Unknown tool")
        return {"content": [{"type": "text", "text": out}]}
    except Exception as e:
        return {"content": [{"type": "text", "text": f"Error: {e}"}], "isError": True}

@app.get("/.well-known/tool-schema.json")
@app.get("/mcp/schema")
async def openapi_schema():
    return app.openapi()

# ─────────────────────────────────────────────────────────────────────────────
# 8. Start Uvicorn server and Ngrok
# ─────────────────────────────────────────────────────────────────────────────
def start_server():
    Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000), daemon=True).start()
    time.sleep(2)
    url = ngrok.connect(8000).public_url
    print(f"\n🔗 MCP Server Public URL: {url}")
    print("📡 Key Endpoints:")
    for label, route in [
        ("Health", "/health"),
        ("MCP Tool Call", "/mcp/tools/call"),
        ("Download File", "/download/<file>"),
        ("Memory View", "/memory/view"),
        ("Tool Chat UI", "/site/toolchat.html"),
        ("OpenAPI Schema", "/mcp/schema"),
    ]:
        print(f"• {label:<20} {url}{route}")
    return url

public_url = start_server()

Exception in thread Thread-12 (<lambda>):
Traceback (most recent call last):
  File "/usr/lib/python3.11/threading.py", line 1045, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.11/threading.py", line 982, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipython-input-2-665449556.py", line 259, in <lambda>
NameError: name 'uvicorn' is not defined


Mounted at /content/drive

🔗 MCP Server Public URL: https://10de-34-126-91-119.ngrok-free.app
📡 Key Endpoints:
• Health               https://10de-34-126-91-119.ngrok-free.app/health
• MCP Tool Call        https://10de-34-126-91-119.ngrok-free.app/mcp/tools/call
• Download File        https://10de-34-126-91-119.ngrok-free.app/download/<file>
• Memory View          https://10de-34-126-91-119.ngrok-free.app/memory/view
• Tool Chat UI         https://10de-34-126-91-119.ngrok-free.app/site/toolchat.html
• OpenAPI Schema       https://10de-34-126-91-119.ngrok-free.app/mcp/schema


In [ ]:
# ╔════════════════════════════════════════════════════════════════════════════════════╗
# ║ 🤖 Colab MCP Server v1.6 — Full FastAPI Tool Server with Claude & OpenAI Support  ║
# ║                                                                                    ║
# ║ • Includes /mcp, /mcp/tools/call, /memory routes, /download/{file_path},          ║
# ║   /logs/view, /site/toolchat.html, and full OpenAPI schema with tool descriptions.║
# ║ • Sandboxes file access to Google Drive.                                          ║
# ║ • Designed for compatibility with Claude and OpenAI plugins (ActionsGPT).         ║
# ╚════════════════════════════════════════════════════════════════════════════════════╝

# ─────────────────────────────────────────────────────────────────────────────
# 1. Install dependencies
# ─────────────────────────────────────────────────────────────────────────────
!pip install --quiet fastapi uvicorn pyngrok nest-asyncio sse-starlette requests

# ─────────────────────────────────────────────────────────────────────────────
# 2. Mount Google Drive
# ─────────────────────────────────────────────────────────────────────────────
import nest_asyncio; nest_asyncio.apply()
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

# ─────────────────────────────────────────────────────────────────────────────
# 3. Imports and server setup
# ─────────────────────────────────────────────────────────────────────────────
import os, time, json, asyncio, logging, subprocess, sys
import uvicorn
from pathlib import Path
from threading import Thread
from fastapi import FastAPI, Request, HTTPException
from fastapi.responses import FileResponse, HTMLResponse
from fastapi.middleware.cors import CORSMiddleware
from fastapi.staticfiles import StaticFiles
from sse_starlette.sse import EventSourceResponse
from pyngrok import ngrok

SAFE_DIR = Path("/content/drive/MyDrive/ColabMCPFiles")
SAFE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(SAFE_DIR)

ngrok.set_auth_token("2yuUKpiAlYXCez40i8UktLZ91s7_5Rv1GjG5HMFw5KKbk5sHz")

app = FastAPI(title="Colab MCP Tool Server", version="1.6")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])
app.mount("/site", StaticFiles(directory=SAFE_DIR / "site", html=True), name="site")
# ─────────────────────────────────────────────────────────────────────────────
# 4. Logging system and broadcast
# ─────────────────────────────────────────────────────────────────────────────
SERVER_LOG = SAFE_DIR / "server_log.txt"; SERVER_LOG.touch()
CHAT_LOG   = SAFE_DIR / "chat_log.txt";   CHAT_LOG.touch()
subscribers: set[asyncio.Queue] = set()

class BroadcastHandler(logging.Handler):
    def emit(self, record):
        msg = self.format(record)
        with SERVER_LOG.open("a") as f: f.write(msg + "\n")
        for q in list(subscribers):
            try: asyncio.get_event_loop().call_soon_threadsafe(q.put_nowait, msg)
            except: pass

logger = logging.getLogger("mcp")
logger.setLevel(logging.INFO)
handler = BroadcastHandler()
handler.setFormatter(logging.Formatter("%(asctime)s %(message)s"))
logger.addHandler(handler)

# ─────────────────────────────────────────────────────────────────────────────
# 5. Tool definitions
# ─────────────────────────────────────────────────────────────────────────────
TOOLS = [
    {
        "name": "execute_python",
        "description": "Execute arbitrary Python code and return its output as a string.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "code": {"type": "string", "description": "Python code to execute."}
            },
            "required": ["code"]
        }
    },
    {
        "name": "install_package",
        "description": "Install a pip package using subprocess and return output.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "package": {"type": "string", "description": "Name of pip package to install."}
            },
            "required": ["package"]
        }
    },
    {
        "name": "list_files",
        "description": "List files and folders inside a given directory in the sandbox.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "path": {"type": "string", "description": "Relative path inside sandbox.", "default": "."}
            }
        }
    },
    {
        "name": "read_file",
        "description": "Return full contents of a UTF-8 file within the sandbox.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "filepath": {"type": "string", "description": "Relative file path to read."}
            },
            "required": ["filepath"]
        }
    },
    {
        "name": "write_file",
        "description": "Write content to a file inside the sandboxed directory.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "filepath": {"type": "string", "description": "File path to write to."},
                "content": {"type": "string", "description": "Text content to write."}
            },
            "required": ["filepath", "content"]
        }
    },
    {
        "name": "print_file_text",
        "description": "Alias of read_file. Prints the full contents of a UTF-8 file.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "filepath": {"type": "string", "description": "Relative file path."}
            },
            "required": ["filepath"]
        }
    },
    {
        "name": "download_file",
        "description": "Generates a direct download URL to a file in the sandbox. "
                       "Use the resulting link from /download/<filepath> to fetch the file.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "filepath": {"type": "string", "description": "Relative file path."}
            },
            "required": ["filepath"]
        }
    }
]

# ─────────────────────────────────────────────────────────────────────────────
# 6. Tool implementations
# ─────────────────────────────────────────────────────────────────────────────
def exec_py(code: str) -> str:
    from io import StringIO
    import contextlib
    buf = StringIO()
    try:
        with contextlib.redirect_stdout(buf):
            exec(code, {})
        return buf.getvalue() or "✓ Code ran with no output."
    except Exception as e:
        return f"❌ Error: {e}"

def inst_pkg(pkg: str) -> str:
    result = subprocess.run([sys.executable, "-m", "pip", "install", pkg], capture_output=True, text=True)
    return result.stdout or result.stderr

def safe_path(p: str) -> Path:
    target = SAFE_DIR / p
    if not target.resolve().is_relative_to(SAFE_DIR.resolve()):
        raise ValueError("Unsafe path.")
    return target

def ls_files(path: str = ".") -> str:
    p = safe_path(path)
    if not p.exists(): return "⚠️ Not found."
    if p.is_file(): return p.read_text()
    return "\n".join(
        f"{'📄' if it.is_file() else '📁'} {it.name}/" if it.is_dir() else f"{it.name}"
        for it in sorted(p.iterdir())
    ) or "(empty folder)"

def rd_file(fp: str) -> str:
    return safe_path(fp).read_text()

def wr_file(fp: str, content: str) -> str:
    p = safe_path(fp)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content)
    return f"✅ Written to {p.name}"



Mounted at /content/drive


In [ ]:
# ╔════════════════════════════════════════════════════════════════════════════════════╗
# ║ 🤖 Colab MCP Server v1.6 — Full FastAPI Tool Server with Claude & OpenAI Support  ║
# ║                                                                                    ║
# ║ • Includes /mcp, /mcp/tools/call, /memory routes, /download/{file_path},          ║
# ║   /logs/view, /site/toolchat.html, and full OpenAPI schema with tool descriptions.║
# ║ • Sandboxes file access to Google Drive.                                          ║
# ║ • Designed for compatibility with Claude and OpenAI plugins (ActionsGPT).         ║
# ╚════════════════════════════════════════════════════════════════════════════════════╝

# ─────────────────────────────────────────────────────────────────────────────
# 1. Install dependencies
# ─────────────────────────────────────────────────────────────────────────────
!pip install --quiet fastapi uvicorn pyngrok nest-asyncio sse-starlette requests

# ─────────────────────────────────────────────────────────────────────────────
# 2. Mount Google Drive
# ─────────────────────────────────────────────────────────────────────────────
import nest_asyncio; nest_asyncio.apply()
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

# ─────────────────────────────────────────────────────────────────────────────
# 3. Imports and server setup
# ─────────────────────────────────────────────────────────────────────────────
import os, time, json, asyncio, logging, subprocess, sys
import uvicorn
from pathlib import Path
from threading import Thread
from fastapi import FastAPI, Request, HTTPException
from fastapi.responses import FileResponse, HTMLResponse
from fastapi.middleware.cors import CORSMiddleware
from fastapi.staticfiles import StaticFiles
from sse_starlette.sse import EventSourceResponse
from pyngrok import ngrok

SAFE_DIR = Path("/content/drive/MyDrive/ColabMCPFiles")
SAFE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(SAFE_DIR)

ngrok.set_auth_token("2yuUKpiAlYXCez40i8UktLZ91s7_5Rv1GjG5HMFw5KKbk5sHz")

app = FastAPI(title="Colab MCP Tool Server", version="1.6")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])
app.mount("/site", StaticFiles(directory=SAFE_DIR / "site", html=True), name="site")

# ─────────────────────────────────────────────────────────────────────────────
# 4. Logging system and broadcast
# ─────────────────────────────────────────────────────────────────────────────
SERVER_LOG = SAFE_DIR / "server_log.txt"; SERVER_LOG.touch()
CHAT_LOG   = SAFE_DIR / "chat_log.txt";   CHAT_LOG.touch()
subscribers: set[asyncio.Queue] = set()

class BroadcastHandler(logging.Handler):
    def emit(self, record):
        msg = self.format(record)
        with SERVER_LOG.open("a") as f: f.write(msg + "\n")
        for q in list(subscribers):
            try: asyncio.get_event_loop().call_soon_threadsafe(q.put_nowait, msg)
            except: pass

logger = logging.getLogger("mcp")
logger.setLevel(logging.INFO)
handler = BroadcastHandler()
handler.setFormatter(logging.Formatter("%(asctime)s %(message)s"))
logger.addHandler(handler)

# ─────────────────────────────────────────────────────────────────────────────
# 5. Tool definitions
# ─────────────────────────────────────────────────────────────────────────────
TOOLS = [
    {
        "name": "execute_python",
        "description": "Execute arbitrary Python code and return its output as a string.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "code": {"type": "string", "description": "Python code to execute."}
            },
            "required": ["code"]
        }
    },
    {
        "name": "install_package",
        "description": "Install a pip package using subprocess and return output.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "package": {"type": "string", "description": "Name of pip package to install."}
            },
            "required": ["package"]
        }
    },
    {
        "name": "list_files",
        "description": "List files and folders inside a given directory in the sandbox.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "path": {"type": "string", "description": "Relative path inside sandbox.", "default": "."}
            }
        }
    },
    {
        "name": "read_file",
        "description": "Return full contents of a UTF-8 file within the sandbox.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "filepath": {"type": "string", "description": "Relative file path to read."}
            },
            "required": ["filepath"]
        }
    },
    {
        "name": "write_file",
        "description": "Write content to a file inside the sandboxed directory.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "filepath": {"type": "string", "description": "File path to write to."},
                "content": {"type": "string", "description": "Text content to write."}
            },
            "required": ["filepath", "content"]
        }
    },
    {
        "name": "print_file_text",
        "description": "Alias of read_file. Prints the full contents of a UTF-8 file.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "filepath": {"type": "string", "description": "Relative file path."}
            },
            "required": ["filepath"]
        }
    },
    {
        "name": "download_file",
        "description": "Generates a direct download URL to a file in the sandbox. "
                       "Use the resulting link from /download/<filepath> to fetch the file.",
        "inputSchema": {
            "type": "object",
            "properties": {
                "filepath": {"type": "string", "description": "Relative file path."}
            },
            "required": ["filepath"]
        }
    }
]

# ─────────────────────────────────────────────────────────────────────────────
# 6. Tool implementations
# ─────────────────────────────────────────────────────────────────────────────
def exec_py(code: str) -> str:
    from io import StringIO
    import contextlib
    buf = StringIO()
    try:
        with contextlib.redirect_stdout(buf):
            exec(code, {})
        return buf.getvalue() or "✓ Code ran with no output."
    except Exception as e:
        return f"❌ Error: {e}"

def inst_pkg(pkg: str) -> str:
    result = subprocess.run([sys.executable, "-m", "pip", "install", pkg], capture_output=True, text=True)
    return result.stdout or result.stderr

def safe_path(p: str) -> Path:
    target = SAFE_DIR / p
    if not target.resolve().is_relative_to(SAFE_DIR.resolve()):
        raise ValueError("Unsafe path.")
    return target

def ls_files(path: str = ".") -> str:
    p = safe_path(path)
    if not p.exists(): return "⚠️ Not found."
    if p.is_file(): return p.read_text()
    return "\n".join(
        f"{'📄' if it.is_file() else '📁'} {it.name}/" if it.is_dir() else f"{it.name}"
        for it in sorted(p.iterdir())
    ) or "(empty folder)"

def rd_file(fp: str) -> str:
    return safe_path(fp).read_text()

def wr_file(fp: str, content: str) -> str:
    p = safe_path(fp)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content)
    return f"✅ Written to {p.name}"

# ─────────────────────────────────────────────────────────────────────────────
# 7. API Routes
# ─────────────────────────────────────────────────────────────────────────────

@app.get("/")
async def root():
    return {"message": "MCP server running.", "version": "1.6"}

@app.get("/health")
async def health():
    return {"status": "ok", "tools": len(TOOLS)}

@app.get("/logs/view")
async def stream_logs(request: Request):
    queue = asyncio.Queue()
    subscribers.add(queue)
    try:
        with SERVER_LOG.open("r") as f:
            history = "".join(f.readlines()[-40:])
        async def log_generator():
            yield f"data: {history}\n\n"
            while True:
                if await request.is_disconnected():
                    break
                try:
                    msg = await asyncio.wait_for(queue.get(), timeout=30.0)
                    yield f"data: {msg}\n\n"
                except asyncio.TimeoutError:
                    continue
        return EventSourceResponse(log_generator())
    finally:
        subscribers.discard(queue)

@app.get("/memory/view")
async def memory_view():
    mem_path = SAFE_DIR / "memory.json"
    if not mem_path.exists():
        mem_path.write_text("{}")
    return json.loads(mem_path.read_text())

@app.get("/memory/summary")
async def memory_summary():
    mem_path = SAFE_DIR / "memory.json"
    if not mem_path.exists(): return {"summary": "(empty)"}
    try:
        data = json.loads(mem_path.read_text())
        return {"summary": data.get("summary", "(no summary key)")}
    except:
        return {"summary": "(invalid JSON)"}

@app.post("/memory/update")
async def memory_update(req: Request):
    mem_path = SAFE_DIR / "memory.json"
    mem = json.loads(mem_path.read_text()) if mem_path.exists() else {}
    update = await req.json()
    mem.update(update)
    mem_path.write_text(json.dumps(mem, indent=2))
    return mem

@app.post("/memory/clear")
async def memory_clear():
    mem_path = SAFE_DIR / "memory.json"
    mem_path.write_text("{}")
    return {"status": "cleared"}

@app.get("/download/{file_path:path}")
async def download_file(file_path: str):
    fp = safe_path(file_path)
    if not fp.exists():
        raise HTTPException(status_code=404, detail="File not found")
    return FileResponse(fp)

@app.get("/mcp/tools/list")
async def list_tools():
    return {"tools": TOOLS}

@app.post("/mcp/tools/call")
async def call_tool(req: Request):
    data = await req.json()
    name = data.get("name")
    args = data.get("input") or data.get("arguments") or {}
    logger.info(f"CALL {name} {args!r}")
    try:
        if name == "execute_python": out = exec_py(args["code"])
        elif name == "install_package": out = inst_pkg(args["package"])
        elif name == "list_files": out = ls_files(args.get("path", "."))
        elif name == "read_file": out = rd_file(args["filepath"])
        elif name == "write_file": out = wr_file(args["filepath"], args["content"])
        elif name == "print_file_text": out = rd_file(args["filepath"])
        elif name == "download_file":
            safe_path(args["filepath"])  # validate only
            out = f"{public_url}/download/{args['filepath']}"
        else:
            raise Exception("Unknown tool")
        return {"content": [{"type": "text", "text": out}]}
    except Exception as e:
        return {"content": [{"type": "text", "text": f"❌ Error: {e}"}], "isError": True}

@app.get("/.well-known/tool-schema.json")
@app.get("/mcp/schema")
async def openapi_schema():
    return app.openapi()

# ─────────────────────────────────────────────────────────────────────────────
# 8. Start server and print endpoints
# ─────────────────────────────────────────────────────────────────────────────

def start_server():
    Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000), daemon=True).start()
    time.sleep(2)
    url = ngrok.connect(8000).public_url
    print(f"\n🔗 MCP Server Public URL: {url}")
    print("📡 Key Endpoints:")
    for label, route in [
        ("Health", "/health"),
        ("Logs View", "/logs/view"),
        ("MCP Tool Call", "/mcp/tools/call"),
        ("Download File", "/download/<file>"),
        ("Memory View", "/memory/view"),
        ("Memory Summary", "/memory/summary"),
        ("Tool Chat UI", "/site/toolchat.html"),
        ("OpenAPI Schema", "/mcp/schema"),
        ("Claude Plugin", "/.well-known/tool-schema.json")
    ]:
        print(f"• {label:<20} {url}{route}")
    return url

public_url = start_server()

INFO:     Started server process [3368]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Mounted at /content/drive

🔗 MCP Server Public URL: https://89ea-35-189-186-49.ngrok-free.app
📡 Key Endpoints:
• Health               https://89ea-35-189-186-49.ngrok-free.app/health
• Logs View            https://89ea-35-189-186-49.ngrok-free.app/logs/view
• MCP Tool Call        https://89ea-35-189-186-49.ngrok-free.app/mcp/tools/call
• Download File        https://89ea-35-189-186-49.ngrok-free.app/download/<file>
• Memory View          https://89ea-35-189-186-49.ngrok-free.app/memory/view
• Memory Summary       https://89ea-35-189-186-49.ngrok-free.app/memory/summary
• Tool Chat UI         https://89ea-35-189-186-49.ngrok-free.app/site/toolchat.html
• OpenAPI Schema       https://89ea-35-189-186-49.ngrok-free.app/mcp/schema
• Claude Plugin        https://89ea-35-189-186-49.ngrok-free.app/.well-known/tool-schema.json


In [ ]:
# ╔════════════════════════════════════════════════════════════════════════════════════╗
# ║ 🤖 Colab MCP Server v1.7 — Streaming Python + Claude/OpenAI Plugin Schema Server  ║
# ║                                                                                    ║
# ║ • Adds real-time code execution streaming, plugin manifest endpoints,             ║
# ║   OpenAPI 3.1.0 schema, /logs/view, /memory/keys, and sandboxed I/O tools.         ║
# ║ • Designed for Claude ActionsGPT and OpenAI tools compatibility.                  ║
# ╚════════════════════════════════════════════════════════════════════════════════════╝

# ─────────────────────────────────────────────────────────────────────────────
# 1. Install deps (in Colab)
# ─────────────────────────────────────────────────────────────────────────────
!pip install --quiet fastapi uvicorn pyngrok nest-asyncio sse-starlette requests

# ─────────────────────────────────────────────────────────────────────────────
# 2. Google Drive mount
# ─────────────────────────────────────────────────────────────────────────────
import nest_asyncio; nest_asyncio.apply()
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

# ─────────────────────────────────────────────────────────────────────────────
# 3. Imports and setup
# ─────────────────────────────────────────────────────────────────────────────
import os, json, time, asyncio, logging, subprocess, sys
from pathlib import Path
from threading import Thread
from fastapi import FastAPI, Request, HTTPException
from fastapi.responses import FileResponse, HTMLResponse, JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from fastapi.staticfiles import StaticFiles
from sse_starlette.sse import EventSourceResponse
from pyngrok import ngrok
import uvicorn

SAFE_DIR = Path("/content/drive/MyDrive/ColabMCPFiles")
SAFE_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(SAFE_DIR)
ngrok.set_auth_token("2yuUKpiAlYXCez40i8UktLZ91s7_5Rv1GjG5HMFw5KKbk5sHz")

app = FastAPI(title="Colab MCP Tool Server", version="1.7")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])
app.mount("/site", StaticFiles(directory=SAFE_DIR / "site", html=True), name="site")

# ─────────────────────────────────────────────────────────────────────────────
# 4. Logging and streaming
# ─────────────────────────────────────────────────────────────────────────────
SERVER_LOG = SAFE_DIR / "server_log.txt"; SERVER_LOG.touch()
subscribers: set[asyncio.Queue] = set()

class BroadcastHandler(logging.Handler):
    def emit(self, record):
        msg = self.format(record)
        with SERVER_LOG.open("a") as f: f.write(msg + "\n")
        for q in list(subscribers):
            try: asyncio.get_event_loop().call_soon_threadsafe(q.put_nowait, msg)
            except: pass

logger = logging.getLogger("mcp")
logger.setLevel(logging.INFO)
handler = BroadcastHandler()
handler.setFormatter(logging.Formatter("%(asctime)s %(message)s"))
logger.addHandler(handler)

# ─────────────────────────────────────────────────────────────────────────────
# 5. Tool definitions
# ─────────────────────────────────────────────────────────────────────────────
TOOLS = [
    {"name": "execute_python", "description": "Run Python code", "inputSchema": {"type": "object", "properties": {"code": {"type": "string"}}, "required": ["code"]}},
    {"name": "install_package", "description": "Install pip package", "inputSchema": {"type": "object", "properties": {"package": {"type": "string"}}, "required": ["package"]}},
    {"name": "list_files", "description": "List files in sandbox", "inputSchema": {"type": "object", "properties": {"path": {"type": "string", "default": "."}}}},
    {"name": "read_file", "description": "Read file content", "inputSchema": {"type": "object", "properties": {"filepath": {"type": "string"}}, "required": ["filepath"]}},
    {"name": "write_file", "description": "Write file", "inputSchema": {"type": "object", "properties": {"filepath": {"type": "string"}, "content": {"type": "string"}}, "required": ["filepath", "content"]}},
    {"name": "download_file", "description": "Download file via link", "inputSchema": {"type": "object", "properties": {"filepath": {"type": "string"}}, "required": ["filepath"]}},
]

# ─────────────────────────────────────────────────────────────────────────────
# 6. Tool execution
# ─────────────────────────────────────────────────────────────────────────────
def exec_py(code: str) -> str:
    from io import StringIO
    import contextlib
    buf = StringIO()
    try:
        with contextlib.redirect_stdout(buf):
            exec(code, {})
        return buf.getvalue() or "✓ Code ran with no output."
    except Exception as e:
        return f"❌ {e}"

def inst_pkg(pkg: str) -> str:
    r = subprocess.run([sys.executable, "-m", "pip", "install", pkg], capture_output=True, text=True)
    return r.stdout or r.stderr

def safe_path(p: str) -> Path:
    target = SAFE_DIR / p
    if not target.resolve().is_relative_to(SAFE_DIR.resolve()):
        raise ValueError("Unsafe path.")
    return target

def read_file(fp: str) -> str:
    return safe_path(fp).read_text()

def write_file(fp: str, content: str) -> str:
    p = safe_path(fp); p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content)
    return f"✅ Written to {p.name}"

# ─────────────────────────────────────────────────────────────────────────────
# 7. Routes — health, logs, memory, tools, schema
# ─────────────────────────────────────────────────────────────────────────────
@app.get("/")
def root(): return {"status": "ok", "version": "1.7"}

@app.get("/health")
def health(): return {"ok": True}

@app.get("/logs/view")
async def view_logs(request: Request):
    q = asyncio.Queue()
    subscribers.add(q)
    try:
        with SERVER_LOG.open("r") as f:
            history = "".join(f.readlines()[-40:])
        async def logstream():
            yield f"data: {history}\n\n"
            while True:
                if await request.is_disconnected(): break
                try:
                    msg = await asyncio.wait_for(q.get(), timeout=30)
                    yield f"data: {msg}\n\n"
                except asyncio.TimeoutError: continue
        return EventSourceResponse(logstream())
    finally:
        subscribers.discard(q)

@app.get("/memory/view")
def view_memory():
    p = SAFE_DIR / "memory.json"
    return json.loads(p.read_text()) if p.exists() else {}

@app.get("/memory/keys")
def keys_memory():
    p = SAFE_DIR / "memory.json"
    return list(json.loads(p.read_text()).keys()) if p.exists() else []

@app.post("/memory/update")
async def update_memory(request: Request):
    data = await request.json()
    p = SAFE_DIR / "memory.json"
    memory = json.loads(p.read_text()) if p.exists() else {}
    memory.update(data)
    p.write_text(json.dumps(memory, indent=2))
    return memory

@app.post("/memory/clear")
def clear_memory():
    p = SAFE_DIR / "memory.json"; p.write_text("{}")
    return {"status": "cleared"}

@app.get("/download/{file_path:path}")
def download_file(file_path: str):
    p = safe_path(file_path)
    if not p.exists(): raise HTTPException(404)
    return FileResponse(p)

@app.get("/mcp/tools/list")
def list_tools(): return {"tools": TOOLS}

@app.post("/mcp/tools/call")
async def call_tool(request: Request):
    data = await request.json()
    name, args = data.get("name"), data.get("input", {})
    logger.info(f"CALL {name} {args!r}")
    try:
        if name == "execute_python": out = exec_py(args["code"])
        elif name == "install_package": out = inst_pkg(args["package"])
        elif name == "list_files": out = "\n".join(f.name for f in safe_path(args.get("path", ".")).iterdir())
        elif name == "read_file": out = read_file(args["filepath"])
        elif name == "write_file": out = write_file(args["filepath"], args["content"])
        elif name == "download_file":
            safe_path(args["filepath"])
            out = f"{public_url}/download/{args['filepath']}"
        else: raise ValueError("Unknown tool")
        return {"content": [{"type": "text", "text": out}]}
    except Exception as e:
        return {"content": [{"type": "text", "text": f"❌ {e}"}], "isError": True}

@app.get("/.well-known/ai-plugin.json")
def serve_openai_manifest():
    return JSONResponse({
        "schema_version": "v1",
        "name_for_human": "Colab MCP",
        "name_for_model": "mcp",
        "description_for_model": "Run tools via /mcp/tools/call",
        "description_for_human": "Run Python, install packages, file I/O, memory ops",
        "auth": {"type": "none"},
        "api": {"type": "openapi", "url": f"{public_url}/mcp/schema"},
        "logo_url": "", "contact_email": "", "legal_info_url": ""
    })

@app.get("/.well-known/tool-schema.json")
@app.get("/mcp/schema")
def serve_schema(): return app.openapi()

# ─────────────────────────────────────────────────────────────────────────────
# 8. Launch server and expose endpoints
# ─────────────────────────────────────────────────────────────────────────────
def launch():
    Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000), daemon=True).start()
    time.sleep(2)
    url = ngrok.connect(8000).public_url
    print(f"🔗 Public URL: {url}")
    print("Endpoints:")
    for k, r in [
        ("Tool Call", "/mcp/tools/call"),
        ("Logs", "/logs/view"),
        ("Memory", "/memory/view"),
        ("Download", "/download/<file>"),
        ("Schema", "/mcp/schema"),
        ("Claude", "/.well-known/tool-schema.json"),
        ("OpenAI", "/.well-known/ai-plugin.json")
    ]: print(f"• {k:<12} {url}{r}")
    return url

public_url = launch()

INFO:     Started server process [1742]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Mounted at /content/drive
🔗 Public URL: https://fcc6-34-91-173-170.ngrok-free.app
Endpoints:
• Tool Call    https://fcc6-34-91-173-170.ngrok-free.app/mcp/tools/call
• Logs         https://fcc6-34-91-173-170.ngrok-free.app/logs/view
• Memory       https://fcc6-34-91-173-170.ngrok-free.app/memory/view
• Download     https://fcc6-34-91-173-170.ngrok-free.app/download/<file>
• Schema       https://fcc6-34-91-173-170.ngrok-free.app/mcp/schema
• Claude       https://fcc6-34-91-173-170.ngrok-free.app/.well-known/tool-schema.json
• OpenAI       https://fcc6-34-91-173-170.ngrok-free.app/.well-known/ai-plugin.json


In [ ]:
# ╔════════════════════════════════════════════════════════════════════════════════════╗
# ║ 🤖 Colab MCP Server v1.7.1 — All Routes Restored + Streaming + Plugin Manifests   ║
# ║                                                                                    ║
# ║ • Includes /mcp, /mcp/tools/call, memory, logs, download, toolchat, Claude/OpenAI ║
# ║ • Tool set: code exec, pip install, file ops, memory ops, download, read/print    ║
# ║ • Live logging, OpenAPI 3.1.0, YAML plugin files, sandbox-safe I/O                ║
# ╚════════════════════════════════════════════════════════════════════════════════════╝

# ─────────────────────────────────────────────────────────────────────────────
# 1. Install dependencies (Colab)
# ─────────────────────────────────────────────────────────────────────────────
!pip install --quiet fastapi uvicorn pyngrok nest-asyncio sse-starlette requests

# ─────────────────────────────────────────────────────────────────────────────
# 2. Mount Google Drive
# ─────────────────────────────────────────────────────────────────────────────
import nest_asyncio; nest_asyncio.apply()
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

# ─────────────────────────────────────────────────────────────────────────────
# 3. Imports and setup
# ─────────────────────────────────────────────────────────────────────────────
import os, json, time, asyncio, logging, subprocess, sys
from pathlib import Path
from threading import Thread
from fastapi import FastAPI, Request, HTTPException, Form
from fastapi.responses import FileResponse, HTMLResponse, JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from fastapi.staticfiles import StaticFiles
from sse_starlette.sse import EventSourceResponse
from pyngrok import ngrok
import uvicorn

SAFE_DIR = Path("/content/drive/MyDrive/ColabMCPFiles")
PLUGIN_DIR = SAFE_DIR / ".well-known"
for d in [SAFE_DIR, PLUGIN_DIR]: d.mkdir(parents=True, exist_ok=True)
os.chdir(SAFE_DIR)
ngrok.set_auth_token("2yuUKpiAlYXCez40i8UktLZ91s7_5Rv1GjG5HMFw5KKbk5sHz")

app = FastAPI(title="Colab MCP Tool Server", version="1.7.1")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])
app.mount("/site", StaticFiles(directory=SAFE_DIR / "site", html=True), name="site")

# ─────────────────────────────────────────────────────────────────────────────
# 4. Logging + log streaming
# ─────────────────────────────────────────────────────────────────────────────
SERVER_LOG = SAFE_DIR / "server_log.txt"; SERVER_LOG.touch()
CHAT_LOG   = SAFE_DIR / "chat_log.txt";   CHAT_LOG.touch()
subscribers: set[asyncio.Queue] = set()

class BroadcastHandler(logging.Handler):
    def emit(self, record):
        msg = self.format(record)
        with SERVER_LOG.open("a") as f: f.write(msg + "\n")
        for q in list(subscribers):
            try: asyncio.get_event_loop().call_soon_threadsafe(q.put_nowait, msg)
            except: pass

logger = logging.getLogger("mcp")
logger.setLevel(logging.INFO)
handler = BroadcastHandler()
handler.setFormatter(logging.Formatter("%(asctime)s %(message)s"))
logger.addHandler(handler)

# ─────────────────────────────────────────────────────────────────────────────
# 5. Tool definitions (full restored set)
# ─────────────────────────────────────────────────────────────────────────────
TOOLS = [
    {"name": "execute_python", "description": "Run Python code", "inputSchema": {"type": "object", "properties": {"code": {"type": "string"}}, "required": ["code"]}},
    {"name": "install_package", "description": "Install pip package", "inputSchema": {"type": "object", "properties": {"package": {"type": "string"}}, "required": ["package"]}},
    {"name": "list_files", "description": "List files in sandbox", "inputSchema": {"type": "object", "properties": {"path": {"type": "string", "default": "."}}}},
    {"name": "read_file", "description": "Read file content", "inputSchema": {"type": "object", "properties": {"filepath": {"type": "string"}}, "required": ["filepath"]}},
    {"name": "write_file", "description": "Write content to file", "inputSchema": {"type": "object", "properties": {"filepath": {"type": "string"}, "content": {"type": "string"}}, "required": ["filepath", "content"]}},
    {"name": "print_file_text", "description": "Alias of read_file", "inputSchema": {"type": "object", "properties": {"filepath": {"type": "string"}}, "required": ["filepath"]}},
    {"name": "download_file", "description": "Get download URL", "inputSchema": {"type": "object", "properties": {"filepath": {"type": "string"}}, "required": ["filepath"]}},
]
# ─────────────────────────────────────────────────────────────────────────────
# 6. Tool execution handlers
# ─────────────────────────────────────────────────────────────────────────────
def exec_py(code: str) -> str:
    from io import StringIO
    import contextlib
    buf = StringIO()
    try:
        with contextlib.redirect_stdout(buf):
            exec(code, {})
        return buf.getvalue() or "✓ Code ran with no output."
    except Exception as e:
        return f"❌ Error: {e}"

def inst_pkg(pkg: str) -> str:
    result = subprocess.run([sys.executable, "-m", "pip", "install", pkg], capture_output=True, text=True)
    return result.stdout or result.stderr

def safe_path(p: str) -> Path:
    target = SAFE_DIR / p
    if not target.resolve().is_relative_to(SAFE_DIR.resolve()):
        raise ValueError("Unsafe path.")
    return target

def read_file(fp: str) -> str:
    return safe_path(fp).read_text()

def write_file(fp: str, content: str) -> str:
    p = safe_path(fp)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content)
    return f"✅ Written to {p.name}"

# ─────────────────────────────────────────────────────────────────────────────
# 7. API routes
# ─────────────────────────────────────────────────────────────────────────────
@app.get("/")
def root(): return {"status": "ok", "version": "1.7.1"}

@app.get("/health")
def health(): return {"status": "healthy", "tools": [t["name"] for t in TOOLS]}

@app.get("/logs/view")
async def view_logs(request: Request):
    q = asyncio.Queue()
    subscribers.add(q)
    try:
        with SERVER_LOG.open("r") as f:
            history = "".join(f.readlines()[-50:])
        async def logstream():
            yield f"data: {history}\n\n"
            while True:
                if await request.is_disconnected(): break
                try:
                    msg = await asyncio.wait_for(q.get(), timeout=30)
                    yield f"data: {msg}\n\n"
                except asyncio.TimeoutError: continue
        return EventSourceResponse(logstream())
    finally:
        subscribers.discard(q)

@app.get("/memory/view")
def memory_view():
    mem_path = SAFE_DIR / "memory.json"
    return json.loads(mem_path.read_text()) if mem_path.exists() else {}

@app.get("/memory/summary")
def memory_summary():
    mem_path = SAFE_DIR / "memory.json"
    if not mem_path.exists(): return {"summary": "(none)"}
    try:
        data = json.loads(mem_path.read_text())
        return {"summary": data.get("summary", "(no summary key)")}
    except: return {"summary": "(error reading JSON)"}

@app.get("/memory/keys")
def memory_keys():
    mem_path = SAFE_DIR / "memory.json"
    return list(json.loads(mem_path.read_text()).keys()) if mem_path.exists() else []

@app.post("/memory/update")
async def memory_update(request: Request):
    mem_path = SAFE_DIR / "memory.json"
    memory = json.loads(mem_path.read_text()) if mem_path.exists() else {}
    update = await request.json()
    memory.update(update)
    mem_path.write_text(json.dumps(memory, indent=2))
    return memory

@app.post("/memory/clear")
def memory_clear():
    mem_path = SAFE_DIR / "memory.json"
    mem_path.write_text("{}")
    return {"status": "cleared"}

@app.get("/download/{file_path:path}")
def download_file(file_path: str):
    p = safe_path(file_path)
    if not p.exists(): raise HTTPException(404, "File not found")
    return FileResponse(p)

@app.get("/mcp/tools/list")
def list_tools(): return {"tools": TOOLS}

@app.post("/mcp/tools/call")
async def call_tool(request: Request):
    data = await request.json()
    name, args = data.get("name"), data.get("input", {})
    logger.info(f"CALL {name} {args}")
    try:
        if name == "execute_python": out = exec_py(args["code"])
        elif name == "install_package": out = inst_pkg(args["package"])
        elif name == "list_files": out = "\n".join(f.name for f in safe_path(args.get("path", ".")).iterdir())
        elif name == "read_file": out = read_file(args["filepath"])
        elif name == "write_file": out = write_file(args["filepath"], args["content"])
        elif name == "print_file_text": out = read_file(args["filepath"])
        elif name == "download_file":
            safe_path(args["filepath"])
            out = f"{public_url}/download/{args['filepath']}"
        else: raise Exception("Unknown tool")
        return {"content": [{"type": "text", "text": out}]}
    except Exception as e:
        return {"content": [{"type": "text", "text": f"❌ {e}"}], "isError": True}

@app.get("/site/toolchat.html")
def serve_toolchat():
    html = """
    <html><head><title>MCP Tool Chat</title></head>
    <body style='font-family: sans-serif'>
    <h2>🧪 MCP Tool Call UI</h2>
    <form method="post" action="/mcp/tools/call">
      <label>Tool:</label><input name="name" value="execute_python"><br><br>
      <label>Input JSON:</label><br>
      <textarea name="input" rows="6" cols="60">{"code": "print('Hello!')"}</textarea><br><br>
      <button type="submit">Run Tool</button>
    </form>
    </body></html>
    """
    return HTMLResponse(content=html)

@app.get("/.well-known/ai-plugin.json")
def serve_openai_manifest():
    data = {
        "schema_version": "v1",
        "name_for_human": "Colab MCP",
        "name_for_model": "mcp",
        "description_for_model": "Tool call server with file/memory/code functions",
        "description_for_human": "Run tools in Colab via OpenAPI",
        "auth": {"type": "none"},
        "api": {"type": "openapi", "url": f"{public_url}/mcp/schema"},
        "logo_url": "", "contact_email": "", "legal_info_url": ""
    }
    (PLUGIN_DIR / "ai-plugin.json").write_text(json.dumps(data, indent=2))
    return JSONResponse(data)

@app.get("/.well-known/tool-schema.json")
@app.get("/mcp/schema")
def serve_openapi_schema(): return app.openapi()

@app.get("/.well-known/claude.yaml")
def serve_claude_yaml():
    yaml = f"""schema_version: 1.0
name: colab-mcp
description: Claude plugin for tool calling via MCP
api:
  type: openapi
  url: {public_url}/mcp/schema
auth:
  type: none
"""
    return HTMLResponse(content=yaml, media_type="text/yaml")

# ─────────────────────────────────────────────────────────────────────────────
# 8. Server launcher
# ─────────────────────────────────────────────────────────────────────────────
def launch():
    Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000), daemon=True).start()
    time.sleep(2)
    url = ngrok.connect(8000).public_url
    print(f"🔗 Public URL: {url}")
    for label, route in [
        ("Tool UI", "/site/toolchat.html"),
        ("Tool Call", "/mcp/tools/call"),
        ("Logs", "/logs/view"),
        ("Memory", "/memory/view"),
        ("Summary", "/memory/summary"),
        ("Schema", "/mcp/schema"),
        ("Claude", "/.well-known/tool-schema.json"),
        ("OpenAI", "/.well-known/ai-plugin.json"),
        ("Claude YAML", "/.well-known/claude.yaml"),
    ]:
        print(f"• {label:<12} {url}{route}")
    return url

public_url = launch()

print("\n🤖 Example Prompts for LLM Tool Use (Claude or OpenAI-compatible):\n")
print("1. Run Python code:")
print("   → Call the 'execute_python' tool with:")
print("     {\"code\": \"print('Hello from Claude!')\"}\n")

print("2. Install a pip package:")
print("   → Call 'install_package' with:")
print("     {\"package\": \"beautifulsoup4\"}\n")

print("3. Read a file from sandboxed drive:")
print("   → Call 'read_file' or 'print_file_text' with:")
print("     {\"filepath\": \"notes/example.txt\"}\n")

print("4. Write a file:")
print("   → Call 'write_file' with:")
print("     {\"filepath\": \"notes/output.txt\", \"content\": \"Hello, world.\"}\n")

print("5. List files in a folder:")
print("   → Call 'list_files' with:")
print("     {\"path\": \".\"} or {\"path\": \"data\"}\n")

print("6. Recall a stored memory value:")
print("   → First call 'memory/update' to store:")
print("     {\"project\": \"AI Assistant\"}\n")
print("   → Then call 'memory/view' or 'memory/summary'\n")

print("7. Download a file:")
print("   → Call 'download_file' with:")
print("     {\"filepath\": \"output/results.csv\"}")
print("   → Result will be a direct download URL.\n")

print("🔧 Tool schema at:")
print(f"   {public_url}/mcp/schema")
print("📘 Plugin manifest at:")
print(f"   {public_url}/.well-known/tool-schema.json")


Mounted at /content/drive


INFO:     Started server process [696]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


🔗 Public URL: https://80ce-34-139-224-82.ngrok-free.app
• Tool UI      https://80ce-34-139-224-82.ngrok-free.app/site/toolchat.html
• Tool Call    https://80ce-34-139-224-82.ngrok-free.app/mcp/tools/call
• Logs         https://80ce-34-139-224-82.ngrok-free.app/logs/view
• Memory       https://80ce-34-139-224-82.ngrok-free.app/memory/view
• Summary      https://80ce-34-139-224-82.ngrok-free.app/memory/summary
• Schema       https://80ce-34-139-224-82.ngrok-free.app/mcp/schema
• Claude       https://80ce-34-139-224-82.ngrok-free.app/.well-known/tool-schema.json
• OpenAI       https://80ce-34-139-224-82.ngrok-free.app/.well-known/ai-plugin.json
• Claude YAML  https://80ce-34-139-224-82.ngrok-free.app/.well-known/claude.yaml

🤖 Example Prompts for LLM Tool Use (Claude or OpenAI-compatible):

1. Run Python code:
   → Call the 'execute_python' tool with:
     {"code": "print('Hello from Claude!')"}

2. Install a pip package:
   → Call 'install_package' with:
     {"package": "beautifulsoup4"

In [ ]:
# ╔════════════════════════════════════════════════════════════════════════════════════╗
# ║ 🤖 Colab MCP Server v1.7.1 — All Routes Restored + Streaming + Plugin Manifests   ║
# ║                                                                                    ║
# ║ • Includes /mcp, /mcp/tools/call, memory, logs, download, toolchat, Claude/OpenAI ║
# ║ • Tool set: code exec, pip install, file ops, memory ops, download, read/print    ║
# ║ • Live logging, OpenAPI 3.1.0, YAML plugin files, sandbox-safe I/O                ║
# ║ • Logging fixed to stream valid log lines (not duplicated `data: data:`)          ║
# ╚════════════════════════════════════════════════════════════════════════════════════╝

# ─────────────────────────────────────────────────────────────────────────────
# 1. Install dependencies (Colab)
# ─────────────────────────────────────────────────────────────────────────────
!pip install --quiet fastapi uvicorn pyngrok nest-asyncio sse-starlette requests

# ─────────────────────────────────────────────────────────────────────────────
# 2. Mount Google Drive
# ─────────────────────────────────────────────────────────────────────────────
import nest_asyncio; nest_asyncio.apply()
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

# ─────────────────────────────────────────────────────────────────────────────
# 3. Imports and setup
# ─────────────────────────────────────────────────────────────────────────────
import os, json, time, asyncio, logging, subprocess, sys
from pathlib import Path
from threading import Thread
from fastapi import FastAPI, Request, HTTPException, Form
from fastapi.responses import FileResponse, HTMLResponse, JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from fastapi.staticfiles import StaticFiles
from sse_starlette.sse import EventSourceResponse
from pyngrok import ngrok
import uvicorn

SAFE_DIR = Path("/content/drive/MyDrive/ColabMCPFiles")
PLUGIN_DIR = SAFE_DIR / ".well-known"
for d in [SAFE_DIR, PLUGIN_DIR]: d.mkdir(parents=True, exist_ok=True)
os.chdir(SAFE_DIR)
ngrok.set_auth_token("2yuUKpiAlYXCez40i8UktLZ91s7_5Rv1GjG5HMFw5KKbk5sHz")

app = FastAPI(title="Colab MCP Tool Server", version="1.7.1")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])
app.mount("/site", StaticFiles(directory=SAFE_DIR / "site", html=True), name="site")

# ─────────────────────────────────────────────────────────────────────────────
# 4. Logging + log streaming (FIXED: avoids `data: data:` duplication)
# ─────────────────────────────────────────────────────────────────────────────
SERVER_LOG = SAFE_DIR / "server_log.txt"; SERVER_LOG.touch()
CHAT_LOG   = SAFE_DIR / "chat_log.txt";   CHAT_LOG.touch()
subscribers: set[asyncio.Queue] = set()

class BroadcastHandler(logging.Handler):
    def emit(self, record):
        msg = self.format(record)
        with SERVER_LOG.open("a", encoding="utf-8") as f:
            f.write(msg + "\n")
        for q in list(subscribers):
            try: asyncio.get_event_loop().call_soon_threadsafe(q.put_nowait, msg)
            except: pass

logger = logging.getLogger("mcp")
logger.setLevel(logging.INFO)
if not any(isinstance(h, BroadcastHandler) for h in logger.handlers):
    handler = BroadcastHandler()
    handler.setFormatter(logging.Formatter("%(asctime)s %(message)s"))
    logger.addHandler(handler)

# ─────────────────────────────────────────────────────────────────────────────
# 5. Tool definitions (full restored set)
# ─────────────────────────────────────────────────────────────────────────────
TOOLS = [
    {"name": "execute_python", "description": "Run Python code", "inputSchema": {"type": "object", "properties": {"code": {"type": "string"}}, "required": ["code"]}},
    {"name": "install_package", "description": "Install pip package", "inputSchema": {"type": "object", "properties": {"package": {"type": "string"}}, "required": ["package"]}},
    {"name": "list_files", "description": "List files in sandbox", "inputSchema": {"type": "object", "properties": {"path": {"type": "string", "default": "."}}}},
    {"name": "read_file", "description": "Read file content", "inputSchema": {"type": "object", "properties": {"filepath": {"type": "string"}}, "required": ["filepath"]}},
    {"name": "write_file", "description": "Write content to file", "inputSchema": {"type": "object", "properties": {"filepath": {"type": "string"}, "content": {"type": "string"}}, "required": ["filepath", "content"]}},
    {"name": "print_file_text", "description": "Alias of read_file", "inputSchema": {"type": "object", "properties": {"filepath": {"type": "string"}}, "required": ["filepath"]}},
    {"name": "download_file", "description": "Get download URL", "inputSchema": {"type": "object", "properties": {"filepath": {"type": "string"}}, "required": ["filepath"]}},
]
# ( in next message...)

# ╔════════════════════════════════════════════════════════════════════════════════════╗
# ║ 🤖 Colab MCP Server v1.7.1 — All Routes Restored + Streaming + Plugin Manifests   ║
# ║                                                                                    ║
# ║ • Includes /mcp, /mcp/tools/call, memory, logs, download, toolchat, Claude/OpenAI ║
# ║ • Tool set: code exec, pip install, file ops, memory ops, download, read/print    ║
# ║ • Live logging, OpenAPI 3.1.0, YAML plugin files, sandbox-safe I/O                ║
# ║ • Logging fixed to stream valid log lines (not duplicated `data: data:`)          ║
# ╚════════════════════════════════════════════════════════════════════════════════════╝

# ─────────────────────────────────────────────────────────────────────────────
# 1. Install dependencies (Colab)
# ─────────────────────────────────────────────────────────────────────────────
!pip install --quiet fastapi uvicorn pyngrok nest-asyncio sse-starlette requests

# ─────────────────────────────────────────────────────────────────────────────
# 2. Mount Google Drive
# ─────────────────────────────────────────────────────────────────────────────
import nest_asyncio; nest_asyncio.apply()
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

# ─────────────────────────────────────────────────────────────────────────────
# 3. Imports and setup
# ─────────────────────────────────────────────────────────────────────────────
import os, json, time, asyncio, logging, subprocess, sys
from pathlib import Path
from threading import Thread
from fastapi import FastAPI, Request, HTTPException, Form
from fastapi.responses import FileResponse, HTMLResponse, JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from fastapi.staticfiles import StaticFiles
from sse_starlette.sse import EventSourceResponse
from pyngrok import ngrok
import uvicorn

SAFE_DIR = Path("/content/drive/MyDrive/ColabMCPFiles")
PLUGIN_DIR = SAFE_DIR / ".well-known"
for d in [SAFE_DIR, PLUGIN_DIR]: d.mkdir(parents=True, exist_ok=True)
os.chdir(SAFE_DIR)
ngrok.set_auth_token("2yuUKpiAlYXCez40i8UktLZ91s7_5Rv1GjG5HMFw5KKbk5sHz")

app = FastAPI(title="Colab MCP Tool Server", version="1.7.1")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])
app.mount("/site", StaticFiles(directory=SAFE_DIR / "site", html=True), name="site")

# ─────────────────────────────────────────────────────────────────────────────
# 4. Logging + log streaming (FIXED: avoids `data: data:` duplication)
# ─────────────────────────────────────────────────────────────────────────────
SERVER_LOG = SAFE_DIR / "server_log.txt"; SERVER_LOG.touch()
CHAT_LOG   = SAFE_DIR / "chat_log.txt";   CHAT_LOG.touch()
subscribers: set[asyncio.Queue] = set()

class BroadcastHandler(logging.Handler):
    def emit(self, record):
        msg = self.format(record)
        with SERVER_LOG.open("a", encoding="utf-8") as f:
            f.write(msg + "\n")
        for q in list(subscribers):
            try: asyncio.get_event_loop().call_soon_threadsafe(q.put_nowait, msg)
            except: pass

logger = logging.getLogger("mcp")
logger.setLevel(logging.INFO)
if not any(isinstance(h, BroadcastHandler) for h in logger.handlers):
    handler = BroadcastHandler()
    handler.setFormatter(logging.Formatter("%(asctime)s %(message)s"))
    logger.addHandler(handler)

#
# 5. Tool definitions (full restored set)
# ─────────────────────────────────────────────────────────────────────────────
TOOLS = [
    {"name": "execute_python", "description": "Run Python code", "inputSchema": {"type": "object", "properties": {"code": {"type": "string"}}, "required": ["code"]}},
    {"name": "install_package", "description": "Install pip package", "inputSchema": {"type": "object", "properties": {"package": {"type": "string"}}, "required": ["package"]}},
    {"name": "list_files", "description": "List files in sandbox", "inputSchema": {"type": "object", "properties": {"path": {"type": "string", "default": "."}}}},
    {"name": "read_file", "description": "Read file content", "inputSchema": {"type": "object", "properties": {"filepath": {"type": "string"}}, "required": ["filepath"]}},
    {"name": "write_file", "description": "Write content to file", "inputSchema": {"type": "object", "properties": {"filepath": {"type": "string"}, "content": {"type": "string"}}, "required": ["filepath", "content"]}},
    {"name": "print_file_text", "description": "Alias of read_file", "inputSchema": {"type": "object", "properties": {"filepath": {"type": "string"}}, "required": ["filepath"]}},
    {"name": "download_file", "description": "Get download URL", "inputSchema": {"type": "object", "properties": {"filepath": {"type": "string"}}, "required": ["filepath"]}},
]
# (Continued in next message...)
# ─────────────────────────────────────────────────────────────────────────────
# 6. Tool execution handlers
# ─────────────────────────────────────────────────────────────────────────────
def exec_py(code: str) -> str:
    from io import StringIO
    import contextlib
    buf = StringIO()
    try:
        with contextlib.redirect_stdout(buf):
            exec(code, {})
        return buf.getvalue() or "✓ Code ran with no output."
    except Exception as e:
        return f"❌ Error: {e}"

def inst_pkg(pkg: str) -> str:
    result = subprocess.run([sys.executable, "-m", "pip", "install", pkg], capture_output=True, text=True)
    return result.stdout or result.stderr

def safe_path(p: str) -> Path:
    target = SAFE_DIR / p
    if not target.resolve().is_relative_to(SAFE_DIR.resolve()):
        raise ValueError("Unsafe path.")
    return target

def read_file(fp: str) -> str:
    return safe_path(fp).read_text()

def write_file(fp: str, content: str) -> str:
    p = safe_path(fp)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content)
    return f"✅ Written to {p.name}"

# ─────────────────────────────────────────────────────────────────────────────
# 7. API routes
# ─────────────────────────────────────────────────────────────────────────────
@app.get("/")
def root(): return {"status": "ok", "version": "1.7.1"}

@app.get("/health")
def health(): return {"status": "healthy", "tools": [t["name"] for t in TOOLS]}

@app.get("/logs/view")
async def view_logs(request: Request):
    q = asyncio.Queue()
    subscribers.add(q)
    try:
        with SERVER_LOG.open("r", encoding="utf-8") as f:
            history = f.readlines()[-50:]
        async def logstream():
            for line in history:
                yield f"data: {line.strip()}\n\n"
            while True:
                if await request.is_disconnected(): break
                try:
                    msg = await asyncio.wait_for(q.get(), timeout=30)
                    yield f"data: {msg.strip()}\n\n"
                except asyncio.TimeoutError:
                    continue
        return EventSourceResponse(logstream())
    finally:
        subscribers.discard(q)

@app.get("/memory/view")
def memory_view():
    mem_path = SAFE_DIR / "memory.json"
    return json.loads(mem_path.read_text()) if mem_path.exists() else {}

@app.get("/memory/summary")
def memory_summary():
    mem_path = SAFE_DIR / "memory.json"
    if not mem_path.exists(): return {"summary": "(none)"}
    try:
        data = json.loads(mem_path.read_text())
        return {"summary": data.get("summary", "(no summary key)")}
    except: return {"summary": "(error reading JSON)"}

@app.get("/memory/keys")
def memory_keys():
    mem_path = SAFE_DIR / "memory.json"
    return list(json.loads(mem_path.read_text()).keys()) if mem_path.exists() else []

@app.post("/memory/update")
async def memory_update(request: Request):
    mem_path = SAFE_DIR / "memory.json"
    memory = json.loads(mem_path.read_text()) if mem_path.exists() else {}
    update = await request.json()
    memory.update(update)
    mem_path.write_text(json.dumps(memory, indent=2))
    return memory

@app.post("/memory/clear")
def memory_clear():
    mem_path = SAFE_DIR / "memory.json"
    mem_path.write_text("{}")
    return {"status": "cleared"}

@app.get("/download/{file_path:path}")
def download_file(file_path: str):
    p = safe_path(file_path)
    if not p.exists(): raise HTTPException(404, "File not found")
    return FileResponse(p)

@app.get("/mcp/tools/list")
def list_tools(): return {"tools": TOOLS}

@app.post("/mcp/tools/call")
async def call_tool(request: Request):
    data = await request.json()
    name, args = data.get("name"), data.get("input", {})
    logger.info(f"CALL {name} {args}")
    try:
        if name == "execute_python": out = exec_py(args["code"])
        elif name == "install_package": out = inst_pkg(args["package"])
        elif name == "list_files": out = "\n".join(f.name for f in safe_path(args.get("path", ".")).iterdir())
        elif name == "read_file": out = read_file(args["filepath"])
        elif name == "write_file": out = write_file(args["filepath"], args["content"])
        elif name == "print_file_text": out = read_file(args["filepath"])
        elif name == "download_file":
            safe_path(args["filepath"])
            out = f"{public_url}/download/{args['filepath']}"
        else: raise Exception("Unknown tool")
        return {"content": [{"type": "text", "text": out}]}
    except Exception as e:
        return {"content": [{"type": "text", "text": f"❌ {e}"}], "isError": True}

@app.get("/site/toolchat.html")
def serve_toolchat():
    html = """
    <html><head><title>MCP Tool Chat</title></head>
    <body style='font-family: sans-serif'>
    <h2>🧪 MCP Tool Call UI</h2>
    <form method="post" action="/mcp/tools/call">
      <label>Tool:</label><input name="name" value="execute_python"><br><br>
      <label>Input JSON:</label><br>
      <textarea name="input" rows="6" cols="60">{"code": "print('Hello!')"}</textarea><br><br>
      <button type="submit">Run Tool</button>
    </form>
    </body></html>
    """
    return HTMLResponse(content=html)

@app.get("/.well-known/ai-plugin.json")
def serve_openai_manifest():
    data = {
        "schema_version": "v1",
        "name_for_human": "Colab MCP",
        "name_for_model": "mcp",
        "description_for_model": "Tool call server with file/memory/code functions",
        "description_for_human": "Run tools in Colab via OpenAPI",
        "auth": {"type": "none"},
        "api": {"type": "openapi", "url": f"{public_url}/mcp/schema"},
        "logo_url": "", "contact_email": "", "legal_info_url": ""
    }
    (PLUGIN_DIR / "ai-plugin.json").write_text(json.dumps(data, indent=2))
    return JSONResponse(data)

@app.get("/.well-known/tool-schema.json")
@app.get("/mcp/schema")
def serve_openapi_schema(): return app.openapi()

@app.get("/.well-known/claude.yaml")
def serve_claude_yaml():
    yaml = f"""schema_version: 1.0
name: colab-mcp
description: Claude plugin for tool calling via MCP
api:
  type: openapi
  url: {public_url}/mcp/schema
auth:
  type: none
"""
    return HTMLResponse(content=yaml, media_type="text/yaml")

# ─────────────────────────────────────────────────────────────────────────────
# 8. Server launcher
# ─────────────────────────────────────────────────────────────────────────────
def launch():
    Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000), daemon=True).start()
    time.sleep(2)
    url = ngrok.connect(8000).public_url
    print(f"🔗 Public URL: {url}")
    for label, route in [
        ("Tool UI", "/site/toolchat.html"),
        ("Tool Call", "/mcp/tools/call"),
        ("Logs", "/logs/view"),
        ("Memory", "/memory/view"),
        ("Summary", "/memory/summary"),
        ("Schema", "/mcp/schema"),
        ("Claude", "/.well-known/tool-schema.json"),
        ("OpenAI", "/.well-known/ai-plugin.json"),
        ("Claude YAML", "/.well-known/claude.yaml"),
    ]:
        print(f"• {label:<12} {url}{route}")
    return url

public_url = launch()

print("\n🤖 Example Prompts for LLM Tool Use (Claude or OpenAI-compatible):\n")
print("1. Run Python code:")
print("   → Call the 'execute_python' tool with:")
print("     {\"code\": \"print('Hello from Claude!')\"}\n")

print("2. Install a pip package:")
print("   → Call 'install_package' with:")
print("     {\"package\": \"beautifulsoup4\"}\n")

print("3. Read a file from sandboxed drive:")
print("   → Call 'read_file' or 'print_file_text' with:")
print("     {\"filepath\": \"notes/example.txt\"}\n")

print("4. Write a file:")
print("   → Call 'write_file' with:")
print("     {\"filepath\": \"notes/output.txt\", \"content\": \"Hello, world.\"}\n")

print("5. List files in a folder:")
print("   → Call 'list_files' with:")
print("     {\"path\": \".\"} or {\"path\": \"data\"}\n")

print("6. Recall a stored memory value:")
print("   → First call 'memory/update' to store:")
print("     {\"project\": \"AI Assistant\"}\n")
print("   → Then call 'memory/view' or 'memory/summary'\n")

print("7. Download a file:")
print("   → Call 'download_file' with:")
print("     {\"filepath\": \"output/results.csv\"}")
print("   → Result will be a direct download URL.\n")

print("🔧 Tool schema at:")
print(f"   {public_url}/mcp/schema")
print("📘 Plugin manifest at:")
print(f"   {public_url}/.well-known/tool-schema.json")

Mounted at /content/drive
Mounted at /content/drive


INFO:     Started server process [696]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


🔗 Public URL: https://e335-34-139-224-82.ngrok-free.app
• Tool UI      https://e335-34-139-224-82.ngrok-free.app/site/toolchat.html
• Tool Call    https://e335-34-139-224-82.ngrok-free.app/mcp/tools/call
• Logs         https://e335-34-139-224-82.ngrok-free.app/logs/view
• Memory       https://e335-34-139-224-82.ngrok-free.app/memory/view
• Summary      https://e335-34-139-224-82.ngrok-free.app/memory/summary
• Schema       https://e335-34-139-224-82.ngrok-free.app/mcp/schema
• Claude       https://e335-34-139-224-82.ngrok-free.app/.well-known/tool-schema.json
• OpenAI       https://e335-34-139-224-82.ngrok-free.app/.well-known/ai-plugin.json
• Claude YAML  https://e335-34-139-224-82.ngrok-free.app/.well-known/claude.yaml

🤖 Example Prompts for LLM Tool Use (Claude or OpenAI-compatible):

1. Run Python code:
   → Call the 'execute_python' tool with:
     {"code": "print('Hello from Claude!')"}

2. Install a pip package:
   → Call 'install_package' with:
     {"package": "beautifulsoup4"

In [ ]:
# ╔════════════════════════════════════════════════════════════════════════════════════╗
# ║ 🤖 Colab MCP Server v1.7.1 — All Routes Restored + Streaming + Plugin Manifests   ║
# ║                                                                                    ║
# ║ • Includes /mcp, /mcp/tools/call, memory, logs, download, toolchat, Claude/OpenAI ║
# ║ • Tool set: code exec, pip install, file ops, memory ops, download, read/print    ║
# ║ • Live logging, OpenAPI 3.1.0, YAML plugin files, sandbox-safe I/O                ║
# ║ • Logging fixed to stream valid log lines (not duplicated `data: data:`)          ║
# ╚════════════════════════════════════════════════════════════════════════════════════╝

# ─────────────────────────────────────────────────────────────────────────────
# 1. Install dependencies (Colab)
# ─────────────────────────────────────────────────────────────────────────────
!pip install --quiet fastapi uvicorn pyngrok nest-asyncio sse-starlette requests

# ─────────────────────────────────────────────────────────────────────────────
# 2. Mount Google Drive
# ─────────────────────────────────────────────────────────────────────────────
import nest_asyncio; nest_asyncio.apply()
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

# ─────────────────────────────────────────────────────────────────────────────
# 3. Imports and setup
# ─────────────────────────────────────────────────────────────────────────────
import os, json, time, asyncio, logging, subprocess, sys
from pathlib import Path
from threading import Thread
from fastapi import FastAPI, Request, HTTPException, Form
from fastapi.responses import FileResponse, HTMLResponse, JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from fastapi.staticfiles import StaticFiles
from sse_starlette.sse import EventSourceResponse
from pyngrok import ngrok
import uvicorn

SAFE_DIR = Path("/content/drive/MyDrive/ColabMCPFiles")
PLUGIN_DIR = SAFE_DIR / ".well-known"
for d in [SAFE_DIR, PLUGIN_DIR]: d.mkdir(parents=True, exist_ok=True)
os.chdir(SAFE_DIR)
ngrok.set_auth_token("2yuUKpiAlYXCez40i8UktLZ91s7_5Rv1GjG5HMFw5KKbk5sHz")

app = FastAPI(title="Colab MCP Tool Server", version="1.7.2")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])
app.mount("/site", StaticFiles(directory=SAFE_DIR / "site", html=True), name="site")

# ─────────────────────────────────────────────────────────────────────────────
# 4. Logging + log streaming (FIXED: avoids `data: data:` duplication)
# ─────────────────────────────────────────────────────────────────────────────
SERVER_LOG = SAFE_DIR / "server_log.txt"; SERVER_LOG.touch()
CHAT_LOG   = SAFE_DIR / "chat_log.txt";   CHAT_LOG.touch()
subscribers: set[asyncio.Queue] = set()

class BroadcastHandler(logging.Handler):
    def emit(self, record):
        msg = self.format(record)
        with SERVER_LOG.open("a", encoding="utf-8") as f:
            f.write(msg + "\n")
        for q in list(subscribers):
            try: asyncio.get_event_loop().call_soon_threadsafe(q.put_nowait, msg)
            except: pass

logger = logging.getLogger("mcp")
logger.setLevel(logging.INFO)
if not any(isinstance(h, BroadcastHandler) for h in logger.handlers):
    handler = BroadcastHandler()
    handler.setFormatter(logging.Formatter("%(asctime)s %(message)s"))
    logger.addHandler(handler)

# ─────────────────────────────────────────────────────────────────────────────
# 5. Tool definitions (fully working set)
# ─────────────────────────────────────────────────────────────────────────────
TOOLS = [
    {"name": "execute_python", "description": "Run Python code", "inputSchema": {"type": "object", "properties": {"code": {"type": "string"}}, "required": ["code"]}},
    {"name": "install_package", "description": "Install pip package", "inputSchema": {"type": "object", "properties": {"package": {"type": "string"}}, "required": ["package"]}},
    {"name": "list_files", "description": "List files in sandbox", "inputSchema": {"type": "object", "properties": {"path": {"type": "string", "default": "."}}}},
    {"name": "read_file", "description": "Read file content", "inputSchema": {"type": "object", "properties": {"filepath": {"type": "string"}}, "required": ["filepath"]}},
    {"name": "write_file", "description": "Write content to file", "inputSchema": {"type": "object", "properties": {"filepath": {"type": "string"}, "content": {"type": "string"}}, "required": ["filepath", "content"]}},
    {"name": "print_file_text", "description": "Alias of read_file", "inputSchema": {"type": "object", "properties": {"filepath": {"type": "string"}}, "required": ["filepath"]}},
    {"name": "download_file", "description": "Get download URL", "inputSchema": {"type": "object", "properties": {"filepath": {"type": "string"}}, "required": ["filepath"]}},
]

# ─────────────────────────────────────────────────────────────────────────────
# 6. Tool execution logic
# ─────────────────────────────────────────────────────────────────────────────
def exec_py(code: str) -> str:
    from io import StringIO
    import contextlib
    buf = StringIO()
    try:
        with contextlib.redirect_stdout(buf):
            exec(code, {})
        return buf.getvalue() or "✓ Code ran with no output."
    except Exception as e:
        return f"❌ Error: {e}"

def inst_pkg(pkg: str) -> str:
    result = subprocess.run([sys.executable, "-m", "pip", "install", pkg], capture_output=True, text=True)
    return result.stdout or result.stderr

def safe_path(p: str) -> Path:
    target = SAFE_DIR / p
    if not target.resolve().is_relative_to(SAFE_DIR.resolve()):
        raise ValueError("Unsafe path.")
    return target

def read_file(fp: str) -> str:
    return safe_path(fp).read_text()

def write_file(fp: str, content: str) -> str:
    p = safe_path(fp)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content)
    return f"✅ Written to {p.name}"

# ─────────────────────────────────────────────────────────────────────────────
# 7. API routes
# ─────────────────────────────────────────────────────────────────────────────
@app.get("/")
def root(): return {"status": "ok", "version": "1.7.2"}

@app.get("/health")
def health(): return {"status": "healthy", "tools": [t["name"] for t in TOOLS]}

@app.get("/logs/view")
async def view_logs(request: Request):
    q = asyncio.Queue()
    subscribers.add(q)
    try:
        with SERVER_LOG.open("r", encoding="utf-8") as f:
            history = f.readlines()[-50:]
        async def logstream():
            for line in history:
                yield f"data: {line.strip()}\n\n"
            while True:
                if await request.is_disconnected(): break
                try:
                    msg = await asyncio.wait_for(q.get(), timeout=30)
                    yield f"data: {msg.strip()}\n\n"
                except asyncio.TimeoutError:
                    continue
        return EventSourceResponse(logstream())
    finally:
        subscribers.discard(q)

@app.get("/memory/view")
def memory_view():
    mem_path = SAFE_DIR / "memory.json"
    return json.loads(mem_path.read_text()) if mem_path.exists() else {}

@app.get("/memory/summary")
def memory_summary():
    mem_path = SAFE_DIR / "memory.json"
    if not mem_path.exists(): return {"summary": "(none)"}
    try:
        data = json.loads(mem_path.read_text())
        return {"summary": data.get("summary", "(no summary key)")}
    except: return {"summary": "(error reading JSON)"}

@app.get("/memory/keys")
def memory_keys():
    mem_path = SAFE_DIR / "memory.json"
    return list(json.loads(mem_path.read_text()).keys()) if mem_path.exists() else []

@app.post("/memory/update")
async def memory_update(request: Request):
    mem_path = SAFE_DIR / "memory.json"
    memory = json.loads(mem_path.read_text()) if mem_path.exists() else {}
    update = await request.json()
    memory.update(update)
    mem_path.write_text(json.dumps(memory, indent=2))
    return memory

@app.post("/memory/clear")
def memory_clear():
    mem_path = SAFE_DIR / "memory.json"
    mem_path.write_text("{}")
    return {"status": "cleared"}

@app.get("/download/{file_path:path}")
def download_file(file_path: str):
    p = safe_path(file_path)
    if not p.exists(): raise HTTPException(404, "File not found")
    return FileResponse(p)

@app.get("/mcp/tools/list")
def list_tools(): return {"tools": TOOLS}

@app.post("/mcp/tools/call")
async def call_tool(request: Request):
    data = await request.json()
    name, args = data.get("name"), data.get("input", {})
    logger.info(f"CALL {name} {args}")
    try:
        if name == "execute_python": out = exec_py(args["code"])
        elif name == "install_package": out = inst_pkg(args["package"])
        elif name == "list_files": out = "\n".join(f.name for f in safe_path(args.get("path", ".")).iterdir())
        elif name == "read_file": out = read_file(args["filepath"])
        elif name == "write_file": out = write_file(args["filepath"], args["content"])
        elif name == "print_file_text": out = read_file(args["filepath"])
        elif name == "download_file":
            safe_path(args["filepath"])
            out = f"{public_url}/download/{args['filepath']}"
        else: raise Exception("Unknown tool")
        return {"content": [{"type": "text", "text": out}]}
    except Exception as e:
        return {"content": [{"type": "text", "text": f"❌ {e}"}], "isError": True}

@app.get("/site/toolchat.html")
def serve_toolchat():
    html = """
    <html><head><title>MCP Tool Chat</title></head>
    <body style='font-family: sans-serif'>
    <h2>🧪 MCP Tool Call UI</h2>
    <form method="post" action="/mcp/tools/call">
      <label>Tool:</label><input name="name" value="execute_python"><br><br>
      <label>Input JSON:</label><br>
      <textarea name="input" rows="6" cols="60">{"code": "print('Hello!')"}</textarea><br><br>
      <button type="submit">Run Tool</button>
    </form>
    </body></html>
    """
    return HTMLResponse(content=html)

@app.get("/.well-known/ai-plugin.json")
def serve_openai_manifest():
    data = {
        "schema_version": "v1",
        "name_for_human": "Colab MCP",
        "name_for_model": "mcp",
        "description_for_model": "Tool call server with file/memory/code functions",
        "description_for_human": "Run tools in Colab via OpenAPI",
        "auth": {"type": "none"},
        "api": {"type": "openapi", "url": f"{public_url}/mcp/schema"},
        "logo_url": "", "contact_email": "", "legal_info_url": ""
    }
    (PLUGIN_DIR / "ai-plugin.json").write_text(json.dumps(data, indent=2))
    return JSONResponse(data)

@app.get("/.well-known/tool-schema.json")
@app.get("/mcp/schema")
def serve_openapi_schema(): return app.openapi()

@app.get("/.well-known/claude.yaml")
def serve_claude_yaml():
    yaml = f"""schema_version: 1.0
name: colab-mcp
description: Claude plugin for tool calling via MCP
api:
  type: openapi
  url: {public_url}/mcp/schema
auth:
  type: none
"""
    return HTMLResponse(content=yaml, media_type="text/yaml")

# ─────────────────────────────────────────────────────────────────────────────
# 8. Launch server and print prompt tips
# ─────────────────────────────────────────────────────────────────────────────
def launch():
    Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000), daemon=True).start()
    time.sleep(2)
    url = ngrok.connect(8000).public_url
    print(f"🔗 Public URL: {url}")
    for label, route in [
        ("Tool UI", "/site/toolchat.html"),
        ("Tool Call", "/mcp/tools/call"),
        ("Logs", "/logs/view"),
        ("Memory", "/memory/view"),
        ("Summary", "/memory/summary"),
        ("Schema", "/mcp/schema"),
        ("Claude", "/.well-known/tool-schema.json"),
        ("OpenAI", "/.well-known/ai-plugin.json"),
        ("Claude YAML", "/.well-known/claude.yaml"),
    ]:
        print(f"• {label:<12} {url}{route}")
    return url

public_url = launch()

# Final example block
print("\n🤖 Example Prompts for LLM Tool Use (Claude or OpenAI-compatible):\n")
print("1. Run Python code:")
print("   → Call the 'execute_python' tool with:")
print("     {\"code\": \"print('Hello from Claude!')\"}\n")
print("2. Install a pip package:")
print("   → Call 'install_package' with:")
print("     {\"package\": \"beautifulsoup4\"}\n")
print("3. Read a file from sandboxed drive:")
print("   → Call 'read_file' or 'print_file_text' with:")
print("     {\"filepath\": \"notes/example.txt\"}\n")
print("4. Write a file:")
print("   → Call 'write_file' with:")
print("     {\"filepath\": \"notes/output.txt\", \"content\": \"Hello, world.\"}\n")
print("5. List files in a folder:")
print("   → Call 'list_files' with:")
print("     {\"path\": \".\"} or {\"path\": \"data\"}\n")
print("6. Recall a stored memory value:")
print("   → First call 'memory/update' to store:")
print("     {\"project\": \"AI Assistant\"}\n")
print("   → Then call 'memory/view' or 'memory/summary'\n")
print("7. Download a file:")
print("   → Call 'download_file' with:")
print("     {\"filepath\": \"output/results.csv\"}")
print("   → Result will be a direct download URL.\n")

Mounted at /content/drive


INFO:     Started server process [4719]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


🔗 Public URL: https://954b-34-143-248-139.ngrok-free.app
• Tool UI      https://954b-34-143-248-139.ngrok-free.app/site/toolchat.html
• Tool Call    https://954b-34-143-248-139.ngrok-free.app/mcp/tools/call
• Logs         https://954b-34-143-248-139.ngrok-free.app/logs/view
• Memory       https://954b-34-143-248-139.ngrok-free.app/memory/view
• Summary      https://954b-34-143-248-139.ngrok-free.app/memory/summary
• Schema       https://954b-34-143-248-139.ngrok-free.app/mcp/schema
• Claude       https://954b-34-143-248-139.ngrok-free.app/.well-known/tool-schema.json
• OpenAI       https://954b-34-143-248-139.ngrok-free.app/.well-known/ai-plugin.json
• Claude YAML  https://954b-34-143-248-139.ngrok-free.app/.well-known/claude.yaml

🤖 Example Prompts for LLM Tool Use (Claude or OpenAI-compatible):

1. Run Python code:
   → Call the 'execute_python' tool with:
     {"code": "print('Hello from Claude!')"}

2. Install a pip package:
   → Call 'install_package' with:
     {"package": "beaut

In [ ]:
# ╔════════════════════════════════════════════════════════════════════════════════════╗
# ║ 🤖 Colab MCP Server v1.7.1 — All Routes Restored + Streaming + Plugin Manifests   ║
# ║                                                                                    ║
# ║ • Includes /mcp, /mcp/tools/call, memory, logs, download, toolchat, Claude/OpenAI ║
# ║ • Tool set: code exec, pip install, file ops, memory ops, download, read/print    ║
# ║ • Live logging, OpenAPI 3.1.0, YAML plugin files, sandbox-safe I/O                ║
# ║ • Logging fixed to stream valid log lines (not duplicated `data: data:`)          ║
# ║ • UPDATED: Logs saved to Drive and /logs/view now streams last 20 actions         ║
# ╚════════════════════════════════════════════════════════════════════════════════════╝

# ─────────────────────────────────────────────────────────────────────────────
# 1. Install dependencies (Colab)
# ─────────────────────────────────────────────────────────────────────────────
!pip install --quiet fastapi uvicorn pyngrok nest-asyncio sse-starlette requests

# ─────────────────────────────────────────────────────────────────────────────
# 2. Mount Google Drive
# ─────────────────────────────────────────────────────────────────────────────
import nest_asyncio; nest_asyncio.apply()
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

# ─────────────────────────────────────────────────────────────────────────────
# 3. Imports and setup
# ─────────────────────────────────────────────────────────────────────────────
import os, json, time, asyncio, logging, subprocess, sys
from pathlib import Path
from threading import Thread
from fastapi import FastAPI, Request, HTTPException, Form
from fastapi.responses import FileResponse, HTMLResponse, JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from fastapi.staticfiles import StaticFiles
from sse_starlette.sse import EventSourceResponse
from pyngrok import ngrok
import uvicorn

SAFE_DIR = Path("/content/drive/MyDrive/ColabMCPFiles")
PLUGIN_DIR = SAFE_DIR / ".well-known"
for d in [SAFE_DIR, PLUGIN_DIR]: d.mkdir(parents=True, exist_ok=True)
os.chdir(SAFE_DIR)
ngrok.set_auth_token("2yuUKpiAlYXCez40i8UktLZ91s7_5Rv1GjG5HMFw5KKbk5sHz")

app = FastAPI(title="Colab MCP Tool Server", version="1.7.2")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])
app.mount("/site", StaticFiles(directory=SAFE_DIR / "site", html=True), name="site")

# ─────────────────────────────────────────────────────────────────────────────
# 4. Logging + log streaming (FIXED + Drive log + 20-line history)
# ─────────────────────────────────────────────────────────────────────────────
SERVER_LOG = SAFE_DIR / "server_log.txt"; SERVER_LOG.touch()
CHAT_LOG   = SAFE_DIR / "chat_log.txt";   CHAT_LOG.touch()
subscribers: set[asyncio.Queue] = set()

class BroadcastHandler(logging.Handler):
    def emit(self, record):
        msg = self.format(record)
        with SERVER_LOG.open("a", encoding="utf-8") as f:
            f.write(msg + "\n")
        for q in list(subscribers):
            try: asyncio.get_event_loop().call_soon_threadsafe(q.put_nowait, msg)
            except: pass

logger = logging.getLogger("mcp")
logger.setLevel(logging.INFO)
if not any(isinstance(h, BroadcastHandler) for h in logger.handlers):
    handler = BroadcastHandler()
    handler.setFormatter(logging.Formatter("%(asctime)s %(message)s"))
    logger.addHandler(handler)

# ─────────────────────────────────────────────────────────────────────────────
# 5. Tool definitions (fully working set)
# ─────────────────────────────────────────────────────────────────────────────
TOOLS = [
    {"name": "execute_python", "description": "Run Python code", "inputSchema": {"type": "object", "properties": {"code": {"type": "string"}}, "required": ["code"]}},
    {"name": "install_package", "description": "Install pip package", "inputSchema": {"type": "object", "properties": {"package": {"type": "string"}}, "required": ["package"]}},
    {"name": "list_files", "description": "List files in sandbox", "inputSchema": {"type": "object", "properties": {"path": {"type": "string", "default": "."}}}},
    {"name": "read_file", "description": "Read file content", "inputSchema": {"type": "object", "properties": {"filepath": {"type": "string"}}, "required": ["filepath"]}},
    {"name": "write_file", "description": "Write content to file", "inputSchema": {"type": "object", "properties": {"filepath": {"type": "string"}, "content": {"type": "string"}}, "required": ["filepath", "content"]}},
    {"name": "print_file_text", "description": "Alias of read_file", "inputSchema": {"type": "object", "properties": {"filepath": {"type": "string"}}, "required": ["filepath"]}},
    {"name": "download_file", "description": "Get download URL", "inputSchema": {"type": "object", "properties": {"filepath": {"type": "string"}}, "required": ["filepath"]}},
]

# ─────────────────────────────────────────────────────────────────────────────
# 6. Tool execution logic
# ─────────────────────────────────────────────────────────────────────────────
def exec_py(code: str) -> str:
    from io import StringIO
    import contextlib
    buf = StringIO()
    try:
        with contextlib.redirect_stdout(buf):
            exec(code, {})
        return buf.getvalue() or "✓ Code ran with no output."
    except Exception as e:
        return f"❌ Error: {e}"

def inst_pkg(pkg: str) -> str:
    result = subprocess.run([sys.executable, "-m", "pip", "install", pkg], capture_output=True, text=True)
    return result.stdout or result.stderr

def safe_path(p: str) -> Path:
    target = SAFE_DIR / p
    if not target.resolve().is_relative_to(SAFE_DIR.resolve()):
        raise ValueError("Unsafe path.")
    return target

def read_file(fp: str) -> str:
    return safe_path(fp).read_text()

def write_file(fp: str, content: str) -> str:
    p = safe_path(fp)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content)
    return f"✅ Written to {p.name}"

# ─────────────────────────────────────────────────────────────────────────────
# 7. API routes
# ─────────────────────────────────────────────────────────────────────────────
@app.get("/")
def root(): return {"status": "ok", "version": "1.7.2"}

@app.get("/health")
def health(): return {"status": "healthy", "tools": [t["name"] for t in TOOLS]}

@app.get("/logs/view")
async def view_logs(request: Request):
    q = asyncio.Queue()
    subscribers.add(q)
    try:
        with SERVER_LOG.open("r", encoding="utf-8") as f:
            history = f.readlines()[-20:]
        async def logstream():
            for line in history:
                yield f"data: {line.strip()}\n\n"
            while True:
                if await request.is_disconnected(): break
                try:
                    msg = await asyncio.wait_for(q.get(), timeout=30)
                    yield f"data: {msg.strip()}\n\n"
                except asyncio.TimeoutError:
                    continue
        return EventSourceResponse(logstream())
    finally:
        subscribers.discard(q)

@app.get("/memory/view")
def memory_view():
    mem_path = SAFE_DIR / "memory.json"
    return json.loads(mem_path.read_text()) if mem_path.exists() else {}

@app.get("/memory/summary")
def memory_summary():
    mem_path = SAFE_DIR / "memory.json"
    if not mem_path.exists(): return {"summary": "(none)"}
    try:
        data = json.loads(mem_path.read_text())
        return {"summary": data.get("summary", "(no summary key)")}
    except: return {"summary": "(error reading JSON)"}

@app.get("/memory/keys")
def memory_keys():
    mem_path = SAFE_DIR / "memory.json"
    return list(json.loads(mem_path.read_text()).keys()) if mem_path.exists() else []

@app.post("/memory/update")
async def memory_update(request: Request):
    mem_path = SAFE_DIR / "memory.json"
    memory = json.loads(mem_path.read_text()) if mem_path.exists() else {}
    update = await request.json()
    memory.update(update)
    mem_path.write_text(json.dumps(memory, indent=2))
    return memory

@app.post("/memory/clear")
def memory_clear():
    mem_path = SAFE_DIR / "memory.json"
    mem_path.write_text("{}")
    return {"status": "cleared"}

@app.get("/download/{file_path:path}")
def download_file(file_path: str):
    p = safe_path(file_path)
    if not p.exists(): raise HTTPException(404, "File not found")
    return FileResponse(p)

@app.get("/mcp/tools/list")
def list_tools(): return {"tools": TOOLS}

@app.post("/mcp/tools/call")
async def call_tool(request: Request):
    data = await request.json()
    name, args = data.get("name"), data.get("input", {})
    logger.info(f"CALL {name} {args}")
    try:
        if name == "execute_python": out = exec_py(args["code"])
        elif name == "install_package": out = inst_pkg(args["package"])
        elif name == "list_files": out = "\n".join(f.name for f in safe_path(args.get("path", ".")).iterdir())
        elif name == "read_file": out = read_file(args["filepath"])
        elif name == "write_file": out = write_file(args["filepath"], args["content"])
        elif name == "print_file_text": out = read_file(args["filepath"])
        elif name == "download_file":
            safe_path(args["filepath"])
            out = f"{public_url}/download/{args['filepath']}"
        else: raise Exception("Unknown tool")
        return {"content": [{"type": "text", "text": out}]}
    except Exception as e:
        return {"content": [{"type": "text", "text": f"❌ {e}"}], "isError": True}

@app.get("/site/toolchat.html")
def serve_toolchat():
    html = """
    <html><head><title>MCP Tool Chat</title></head>
    <body style='font-family: sans-serif'>
    <h2>🧪 MCP Tool Call UI</h2>
    <form method="post" action="/mcp/tools/call">
      <label>Tool:</label><input name="name" value="execute_python"><br><br>
      <label>Input JSON:</label><br>
      <textarea name="input" rows="6" cols="60">{"code": "print('Hello!')"}</textarea><br><br>
      <button type="submit">Run Tool</button>
    </form>
    </body></html>
    """
    return HTMLResponse(content=html)

@app.get("/.well-known/ai-plugin.json")
def serve_openai_manifest():
    data = {
        "schema_version": "v1",
        "name_for_human": "Colab MCP",
        "name_for_model": "mcp",
        "description_for_model": "Tool call server with file/memory/code functions",
        "description_for_human": "Run tools in Colab via OpenAPI",
        "auth": {"type": "none"},
        "api": {"type": "openapi", "url": f"{public_url}/mcp/schema"},
        "logo_url": "", "contact_email": "", "legal_info_url": ""
    }
    (PLUGIN_DIR / "ai-plugin.json").write_text(json.dumps(data, indent=2))
    return JSONResponse(data)

@app.get("/.well-known/tool-schema.json")
@app.get("/mcp/schema")
def serve_openapi_schema(): return app.openapi()

@app.get("/.well-known/claude.yaml")
def serve_claude_yaml():
    yaml = f"""schema_version: 1.0
name: colab-mcp
description: Claude plugin for tool calling via MCP
api:
  type: openapi
  url: {public_url}/mcp/schema
auth:
  type: none
"""
    return HTMLResponse(content=yaml, media_type="text/yaml")

# ─────────────────────────────────────────────────────────────────────────────
# 8. Launch server and print prompt tips
# ─────────────────────────────────────────────────────────────────────────────
def launch():
    Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000), daemon=True).start()
    time.sleep(2)
    url = ngrok.connect(8000).public_url
    print(f"🔗 Public URL: {url}")
    for label, route in [
        ("Tool UI", "/site/toolchat.html"),
        ("Tool Call", "/mcp/tools/call"),
        ("Logs", "/logs/view"),
        ("Memory", "/memory/view"),
        ("Summary", "/memory/summary"),
        ("Schema", "/mcp/schema"),
        ("Claude", "/.well-known/tool-schema.json"),
        ("OpenAI", "/.well-known/ai-plugin.json"),
        ("Claude YAML", "/.well-known/claude.yaml"),
    ]:
        print(f"• {label:<12} {url}{route}")
    return url

public_url = launch()

print("\n🤖 Example Prompts for LLM Tool Use (Claude or OpenAI-compatible):\n")
print("1. Run Python code:")
print("   → Call the 'execute_python' tool with:")
print("     {\"code\": \"print('Hello from Claude!')\"}\n")
print("2. Install a pip package:")
print("   → Call 'install_package' with:")
print("     {\"package\": \"beautifulsoup4\"}\n")
print("3. Read a file from sandboxed drive:")
print("   → Call 'read_file' or 'print_file_text' with:")
print("     {\"filepath\": \"notes/example.txt\"}\n")
print("4. Write a file:")
print("   → Call 'write_file' with:")
print("     {\"filepath\": \"notes/output.txt\", \"content\": \"Hello, world.\"}\n")
print("5. List files in a folder:")
print("   → Call 'list_files' with:")
print("     {\"path\": \".\"} or {\"path\": \"data\"}\n")
print("6. Recall a stored memory value:")
print("   → First call 'memory/update' to store:")
print("     {\"project\": \"AI Assistant\"}\n")
print("   → Then call 'memory/view' or 'memory/summary'\n")
print("7. Download a file:")
print("   → Call 'download_file' with:")
print("     {\"filepath\": \"output/results.csv\"}")
print("   → Result will be a direct download URL.\n")

Mounted at /content/drive


INFO:     Started server process [6681]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


🔗 Public URL: https://9ee2-35-185-150-247.ngrok-free.app
• Tool UI      https://9ee2-35-185-150-247.ngrok-free.app/site/toolchat.html
• Tool Call    https://9ee2-35-185-150-247.ngrok-free.app/mcp/tools/call
• Logs         https://9ee2-35-185-150-247.ngrok-free.app/logs/view
• Memory       https://9ee2-35-185-150-247.ngrok-free.app/memory/view
• Summary      https://9ee2-35-185-150-247.ngrok-free.app/memory/summary
• Schema       https://9ee2-35-185-150-247.ngrok-free.app/mcp/schema
• Claude       https://9ee2-35-185-150-247.ngrok-free.app/.well-known/tool-schema.json
• OpenAI       https://9ee2-35-185-150-247.ngrok-free.app/.well-known/ai-plugin.json
• Claude YAML  https://9ee2-35-185-150-247.ngrok-free.app/.well-known/claude.yaml

🤖 Example Prompts for LLM Tool Use (Claude or OpenAI-compatible):

1. Run Python code:
   → Call the 'execute_python' tool with:
     {"code": "print('Hello from Claude!')"}

2. Install a pip package:
   → Call 'install_package' with:
     {"package": "beaut

In [ ]:
# ╔════════════════════════════════════════════════════════════════════════════════════╗
# ║ 🤖 Colab MCP Server v1.7.2 — All Routes Restored + Streaming + Plugin Manifests   ║
# ║                                                                                    ║
# ║ • Includes /mcp, /mcp/tools/call, memory, logs, download, toolchat, Claude/OpenAI ║
# ║ • Tool set: code exec, pip install, file ops, memory ops, download, read/print    ║
# ║ • Live logging, OpenAPI 3.1.0, YAML plugin files, sandbox-safe I/O                ║
# ║ • Logging fixed to stream valid log lines (not duplicated `data: data:`)          ║
# ║ • UPDATED: Logs saved to Drive and /logs/view now streams last 20 actions         ║
# ╚════════════════════════════════════════════════════════════════════════════════════╝

# ─────────────────────────────────────────────────────────────────────────────
# 1. Install dependencies (Colab)
# ─────────────────────────────────────────────────────────────────────────────
!pip install --quiet fastapi uvicorn pyngrok nest-asyncio sse-starlette requests

# ─────────────────────────────────────────────────────────────────────────────
# 2. Mount Google Drive
# ─────────────────────────────────────────────────────────────────────────────
import nest_asyncio; nest_asyncio.apply()
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

# ─────────────────────────────────────────────────────────────────────────────
# 3. Imports and setup
# ─────────────────────────────────────────────────────────────────────────────
import os, json, time, asyncio, logging, subprocess, sys
from pathlib import Path
from threading import Thread
from fastapi import FastAPI, Request, HTTPException
from fastapi.responses import FileResponse, HTMLResponse, JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from fastapi.staticfiles import StaticFiles
from sse_starlette.sse import EventSourceResponse
from pyngrok import ngrok
import uvicorn

SAFE_DIR = Path("/content/drive/MyDrive/ColabMCPFiles")
PLUGIN_DIR = SAFE_DIR / ".well-known"
for d in [SAFE_DIR, PLUGIN_DIR]: d.mkdir(parents=True, exist_ok=True)
os.chdir(SAFE_DIR)
ngrok.set_auth_token("2yuUKpiAlYXCez40i8UktLZ91s7_5Rv1GjG5HMFw5KKbk5sHz")

app = FastAPI(title="Colab MCP Tool Server", version="1.7.2")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])
app.mount("/site", StaticFiles(directory=SAFE_DIR / "site", html=True), name="site")

# ─────────────────────────────────────────────────────────────────────────────
# 4. Logging + log streaming (FIXED + Drive log + 20-line history)
# ─────────────────────────────────────────────────────────────────────────────
SERVER_LOG = SAFE_DIR / "server_log.txt"; SERVER_LOG.touch()
CHAT_LOG   = SAFE_DIR / "chat_log.txt";   CHAT_LOG.touch()
subscribers: set[asyncio.Queue] = set()

class BroadcastHandler(logging.Handler):
    def emit(self, record):
        msg = self.format(record)
        with SERVER_LOG.open("a", encoding="utf-8") as f:
            f.write(msg + "\n")
        for q in list(subscribers):
            try: asyncio.get_event_loop().call_soon_threadsafe(q.put_nowait, msg)
            except: pass

logger = logging.getLogger("mcp")
logger.setLevel(logging.INFO)
if not any(isinstance(h, BroadcastHandler) for h in logger.handlers):
    handler = BroadcastHandler()
    handler.setFormatter(logging.Formatter("%(asctime)s %(message)s"))
    logger.addHandler(handler)

# ─────────────────────────────────────────────────────────────────────────────
# 5. Tool definitions (fully working set)
# ─────────────────────────────────────────────────────────────────────────────
TOOLS = [
    {"name": "execute_python", "description": "Run Python code", "inputSchema": {"type": "object", "properties": {"code": {"type": "string"}}, "required": ["code"]}},
    {"name": "install_package", "description": "Install pip package", "inputSchema": {"type": "object", "properties": {"package": {"type": "string"}}, "required": ["package"]}},
    {"name": "list_files", "description": "List files in sandbox", "inputSchema": {"type": "object", "properties": {"path": {"type": "string", "default": "."}}}},
    {"name": "read_file", "description": "Read file content", "inputSchema": {"type": "object", "properties": {"filepath": {"type": "string"}}, "required": ["filepath"]}},
    {"name": "write_file", "description": "Write content to file", "inputSchema": {"type": "object", "properties": {"filepath": {"type": "string"}, "content": {"type": "string"}}, "required": ["filepath", "content"]}},
    {"name": "print_file_text", "description": "Alias of read_file", "inputSchema": {"type": "object", "properties": {"filepath": {"type": "string"}}, "required": ["filepath"]}},
    {"name": "download_file", "description": "Get download URL", "inputSchema": {"type": "object", "properties": {"filepath": {"type": "string"}}, "required": ["filepath"]}},
]
# ─────────────────────────────────────────────────────────────────────────────
# 6. Tool execution logic
# ─────────────────────────────────────────────────────────────────────────────
def exec_py(code: str) -> str:
    from io import StringIO
    import contextlib
    buf = StringIO()
    try:
        with contextlib.redirect_stdout(buf):
            exec(code, {})
        return buf.getvalue() or "✓ Code ran with no output."
    except Exception as e:
        return f"❌ Error: {e}"

def inst_pkg(pkg: str) -> str:
    result = subprocess.run([sys.executable, "-m", "pip", "install", pkg], capture_output=True, text=True)
    return result.stdout or result.stderr

def safe_path(p: str) -> Path:
    target = SAFE_DIR / p
    if not target.resolve().is_relative_to(SAFE_DIR.resolve()):
        raise ValueError("Unsafe path.")
    return target

def read_file(fp: str) -> str:
    return safe_path(fp).read_text()

def write_file(fp: str, content: str) -> str:
    p = safe_path(fp)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content)
    return f"✅ Written to {p.name}"

# ─────────────────────────────────────────────────────────────────────────────
# 7. API routes
# ─────────────────────────────────────────────────────────────────────────────
@app.get("/")
def root(): return {"status": "ok", "version": "1.7.2"}

@app.get("/health")
def health(): return {"status": "healthy", "tools": [t["name"] for t in TOOLS]}

@app.get("/logs/view")
async def view_logs(request: Request):
    q = asyncio.Queue()
    subscribers.add(q)
    try:
        with SERVER_LOG.open("r", encoding="utf-8") as f:
            history = f.readlines()[-20:]
        async def logstream():
            for line in history:
                yield f"data: {line.strip()}\n\n"
            while True:
                if await request.is_disconnected(): break
                try:
                    msg = await asyncio.wait_for(q.get(), timeout=30)
                    yield f"data: {msg.strip()}\n\n"
                except asyncio.TimeoutError:
                    continue
        return EventSourceResponse(logstream())
    finally:
        subscribers.discard(q)

@app.get("/memory/view")
def memory_view():
    mem_path = SAFE_DIR / "memory.json"
    return json.loads(mem_path.read_text()) if mem_path.exists() else {}

@app.get("/memory/summary")
def memory_summary():
    mem_path = SAFE_DIR / "memory.json"
    if not mem_path.exists(): return {"summary": "(none)"}
    try:
        data = json.loads(mem_path.read_text())
        return {"summary": data.get("summary", "(no summary key)")}
    except: return {"summary": "(error reading JSON)"}

@app.get("/memory/keys")
def memory_keys():
    mem_path = SAFE_DIR / "memory.json"
    return list(json.loads(mem_path.read_text()).keys()) if mem_path.exists() else []

@app.post("/memory/update")
async def memory_update(request: Request):
    mem_path = SAFE_DIR / "memory.json"
    memory = json.loads(mem_path.read_text()) if mem_path.exists() else {}
    update = await request.json()
    memory.update(update)
    mem_path.write_text(json.dumps(memory, indent=2))
    return memory

@app.post("/memory/clear")
def memory_clear():
    mem_path = SAFE_DIR / "memory.json"
    mem_path.write_text("{}")
    return {"status": "cleared"}

@app.get("/download/{file_path:path}")
def download_file(file_path: str):
    p = safe_path(file_path)
    if not p.exists(): raise HTTPException(404, "File not found")
    return FileResponse(p)

@app.get("/mcp/tools/list")
def list_tools(): return {"tools": TOOLS}

@app.post("/mcp/tools/call")
async def call_tool(request: Request):
    data = await request.json()
    name, args = data.get("name"), data.get("input", {})
    logger.info(f"CALL {name} {args}")
    try:
        if name == "execute_python": out = exec_py(args["code"])
        elif name == "install_package": out = inst_pkg(args["package"])
        elif name == "list_files": out = "\n".join(f.name for f in safe_path(args.get("path", ".")).iterdir())
        elif name == "read_file": out = read_file(args["filepath"])
        elif name == "write_file": out = write_file(args["filepath"], args["content"])
        elif name == "print_file_text": out = read_file(args["filepath"])
        elif name == "download_file":
            safe_path(args["filepath"])
            out = f"{public_url}/download/{args['filepath']}"
        else: raise Exception("Unknown tool")
        return {"content": [{"type": "text", "text": out}]}
    except Exception as e:
        return {"content": [{"type": "text", "text": f"❌ {e}"}], "isError": True}

@app.get("/site/toolchat.html")
def serve_toolchat():
    html = """
    <html><head><title>MCP Tool Chat</title></head>
    <body style='font-family: sans-serif'>
    <h2>🧪 MCP Tool Call UI</h2>
    <form method="post" action="/mcp/tools/call">
      <label>Tool:</label><input name="name" value="execute_python"><br><br>
      <label>Input JSON:</label><br>
      <textarea name="input" rows="6" cols="60">{"code": "print('Hello!')"}</textarea><br><br>
      <button type="submit">Run Tool</button>
    </form>
    </body></html>
    """
    return HTMLResponse(content=html)

@app.get("/.well-known/ai-plugin.json")
def serve_openai_manifest():
    data = {
        "schema_version": "v1",
        "name_for_human": "Colab MCP",
        "name_for_model": "mcp",
        "description_for_model": "Tool call server with file/memory/code functions",
        "description_for_human": "Run tools in Colab via OpenAPI",
        "auth": {"type": "none"},
        "api": {"type": "openapi", "url": f"{public_url}/mcp/schema"},
        "logo_url": "", "contact_email": "", "legal_info_url": ""
    }
    (PLUGIN_DIR / "ai-plugin.json").write_text(json.dumps(data, indent=2))
    return JSONResponse(data)

@app.get("/.well-known/tool-schema.json")
@app.get("/mcp/schema")
def serve_openapi_schema(): return app.openapi()

@app.get("/.well-known/claude.yaml")
def serve_claude_yaml():
    yaml = f"""schema_version: 1.0
name: colab-mcp
description: Claude plugin for tool calling via MCP
api:
  type: openapi
  url: {public_url}/mcp/schema
auth:
  type: none
"""
    return HTMLResponse(content=yaml, media_type="text/yaml")

# ─────────────────────────────────────────────────────────────────────────────
# 8. Launch server and print prompt tips
# ─────────────────────────────────────────────────────────────────────────────
def launch():
    Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000), daemon=True).start()
    time.sleep(2)
    url = ngrok.connect(8000).public_url
    print(f"🔗 Public URL: {url}")
    for label, route in [
        ("Tool UI", "/site/toolchat.html"),
        ("Tool Call", "/mcp/tools/call"),
        ("Logs", "/logs/view"),
        ("Memory", "/memory/view"),
        ("Summary", "/memory/summary"),
        ("Schema", "/mcp/schema"),
        ("Claude", "/.well-known/tool-schema.json"),
        ("OpenAI", "/.well-known/ai-plugin.json"),
        ("Claude YAML", "/.well-known/claude.yaml"),
    ]:
        print(f"• {label:<12} {url}{route}")
    return url

public_url = launch()

print("\n🤖 Example Prompts for LLM Tool Use (Claude or OpenAI-compatible):\n")
print("1. Run Python code:")
print("   → Call the 'execute_python' tool with:")
print("     {\"code\": \"print('Hello from Claude!')\"}\n")
print("2. Install a pip package:")
print("   → Call 'install_package' with:")
print("     {\"package\": \"beautifulsoup4\"}\n")
print("3. Read a file from sandboxed drive:")
print("   → Call 'read_file' or 'print_file_text' with:")
print("     {\"filepath\": \"notes/example.txt\"}\n")
print("4. Write a file:")
print("   → Call 'write_file' with:")
print("     {\"filepath\": \"notes/output.txt\", \"content\": \"Hello, world.\"}\n")
print("5. List files in a folder:")
print("   → Call 'list_files' with:")
print("     {\"path\": \".\"} or {\"path\": \"data\"}\n")
print("6. Recall a stored memory value:")
print("   → First call 'memory/update' to store:")
print("     {\"project\": \"AI Assistant\"}\n")
print("   → Then call 'memory/view' or 'memory/summary'\n")
print("7. Download a file:")
print("   → Call 'download_file' with:")
print("     {\"filepath\": \"output/results.csv\"}")
print("   → Result will be a direct download URL.\n")

ValueError: mount failed

In [ ]:
# ╔════════════════════════════════════════════════════════════════════════════════════╗
# ║ 🤖 Colab MCP Server v1.7.1 — All Routes Restored + Streaming + Plugin Manifests   ║
# ║                                                                                    ║
# ║ • Includes /mcp, /mcp/tools/call, memory, logs, download, toolchat, Claude/OpenAI ║
# ║ • Tool set: code exec, pip install, file ops, memory ops, download, read/print    ║
# ║ • Live logging, OpenAPI 3.1.0, YAML plugin files, sandbox-safe I/O                ║
# ║ • Logging fixed to stream valid log lines (not duplicated `data: data:`)          ║
# ║ • UPDATED: Logs saved to Drive and /logs/view now streams last 20 actions         ║
# ╚════════════════════════════════════════════════════════════════════════════════════╝

# ─────────────────────────────────────────────────────────────────────────────
# 1. Install dependencies (Colab)
# ─────────────────────────────────────────────────────────────────────────────
!pip install --quiet fastapi uvicorn pyngrok nest-asyncio sse-starlette requests

# ─────────────────────────────────────────────────────────────────────────────
# 2. Mount Google Drive
# ─────────────────────────────────────────────────────────────────────────────
import nest_asyncio; nest_asyncio.apply()
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

# ─────────────────────────────────────────────────────────────────────────────
# 3. Imports and setup
# ─────────────────────────────────────────────────────────────────────────────
import os, json, time, asyncio, logging, subprocess, sys
from pathlib import Path
from threading import Thread
from fastapi import FastAPI, Request, HTTPException, Form
from fastapi.responses import FileResponse, HTMLResponse, JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from fastapi.staticfiles import StaticFiles
from sse_starlette.sse import EventSourceResponse
from pyngrok import ngrok
import uvicorn

SAFE_DIR = Path("/content/drive/MyDrive/ColabMCPFiles")
PLUGIN_DIR = SAFE_DIR / ".well-known"
for d in [SAFE_DIR, PLUGIN_DIR]: d.mkdir(parents=True, exist_ok=True)
os.chdir(SAFE_DIR)
ngrok.set_auth_token("2yuUKpiAlYXCez40i8UktLZ91s7_5Rv1GjG5HMFw5KKbk5sHz")

app = FastAPI(title="Colab MCP Tool Server", version="1.7.2")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])
app.mount("/site", StaticFiles(directory=SAFE_DIR / "site", html=True), name="site")

# ─────────────────────────────────────────────────────────────────────────────
# 4. Logging + log streaming (FIXED + Drive log + 20-line history)
# ─────────────────────────────────────────────────────────────────────────────
SERVER_LOG = SAFE_DIR / "server_log.txt"; SERVER_LOG.touch()
CHAT_LOG   = SAFE_DIR / "chat_log.txt";   CHAT_LOG.touch()
subscribers: set[asyncio.Queue] = set()

class BroadcastHandler(logging.Handler):
    def emit(self, record):
        msg = self.format(record)
        with SERVER_LOG.open("a", encoding="utf-8") as f:
            f.write(msg + "\n")
        for q in list(subscribers):
            try: asyncio.get_event_loop().call_soon_threadsafe(q.put_nowait, msg)
            except: pass

logger = logging.getLogger("mcp")
logger.setLevel(logging.INFO)
if not any(isinstance(h, BroadcastHandler) for h in logger.handlers):
    handler = BroadcastHandler()
    handler.setFormatter(logging.Formatter("%(asctime)s %(message)s"))
    logger.addHandler(handler)

# ─────────────────────────────────────────────────────────────────────────────
# 5. Tool definitions (fully working set)
# ─────────────────────────────────────────────────────────────────────────────
TOOLS = [
    {"name": "execute_python", "description": "Run Python code", "inputSchema": {"type": "object", "properties": {"code": {"type": "string"}}, "required": ["code"]}},
    {"name": "install_package", "description": "Install pip package", "inputSchema": {"type": "object", "properties": {"package": {"type": "string"}}, "required": ["package"]}},
    {"name": "list_files", "description": "List files in sandbox", "inputSchema": {"type": "object", "properties": {"path": {"type": "string", "default": "."}}}},
    {"name": "read_file", "description": "Read file content", "inputSchema": {"type": "object", "properties": {"filepath": {"type": "string"}}, "required": ["filepath"]}},
    {"name": "write_file", "description": "Write content to file", "inputSchema": {"type": "object", "properties": {"filepath": {"type": "string"}, "content": {"type": "string"}}, "required": ["filepath", "content"]}},
    {"name": "print_file_text", "description": "Alias of read_file", "inputSchema": {"type": "object", "properties": {"filepath": {"type": "string"}}, "required": ["filepath"]}},
    {"name": "download_file", "description": "Get download URL", "inputSchema": {"type": "object", "properties": {"filepath": {"type": "string"}}, "required": ["filepath"]}},
]

# ─────────────────────────────────────────────────────────────────────────────
# 6. Tool execution logic
# ─────────────────────────────────────────────────────────────────────────────
def exec_py(code: str) -> str:
    from io import StringIO
    import contextlib
    buf = StringIO()
    try:
        with contextlib.redirect_stdout(buf):
            exec(code, {})
        return buf.getvalue() or "✓ Code ran with no output."
    except Exception as e:
        return f"❌ Error: {e}"

def inst_pkg(pkg: str) -> str:
    result = subprocess.run([sys.executable, "-m", "pip", "install", pkg], capture_output=True, text=True)
    return result.stdout or result.stderr

def safe_path(p: str) -> Path:
    target = SAFE_DIR / p
    if not target.resolve().is_relative_to(SAFE_DIR.resolve()):
        raise ValueError("Unsafe path.")
    return target

def read_file(fp: str) -> str:
    return safe_path(fp).read_text()

def write_file(fp: str, content: str) -> str:
    p = safe_path(fp)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content)
    return f"✅ Written to {p.name}"

    # ─────────────────────────────────────────────────────────────────────────────
# 7. API routes
# ─────────────────────────────────────────────────────────────────────────────
@app.get("/")
def root(): return {"status": "ok", "version": "1.7.2"}

@app.get("/health")
def health(): return {"status": "healthy", "tools": [t["name"] for t in TOOLS]}

@app.get("/logs/view")
async def view_logs(request: Request):
    q = asyncio.Queue()
    subscribers.add(q)
    try:
        with SERVER_LOG.open("r", encoding="utf-8") as f:
            history = f.readlines()[-20:]
        async def logstream():
            for line in history:
                yield f"data: {line.strip()}\n\n"
            while True:
                if await request.is_disconnected(): break
                try:
                    msg = await asyncio.wait_for(q.get(), timeout=30)
                    yield f"data: {msg.strip()}\n\n"
                except asyncio.TimeoutError:
                    continue
        return EventSourceResponse(logstream())
    finally:
        subscribers.discard(q)

@app.get("/memory/view")
def memory_view():
    mem_path = SAFE_DIR / "memory.json"
    return json.loads(mem_path.read_text()) if mem_path.exists() else {}

@app.get("/memory/summary")
def memory_summary():
    mem_path = SAFE_DIR / "memory.json"
    if not mem_path.exists(): return {"summary": "(none)"}
    try:
        data = json.loads(mem_path.read_text())
        return {"summary": data.get("summary", "(no summary key)")}
    except: return {"summary": "(error reading JSON)"}

@app.get("/memory/keys")
def memory_keys():
    mem_path = SAFE_DIR / "memory.json"
    return list(json.loads(mem_path.read_text()).keys()) if mem_path.exists() else []

@app.post("/memory/update")
async def memory_update(request: Request):
    mem_path = SAFE_DIR / "memory.json"
    memory = json.loads(mem_path.read_text()) if mem_path.exists() else {}
    update = await request.json()
    memory.update(update)
    mem_path.write_text(json.dumps(memory, indent=2))
    return memory

@app.post("/memory/clear")
def memory_clear():
    mem_path = SAFE_DIR / "memory.json"
    mem_path.write_text("{}")
    return {"status": "cleared"}

@app.get("/download/{file_path:path}")
def download_file(file_path: str):
    p = safe_path(file_path)
    if not p.exists(): raise HTTPException(404, "File not found")
    return FileResponse(p)

@app.get("/mcp/tools/list")
def list_tools(): return {"tools": TOOLS}

@app.post("/mcp/tools/call")
async def call_tool(request: Request):
    data = await request.json()
    name, args = data.get("name"), data.get("input", {})
    logger.info(f"CALL {name} {args}")
    try:
        if name == "execute_python": out = exec_py(args["code"])
        elif name == "install_package": out = inst_pkg(args["package"])
        elif name == "list_files": out = "\n".join(f.name for f in safe_path(args.get("path", ".")).iterdir())
        elif name == "read_file": out = read_file(args["filepath"])
        elif name == "write_file": out = write_file(args["filepath"], args["content"])
        elif name == "print_file_text": out = read_file(args["filepath"])
        elif name == "download_file":
            safe_path(args["filepath"])
            out = f"{public_url}/download/{args['filepath']}"
        else: raise Exception("Unknown tool")
        return {"content": [{"type": "text", "text": out}]}
    except Exception as e:
        return {"content": [{"type": "text", "text": f"❌ {e}"}], "isError": True}

@app.get("/site/toolchat.html")
def serve_toolchat():
    html = """
    <html><head><title>MCP Tool Chat</title></head>
    <body style='font-family: sans-serif'>
    <h2>🧪 MCP Tool Call UI</h2>
    <form method="post" action="/mcp/tools/call">
      <label>Tool:</label><input name="name" value="execute_python"><br><br>
      <label>Input JSON:</label><br>
      <textarea name="input" rows="6" cols="60">{"code": "print('Hello!')"}</textarea><br><br>
      <button type="submit">Run Tool</button>
    </form>
    </body></html>
    """
    return HTMLResponse(content=html)

@app.get("/.well-known/ai-plugin.json")
def serve_openai_manifest():
    data = {
        "schema_version": "v1",
        "name_for_human": "Colab MCP",
        "name_for_model": "mcp",
        "description_for_model": "Tool call server with file/memory/code functions",
        "description_for_human": "Run tools in Colab via OpenAPI",
        "auth": {"type": "none"},
        "api": {"type": "openapi", "url": f"{public_url}/mcp/schema"},
        "logo_url": "", "contact_email": "", "legal_info_url": ""
    }
    (PLUGIN_DIR / "ai-plugin.json").write_text(json.dumps(data, indent=2))
    return JSONResponse(data)

@app.get("/.well-known/tool-schema.json")
@app.get("/mcp/schema")
def serve_openapi_schema(): return app.openapi()

@app.get("/.well-known/claude.yaml")
def serve_claude_yaml():
    yaml = f"""schema_version: 1.0
name: colab-mcp
description: Claude plugin for tool calling via MCP
api:
  type: openapi
  url: {public_url}/mcp/schema
auth:
  type: none
"""
    return HTMLResponse(content=yaml, media_type="text/yaml")

# ─────────────────────────────────────────────────────────────────────────────
# 8. Launch server and print prompt tips
# ─────────────────────────────────────────────────────────────────────────────
def launch():
    Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000), daemon=True).start()
    time.sleep(2)
    url = ngrok.connect(8000).public_url
    print(f"🔗 Public URL: {url}")
    for label, route in [
        ("Tool UI", "/site/toolchat.html"),
        ("Tool Call", "/mcp/tools/call"),
        ("Logs", "/logs/view"),
        ("Memory", "/memory/view"),
        ("Summary", "/memory/summary"),
        ("Schema", "/mcp/schema"),
        ("Claude", "/.well-known/tool-schema.json"),
        ("OpenAI", "/.well-known/ai-plugin.json"),
        ("Claude YAML", "/.well-known/claude.yaml"),
    ]:
        print(f"• {label:<12} {url}{route}")
    return url

public_url = launch()

print("\n🤖 Example Prompts for LLM Tool Use (Claude or OpenAI-compatible):\n")
print("1. Run Python code:")
print("   → Call the 'execute_python' tool with:")
print("     {\"code\": \"print('Hello from Claude!')\"}\n")
print("2. Install a pip package:")
print("   → Call 'install_package' with:")
print("     {\"package\": \"beautifulsoup4\"}\n")
print("3. Read a file from sandboxed drive:")
print("   → Call 'read_file' or 'print_file_text' with:")
print("     {\"filepath\": \"notes/example.txt\"}\n")
print("4. Write a file:")
print("   → Call 'write_file' with:")
print("     {\"filepath\": \"notes/output.txt\", \"content\": \"Hello, world.\"}\n")
print("5. List files in a folder:")
print("   → Call 'list_files' with:")
print("     {\"path\": \".\"} or {\"path\": \"data\"}\n")
print("6. Recall a stored memory value:")
print("   → First call 'memory/update' to store:")
print("     {\"project\": \"AI Assistant\"}\n")
print("   → Then call 'memory/view' or 'memory/summary'\n")
print("7. Download a file:")
print("   → Call 'download_file' with:")
print("     {\"filepath\": \"output/results.csv\"}")
print("   → Result will be a direct download URL.\n")


Mounted at /content/drive


INFO:     Started server process [5841]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


🔗 Public URL: https://d497-104-196-191-228.ngrok-free.app
• Tool UI      https://d497-104-196-191-228.ngrok-free.app/site/toolchat.html
• Tool Call    https://d497-104-196-191-228.ngrok-free.app/mcp/tools/call
• Logs         https://d497-104-196-191-228.ngrok-free.app/logs/view
• Memory       https://d497-104-196-191-228.ngrok-free.app/memory/view
• Summary      https://d497-104-196-191-228.ngrok-free.app/memory/summary
• Schema       https://d497-104-196-191-228.ngrok-free.app/mcp/schema
• Claude       https://d497-104-196-191-228.ngrok-free.app/.well-known/tool-schema.json
• OpenAI       https://d497-104-196-191-228.ngrok-free.app/.well-known/ai-plugin.json
• Claude YAML  https://d497-104-196-191-228.ngrok-free.app/.well-known/claude.yaml

🤖 Example Prompts for LLM Tool Use (Claude or OpenAI-compatible):

1. Run Python code:
   → Call the 'execute_python' tool with:
     {"code": "print('Hello from Claude!')"}

2. Install a pip package:
   → Call 'install_package' with:
     {"packag